In [1]:
from pathlib import Path

PROJECT_ROOT = Path(r"D:\BDHS2022_FeverCare_SpatialEquity")

if not Path("D:/").exists():
    raise FileNotFoundError("The D: drive is not available.")

DIRECTORIES = [
    "data/raw",
    "data/processed",
    "notebooks",
    "results/tables",
    "results/figures",
    "results/models",
    "manuscript",
    "logs",
]

# Initialize the project directory structure.
for directory in DIRECTORIES:
    (PROJECT_ROOT / directory).mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print("\nDirectory structure:")

for directory in DIRECTORIES:
    path = PROJECT_ROOT / directory
    print(f"  {path.relative_to(PROJECT_ROOT)}")

print("\nWorkspace initialization completed.")

Project root: D:\BDHS2022_FeverCare_SpatialEquity

Directory structure:
  data\raw
  data\processed
  notebooks
  results\tables
  results\figures
  results\models
  manuscript
  logs

Workspace initialization completed.


In [2]:
from pathlib import Path
import hashlib
import json
import platform
import zipfile

import pandas as pd

PROJECT_ROOT = Path(r"D:\BDHS2022_FeverCare_SpatialEquity")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
LOG_DIR = PROJECT_ROOT / "logs"

LOG_DIR.mkdir(parents=True, exist_ok=True)

kr_path = RAW_DIR / "BDKR81FL.DTA"

gps_candidates = [
    RAW_DIR / "BDGE81FL.zip",
    RAW_DIR / "BDGE81FL (1).zip",
]

gps_path = next(
    (path for path in gps_candidates if path.is_file()),
    None,
)

# Verify that both source files are available.
if not kr_path.is_file() or gps_path is None:
    missing = []

    if not kr_path.is_file():
        missing.append("BDKR81FL.DTA")

    if gps_path is None:
        missing.append("BDGE81FL.zip")

    message = f"Missing source files: {', '.join(missing)}"
    (LOG_DIR / "01_input_audit.txt").write_text(
        message, encoding="utf-8"
    )

    print(message)
    raise FileNotFoundError(message)


def file_checksum(path):
    digest = hashlib.sha256()

    with path.open("rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


# Inspect the GPS archive without modifying the source files.
with zipfile.ZipFile(gps_path) as archive:
    archive_members = sorted(archive.namelist())
    archive_error = archive.testzip()

required_components = [
    "BDGE81FL.shp",
    "BDGE81FL.shx",
    "BDGE81FL.dbf",
    "BDGE81FL.prj",
]

archive_names = {Path(name).name for name in archive_members}

missing_components = [
    name for name in required_components
    if name not in archive_names
]


# Inspect Stata metadata and read the variables required for cohort screening.
required_variables = [
    "v001", "v005", "v021", "v023",
    "v024", "v025", "sdist",
    "h22", "h32y", "h46a", "h46b",
]

with pd.read_stata(
    kr_path,
    iterator=True,
    convert_categoricals=False,
) as reader:

    variable_labels = reader.variable_labels()

    missing_variables = [
        name for name in required_variables
        if name not in variable_labels
    ]

    available_variables = [
        name for name in required_variables
        if name in variable_labels
    ]

    data = reader.read(columns=available_variables)


# Construct the fever cohort using the DHS fever indicator.
if "h22" in data.columns:
    fever = data.loc[data["h22"] == 1].copy()
else:
    fever = pd.DataFrame()

cohort_summary = {
    "total_records": int(len(data)),
    "total_variables": int(len(variable_labels)),
    "fever_records": int(len(fever)),
}

if not fever.empty:
    cohort_summary.update({
        "fever_clusters": int(fever["v001"].nunique()),
        "h32y_code_0": int((fever["h32y"] == 0).sum()),
        "h32y_code_1": int((fever["h32y"] == 1).sum()),
        "first_care_source_recorded": int(fever["h46a"].notna().sum()),
        "pharmacy_first_code_36": int((fever["h46a"] == 36).sum()),
        "care_delay_recorded": int(fever["h46b"].notna().sum()),
    })


audit = {
    "environment": {
        "python_version": platform.python_version(),
        "pandas_version": pd.__version__,
    },
    "source_files": {
        "kr_filename": kr_path.name,
        "kr_sha256": file_checksum(kr_path),
        "gps_filename": gps_path.name,
        "gps_sha256": file_checksum(gps_path),
    },
    "gps_archive": {
        "member_count": len(archive_members),
        "integrity_ok": archive_error is None,
        "missing_components": missing_components,
    },
    "stata_variables": {
        "missing_required_variables": missing_variables,
    },
    "cohort_summary": cohort_summary,
}

report_lines = [
    "STEP 01 | RAW DATA INTEGRITY AUDIT",
    "=" * 45,
    f"Python version: {platform.python_version()}",
    f"Pandas version: {pd.__version__}",
    "",
    "SOURCE FILES",
    f"KR dataset: {kr_path.name}",
    f"GPS archive: {gps_path.name}",
    f"GPS archive integrity: {archive_error is None}",
    f"Missing GPS components: {missing_components}",
    f"Missing required variables: {missing_variables}",
    "",
    "COHORT SUMMARY",
]

report_lines.extend(
    f"{key}: {value}"
    for key, value in cohort_summary.items()
)

report_lines.extend([
    "",
    f"JSON report: {LOG_DIR / '01_input_audit.json'}",
    f"Text report: {LOG_DIR / '01_input_audit.txt'}",
])

report = "\n".join(report_lines)

# Preserve the displayed report and its structured metadata.
(LOG_DIR / "01_input_audit.txt").write_text(
    report + "\n",
    encoding="utf-8",
)

(LOG_DIR / "01_input_audit.json").write_text(
    json.dumps(audit, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)

print(report)

STEP 01 | RAW DATA INTEGRITY AUDIT
Python version: 3.14.6
Pandas version: 3.0.3

SOURCE FILES
KR dataset: BDKR81FL.DTA
GPS archive: BDGE81FL.zip
GPS archive integrity: True
Missing GPS components: []
Missing required variables: []

COHORT SUMMARY
total_records: 8784
total_variables: 1238
fever_records: 2630
fever_clusters: 642
h32y_code_0: 1974
h32y_code_1: 656
first_care_source_recorded: 1974
pharmacy_first_code_36: 1120
care_delay_recorded: 1974

JSON report: D:\BDHS2022_FeverCare_SpatialEquity\logs\01_input_audit.json
Text report: D:\BDHS2022_FeverCare_SpatialEquity\logs\01_input_audit.txt


In [4]:
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO
import importlib
import json
import subprocess
import sys

import numpy as np
import pandas as pd
from IPython.display import display

try:
    import shapefile
except ModuleNotFoundError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "--disable-pip-version-check", "pyshp"
    ])
    shapefile = importlib.import_module("shapefile")


PROJECT_ROOT = Path(r"D:\BDHS2022_FeverCare_SpatialEquity")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
LOG_DIR = PROJECT_ROOT / "logs"
TABLE_DIR = PROJECT_ROOT / "results" / "tables"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

for directory in (LOG_DIR, TABLE_DIR, PROCESSED_DIR):
    directory.mkdir(parents=True, exist_ok=True)

kr_path = RAW_DIR / "BDKR81FL.DTA"
gps_path = RAW_DIR / "BDGE81FL.zip"

for path in (kr_path, gps_path):
    if not path.is_file():
        raise FileNotFoundError(
            f"Required source file not found: {path}"
        )


kr_columns = [
    "v001", "v021", "v023", "v024", "v025", "v005",
    "sdist", "h22", "h32y", "h46a", "h46b",
]

kr = pd.read_stata(
    kr_path,
    columns=kr_columns,
    convert_categoricals=False,
)


# Read GPS attributes and point geometry from the original archive.
with ZipFile(gps_path) as archive:

    names = set(archive.namelist())

    required_members = {
        "BDGE81FL.shp",
        "BDGE81FL.shx",
        "BDGE81FL.dbf",
        "BDGE81FL.prj",
    }

    if not required_members.issubset(names):
        raise ValueError(
            f"Missing GPS components: "
            f"{sorted(required_members - names)}"
        )

    if archive.testzip() is not None:
        raise ValueError(
            "GPS archive failed integrity verification."
        )

    projection = archive.read(
        "BDGE81FL.prj"
    ).decode("utf-8-sig").upper()

    reader = shapefile.Reader(
        shp=BytesIO(archive.read("BDGE81FL.shp")),
        shx=BytesIO(archive.read("BDGE81FL.shx")),
        dbf=BytesIO(archive.read("BDGE81FL.dbf")),
        encoding="utf-8",
    )

    gps_rows = []

    for item in reader.iterShapeRecords():

        attributes = item.record.as_dict()

        if (
            item.shape.shapeType != shapefile.POINT
            or len(item.shape.points) != 1
        ):
            raise ValueError(
                "Unexpected GPS geometry: a point feature is required."
            )

        longitude, latitude = item.shape.points[0]

        gps_rows.append({
            "cluster_id": attributes["DHSCLUST"],
            "gps_division": attributes["ADM1DHS"],
            "gps_residence": attributes["URBAN_RURA"],
            "gps_latitude": attributes["LATNUM"],
            "gps_longitude": attributes["LONGNUM"],
            "geometry_latitude": latitude,
            "geometry_longitude": longitude,
            "country": attributes["DHSCC"],
            "survey_year": attributes["DHSYEAR"],
        })

    reader.close()


gps = pd.DataFrame.from_records(gps_rows)

valid_ids = bool(
    gps["cluster_id"].notna().all()
    and (gps["cluster_id"] % 1 == 0).all()
)

if not valid_ids:
    raise ValueError(
        "GPS cluster IDs contain missing or non-integer values."
    )

gps["cluster_id"] = gps["cluster_id"].astype("int64")


# Validate the child-to-cluster linkage.
lookup_columns = [
    "cluster_id",
    "gps_division",
    "gps_residence",
    "gps_latitude",
    "gps_longitude",
]

lookup = gps[lookup_columns]

linked = kr.merge(
    lookup,
    how="left",
    left_on="v001",
    right_on="cluster_id",
    validate="many_to_one",
    indicator=True,
)

fever = linked.loc[linked["h22"].eq(1)].copy()

care_sought = fever["h32y"].eq(0)
no_care = fever["h32y"].eq(1)


# Validate geographic coordinates against shapefile geometry.
coordinate_valid = (
    gps["gps_latitude"].between(20, 27)
    & gps["gps_longitude"].between(88, 93)
)

geometry_valid = bool(
    np.allclose(
        gps["gps_latitude"],
        gps["geometry_latitude"],
        rtol=0,
        atol=1e-6,
    )
    and np.allclose(
        gps["gps_longitude"],
        gps["geometry_longitude"],
        rtol=0,
        atol=1e-6,
    )
)


metrics = {
    "kr_records": int(len(kr)),
    "kr_clusters": int(kr["v001"].nunique()),
    "gps_clusters": int(gps["cluster_id"].nunique()),

    "duplicate_gps_cluster_ids": int(
        gps["cluster_id"].duplicated().sum()
    ),

    "unmatched_kr_records": int(
        linked["_merge"].eq("left_only").sum()
    ),

    "unmatched_gps_clusters": int(
        (~gps["cluster_id"].isin(kr["v001"])).sum()
    ),

    "psu_id_mismatches": int(
        kr["v001"].ne(kr["v021"]).sum()
    ),

    "division_mismatches": int(
        linked["v024"].ne(linked["gps_division"]).sum()
    ),

    "residence_mismatches": int(
        (
            (linked["v025"] == 1)
            != (linked["gps_residence"] == "U")
        ).sum()
    ),

    "invalid_gps_coordinates": int(
        (~coordinate_valid).sum()
    ),

    "fever_children": int(len(fever)),

    "fever_clusters": int(
        fever["v001"].nunique()
    ),

    "fever_children_without_gps": int(
        fever["cluster_id"].isna().sum()
    ),

    "fever_unknown_care_status": int(
        (~(care_sought | no_care)).sum()
    ),

    "care_source_skip_errors": int(
        (care_sought != fever["h46a"].notna()).sum()
    ),

    "care_delay_skip_errors": int(
        (care_sought != fever["h46b"].notna()).sum()
    ),

    "care_sought": int(care_sought.sum()),

    "no_care_sought": int(no_care.sum()),

    "pharmacy_first": int(
        fever["h46a"].eq(36).sum()
    ),
}


checks = {
    "WGS84 geographic coordinates": (
        "GCS_WGS_1984" in projection
        and "WGS_1984" in projection
    ),

    "GPS survey identity": bool(
        gps["country"].eq("BD").all()
        and gps["survey_year"].eq(2022).all()
    ),

    "Unique GPS cluster IDs":
        metrics["duplicate_gps_cluster_ids"] == 0,

    "Complete KR-GPS linkage": (
        metrics["unmatched_kr_records"] == 0
        and metrics["unmatched_gps_clusters"] == 0
    ),

    "Consistent PSU identifiers":
        metrics["psu_id_mismatches"] == 0,

    "Consistent division identifiers":
        metrics["division_mismatches"] == 0,

    "Consistent residence classification": bool(
        metrics["residence_mismatches"] == 0
        and gps["gps_residence"].isin(["U", "R"]).all()
    ),

    "Valid GPS coordinates and geometries": bool(
        metrics["invalid_gps_coordinates"] == 0
        and geometry_valid
    ),

    "Complete fever GPS linkage":
        metrics["fever_children_without_gps"] == 0,

    "Known fever care status":
        metrics["fever_unknown_care_status"] == 0,

    "Valid care-source skip logic":
        metrics["care_source_skip_errors"] == 0,

    "Valid care-delay skip logic":
        metrics["care_delay_skip_errors"] == 0,

    "Complete survey design variables": bool(
        kr[["v005", "v021", "v023"]].notna().all().all()
        and kr["v005"].gt(0).all()
    ),
}


# Preserve validation results and software metadata.
checks_table = pd.DataFrame({
    "verification": list(checks),
    "status": [
        "PASS" if passed else "FAIL"
        for passed in checks.values()
    ],
})

all_passed = all(checks.values())

checks_table.to_csv(
    TABLE_DIR / "02_gps_linkage_checks.csv",
    index=False,
)

audit = {
    "software": {
        "pandas": pd.__version__,
        "pyshp": shapefile.__version__,
    },
    "metrics": metrics,
    "checks": checks,
}

(LOG_DIR / "02_gps_linkage_audit.json").write_text(
    json.dumps(audit, indent=2) + "\n",
    encoding="utf-8",
)

report_lines = [
    "STEP 02 | GPS LINKAGE AND COHORT CONSISTENCY",
    "=" * 48,
    *(
        f"{name}: {value}"
        for name, value in metrics.items()
    ),
    "",
    f"Overall verification: {'PASS' if all_passed else 'FAIL'}",
    f"PyShp version: {shapefile.__version__}",
]

report = "\n".join(report_lines)

(LOG_DIR / "02_gps_linkage_report.txt").write_text(
    report + "\n",
    encoding="utf-8",
)


# Release the validated GPS lookup only after all checks pass.
if all_passed:
    lookup.sort_values("cluster_id").to_csv(
        PROCESSED_DIR / "02_cluster_gps_lookup.csv",
        index=False,
    )


print(report)

print(
    f"\nSaved audit: "
    f"{LOG_DIR / '02_gps_linkage_audit.json'}"
)

print(
    f"Saved checks: "
    f"{TABLE_DIR / '02_gps_linkage_checks.csv'}"
)

if all_passed:
    print(
        f"Saved GPS lookup: "
        f"{PROCESSED_DIR / '02_cluster_gps_lookup.csv'}"
    )

display(checks_table)

if not all_passed:
    raise ValueError(
        "GPS linkage validation failed; review the audit files."
    )

STEP 02 | GPS LINKAGE AND COHORT CONSISTENCY
kr_records: 8784
kr_clusters: 674
gps_clusters: 674
duplicate_gps_cluster_ids: 0
unmatched_kr_records: 0
unmatched_gps_clusters: 0
psu_id_mismatches: 0
division_mismatches: 0
residence_mismatches: 0
invalid_gps_coordinates: 0
fever_children: 2630
fever_clusters: 642
fever_children_without_gps: 0
fever_unknown_care_status: 0
care_source_skip_errors: 0
care_delay_skip_errors: 0
care_sought: 1974
no_care_sought: 656
pharmacy_first: 1120

Overall verification: PASS
PyShp version: 3.1.9

Saved audit: D:\BDHS2022_FeverCare_SpatialEquity\logs\02_gps_linkage_audit.json
Saved checks: D:\BDHS2022_FeverCare_SpatialEquity\results\tables\02_gps_linkage_checks.csv
Saved GPS lookup: D:\BDHS2022_FeverCare_SpatialEquity\data\processed\02_cluster_gps_lookup.csv


,verification,status
0,WGS84 geographic coordinates,PASS
1,GPS survey identity,PASS
2,Unique GPS cluster IDs,PASS
3,Complete KR-GPS linkage,PASS
4,Consistent PSU identifiers,PASS
5,Consistent division identifiers,PASS
6,Consistent residence classification,PASS
7,Valid GPS coordinates and geometries,PASS
8,Complete fever GPS linkage,PASS
9,Known fever care status,PASS


In [5]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path(r"D:\BDHS2022_FeverCare_SpatialEquity")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
LOG_DIR = PROJECT_ROOT / "logs"
TABLE_DIR = PROJECT_ROOT / "results" / "tables"

LOG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

source_path = RAW_DIR / "BDKR81FL.DTA"

if not source_path.is_file():
    raise FileNotFoundError(
        f"Source dataset not found: {source_path}"
    )


# Define the variables required for cohort and predictor screening.
variable_groups = {
    "Survey design": [
        "v001", "v005", "v021", "v022",
        "v023", "v024", "v025", "sdist"
    ],
    "Eligibility and outcomes": [
        "h22", "h32y", "h46a", "h46b"
    ],
    "Child characteristics": [
        "b19", "b4", "h31", "h31b"
    ],
    "Household and maternal characteristics": [
        "v012", "v106", "v149", "v190",
        "v467b", "v467c", "v467d", "v467f"
    ],
    "Antibiotic variables for secondary analysis": [
        "h37z", "h37o", "h37p", "h37q",
        "h37r", "s631j", "s631k"
    ],
}

variables = [
    name
    for names in variable_groups.values()
    for name in names
]

with pd.read_stata(
    source_path,
    iterator=True,
    convert_categoricals=False
) as reader:
    stata_labels = reader.variable_labels()

missing_variables = [
    name for name in variables
    if name not in stata_labels
]

if missing_variables:
    raise ValueError(
        f"Required variables unavailable: {missing_variables}"
    )


# Read the original numeric codes and labelled responses separately.
raw = pd.read_stata(
    source_path,
    columns=variables,
    convert_categoricals=False
)

labeled = pd.read_stata(
    source_path,
    columns=variables,
    convert_categoricals=True
)

fever_mask = raw["h22"].eq(1)

fever = raw.loc[fever_mask]
fever_labels = labeled.loc[fever_mask]

care_mask = fever["h32y"].eq(0)
no_care_mask = fever["h32y"].eq(1)

conditional_variables = {"h46a", "h46b"}

frequency_variables = [
    "h22", "h32y", "h46a", "h46b",
    "b4", "h31", "h31b",
    "v024", "v025",
    "v106", "v149", "v190",
    "v467b", "v467c", "v467d", "v467f",
    "h37z", "h37o", "h37p", "h37q",
    "h37r", "s631j", "s631k"
]


# Distinguish structural non-applicability from unexplained missingness.
rows = []

for group, names in variable_groups.items():
    for variable in names:

        eligible = (
            care_mask
            if variable in conditional_variables
            else pd.Series(True, index=fever.index)
        )

        observed = fever.loc[eligible, variable].notna()

        n_eligible = int(eligible.sum())
        n_observed = int(observed.sum())
        n_missing = n_eligible - n_observed

        rows.append({
            "group": group,
            "variable": variable,
            "stata_label": stata_labels[variable],
            "data_type": str(fever[variable].dtype),
            "eligible_n": n_eligible,
            "observed_n": n_observed,
            "unexplained_missing_n": n_missing,
            "unexplained_missing_pct": round(
                100 * n_missing / n_eligible, 2
            ),
            "structural_not_applicable_n": int(
                (~eligible).sum()
            ),
            "distinct_observed_values": int(
                fever.loc[eligible, variable].nunique(
                    dropna=True
                )
            )
        })

dictionary = pd.DataFrame(rows)


# Tabulate observed response codes without altering their definitions.
frequency_rows = []

for variable in frequency_variables:

    eligible = (
        care_mask
        if variable in conditional_variables
        else pd.Series(True, index=fever.index)
    )

    index = fever.index[
        eligible & fever[variable].notna()
    ]

    values = pd.DataFrame({
        "raw_code": fever.loc[
            index, variable
        ].to_numpy(),

        "stata_value_label": fever_labels.loc[
            index, variable
        ].astype("string").to_numpy()
    })

    distribution = (
        values.groupby(
            ["raw_code", "stata_value_label"],
            dropna=False,
            observed=True
        )
        .size()
        .reset_index(name="n")
        .sort_values("raw_code")
    )

    for record in distribution.itertuples(index=False):

        code = (
            str(int(record.raw_code))
            if float(record.raw_code).is_integer()
            else str(record.raw_code)
        )

        frequency_rows.append({
            "variable": variable,
            "raw_code": code,
            "stata_value_label": str(
                record.stata_value_label
            ),
            "n": int(record.n),
            "denominator_n": int(eligible.sum()),
            "percent": round(
                100 * record.n / int(eligible.sum()), 2
            )
        })

codebook = pd.DataFrame(frequency_rows)


# Verify cohort counts, eligibility and survey-design consistency.
metrics = {
    "all_children": int(len(raw)),
    "fever_children": int(len(fever)),
    "fever_clusters": int(
        fever["v021"].nunique()
    ),
    "care_sought": int(care_mask.sum()),
    "no_care_sought": int(no_care_mask.sum()),
    "pharmacy_first_among_care_seekers": int(
        fever.loc[care_mask, "h46a"].eq(36).sum()
    ),
    "other_first_source_among_care_seekers": int(
        fever.loc[care_mask, "h46a"].ne(36).sum()
    ),
    "care_source_other_code_96": int(
        fever["h46a"].eq(96).sum()
    ),
    "short_rapid_breathing_unknown_code_8": int(
        fever["h31b"].eq(8).sum()
    ),
    "medication_unknown_h37z_code_1": int(
        fever["h37z"].eq(1).sum()
    ),
    "survey_strata": int(
        fever["v023"].nunique()
    ),
    "nonpositive_or_missing_weights": int(
        (~fever["v005"].gt(0)).sum()
    )
}

checks = {
    "Fever cohort and cluster counts": (
        metrics["fever_children"] == 2630
        and metrics["fever_clusters"] == 642
    ),

    "Care-status codes are 0 or 1": bool(
        fever["h32y"].isin([0, 1]).all()
    ),

    "Care-seeking counts are reconciled": (
        metrics["care_sought"]
        + metrics["no_care_sought"]
        == metrics["fever_children"]
    ),

    "First-care source observed only for care seekers": bool(
        fever["h46a"].notna().eq(care_mask).all()
    ),

    "Care delay observed only for care seekers": bool(
        fever["h46b"].notna().eq(care_mask).all()
    ),

    "First-care source categories are mutually exclusive": (
        metrics["pharmacy_first_among_care_seekers"]
        + metrics["other_first_source_among_care_seekers"]
        == metrics["care_sought"]
    ),

    "Stratification variables agree": bool(
        fever["v022"].eq(fever["v023"]).all()
    ),

    "PSU and cluster IDs agree": bool(
        fever["v021"].eq(fever["v001"]).all()
    ),

    "All analytic weights are positive": (
        metrics["nonpositive_or_missing_weights"] == 0
    ),

    "Child age in months is within 0 to 59": bool(
        fever["b19"].between(0, 59).all()
    ),

    "No unexpected missing values in selected variables": bool(
        (dictionary["unexplained_missing_n"] == 0).all()
    )
}

checks_table = pd.DataFrame([
    {
        "check": name,
        "status": "PASS" if result else "FAIL"
    }
    for name, result in checks.items()
])

all_passed = all(checks.values())


# Save the complete audit before proceeding to data transformation.
paths = {
    "dictionary": TABLE_DIR / "03_variable_dictionary.csv",
    "codebook": TABLE_DIR / "03_response_codebook.csv",
    "checks": TABLE_DIR / "03_preprocessing_checks.csv",
    "audit": LOG_DIR / "03_variable_audit.json",
    "report": LOG_DIR / "03_variable_audit.txt"
}

dictionary.to_csv(
    paths["dictionary"],
    index=False,
    encoding="utf-8-sig"
)

codebook.to_csv(
    paths["codebook"],
    index=False,
    encoding="utf-8-sig"
)

checks_table.to_csv(
    paths["checks"],
    index=False,
    encoding="utf-8-sig"
)

audit = {
    "metrics": metrics,
    "checks": checks,
    "notes": {
        "care_status": (
            "h32y=0: care sought; "
            "h32y=1: no treatment reported"
        ),
        "care_source_denominator": (
            "h46a is applicable to care seekers only"
        ),
        "care_delay_denominator": (
            "h46b is applicable to care seekers only"
        ),
        "weights": (
            "v005 is stored with six implied decimal places; "
            "do not use unscaled weights in analyses"
        ),
        "special_codes": (
            "Retain h31b=8 and h37z=1 as explicit "
            "categories during auditing"
        ),
        "geographic_coordinates": (
            "Use displaced DHS GPS coordinates; "
            "do not infer precise facilities or addresses"
        )
    }
}

paths["audit"].write_text(
    json.dumps(
        audit,
        indent=2,
        ensure_ascii=False
    ) + "\n",
    encoding="utf-8"
)

report = "\n".join([
    "STEP 03 | VARIABLE CODING AND ELIGIBILITY AUDIT",
    "=" * 50,
    *(
        f"{key}: {value}"
        for key, value in metrics.items()
    ),
    "",
    f"Variables examined: {len(variables)}",
    (
        "Unexpected missing across audited variables: "
        f"{int(dictionary['unexplained_missing_n'].sum())}"
    ),
    f"Validation: {'PASS' if all_passed else 'FAIL'}",
    "",
    *(
        f"{name}: {path}"
        for name, path in paths.items()
    )
])

paths["report"].write_text(
    report + "\n",
    encoding="utf-8"
)

print(report)

print("\nVALIDATION CHECKS")
display(checks_table)

print("\nVARIABLE DICTIONARY")
display(dictionary)

print("\nRESPONSE CODEBOOK")
display(codebook)

if not all_passed:
    raise ValueError(
        "Variable audit failed; examine the saved checks "
        "before preprocessing."
    )

STEP 03 | VARIABLE CODING AND ELIGIBILITY AUDIT
all_children: 8784
fever_children: 2630
fever_clusters: 642
care_sought: 1974
no_care_sought: 656
pharmacy_first_among_care_seekers: 1120
other_first_source_among_care_seekers: 854
care_source_other_code_96: 33
short_rapid_breathing_unknown_code_8: 6
medication_unknown_h37z_code_1: 66
survey_strata: 16
nonpositive_or_missing_weights: 0

Variables examined: 31
Unexpected missing across audited variables: 0
Validation: PASS

dictionary: D:\BDHS2022_FeverCare_SpatialEquity\results\tables\03_variable_dictionary.csv
codebook: D:\BDHS2022_FeverCare_SpatialEquity\results\tables\03_response_codebook.csv
checks: D:\BDHS2022_FeverCare_SpatialEquity\results\tables\03_preprocessing_checks.csv
audit: D:\BDHS2022_FeverCare_SpatialEquity\logs\03_variable_audit.json
report: D:\BDHS2022_FeverCare_SpatialEquity\logs\03_variable_audit.txt

VALIDATION CHECKS


,check,status
0,Fever cohort and cluster counts,PASS
1,Care-status codes are 0 or 1,PASS
2,Care-seeking counts are reconciled,PASS
3,First-care source observed only for care seekers,PASS
4,Care delay observed only for care seekers,PASS
5,First-care source categories are mutually excl...,PASS
6,Stratification variables agree,PASS
7,PSU and cluster IDs agree,PASS
8,All analytic weights are positive,PASS
9,Child age in months is within 0 to 59,PASS



VARIABLE DICTIONARY


,group,variable,stata_label,data_type,eligible_n,observed_n,unexplained_missing_n,unexplained_missing_pct,structural_not_applicable_n,distinct_observed_values
0,Survey design,v001,cluster number,int16,2630,2630,0,0.0,0,642
1,Survey design,v005,women's individual sample weight (6 decimals),int32,2630,2630,0,0.0,0,628
2,Survey design,v021,primary sampling unit,int16,2630,2630,0,0.0,0,642
3,Survey design,v022,sample strata for sampling errors,int8,2630,2630,0,0.0,0,16
4,Survey design,v023,stratification used in sample design,int8,2630,2630,0,0.0,0,16
5,Survey design,v024,division,int8,2630,2630,0,0.0,0,8
6,Survey design,v025,type of place of residence,int8,2630,2630,0,0.0,0,2
7,Survey design,sdist,district,int8,2630,2630,0,0.0,0,64
8,Eligibility and outcomes,h22,had fever in last two weeks,float64,2630,2630,0,0.0,0,1
9,Eligibility and outcomes,h32y,fever/cough: no treatment,float64,2630,2630,0,0.0,0,2



RESPONSE CODEBOOK


,variable,raw_code,stata_value_label,n,denominator_n,percent
0,h22,1,yes,2630,2630,100.00
1,h32y,0,no: received treatment,1974,2630,75.06
2,h32y,1,yes: no treatment,656,2630,24.94
3,h46a,11,medical college hospital,24,1974,1.22
4,h46a,12,specialized govt hospital,23,1974,1.17
...,...,...,...,...,...,...
82,h37r,1,yes,18,2630,0.68
83,s631j,0,no,2492,2630,94.75
84,s631j,1,yes,138,2630,5.25
85,s631k,0,no,2612,2630,99.32


In [6]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path(r"D:\BDHS2022_FeverCare_SpatialEquity")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLE_DIR = PROJECT_ROOT / "results" / "tables"
LOG_DIR = PROJECT_ROOT / "logs"

kr_path = RAW_DIR / "BDKR81FL.DTA"
dictionary_path = TABLE_DIR / "03_variable_dictionary.csv"
codebook_path = TABLE_DIR / "03_response_codebook.csv"
gps_lookup_path = PROCESSED_DIR / "02_cluster_gps_lookup.csv"

for path in (kr_path, dictionary_path, codebook_path, gps_lookup_path):
    if not path.is_file():
        raise FileNotFoundError(f"Required input not found: {path}")

variable_dictionary = pd.read_csv(dictionary_path)
codebook = pd.read_csv(codebook_path, dtype={"raw_code": "string"})

child_keys = ["v001", "v002", "v003", "bidx"]

source_columns = list(dict.fromkeys(
    child_keys + variable_dictionary["variable"].tolist()
))

kr = pd.read_stata(
    kr_path,
    columns=source_columns,
    convert_categoricals=False,
)

cohort = kr.loc[kr["h22"].eq(1)].copy()
cohort.insert(0, "source_row_number", cohort.index.to_numpy() + 1)
cohort.reset_index(drop=True, inplace=True)

care = cohort["h32y"].eq(0)
no_care = cohort["h32y"].eq(1)
pharmacy = care & cohort["h46a"].eq(36)
other_source = care & cohort["h46a"].ne(36)

# Preserve original response codes alongside the derived outcomes.
cohort["care_sought"] = care.astype("int8")

cohort["care_pathway_code"] = np.select(
    [no_care, pharmacy, other_source],
    [0, 1, 2],
    default=-1,
).astype("int8")

cohort["pharmacy_first_if_care"] = pd.Series(
    pd.NA,
    index=cohort.index,
    dtype="Int8",
)

cohort.loc[care, "pharmacy_first_if_care"] = (
    cohort.loc[care, "h46a"].eq(36).astype("int8")
)

cohort["early_care_within_1d_if_care"] = pd.Series(
    pd.NA,
    index=cohort.index,
    dtype="Int8",
)

cohort.loc[care, "early_care_within_1d_if_care"] = (
    cohort.loc[care, "h46b"].isin([0, 1]).astype("int8")
)

cohort["unspecified_first_source"] = (
    cohort["h46a"].eq(96).astype("int8")
)

cohort["rapid_breathing_unknown"] = (
    cohort["h31b"].eq(8).astype("int8")
)

cohort["medication_unknown"] = (
    cohort["h37z"].eq(1).astype("int8")
)

cohort["survey_weight"] = (
    cohort["v005"].astype("float64") / 1_000_000
)

lookup = pd.read_csv(
    gps_lookup_path,
    usecols=["cluster_id"],
)

codebook["raw_code"] = codebook["raw_code"].str.replace(
    r"\.0$", "", regex=True
)

pharmacy_label = codebook.loc[
    codebook["variable"].eq("h46a")
    & codebook["raw_code"].eq("36"),
    "stata_value_label",
]

hash_value = hashlib.sha256(
    kr_path.read_bytes()
).hexdigest()

input_audit_path = LOG_DIR / "01_input_audit.json"

with input_audit_path.open(encoding="utf-8") as handle:
    original_audit = json.load(handle)

expected_hash = original_audit["source_files"]["kr_sha256"]

checks = {
    "Raw file matches Step 1 checksum":
        hash_value == expected_hash,

    "Fever cohort includes 2,630 children":
        len(cohort) == 2630,

    "Child identifiers are complete and unique": bool(
        cohort[child_keys].notna().all().all()
        and not cohort.duplicated(subset=child_keys).any()
    ),

    "All clusters occur in the validated GPS lookup": bool(
        cohort["v001"].isin(lookup["cluster_id"]).all()
        and not lookup["cluster_id"].duplicated().any()
    ),

    "Original care-status codes are valid": bool(
        cohort["h32y"].isin([0, 1]).all()
    ),

    "First-care and delay responses follow skip logic": bool(
        cohort["h46a"].notna().eq(care).all()
        and cohort["h46b"].notna().eq(care).all()
    ),

    "Pharmacy code is verified against the Stata label": bool(
        pharmacy_label.str.contains(
            "pharmacy", case=False, na=False
        ).any()
    ),

    "Care-pathway groups partition the fever cohort": bool(
        cohort["care_pathway_code"].isin([0, 1, 2]).all()
        and int(
            no_care.sum() + pharmacy.sum() + other_source.sum()
        ) == len(cohort)
    ),

    "Conditional outcomes have structural missingness only": bool(
        cohort.loc[
            no_care,
            ["pharmacy_first_if_care", "early_care_within_1d_if_care"]
        ].isna().all().all()
        and cohort.loc[
            care,
            ["pharmacy_first_if_care", "early_care_within_1d_if_care"]
        ].notna().all().all()
    ),

    "Care delay has valid nonnegative reported values": bool(
        cohort.loc[care, "h46b"].between(0, 15).all()
    ),

    "Survey weights are positive": bool(
        cohort["survey_weight"].gt(0).all()
    ),

    "Previously verified cohort counts are reproduced": bool(
        care.sum() == 1974
        and no_care.sum() == 656
        and pharmacy.sum() == 1120
        and other_source.sum() == 854
        and cohort["v021"].nunique() == 642
    ),
}

check_table = pd.DataFrame({
    "check": list(checks),
    "status": [
        "PASS" if result else "FAIL"
        for result in checks.values()
    ],
})


def summary_row(name, target, eligible, population):
    numerator = cohort.loc[
        target & eligible, "survey_weight"
    ].sum()

    denominator = cohort.loc[
        eligible, "survey_weight"
    ].sum()

    return {
        "indicator": name,
        "population": population,
        "n": int((target & eligible).sum()),
        "denominator_n": int(eligible.sum()),
        "weighted_percent": round(
            100 * numerator / denominator, 2
        ),
    }


all_fever = pd.Series(True, index=cohort.index)
early = care & cohort["h46b"].isin([0, 1])

summary_table = pd.DataFrame([
    summary_row(
        "No reported care", no_care, all_fever, "Fever cohort"
    ),
    summary_row(
        "Care sought", care, all_fever, "Fever cohort"
    ),
    summary_row(
        "Pharmacy first", pharmacy, all_fever, "Fever cohort"
    ),
    summary_row(
        "Other first-care source",
        other_source, all_fever, "Fever cohort"
    ),
    summary_row(
        "Pharmacy first", pharmacy, care, "Care seekers only"
    ),
    summary_row(
        "Care within one day", early, care, "Care seekers only"
    ),
    summary_row(
        "Unspecified first-care source",
        cohort["h46a"].eq(96), care, "Care seekers only"
    ),
])

all_passed = all(checks.values())

checks_path = TABLE_DIR / "04_outcome_construction_checks.csv"
summary_path = TABLE_DIR / "04_outcome_summary.csv"
report_path = LOG_DIR / "04_outcome_construction_report.txt"
audit_path = LOG_DIR / "04_outcome_construction_audit.json"
cohort_path = PROCESSED_DIR / "04_fever_care_analytic.csv"
schema_path = PROCESSED_DIR / "04_fever_care_schema.json"

check_table.to_csv(checks_path, index=False)
summary_table.to_csv(summary_path, index=False)

report = "\n".join([
    "STEP 04 | TWO-STAGE OUTCOME CONSTRUCTION",
    "=" * 46,
    f"Fever children: {len(cohort)}",
    f"Care sought: {int(care.sum())}",
    f"No reported care: {int(no_care.sum())}",
    f"Pharmacy-first: {int(pharmacy.sum())}",
    f"Other first-care source: {int(other_source.sum())}",
    f"Care within one day: {int(early.sum())}",
    (
        "Unspecified first source (h46a=96): "
        f"{int(cohort['unspecified_first_source'].sum())}"
    ),
    (
        "Unknown rapid breathing (h31b=8): "
        f"{int(cohort['rapid_breathing_unknown'].sum())}"
    ),
    (
        "Unknown medication (h37z=1): "
        f"{int(cohort['medication_unknown'].sum())}"
    ),
    f"Validation: {'PASS' if all_passed else 'FAIL'}",
])

report_path.write_text(report + "\n", encoding="utf-8")

audit_path.write_text(
    json.dumps({
        "raw_sha256": hash_value,
        "checks": checks,
        "cohort_n": int(len(cohort)),
        "processed_written": bool(all_passed),
    }, indent=2) + "\n",
    encoding="utf-8",
)

if all_passed:
    cohort.to_csv(cohort_path, index=False)

    schema = {
        "source": "BDKR81FL.DTA",
        "cohort": "Children with h22=1",
        "record_key": child_keys,
        "weights": "survey_weight = v005 / 1,000,000",
        "survey_design": {
            "psu": "v021",
            "stratum": "v023",
        },
        "outcomes": {
            "care_sought": {
                "1": "Care sought",
                "0": "No reported care",
            },
            "care_pathway_code": {
                "0": "No reported care",
                "1": "Pharmacy first",
                "2": (
                    "Other first-care source, "
                    "including unspecified source"
                ),
            },
            "pharmacy_first_if_care": (
                "Conditional on care sought; "
                "undefined otherwise"
            ),
            "early_care_within_1d_if_care": (
                "Care sought on day 0 or day 1; "
                "undefined without care"
            ),
        },
        "special_codes_retained": [
            "h46a=96", "h31b=8", "h37z=1"
        ],
        "gps_data": (
            "Coordinates remain in the separate "
            "validated cluster lookup"
        ),
    }

    schema_path.write_text(
        json.dumps(schema, indent=2) + "\n",
        encoding="utf-8",
    )

print(report)

print("\nVALIDATION CHECKS")
display(check_table)

print("\nDESCRIPTIVE OUTCOME SUMMARY")
display(summary_table)

print("\nSAVED FILES")

for path in (
    checks_path,
    summary_path,
    report_path,
    audit_path,
):
    print(path)

if all_passed:
    print(cohort_path)
    print(schema_path)
else:
    raise ValueError(
        "Outcome construction failed validation; "
        "review the saved checks."
    )

STEP 04 | TWO-STAGE OUTCOME CONSTRUCTION
Fever children: 2630
Care sought: 1974
No reported care: 656
Pharmacy-first: 1120
Other first-care source: 854
Care within one day: 1368
Unspecified first source (h46a=96): 33
Unknown rapid breathing (h31b=8): 6
Unknown medication (h37z=1): 66
Validation: PASS

VALIDATION CHECKS


,check,status
0,Raw file matches Step 1 checksum,PASS
1,"Fever cohort includes 2,630 children",PASS
2,Child identifiers are complete and unique,PASS
3,All clusters occur in the validated GPS lookup,PASS
4,Original care-status codes are valid,PASS
5,First-care and delay responses follow skip logic,PASS
6,Pharmacy code is verified against the Stata label,PASS
7,Care-pathway groups partition the fever cohort,PASS
8,Conditional outcomes have structural missingne...,PASS
9,Care delay has valid nonnegative reported values,PASS



DESCRIPTIVE OUTCOME SUMMARY


,indicator,population,n,denominator_n,weighted_percent
0,No reported care,Fever cohort,656,2630,23.78
1,Care sought,Fever cohort,1974,2630,76.22
2,Pharmacy first,Fever cohort,1120,2630,44.03
3,Other first-care source,Fever cohort,854,2630,32.19
4,Pharmacy first,Care seekers only,1120,1974,57.76
5,Care within one day,Care seekers only,1368,1974,69.51
6,Unspecified first-care source,Care seekers only,33,1974,1.57



SAVED FILES
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\04_outcome_construction_checks.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\04_outcome_summary.csv
D:\BDHS2022_FeverCare_SpatialEquity\logs\04_outcome_construction_report.txt
D:\BDHS2022_FeverCare_SpatialEquity\logs\04_outcome_construction_audit.json
D:\BDHS2022_FeverCare_SpatialEquity\data\processed\04_fever_care_analytic.csv
D:\BDHS2022_FeverCare_SpatialEquity\data\processed\04_fever_care_schema.json


In [7]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path(r"D:\BDHS2022_FeverCare_SpatialEquity")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
TABLE_DIR = PROJECT_ROOT / "results" / "tables"
LOG_DIR = PROJECT_ROOT / "logs"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

kr_path = RAW_DIR / "BDKR81FL.DTA"
codebook_path = TABLE_DIR / "03_response_codebook.csv"

if not codebook_path.exists():
    raise FileNotFoundError(codebook_path)

codebook = pd.read_csv(
    codebook_path,
    dtype={"raw_code": "string"}
)

first_source_labels = codebook.loc[
    codebook["variable"].eq("h46a")
].copy()

first_source_labels["h46a_code"] = pd.to_numeric(
    first_source_labels["raw_code"]
).astype(int)


# Define candidate predictors according to the study framework.
candidate_groups = {
    "core": {
        "b19": "Child age in months",
        "b4": "Child sex",
        "h31": "Cough in the previous two weeks",
        "h31b": "Reported short, rapid breathing",
        "v012": "Maternal age",
        "v106": "Maternal educational level",
        "v190": "Household wealth quintile",
        "bord": "Birth order",
        "v025": "Urban or rural residence",
        "v024": "Survey division",
    },
    "access_barrier": {
        "v467c": "Mother-reported money barrier to her own healthcare",
        "v467d": "Mother-reported distance barrier to her own healthcare",
    },
    "sensitivity": {
        "v467b": "Mother-reported permission barrier",
        "v467f": "Mother-reported concern about going alone",
        "v714": "Maternal current employment",
        "v136": "Household size",
    },
}


# Preserve the first-contact categories used in the survey.
provider_codes = {
    "Public sector": [11, 12, 13, 14, 15, 16, 17, 18, 19],
    "Private clinical source": [31, 32, 33, 34],
    "NGO sector": [41, 45],
    "Pharmacy or drug store": [36],
    "Non-qualified doctor": [35],
    "Other or unspecified source": [96],
}

source_variables = [
    "h22", "h32y", "h46a", "h46b",
    "h37z", "v005", "v021", "v023"
]

source_variables += [
    variable
    for group in candidate_groups.values()
    for variable in group
]

source_variables = list(dict.fromkeys(source_variables))


# Check variable availability before loading the analytic subset.
with pd.read_stata(
    kr_path,
    iterator=True,
    convert_categoricals=False
) as reader:
    labels = reader.variable_labels()

missing_columns = sorted(
    set(source_variables) - set(labels)
)

if missing_columns:
    raise ValueError(
        f"Required columns are missing: {missing_columns}"
    )

kr = pd.read_stata(
    kr_path,
    columns=source_variables,
    convert_categoricals=False
)

fever = kr.loc[kr["h22"].eq(1)].copy()

fever["survey_weight"] = (
    fever["v005"].astype(float) / 1_000_000
)

care_sought = fever["h32y"].eq(0)
no_care = fever["h32y"].eq(1)


# Assign mutually exclusive first-contact categories.
provider_labels = pd.Series(
    pd.NA,
    index=fever.index,
    dtype="string"
)

provider_labels.loc[no_care] = "No reported care"

for name, codes in provider_codes.items():
    eligible = care_sought & fever["h46a"].isin(codes)
    provider_labels.loc[eligible] = name

category_order = [
    "No reported care",
    *provider_codes.keys()
]

weighted_total = fever["survey_weight"].sum()
summary_rows = []

for category in category_order:

    group = provider_labels.eq(category).fillna(False)

    summary_rows.append({
        "first_care_category": category,
        "n": int(group.sum()),
        "weighted_percent_all_fever": round(
            100 * fever.loc[
                group, "survey_weight"
            ].sum() / weighted_total,
            3
        ),
    })

provider_summary = pd.DataFrame(summary_rows)


# Link provider codes to their original Stata value labels.
source_crosswalk = []

for category, codes in provider_codes.items():
    for code in codes:
        source_crosswalk.append({
            "h46a_code": code,
            "source_group": category,
            "n": int(fever["h46a"].eq(code).sum()),
        })

crosswalk = (
    pd.DataFrame(source_crosswalk)
    .merge(
        first_source_labels[
            ["h46a_code", "stata_value_label"]
        ],
        on="h46a_code",
        how="left",
        validate="one_to_one"
    )
    .sort_values("h46a_code")
)


# Record completeness and response structure for each predictor.
predictor_rows = []

for tier, candidates in candidate_groups.items():
    for variable, description in candidates.items():

        values = fever[variable]

        predictor_rows.append({
            "variable": variable,
            "tier": tier,
            "construct": description,
            "stata_label": labels[variable],
            "missing_n": int(values.isna().sum()),
            "distinct_values": int(
                values.nunique(dropna=True)
            ),
            "special_code_8_n": (
                int(values.eq(8).sum())
                if variable == "h31b"
                else pd.NA
            ),
        })

predictors = pd.DataFrame(predictor_rows)


# Examine joint symptom coding before deriving severity indicators.
symptoms = pd.crosstab(
    fever["h31"],
    fever["h31b"],
    dropna=False
)

symptoms.index.name = "cough_code"
symptoms.columns.name = "rapid_breathing_code"


# Validate source classification and preservation of special codes.
checks = {
    "Fever cohort unchanged":
        len(fever) == 2630,

    "No unknown care-status values":
        bool(
            care_sought.sum() + no_care.sum()
            == len(fever)
        ),

    "All reported first-care codes classified":
        bool(provider_labels.notna().all()),

    "Provider categories partition the fever cohort":
        bool(provider_summary["n"].sum() == len(fever)),

    "Weighted provider proportions sum to 100 percent":
        bool(
            abs(
                provider_summary[
                    "weighted_percent_all_fever"
                ].sum() - 100
            ) < 0.01
        ),

    "Every source code has an original value label":
        bool(crosswalk["stata_value_label"].notna().all()),

    "Pharmacy code agrees with original value label":
        bool(
            crosswalk.loc[
                crosswalk["h46a_code"].eq(36),
                "stata_value_label"
            ].str.contains(
                "pharmacy",
                case=False,
                na=False
            ).all()
        ),

    "No repeated provider-code assignments":
        len(crosswalk) == crosswalk["h46a_code"].nunique(),

    "Care seekers identified consistently":
        int(care_sought.sum()) == 1974,

    "Pharmacy-first cases reproduced":
        int(fever["h46a"].eq(36).sum()) == 1120,

    "Unspecified first-source cases preserved":
        int(fever["h46a"].eq(96).sum()) == 33,

    "Unknown breathing responses preserved":
        int(fever["h31b"].eq(8).sum()) == 6,

    "Unknown medication responses preserved":
        int(fever["h37z"].eq(1).sum()) == 66,

    "All sample weights positive":
        bool(fever["survey_weight"].gt(0).all()),
}

checks_table = pd.DataFrame([
    {
        "check": name,
        "status": "PASS" if passed else "FAIL"
    }
    for name, passed in checks.items()
])


# Store the complete audit and its supporting tables.
files = {
    "providers":
        TABLE_DIR / "05_first_care_source_distribution.csv",
    "crosswalk":
        TABLE_DIR / "05_first_care_code_crosswalk.csv",
    "predictors":
        TABLE_DIR / "05_candidate_predictor_audit.csv",
    "symptoms":
        TABLE_DIR / "05_cough_rapid_breathing_crosstab.csv",
    "checks":
        TABLE_DIR / "05_literature_coding_checks.csv",
    "audit":
        LOG_DIR / "05_literature_coding_audit.json",
    "report":
        LOG_DIR / "05_literature_coding_report.txt",
}

provider_summary.to_csv(files["providers"], index=False)
crosswalk.to_csv(files["crosswalk"], index=False)
predictors.to_csv(files["predictors"], index=False)
symptoms.to_csv(files["symptoms"])
checks_table.to_csv(files["checks"], index=False)

all_passed = all(checks.values())

report = "\n".join([
    "STEP 05 | PROVIDER CLASSIFICATION AND PREDICTOR AUDIT",
    "=" * 56,
    f"Fever cohort: {len(fever)}",
    f"Care sought: {int(care_sought.sum())}",
    f"No reported care: {int(no_care.sum())}",
    f"Provider categories: {len(provider_summary)}",
    (
        "Reported first-care codes: "
        f"{fever['h46a'].nunique(dropna=True)}"
    ),
    f"Prespecified candidate predictors: {len(predictors)}",
    f"Unclassified children: {int(provider_labels.isna().sum())}",
    (
        "No cough but rapid breathing reported: "
        f"{int((fever['h31'].eq(0) & fever['h31b'].eq(1)).sum())}"
    ),
    f"Validation: {'PASS' if all_passed else 'FAIL'}",
])

files["report"].write_text(
    report + "\n",
    encoding="utf-8"
)

audit = {
    "checks": checks,
    "provider_code_groups": provider_codes,
    "candidate_predictors": candidate_groups,
    "primary_stage_1": (
        "Care sought versus no reported care, all fever cases"
    ),
    "primary_stage_2": (
        "Pharmacy versus all non-pharmacy sources among care seekers"
    ),
    "provider_other": (
        "Retain reported other source without assigning provider qualification"
    ),
    "unknown_breathing": (
        "Do not recode code 8 as no"
    ),
    "cough_breathing_check": (
        "Review reported rapid breathing among children without cough "
        "before deriving combined symptom severity"
    ),
    "medication_unknown": (
        "Not an exclusion for the primary care-seeking analysis"
    ),
}

files["audit"].write_text(
    json.dumps(
        audit,
        indent=2,
        ensure_ascii=False
    ) + "\n",
    encoding="utf-8"
)

print(report)

print("\nFIRST-CARE SOURCE DISTRIBUTION")
display(provider_summary)

print("\nCANDIDATE PREDICTOR AUDIT")
display(predictors)

print("\nCOUGH AND RAPID BREATHING CODING")
display(symptoms)

print("\nVALIDATION CHECKS")
display(checks_table)

print("\nSaved outputs:")
for file in files.values():
    print(file)

if not all_passed:
    raise ValueError(
        "Provider coding audit failed; inspect the saved checks."
    )

STEP 05 | PROVIDER CLASSIFICATION AND PREDICTOR AUDIT
Fever cohort: 2630
Care sought: 1974
No reported care: 656
Provider categories: 7
Reported first-care codes: 18
Prespecified candidate predictors: 16
Unclassified children: 0
No cough but rapid breathing reported: 39
Validation: PASS

FIRST-CARE SOURCE DISTRIBUTION


,first_care_category,n,weighted_percent_all_fever
0,No reported care,656,23.779
1,Public sector,300,10.779
2,Private clinical source,413,15.793
3,NGO sector,11,0.375
4,Pharmacy or drug store,1120,44.027
5,Non-qualified doctor,97,4.052
6,Other or unspecified source,33,1.195



CANDIDATE PREDICTOR AUDIT


,variable,tier,construct,stata_label,missing_n,distinct_values,special_code_8_n
0,b19,core,Child age in months,current age of child in months (months since b...,0,60,<NA>
1,b4,core,Child sex,sex of child,0,2,<NA>
2,h31,core,Cough in the previous two weeks,had cough in last two weeks,0,2,<NA>
3,h31b,core,"Reported short, rapid breathing","short, rapid breaths",0,3,6
4,v012,core,Maternal age,respondent's current age,0,34,<NA>
5,v106,core,Maternal educational level,highest educational level,0,4,<NA>
6,v190,core,Household wealth quintile,wealth index combined,0,5,<NA>
7,bord,core,Birth order,birth order number,0,9,<NA>
8,v025,core,Urban or rural residence,type of place of residence,0,2,<NA>
9,v024,core,Survey division,division,0,8,<NA>



COUGH AND RAPID BREATHING CODING


rapid_breathing_code,0.0,1.0,8.0
cough_code,,,
0.0,938,39,6
2.0,1329,318,0



VALIDATION CHECKS


,check,status
0,Fever cohort unchanged,PASS
1,No unknown care-status values,PASS
2,All reported first-care codes classified,PASS
3,Provider categories partition the fever cohort,PASS
4,Weighted provider proportions sum to 100 percent,PASS
5,Every source code has an original value label,PASS
6,Pharmacy code agrees with original value label,PASS
7,No repeated provider-code assignments,PASS
8,Care seekers identified consistently,PASS
9,Pharmacy-first cases reproduced,PASS



Saved outputs:
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\05_first_care_source_distribution.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\05_first_care_code_crosswalk.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\05_candidate_predictor_audit.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\05_cough_rapid_breathing_crosstab.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\05_literature_coding_checks.csv
D:\BDHS2022_FeverCare_SpatialEquity\logs\05_literature_coding_audit.json
D:\BDHS2022_FeverCare_SpatialEquity\logs\05_literature_coding_report.txt


In [8]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path(r"D:\BDHS2022_FeverCare_SpatialEquity")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLE_DIR = PROJECT_ROOT / "results" / "tables"
LOG_DIR = PROJECT_ROOT / "logs"

required_files = {
    "source": RAW_DIR / "BDKR81FL.DTA",
    "cohort": PROCESSED_DIR / "04_fever_care_analytic.csv",
    "codebook": TABLE_DIR / "03_response_codebook.csv",
    "crosswalk": TABLE_DIR / "05_first_care_code_crosswalk.csv",
    "provider_summary": TABLE_DIR / "05_first_care_source_distribution.csv",
    "step1_audit": LOG_DIR / "01_input_audit.json",
    "step4_audit": LOG_DIR / "04_outcome_construction_audit.json",
    "step5_audit": LOG_DIR / "05_literature_coding_audit.json",
}

for path in required_files.values():
    if not path.is_file():
        raise FileNotFoundError(f"Required input not found: {path}")

with required_files["step1_audit"].open(encoding="utf-8") as file:
    step1 = json.load(file)

with required_files["step4_audit"].open(encoding="utf-8") as file:
    step4 = json.load(file)

with required_files["step5_audit"].open(encoding="utf-8") as file:
    step5 = json.load(file)

if not all(step4["checks"].values()) or not all(step5["checks"].values()):
    raise ValueError("An earlier cohort or coding audit did not pass.")

with required_files["source"].open("rb") as source_file:
    raw_sha256 = hashlib.file_digest(
        source_file, "sha256"
    ).hexdigest()

if raw_sha256 != step1["source_files"]["kr_sha256"]:
    raise ValueError("The source dataset differs from the Step 1 version.")


# Recover additional predictors using the original child identifiers.
cohort = pd.read_csv(
    required_files["cohort"], low_memory=False
)

keys = ["v001", "v002", "v003", "bidx"]
additional_variables = ["bord", "v714", "v136"]

additional = pd.read_stata(
    required_files["source"],
    columns=keys + ["h22"] + additional_variables,
    convert_categoricals=False,
)

additional = additional.loc[
    additional["h22"].eq(1),
    keys + additional_variables
]

if additional.duplicated(keys).any() or cohort.duplicated(keys).any():
    raise ValueError("Child identifiers are not unique.")

cohort = cohort.merge(
    additional,
    on=keys,
    how="left",
    validate="one_to_one",
    sort=False,
    indicator=True,
)

if not cohort["_merge"].eq("both").all():
    raise ValueError(
        "Some fever records could not be linked to their source records."
    )

cohort = cohort.drop(columns="_merge")


# Verify response definitions against the original Stata codebook.
codebook = pd.read_csv(
    required_files["codebook"],
    dtype={"raw_code": "string"},
)

codebook["raw_code"] = codebook["raw_code"].str.replace(
    r"\.0$", "", regex=True
)

label_requirements = {
    ("h31", "0"): "no",
    ("h31", "2"): "yes, last two weeks",
    ("h31b", "0"): "no",
    ("h31b", "1"): "yes",
    ("h31b", "8"): "don't know",
    ("v467b", "1"): "big problem",
    ("v467b", "2"): "not a big problem",
    ("v467c", "1"): "big problem",
    ("v467c", "2"): "not a big problem",
    ("v467d", "1"): "big problem",
    ("v467d", "2"): "not a big problem",
    ("v467f", "1"): "big problem",
    ("v467f", "2"): "not a big problem",
    ("v025", "1"): "urban",
    ("v025", "2"): "rural",
    ("b4", "1"): "male",
    ("b4", "2"): "female",
}

label_rows = []

for (variable, code), expected_label in label_requirements.items():

    matches = codebook.loc[
        codebook["variable"].eq(variable)
        & codebook["raw_code"].eq(code),
        "stata_value_label",
    ]

    valid = (
        len(matches) == 1
        and str(matches.iloc[0]).strip().lower() == expected_label
    )

    label_rows.append({
        "variable": variable,
        "code": code,
        "expected_label": expected_label,
        "observed_label": (
            matches.iloc[0] if len(matches) == 1 else None
        ),
        "status": "PASS" if valid else "FAIL",
    })

label_table = pd.DataFrame(label_rows)

if not label_table["status"].eq("PASS").all():
    raise ValueError(
        "Value labels differ from the approved coding definitions."
    )


# Restore the detailed first-contact classification.
crosswalk = pd.read_csv(required_files["crosswalk"])

if crosswalk["h46a_code"].duplicated().any():
    raise ValueError(
        "First-care source crosswalk has duplicate response codes."
    )

provider_map = crosswalk.set_index(
    "h46a_code"
)["source_group"].to_dict()

cohort["first_care_source_group"] = (
    cohort["h46a"].map(provider_map).astype("string")
)

cohort.loc[
    cohort["care_sought"].eq(0),
    "first_care_source_group"
] = "No reported care"


# Recode child symptoms without assigning unknown responses to absence.
cohort["cough_reported"] = (
    cohort["h31"].map({0: 0, 2: 1}).astype("Int8")
)

cohort["rapid_breathing_reported"] = (
    cohort["h31b"].map({0: 0, 1: 1}).astype("Int8")
)

cohort["rapid_breathing_response"] = (
    cohort["h31b"].map({
        0: "No",
        1: "Yes",
        8: "Don't know",
    }).astype("string")
)


# Recode demographic characteristics and perceived access barriers.
cohort["female_child"] = (
    cohort["b4"].map({1: 0, 2: 1}).astype("Int8")
)

cohort["urban_residence"] = (
    cohort["v025"].map({1: 1, 2: 0}).astype("Int8")
)

barrier_variables = {
    "v467b": "maternal_permission_barrier",
    "v467c": "maternal_money_barrier",
    "v467d": "maternal_distance_barrier",
    "v467f": "maternal_alone_barrier",
}

for source, target in barrier_variables.items():
    cohort[target] = (
        cohort[source].map({1: 1, 2: 0}).astype("Int8")
    )

cohort["maternal_employed"] = (
    cohort["v714"].map({0: 0, 1: 1}).astype("Int8")
)


# Define analysis-specific eligibility without excluding cohort records.
core_predictors = [
    "b19",
    "female_child",
    "cough_reported",
    "rapid_breathing_reported",
    "v012",
    "v106",
    "v190",
    "bord",
    "urban_residence",
    "v024",
]

complete_covariates = (
    cohort[core_predictors].notna().all(axis=1)
)

care_sought = cohort["care_sought"].eq(1)

cohort["stage1_descriptive_eligible"] = 1

cohort["stage2_descriptive_eligible"] = (
    care_sought.astype("int8")
)

cohort["stage1_core_complete_case"] = (
    complete_covariates.astype("int8")
)

cohort["stage2_core_complete_case"] = (
    (care_sought & complete_covariates).astype("int8")
)

cohort["stage2_specified_source_sensitivity"] = (
    care_sought & cohort["h46a"].ne(96)
).astype("int8")


# Validate cohort retention, response coding and analysis eligibility.
checks = {
    "Fever cohort preserved":
        len(cohort) == 2630,

    "Source records linked without duplication": bool(
        cohort[keys].notna().all().all()
        and not cohort.duplicated(keys).any()
    ),

    "All reported first-care codes classified": bool(
        cohort["first_care_source_group"].notna().all()
    ),

    "First-care groups match the Step 5 audit": bool(
        cohort["first_care_source_group"].value_counts().to_dict()
        == pd.read_csv(
            required_files["provider_summary"]
        ).set_index("first_care_category")["n"].to_dict()
    ),

    "Cough coding is complete": bool(
        cohort["cough_reported"].notna().all()
    ),

    "Unknown breathing is not recoded as absence": bool(
        cohort.loc[
            cohort["h31b"].eq(8),
            "rapid_breathing_reported"
        ].isna().all()
        and cohort["rapid_breathing_reported"].isna().sum() == 6
    ),

    "Breathing without reported cough is preserved": (
        int((
            cohort["cough_reported"].eq(0)
            & cohort["rapid_breathing_reported"].eq(1)
        ).sum()) == 39
    ),

    "Additional source variables are valid": bool(
        cohort["v714"].isin([0, 1]).all()
        and cohort["bord"].ge(1).all()
        and cohort["v136"].ge(1).all()
    ),

    "Demographic and barrier recodes are complete": bool(
        cohort[
            [
                "female_child",
                "urban_residence",
                "maternal_permission_barrier",
                "maternal_money_barrier",
                "maternal_distance_barrier",
                "maternal_alone_barrier",
                "maternal_employed",
            ]
        ].notna().all().all()
    ),

    "Conditional outcomes retain structural missingness": bool(
        cohort.loc[
            ~care_sought,
            [
                "pharmacy_first_if_care",
                "early_care_within_1d_if_care",
            ]
        ].isna().all().all()
    ),

    "Unspecified first sources are retained": (
        int(
            cohort["first_care_source_group"].eq(
                "Other or unspecified source"
            ).sum()
        ) == 33
    ),

    "Analysis-specific complete-case indicators agree": bool(
        cohort["stage1_core_complete_case"].sum() == 2624
        and cohort["stage2_core_complete_case"].sum() == 1969
    ),

    "Survey weights remain unchanged": bool(
        np.allclose(
            cohort["survey_weight"],
            cohort["v005"] / 1_000_000,
            rtol=0,
            atol=1e-10,
        )
    ),
}

check_table = pd.DataFrame({
    "check": list(checks),
    "status": [
        "PASS" if result else "FAIL"
        for result in checks.values()
    ],
})

summary_table = pd.DataFrame([
    {
        "population": "All fever records",
        "n": len(cohort),
    },
    {
        "population": "Stage 1: descriptive",
        "n": int(cohort["stage1_descriptive_eligible"].sum()),
    },
    {
        "population": "Stage 1: core complete case",
        "n": int(cohort["stage1_core_complete_case"].sum()),
    },
    {
        "population": "Stage 2: descriptive",
        "n": int(cohort["stage2_descriptive_eligible"].sum()),
    },
    {
        "population": "Stage 2: core complete case",
        "n": int(cohort["stage2_core_complete_case"].sum()),
    },
    {
        "population": "Stage 2: specified first source",
        "n": int(
            cohort["stage2_specified_source_sensitivity"].sum()
        ),
    },
    {
        "population": "Unknown rapid breathing",
        "n": int(cohort["rapid_breathing_reported"].isna().sum()),
    },
    {
        "population": "Other or unspecified first source",
        "n": int(cohort["h46a"].eq(96).sum()),
    },
])


# Store validation results and the analysis specification.
files = {
    "checks": TABLE_DIR / "06_predictor_recoding_checks.csv",
    "labels": TABLE_DIR / "06_verified_value_labels.csv",
    "summary": TABLE_DIR / "06_analysis_population_counts.csv",
    "audit": LOG_DIR / "06_predictor_recoding_audit.json",
    "report": LOG_DIR / "06_predictor_recoding_report.txt",
    "analytic": PROCESSED_DIR / "06_fever_care_analysis_ready.csv",
    "specification": PROCESSED_DIR / "06_analysis_specification.json",
}

for name, table in (
    ("checks", check_table),
    ("labels", label_table),
    ("summary", summary_table),
):
    table.to_csv(
        files[name],
        index=False,
        encoding="utf-8-sig",
    )

all_passed = all(checks.values())

specification = {
    "design": (
        "BDHS 2022 fever cohort linked to displaced "
        "DHS cluster coordinates"
    ),
    "stage1_outcome": (
        "care_sought: 1 = any advice/treatment sought, 0 = none"
    ),
    "stage2_outcome": (
        "pharmacy_first_if_care: conditional on reported care seeking"
    ),
    "primary_core_predictors": core_predictors,
    "extended_covariates": [
        "maternal_money_barrier",
        "maternal_distance_barrier",
    ],
    "sensitivity_covariates": [
        "maternal_permission_barrier",
        "maternal_alone_barrier",
        "maternal_employed",
        "v136",
    ],
    "unknown_rapid_breathing": (
        "Missing covariate; all observations retained "
        "in descriptive analyses"
    ),
    "unspecified_first_care": (
        "Retained in the non-pharmacy comparison; "
        "excluded only in specified-source sensitivity analysis"
    ),
    "medication_unknown": (
        "Not an exclusion for either primary care-seeking outcome"
    ),
    "sampling": {
        "weight": "survey_weight",
        "psu": "v021",
        "stratum": "v023",
    },
    "gps": (
        "Separate validated lookup; "
        "no exact household locations implied"
    ),
}

files["audit"].write_text(
    json.dumps({
        "checks": checks,
        "core_predictors": core_predictors,
        "source_sha256": raw_sha256,
        "processed_written": all_passed,
    }, indent=2) + "\n",
    encoding="utf-8",
)

report = "\n".join([
    "STEP 06 | PREDICTOR RECODING AND ANALYSIS COHORT",
    "=" * 50,
    *(
        f"{record.population}: {record.n}"
        for record in summary_table.itertuples(index=False)
    ),
    "",
    f"Validation: {'PASS' if all_passed else 'FAIL'}",
    f"Validated checks: {sum(checks.values())} / {len(checks)}",
])

files["report"].write_text(
    report + "\n",
    encoding="utf-8",
)

if all_passed:
    cohort.to_csv(
        files["analytic"],
        index=False,
        encoding="utf-8-sig",
    )

    files["specification"].write_text(
        json.dumps(
            specification,
            ensure_ascii=False,
            indent=2,
        ) + "\n",
        encoding="utf-8",
    )

print(report)

print("\nANALYSIS POPULATIONS")
display(summary_table)

print("\nVALUE-LABEL VERIFICATION")
display(label_table)

print("\nVALIDATION CHECKS")
display(check_table)

print("\nSAVED FILES")
for name, path in files.items():
    if path.is_file():
        print(path)

if not all_passed:
    raise ValueError(
        "Predictor recoding failed validation. "
        "Review the saved checks."
    )

STEP 06 | PREDICTOR RECODING AND ANALYSIS COHORT
All fever records: 2630
Stage 1: descriptive: 2630
Stage 1: core complete case: 2624
Stage 2: descriptive: 1974
Stage 2: core complete case: 1969
Stage 2: specified first source: 1941
Unknown rapid breathing: 6
Other or unspecified first source: 33

Validation: PASS
Validated checks: 13 / 13

ANALYSIS POPULATIONS


,population,n
0,All fever records,2630
1,Stage 1: descriptive,2630
2,Stage 1: core complete case,2624
3,Stage 2: descriptive,1974
4,Stage 2: core complete case,1969
5,Stage 2: specified first source,1941
6,Unknown rapid breathing,6
7,Other or unspecified first source,33



VALUE-LABEL VERIFICATION


,variable,code,expected_label,observed_label,status
0,h31,0,no,no,PASS
1,h31,2,"yes, last two weeks","yes, last two weeks",PASS
2,h31b,0,no,no,PASS
3,h31b,1,yes,yes,PASS
4,h31b,8,don't know,don't know,PASS
5,v467b,1,big problem,big problem,PASS
6,v467b,2,not a big problem,not a big problem,PASS
7,v467c,1,big problem,big problem,PASS
8,v467c,2,not a big problem,not a big problem,PASS
9,v467d,1,big problem,big problem,PASS



VALIDATION CHECKS


,check,status
0,Fever cohort preserved,PASS
1,Source records linked without duplication,PASS
2,All reported first-care codes classified,PASS
3,First-care groups match the Step 5 audit,PASS
4,Cough coding is complete,PASS
5,Unknown breathing is not recoded as absence,PASS
6,Breathing without reported cough is preserved,PASS
7,Additional source variables are valid,PASS
8,Demographic and barrier recodes are complete,PASS
9,Conditional outcomes retain structural missing...,PASS



SAVED FILES
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\06_predictor_recoding_checks.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\06_verified_value_labels.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\06_analysis_population_counts.csv
D:\BDHS2022_FeverCare_SpatialEquity\logs\06_predictor_recoding_audit.json
D:\BDHS2022_FeverCare_SpatialEquity\logs\06_predictor_recoding_report.txt
D:\BDHS2022_FeverCare_SpatialEquity\data\processed\06_fever_care_analysis_ready.csv
D:\BDHS2022_FeverCare_SpatialEquity\data\processed\06_analysis_specification.json


In [9]:
from pathlib import Path
from statistics import NormalDist
import json
import math

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path(r"D:\BDHS2022_FeverCare_SpatialEquity")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLE_DIR = PROJECT_ROOT / "results" / "tables"
LOG_DIR = PROJECT_ROOT / "logs"

analysis_path = PROCESSED_DIR / "06_fever_care_analysis_ready.csv"
survey_path = RAW_DIR / "BDKR81FL.DTA"
step6_path = LOG_DIR / "06_predictor_recoding_audit.json"
codebook_path = TABLE_DIR / "03_response_codebook.csv"

for path in (analysis_path, survey_path, step6_path, codebook_path):
    if not path.is_file():
        raise FileNotFoundError(f"Required source missing: {path}")

with step6_path.open(encoding="utf-8") as file:
    previous_audit = json.load(file)

if not all(previous_audit["checks"].values()):
    raise ValueError("Step 6 verification must pass before survey estimation.")

cohort = pd.read_csv(analysis_path, low_memory=False)

codebook = pd.read_csv(
    codebook_path,
    dtype={"raw_code": "string"}
)

codebook["numeric_code"] = pd.to_numeric(
    codebook["raw_code"], errors="coerce"
)

category_labels = {
    variable: dict(zip(
        codebook.loc[
            codebook["variable"].eq(variable), "numeric_code"
        ],
        codebook.loc[
            codebook["variable"].eq(variable), "stata_value_label"
        ].str.title(),
    ))
    for variable in ("b4", "v106", "v190", "v025", "v024")
}

# Retain all PSUs represented in the KR file for domain variance estimation.
full_kr_design = pd.read_stata(
    survey_path,
    columns=["v021", "v023"],
    convert_categoricals=False,
)

psus = (
    full_kr_design
    .drop_duplicates(["v023", "v021"])
    .sort_values(["v023", "v021"])
    .reset_index(drop=True)
)

stratum_sizes = psus.groupby(
    "v023", observed=True
)["v021"].size()

checks = {
    "Step 6 cohort retained": len(cohort) == 2630,

    "Care seeking and first-contact counts retained": (
        cohort["care_sought"].eq(1).sum() == 1974
        and cohort["pharmacy_first_if_care"].eq(1).sum() == 1120
    ),

    "All sampled PSUs represented in design frame":
        psus["v021"].nunique() == 674,

    "Survey has 16 design strata":
        len(stratum_sizes) == 16,

    "Every stratum has at least two PSUs":
        bool(stratum_sizes.ge(2).all()),

    "Every PSU belongs to one stratum":
        bool(
            psus.groupby("v021")["v023"]
            .nunique().eq(1).all()
        ),

    "Fever-domain PSUs are a subset of the design":
        bool(
            pd.MultiIndex.from_frame(
                cohort[["v023", "v021"]]
            ).isin(
                pd.MultiIndex.from_frame(
                    psus[["v023", "v021"]]
                )
            ).all()
        ),

    "Sampling weights positive and finite":
        bool(
            np.isfinite(cohort["survey_weight"]).all()
            and cohort["survey_weight"].gt(0).all()
        ),

    "First-care source classified for every fever child":
        bool(cohort["first_care_source_group"].notna().all()),
}

if not all(checks.values()):
    failed = [name for name, result in checks.items() if not result]
    raise ValueError(f"Survey design audit failed: {failed}")

psu_index = pd.MultiIndex.from_frame(
    psus[["v023", "v021"]]
)

weights = cohort["survey_weight"].to_numpy(dtype=float)
care = cohort["care_sought"].eq(1).to_numpy()
pharmacy = cohort["pharmacy_first_if_care"].eq(1).to_numpy()
early = cohort["early_care_within_1d_if_care"].eq(1).to_numpy()
all_fever = np.ones(len(cohort), dtype=bool)

critical_value = NormalDist().inv_cdf(0.975)
design_df = int(len(psus) - len(stratum_sizes))


def survey_binary_proportion(outcome, domain):
    outcome = np.asarray(outcome, dtype=bool)
    domain = np.asarray(domain, dtype=bool)

    if len(outcome) != len(cohort) or len(domain) != len(cohort):
        raise ValueError("Outcome and domain must match the fever cohort.")

    n = int(domain.sum())

    if n == 0:
        raise ValueError("Cannot estimate a proportion for an empty domain.")

    denominator = float(np.sum(weights[domain]))

    proportion = float(
        np.sum(weights[domain & outcome]) / denominator
    )

    influence = (
        weights
        * domain
        * (outcome.astype(float) - proportion)
        / denominator
    )

    cluster_totals = pd.DataFrame({
        "v023": cohort["v023"].to_numpy(),
        "v021": cohort["v021"].to_numpy(),
        "influence": influence,
    }).groupby(
        ["v023", "v021"],
        observed=True
    )["influence"].sum()

    # Preserve zero-domain PSUs in the stratified variance calculation.
    complete_totals = cluster_totals.reindex(
        psu_index, fill_value=0.0
    )

    design_scores = psus.assign(
        influence=complete_totals.to_numpy()
    )

    variance = 0.0

    for _, stratum in design_scores.groupby(
        "v023", observed=True
    ):
        scores = stratum["influence"].to_numpy(dtype=float)

        variance += (
            len(scores) / (len(scores) - 1)
            * np.sum((scores - scores.mean()) ** 2)
        )

    se = math.sqrt(max(variance, 0.0))

    # Use logit-scale intervals to keep prevalence limits within 0-100%.
    if 0.0 < proportion < 1.0:
        logit = math.log(proportion / (1.0 - proportion))
        se_logit = se / (proportion * (1.0 - proportion))

        lower = 1.0 / (
            1.0 + math.exp(
                -(logit - critical_value * se_logit)
            )
        )

        upper = 1.0 / (
            1.0 + math.exp(
                -(logit + critical_value * se_logit)
            )
        )
    else:
        lower, upper = np.nan, np.nan

    return {
        "n": n,
        "positive_n": int(np.sum(domain & outcome)),
        "weighted_percent": 100.0 * proportion,
        "se_percentage_points": 100.0 * se,
        "ci95_lower": 100.0 * lower,
        "ci95_upper": 100.0 * upper,
        "domain_psus": int(
            cohort.loc[
                domain, ["v023", "v021"]
            ].drop_duplicates().shape[0]
        ),
    }


primary_specifications = [
    ("Care sought", care, all_fever, "All febrile children"),
    ("No reported care", ~care, all_fever, "All febrile children"),
    ("Pharmacy first", pharmacy, all_fever, "All febrile children"),
    (
        "Non-pharmacy first",
        care & ~pharmacy,
        all_fever,
        "All febrile children",
    ),
    (
        "Pharmacy first among care seekers",
        pharmacy, care, "Care seekers",
    ),
    (
        "Care sought on day 0-1",
        early, care, "Care seekers",
    ),
]

primary_results = pd.DataFrame([
    dict(
        indicator=name,
        population=population,
        **survey_binary_proportion(outcome, domain)
    )
    for name, outcome, domain, population
    in primary_specifications
])


source_order = [
    "No reported care",
    "Public sector",
    "Private clinical source",
    "NGO sector",
    "Pharmacy or drug store",
    "Non-qualified doctor",
    "Other or unspecified source",
]

source_results = pd.DataFrame([
    dict(
        source=source,
        **survey_binary_proportion(
            cohort["first_care_source_group"]
            .eq(source).to_numpy(),
            all_fever,
        )
    )
    for source in source_order
])


# Describe prespecified demographic and socioeconomic subgroups.
cohort["age_group"] = pd.cut(
    cohort["b19"],
    bins=[0, 12, 24, 36, 48, 60],
    right=False,
    labels=[
        "0-11 months",
        "12-23 months",
        "24-35 months",
        "36-47 months",
        "48-59 months",
    ],
)

subgroup_definitions = [
    (
        "Age group",
        "age_group",
        list(cohort["age_group"].cat.categories),
    ),
    ("Sex", "b4", [1, 2]),
    (
        "Maternal education",
        "v106",
        sorted(cohort["v106"].dropna().unique()),
    ),
    ("Wealth quintile", "v190", [1, 2, 3, 4, 5]),
    ("Residence", "v025", [1, 2]),
    (
        "Division",
        "v024",
        sorted(cohort["v024"].dropna().unique()),
    ),
]

labels = {
    "Sex": category_labels["b4"],
    "Maternal education": category_labels["v106"],
    "Wealth quintile": category_labels["v190"],
    "Residence": category_labels["v025"],
    "Division": category_labels["v024"],
}

subgroup_rows = []

for dimension, variable, categories in subgroup_definitions:
    for category in categories:
        selected = (
            cohort[variable]
            .eq(category)
            .fillna(False)
            .to_numpy(dtype=bool)
        )

        if not selected.any():
            continue

        prevalence = survey_binary_proportion(
            selected, all_fever
        )

        stage1 = survey_binary_proportion(
            care, selected
        )

        stage2 = survey_binary_proportion(
            pharmacy, selected & care
        )

        subgroup_rows.append({
            "characteristic": dimension,
            "category": labels.get(
                dimension, {}
            ).get(category, str(category)),
            "fever_n": int(selected.sum()),
            "fever_weighted_share_percent":
                prevalence["weighted_percent"],
            "care_sought_n": stage1["positive_n"],
            "care_sought_percent":
                stage1["weighted_percent"],
            "care_sought_ci95_lower":
                stage1["ci95_lower"],
            "care_sought_ci95_upper":
                stage1["ci95_upper"],
            "care_seeker_n": stage2["n"],
            "pharmacy_first_n": stage2["positive_n"],
            "pharmacy_first_given_care_percent":
                stage2["weighted_percent"],
            "pharmacy_ci95_lower":
                stage2["ci95_lower"],
            "pharmacy_ci95_upper":
                stage2["ci95_upper"],
            "domain_psus": stage1["domain_psus"],
        })

subgroup_results = pd.DataFrame(subgroup_rows)


validation = {
    **checks,

    "Seven first-care groups partition the cohort":
        bool(
            source_results["positive_n"].sum()
            == len(cohort)
        ),

    "Seven first-care proportions sum to 100 percent":
        bool(
            np.isclose(
                source_results["weighted_percent"].sum(),
                100,
                atol=1e-6,
            )
        ),

    "All subgroup categories are labelled":
        bool(subgroup_results["category"].notna().all()),

    "Care-seeking point estimate matches Step 4":
        bool(
            abs(
                primary_results.loc[
                    primary_results["indicator"].eq("Care sought"),
                    "weighted_percent",
                ].iloc[0] - 76.22
            ) < 0.05
        ),

    "Pharmacy-first point estimate matches Step 4":
        bool(
            abs(
                primary_results.loc[
                    primary_results["indicator"].eq("Pharmacy first"),
                    "weighted_percent",
                ].iloc[0] - 44.03
            ) < 0.05
        ),

    "All finite confidence intervals are ordered":
        bool(
            (
                primary_results["ci95_lower"]
                <= primary_results["weighted_percent"]
            ).all()
            and (
                primary_results["weighted_percent"]
                <= primary_results["ci95_upper"]
            ).all()
        ),

    "Design degrees of freedom positive":
        bool(design_df > 0),
}

outputs = {
    "primary":
        TABLE_DIR / "07_primary_survey_estimates.csv",
    "providers":
        TABLE_DIR / "07_first_care_source_survey_estimates.csv",
    "subgroups":
        TABLE_DIR / "07_unadjusted_subgroup_estimates.csv",
    "checks":
        TABLE_DIR / "07_survey_estimation_checks.csv",
    "audit":
        LOG_DIR / "07_survey_estimation_audit.json",
    "report":
        LOG_DIR / "07_survey_estimation_report.txt",
}

validation_table = pd.DataFrame([
    {
        "check": name,
        "status": "PASS" if ok else "FAIL",
    }
    for name, ok in validation.items()
])

for name, table in [
    ("primary", primary_results),
    ("providers", source_results),
    ("subgroups", subgroup_results),
    ("checks", validation_table),
]:
    table.to_csv(
        outputs[name],
        index=False,
        encoding="utf-8-sig",
    )

all_passed = all(validation.values())

report = "\n".join([
    "STEP 07 | COMPLEX-SURVEY DESCRIPTIVE ESTIMATION",
    "=" * 52,
    f"PSUs in the KR design frame: {len(psus)}",
    f"PSUs with fever cases: {cohort['v021'].nunique()}",
    f"Sampling strata: {len(stratum_sizes)}",
    f"Design degrees of freedom: {design_df}",
    f"Fever records: {len(cohort)}",
    "Variance estimator: stratified PSU Taylor linearization",
    "Confidence intervals: 95% logit-normal approximation",
    "Full KR PSU frame retained for domain variances: Yes",
    "Note: descriptive associations are unadjusted",
    f"Validation: {'PASS' if all_passed else 'FAIL'}",
])

outputs["report"].write_text(
    report + "\n", encoding="utf-8"
)

outputs["audit"].write_text(
    json.dumps({
        "checks": {
            key: bool(value)
            for key, value in validation.items()
        },
        "design_psus": len(psus),
        "fever_psus": int(cohort["v021"].nunique()),
        "strata": len(stratum_sizes),
        "design_df": design_df,
        "method": (
            "Stratified PSU Taylor linearization of weighted "
            "ratios; logit-normal 95% confidence intervals; "
            "no finite population correction"
        ),
    }, indent=2) + "\n",
    encoding="utf-8"
)

print(report)

print("\nPRIMARY OUTCOME ESTIMATES")
display(primary_results.round(3))

print("\nFIRST-CARE SOURCE ESTIMATES")
display(
    source_results[
        [
            "source",
            "positive_n",
            "weighted_percent",
            "se_percentage_points",
            "ci95_lower",
            "ci95_upper",
        ]
    ].round(3)
)

print("\nUNADJUSTED SUBGROUP DESCRIPTIVES")
display(subgroup_results.round(2))

print("\nVALIDATION CHECKS")
display(validation_table)

print("\nSAVED FILES")
for path in outputs.values():
    print(path)

if not all_passed:
    raise ValueError(
        "Survey estimation failed validation. Review saved checks."
    )

STEP 07 | COMPLEX-SURVEY DESCRIPTIVE ESTIMATION
PSUs in the KR design frame: 674
PSUs with fever cases: 642
Sampling strata: 16
Design degrees of freedom: 658
Fever records: 2630
Variance estimator: stratified PSU Taylor linearization
Confidence intervals: 95% logit-normal approximation
Full KR PSU frame retained for domain variances: Yes
Note: descriptive associations are unadjusted
Validation: PASS

PRIMARY OUTCOME ESTIMATES


,indicator,population,n,positive_n,weighted_percent,se_percentage_points,ci95_lower,ci95_upper,domain_psus
0,Care sought,All febrile children,2630,1974,76.221,0.946,74.318,78.024,642
1,No reported care,All febrile children,2630,656,23.779,0.946,21.976,25.682,642
2,Pharmacy first,All febrile children,2630,1120,44.027,1.181,41.727,46.352,642
3,Non-pharmacy first,All febrile children,2630,854,32.194,1.115,30.049,34.417,642
4,Pharmacy first among care seekers,Care seekers,1974,1120,57.762,1.369,55.058,60.420,611
5,Care sought on day 0-1,Care seekers,1974,1368,69.515,1.289,66.931,71.981,611



FIRST-CARE SOURCE ESTIMATES


,source,positive_n,weighted_percent,se_percentage_points,ci95_lower,ci95_upper
0,No reported care,656,23.779,0.946,21.976,25.682
1,Public sector,300,10.779,0.734,9.422,12.304
2,Private clinical source,413,15.793,0.865,14.171,17.563
3,NGO sector,11,0.375,0.147,0.174,0.809
4,Pharmacy or drug store,1120,44.027,1.181,41.727,46.352
5,Non-qualified doctor,97,4.052,0.474,3.219,5.090
6,Other or unspecified source,33,1.195,0.230,0.819,1.740



UNADJUSTED SUBGROUP DESCRIPTIVES


,characteristic,category,fever_n,fever_weighted_share_percent,care_sought_n,care_sought_percent,care_sought_ci95_lower,care_sought_ci95_upper,care_seeker_n,pharmacy_first_n,pharmacy_first_given_care_percent,pharmacy_ci95_lower,pharmacy_ci95_upper,domain_psus
0,Age group,0-11 months,611,23.74,476,80.12,76.52,83.30,476,228,49.24,44.21,54.29,388
1,Age group,12-23 months,613,23.47,480,77.71,73.37,81.53,480,256,55.76,50.96,60.45,384
2,Age group,24-35 months,523,19.59,400,77.45,72.57,81.68,400,241,61.44,55.82,66.77,349
3,Age group,36-47 months,455,17.43,330,73.76,69.23,77.84,330,201,60.74,54.92,66.27,311
4,Age group,48-59 months,428,15.77,288,69.32,64.06,74.11,288,194,67.33,60.54,73.46,312
5,Sex,Male,1394,52.68,1069,77.91,75.39,80.23,1069,597,57.71,54.36,60.98,567
6,Sex,Female,1236,47.32,905,74.35,71.59,76.92,905,523,57.83,54.03,61.53,530
7,Maternal education,No Education,143,5.46,103,73.69,65.53,80.49,103,71,66.24,55.20,75.76,111
8,Maternal education,Primary,646,24.69,502,78.98,75.13,82.38,502,315,63.72,58.68,68.48,339
9,Maternal education,Secondary,1434,55.34,1096,76.96,74.42,79.32,1096,611,57.40,54.00,60.73,553



VALIDATION CHECKS


,check,status
0,Step 6 cohort retained,PASS
1,Care seeking and first-contact counts retained,PASS
2,All sampled PSUs represented in design frame,PASS
3,Survey has 16 design strata,PASS
4,Every stratum has at least two PSUs,PASS
5,Every PSU belongs to one stratum,PASS
6,Fever-domain PSUs are a subset of the design,PASS
7,Sampling weights positive and finite,PASS
8,First-care source classified for every fever c...,PASS
9,Seven first-care groups partition the cohort,PASS



SAVED FILES
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\07_primary_survey_estimates.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\07_first_care_source_survey_estimates.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\07_unadjusted_subgroup_estimates.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\07_survey_estimation_checks.csv
D:\BDHS2022_FeverCare_SpatialEquity\logs\07_survey_estimation_audit.json
D:\BDHS2022_FeverCare_SpatialEquity\logs\07_survey_estimation_report.txt


In [11]:

from pathlib import Path
from importlib.metadata import version
import hashlib
import json

import numpy as np
import pandas as pd
import polars as pl
import svy
from IPython.display import display

PROJECT_ROOT = Path(r"D:\BDHS2022_FeverCare_SpatialEquity")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
TABLE_DIR = PROJECT_ROOT / "results" / "tables"
LOG_DIR = PROJECT_ROOT / "logs"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

kr_path = RAW_DIR / "BDKR81FL.DTA"
primary_path = TABLE_DIR / "07_primary_survey_estimates.csv"
provider_path = TABLE_DIR / "07_first_care_source_survey_estimates.csv"
step1_path = LOG_DIR / "01_input_audit.json"
step7_path = LOG_DIR / "07_survey_estimation_audit.json"

required_files = [
    kr_path,
    primary_path,
    provider_path,
    step1_path,
    step7_path,
]

for path in required_files:
    if not path.is_file():
        raise FileNotFoundError(f"Required input not found: {path}")

svy_version = version("svy")

if svy_version != "0.37.0":
    raise RuntimeError(
        f"Expected svy 0.37.0; installed version: {svy_version}"
    )

with step1_path.open(encoding="utf-8") as file:
    step1_audit = json.load(file)

with step7_path.open(encoding="utf-8") as file:
    step7_audit = json.load(file)

if not all(step7_audit["checks"].values()):
    raise ValueError("Step 7 validation has not passed.")

with kr_path.open("rb") as file:
    source_checksum = hashlib.file_digest(
        file, "sha256"
    ).hexdigest()

expected_checksum = step1_audit["source_files"]["kr_sha256"]

if source_checksum != expected_checksum:
    raise ValueError("Source dataset checksum mismatch.")


# Reconstruct the original survey design before defining subpopulations.
survey_data = pd.read_stata(
    kr_path,
    columns=[
        "v021",
        "v023",
        "v005",
        "h22",
        "h32y",
        "h46a",
        "h46b",
    ],
    convert_categoricals=False,
)

survey_data["survey_weight"] = (
    survey_data["v005"].astype("float64") / 1_000_000
)

fever = survey_data["h22"].eq(1)
care = fever & survey_data["h32y"].eq(0)

survey_data["out_care_sought"] = care.astype("int8")

survey_data["out_no_care"] = (
    fever & survey_data["h32y"].eq(1)
).astype("int8")

survey_data["out_pharmacy"] = (
    care & survey_data["h46a"].eq(36)
).astype("int8")

survey_data["out_nonpharmacy"] = (
    care & survey_data["h46a"].ne(36)
).astype("int8")

survey_data["out_early_care"] = (
    care & survey_data["h46b"].isin([0, 1])
).astype("int8")


# Preserve the first-care classification established in Step 5.
provider_codes = {
    "No reported care": None,
    "Public sector": [11, 12, 13, 14, 15, 16, 17, 18, 19],
    "Private clinical source": [31, 32, 33, 34],
    "NGO sector": [41, 45],
    "Pharmacy or drug store": [36],
    "Non-qualified doctor": [35],
    "Other or unspecified source": [96],
}

provider_outcomes = {}
provider_counts = {}

for index, (name, codes) in enumerate(provider_codes.items()):
    if codes is None:
        mask = fever & survey_data["h32y"].eq(1)
    else:
        mask = care & survey_data["h46a"].isin(codes)

    column = f"out_provider_{index}"

    survey_data[column] = mask.astype("int8")
    provider_outcomes[name] = column
    provider_counts[name] = int(mask.sum())


# Construct a native Polars DataFrame for the svy estimation engine.
polars_data = pl.DataFrame({
    column: survey_data[column].tolist()
    for column in survey_data.columns
})

design = svy.Design(
    stratum="v023",
    psu="v021",
    wgt="survey_weight",
)

survey_sample = svy.Sample(
    data=polars_data,
    design=design,
)

fever_domain = svy.col("h22") == 1

care_domain = (
    (svy.col("h22") == 1)
    & (svy.col("h32y") == 0)
)


# Estimate survey-weighted proportions using Taylor linearization.
def extract_estimate(name, variable, domain, population):
    result = survey_sample.estimation.mean(
        y=variable,
        where=domain,
    )

    result_table = result.to_polars()

    required_columns = {
        "est", "se", "lci", "uci", "df", "n"
    }

    if result_table.height != 1:
        raise ValueError(
            f"Unexpected estimate dimensions for {name}."
        )

    if not required_columns.issubset(result_table.columns):
        raise ValueError(
            f"Incomplete survey estimate for {name}: "
            f"{result_table.columns}"
        )

    row = result_table.to_dicts()[0]

    return {
        "indicator": name,
        "population": population,
        "n": int(row["n"]),
        "weighted_percent": 100 * float(row["est"]),
        "se_percentage_points": 100 * float(row["se"]),
        "ci95_lower": 100 * float(row["lci"]),
        "ci95_upper": 100 * float(row["uci"]),
        "design_df": int(row["df"]),
    }


primary_specs = [
    (
        "Care sought",
        "out_care_sought",
        fever_domain,
        "All febrile children",
    ),
    (
        "No reported care",
        "out_no_care",
        fever_domain,
        "All febrile children",
    ),
    (
        "Pharmacy first",
        "out_pharmacy",
        fever_domain,
        "All febrile children",
    ),
    (
        "Non-pharmacy first",
        "out_nonpharmacy",
        fever_domain,
        "All febrile children",
    ),
    (
        "Pharmacy first among care seekers",
        "out_pharmacy",
        care_domain,
        "Care seekers",
    ),
    (
        "Care sought on day 0-1",
        "out_early_care",
        care_domain,
        "Care seekers",
    ),
]

independent_primary = pd.DataFrame([
    extract_estimate(
        name,
        variable,
        domain,
        population,
    )
    for name, variable, domain, population in primary_specs
])

independent_providers = pd.DataFrame([
    extract_estimate(
        name,
        variable,
        fever_domain,
        "All febrile children",
    )
    for name, variable in provider_outcomes.items()
])

independent_providers["positive_n"] = (
    independent_providers["indicator"].map(provider_counts)
)


# Compare the independent estimates against the Step 7 calculations.
manual_primary = pd.read_csv(primary_path)
manual_providers = pd.read_csv(provider_path)


def compare_estimates(independent, reference):
    required = [
        "indicator",
        "n",
        "weighted_percent",
        "se_percentage_points",
    ]

    merged = independent.merge(
        reference[required],
        on="indicator",
        how="outer",
        suffixes=("_svy", "_step7"),
        validate="one_to_one",
        indicator=True,
    )

    if not merged["_merge"].eq("both").all():
        raise ValueError(
            "Independent and reference estimates have different indicators."
        )

    merged = merged.drop(columns="_merge")

    merged["estimate_difference_pp"] = (
        merged["weighted_percent_svy"]
        - merged["weighted_percent_step7"]
    )

    merged["se_difference_pp"] = (
        merged["se_percentage_points_svy"]
        - merged["se_percentage_points_step7"]
    )

    merged["se_relative_difference_pct"] = (
        100
        * merged["se_difference_pp"].abs()
        / merged["se_percentage_points_svy"]
        .abs()
        .clip(lower=1e-12)
    )

    merged["sample_size_matches"] = (
        merged["n_svy"] == merged["n_step7"]
    )

    return merged


primary_comparison = compare_estimates(
    independent_primary,
    manual_primary,
)

provider_reference = manual_providers.rename(
    columns={"source": "indicator"}
)

provider_comparison = compare_estimates(
    independent_providers,
    provider_reference,
)

primary_comparison.insert(
    0, "analysis", "Primary outcomes"
)

provider_comparison.insert(
    0, "analysis", "First-care sources"
)

comparison = pd.concat(
    [primary_comparison, provider_comparison],
    ignore_index=True,
)


# Verify survey structure, sample sizes and numerical agreement.
design_psus = int(
    survey_data["v021"].nunique()
)

design_strata = int(
    survey_data["v023"].nunique()
)

design_df = design_psus - design_strata

all_estimates = pd.concat(
    [independent_primary, independent_providers],
    ignore_index=True,
)

checks = {
    "Original KR sample retained":
        len(survey_data) == 8784,

    "Polars data structure confirmed":
        isinstance(polars_data, pl.DataFrame),

    "Full survey contains 674 PSUs":
        design_psus == 674,

    "Full survey contains 16 strata":
        design_strata == 16,

    "Fever domain contains 2,630 children":
        int(fever.sum()) == 2630,

    "Care-seeking domain contains 1,974 children":
        int(care.sum()) == 1974,

    "Seven provider categories cover the fever cohort":
        sum(provider_counts.values()) == 2630,

    "All estimated sample sizes match Step 7":
        bool(comparison["sample_size_matches"].all()),

    "Reported design degrees of freedom equal 658":
        bool(
            design_df == 658
            and all_estimates["design_df"].eq(658).all()
        ),

    "All primary point estimates agree":
        bool(
            primary_comparison[
                "estimate_difference_pp"
            ].abs().max() < 1e-5
        ),

    "All provider point estimates agree":
        bool(
            provider_comparison[
                "estimate_difference_pp"
            ].abs().max() < 1e-5
        ),

    "Primary standard errors agree within 0.5 percent":
        bool(
            primary_comparison[
                "se_relative_difference_pct"
            ].max() < 0.5
        ),

    "Provider standard errors agree within 0.5 percent":
        bool(
            provider_comparison[
                "se_relative_difference_pct"
            ].max() < 0.5
        ),

    "All estimates and standard errors are finite":
        bool(
            np.isfinite(
                all_estimates[
                    ["weighted_percent", "se_percentage_points"]
                ].to_numpy(dtype=float)
            ).all()
        ),

    "Confidence intervals contain their point estimates":
        bool(
            (
                all_estimates["ci95_lower"]
                <= all_estimates["weighted_percent"]
            ).all()
            and (
                all_estimates["weighted_percent"]
                <= all_estimates["ci95_upper"]
            ).all()
        ),
}

validation_table = pd.DataFrame([
    {
        "check": name,
        "status": "PASS" if passed else "REVIEW",
    }
    for name, passed in checks.items()
])

all_passed = all(checks.values())


# Save estimates and validation results without altering Step 7 outputs.
outputs = {
    "primary":
        TABLE_DIR / "07a_svy_primary_estimates.csv",
    "providers":
        TABLE_DIR / "07a_svy_provider_estimates.csv",
    "comparison":
        TABLE_DIR / "07a_variance_comparison.csv",
    "checks":
        TABLE_DIR / "07a_svy_validation_checks.csv",
    "audit":
        LOG_DIR / "07a_svy_validation_audit.json",
    "report":
        LOG_DIR / "07a_svy_validation_report.txt",
}

for name, table in [
    ("primary", independent_primary),
    ("providers", independent_providers),
    ("comparison", comparison),
    ("checks", validation_table),
]:
    table.to_csv(
        outputs[name],
        index=False,
        encoding="utf-8-sig",
    )

audit = {
    "package": "svy",
    "svy_version": svy_version,
    "polars_version": pl.__version__,
    "pandas_version": pd.__version__,
    "source_sha256": source_checksum,
    "full_sample_records": int(len(survey_data)),
    "design_psus": design_psus,
    "design_strata": design_strata,
    "design_df": design_df,
    "method": "Taylor linearization",
    "domain_estimation": "Full KR survey design retained",
    "confidence_intervals": "svy default mean confidence intervals",
    "checks": {
        key: bool(value)
        for key, value in checks.items()
    },
    "maximum_se_relative_difference_pct": {
        "primary": float(
            primary_comparison[
                "se_relative_difference_pct"
            ].max()
        ),
        "providers": float(
            provider_comparison[
                "se_relative_difference_pct"
            ].max()
        ),
    },
}

outputs["audit"].write_text(
    json.dumps(audit, indent=2) + "\n",
    encoding="utf-8",
)

report = "\n".join([
    "STEP 07.1 | INDEPENDENT SURVEY ESTIMATION VERIFICATION",
    "=" * 60,
    f"Survey package: svy {svy_version}",
    f"Polars version: {pl.__version__}",
    f"Full KR records: {len(survey_data)}",
    f"PSUs: {design_psus}",
    f"Strata: {design_strata}",
    f"Design degrees of freedom: {design_df}",
    "Variance estimator: Taylor linearization",
    "Domain estimation: Full KR survey design retained",
    (
        "Maximum primary SE difference (%): "
        f"{primary_comparison['se_relative_difference_pct'].max():.6f}"
    ),
    (
        "Maximum provider SE difference (%): "
        f"{provider_comparison['se_relative_difference_pct'].max():.6f}"
    ),
    f"Validation: {'PASS' if all_passed else 'REVIEW REQUIRED'}",
])

outputs["report"].write_text(
    report + "\n",
    encoding="utf-8",
)

print(report)

print("\nINDEPENDENT PRIMARY ESTIMATES")
display(independent_primary.round(4))

print("\nINDEPENDENT FIRST-CARE ESTIMATES")
display(independent_providers.round(4))

print("\nVARIANCE COMPARISON")
display(
    comparison[
        [
            "analysis",
            "indicator",
            "estimate_difference_pp",
            "se_difference_pp",
            "se_relative_difference_pct",
            "sample_size_matches",
        ]
    ].round(6)
)

print("\nVALIDATION CHECKS")
display(validation_table)

print("\nSAVED FILES")
for path in outputs.values():
    print(path)

if not all_passed:
    print(
        "\nIndependent estimates require methodological review "
        "before proceeding to adjusted analysis."
    )

STEP 07.1 | INDEPENDENT SURVEY ESTIMATION VERIFICATION
Survey package: svy 0.37.0
Polars version: 2.1.0
Full KR records: 8784
PSUs: 674
Strata: 16
Design degrees of freedom: 658
Variance estimator: Taylor linearization
Domain estimation: Full KR survey design retained
Maximum primary SE difference (%): 0.000000
Maximum provider SE difference (%): 0.000000
Validation: REVIEW REQUIRED

INDEPENDENT PRIMARY ESTIMATES


,indicator,population,n,weighted_percent,se_percentage_points,ci95_lower,ci95_upper,design_df
0,Care sought,All febrile children,2630,76.2209,0.9456,74.3639,78.0779,626
1,No reported care,All febrile children,2630,23.7791,0.9456,21.9221,25.6361,626
2,Pharmacy first,All febrile children,2630,44.0266,1.1809,41.7076,46.3456,626
3,Non-pharmacy first,All febrile children,2630,32.1943,1.1150,30.0047,34.3839,626
4,Pharmacy first among care seekers,Care seekers,1974,57.7618,1.3691,55.0730,60.4506,595
5,Care sought on day 0-1,Care seekers,1974,69.5148,1.2892,66.9828,72.0467,595



INDEPENDENT FIRST-CARE ESTIMATES


,indicator,population,n,weighted_percent,se_percentage_points,ci95_lower,ci95_upper,design_df,positive_n
0,No reported care,All febrile children,2630,23.7791,0.9456,21.9221,25.6361,626,656
1,Public sector,All febrile children,2630,10.7786,0.7342,9.3368,12.2204,626,300
2,Private clinical source,All febrile children,2630,15.7929,0.8649,14.0945,17.4913,626,413
3,NGO sector,All febrile children,2630,0.3755,0.1474,0.0861,0.6649,626,11
4,Pharmacy or drug store,All febrile children,2630,44.0266,1.1809,41.7076,46.3456,626,1120
5,Non-qualified doctor,All febrile children,2630,4.0523,0.4739,3.1216,4.9830,626,97
6,Other or unspecified source,All febrile children,2630,1.1950,0.2297,0.7439,1.6461,626,33



VARIANCE COMPARISON


,analysis,indicator,estimate_difference_pp,se_difference_pp,se_relative_difference_pct,sample_size_matches
0,Primary outcomes,Care sought,0.0,0.0,0.0,True
1,Primary outcomes,Care sought on day 0-1,-0.0,-0.0,0.0,True
2,Primary outcomes,No reported care,0.0,0.0,0.0,True
3,Primary outcomes,Non-pharmacy first,0.0,0.0,0.0,True
4,Primary outcomes,Pharmacy first,0.0,0.0,0.0,True
5,Primary outcomes,Pharmacy first among care seekers,-0.0,0.0,0.0,True
6,First-care sources,NGO sector,0.0,0.0,0.0,True
7,First-care sources,No reported care,0.0,0.0,0.0,True
8,First-care sources,Non-qualified doctor,0.0,0.0,0.0,True
9,First-care sources,Other or unspecified source,0.0,0.0,0.0,True



VALIDATION CHECKS


,check,status
0,Original KR sample retained,PASS
1,Polars data structure confirmed,PASS
2,Full survey contains 674 PSUs,PASS
3,Full survey contains 16 strata,PASS
4,"Fever domain contains 2,630 children",PASS
5,"Care-seeking domain contains 1,974 children",PASS
6,Seven provider categories cover the fever cohort,PASS
7,All estimated sample sizes match Step 7,PASS
8,Reported design degrees of freedom equal 658,REVIEW
9,All primary point estimates agree,PASS



SAVED FILES
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\07a_svy_primary_estimates.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\07a_svy_provider_estimates.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\07a_variance_comparison.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\07a_svy_validation_checks.csv
D:\BDHS2022_FeverCare_SpatialEquity\logs\07a_svy_validation_audit.json
D:\BDHS2022_FeverCare_SpatialEquity\logs\07a_svy_validation_report.txt

Independent estimates require methodological review before proceeding to adjusted analysis.


In [12]:

from pathlib import Path
import json

import numpy as np
import pandas as pd
from scipy import __version__ as scipy_version
from scipy.special import expit, logit
from scipy.stats import t
from IPython.display import display

PROJECT_ROOT = Path(r"D:\BDHS2022_FeverCare_SpatialEquity")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLE_DIR = PROJECT_ROOT / "results" / "tables"
LOG_DIR = PROJECT_ROOT / "logs"

inputs = {
    "kr": RAW_DIR / "BDKR81FL.DTA",
    "cohort": PROCESSED_DIR / "06_fever_care_analysis_ready.csv",
    "primary": TABLE_DIR / "07a_svy_primary_estimates.csv",
    "providers": TABLE_DIR / "07a_svy_provider_estimates.csv",
    "comparison": TABLE_DIR / "07a_variance_comparison.csv",
}

for path in inputs.values():
    if not path.is_file():
        raise FileNotFoundError(f"Required input not found: {path}")

cohort = pd.read_csv(inputs["cohort"], low_memory=False)
primary = pd.read_csv(inputs["primary"])
providers = pd.read_csv(inputs["providers"])
comparison = pd.read_csv(inputs["comparison"])

full_design = pd.read_stata(
    inputs["kr"],
    columns=["v021", "v023"],
    convert_categoricals=False,
)


def design_counts(data):
    units = data[["v023", "v021"]].drop_duplicates()

    n_psus = int(len(units))
    n_strata = int(units["v023"].nunique())

    return {
        "psus": n_psus,
        "strata": n_strata,
        "df": n_psus - n_strata,
    }


full_counts = design_counts(full_design)
fever_counts = design_counts(cohort)

care_seekers = cohort.loc[cohort["care_sought"].eq(1)]
care_counts = design_counts(care_seekers)

domain_summary = pd.DataFrame([
    {
        "population": "Full KR sample",
        "n": len(full_design),
        **full_counts,
    },
    {
        "population": "All febrile children",
        "n": len(cohort),
        **fever_counts,
    },
    {
        "population": "Care seekers",
        "n": len(care_seekers),
        **care_counts,
    },
])

domain_lookup = domain_summary.set_index("population")


# Reconstruct the intervals using the degrees of freedom of each domain.
estimates = pd.concat(
    [
        primary.assign(analysis="Primary outcome"),
        providers.assign(analysis="First-care source"),
    ],
    ignore_index=True,
)

estimates["expected_df"] = estimates["population"].map(
    domain_lookup["df"]
)

estimates["expected_n"] = estimates["population"].map(
    domain_lookup["n"]
)

estimates["t_critical"] = t.ppf(
    0.975,
    estimates["expected_df"].to_numpy(dtype=float),
)

p = estimates["weighted_percent"].to_numpy(dtype=float) / 100.0
se = estimates["se_percentage_points"].to_numpy(dtype=float) / 100.0
critical = estimates["t_critical"].to_numpy(dtype=float)

if not np.isfinite(p).all() or not np.isfinite(se).all():
    raise ValueError("Non-finite survey estimates detected.")

if not ((p > 0) & (p < 1)).all():
    raise ValueError(
        "Boundary proportions require a separate interval method."
    )

if not (se > 0).all():
    raise ValueError("Invalid survey standard errors detected.")

estimates["wald_t_lower"] = 100 * (
    p - critical * se
)

estimates["wald_t_upper"] = 100 * (
    p + critical * se
)

logit_se = se / (p * (1 - p))

estimates["logit_t_lower"] = 100 * expit(
    logit(p) - critical * logit_se
)

estimates["logit_t_upper"] = 100 * expit(
    logit(p) + critical * logit_se
)

estimates["wald_lower_difference_pp"] = (
    estimates["wald_t_lower"] - estimates["ci95_lower"]
).abs()

estimates["wald_upper_difference_pp"] = (
    estimates["wald_t_upper"] - estimates["ci95_upper"]
).abs()

estimates["domain_df_matches"] = (
    estimates["design_df"] == estimates["expected_df"]
)

estimates["domain_n_matches"] = (
    estimates["n"] == estimates["expected_n"]
)

estimates["sparse_source_flag"] = (
    estimates["analysis"].eq("First-care source")
    & estimates["positive_n"].fillna(np.inf).lt(30)
)


# Validate the domain-specific inference and independent variance estimates.
max_estimate_difference = float(
    comparison["estimate_difference_pp"].abs().max()
)

max_se_relative_difference = float(
    comparison["se_relative_difference_pct"].max()
)

max_wald_difference = float(
    estimates[
        [
            "wald_lower_difference_pp",
            "wald_upper_difference_pp",
        ]
    ].to_numpy().max()
)

checks = {
    "Full KR survey contains 8,784 records":
        len(full_design) == 8784,

    "Full design has 674 PSUs and 16 strata":
        full_counts["psus"] == 674
        and full_counts["strata"] == 16,

    "Full design degrees of freedom equal 658":
        full_counts["df"] == 658,

    "Fever domain has 642 PSUs and 16 strata":
        fever_counts["psus"] == 642
        and fever_counts["strata"] == 16,

    "Fever domain degrees of freedom equal 626":
        fever_counts["df"] == 626,

    "Care-seeker domain has 611 PSUs and 16 strata":
        care_counts["psus"] == 611
        and care_counts["strata"] == 16,

    "Care-seeker domain degrees of freedom equal 595":
        care_counts["df"] == 595,

    "All 13 independent estimates are available":
        len(estimates) == 13,

    "All estimate-specific degrees of freedom agree":
        bool(estimates["domain_df_matches"].all()),

    "All estimate-specific sample sizes agree":
        bool(estimates["domain_n_matches"].all()),

    "Point estimates agree with Step 7":
        max_estimate_difference < 1e-5,

    "Standard errors agree within 0.5 percent":
        max_se_relative_difference < 0.5,

    "Package Wald intervals reproduce using domain t values":
        max_wald_difference < 0.001,

    "Logit-transformed intervals are bounded":
        bool(
            (
                estimates["logit_t_lower"]
                >= 0
            ).all()
            and (
                estimates["logit_t_upper"]
                <= 100
            ).all()
            and (
                estimates["logit_t_lower"]
                <= estimates["weighted_percent"]
            ).all()
            and (
                estimates["weighted_percent"]
                <= estimates["logit_t_upper"]
            ).all()
        ),

    "Provider categories cover all fever records":
        int(providers["positive_n"].sum()) == 2630,
}

validation_table = pd.DataFrame([
    {
        "verification": name,
        "status": "PASS" if result else "REVIEW",
    }
    for name, result in checks.items()
])

all_passed = all(checks.values())

primary_final = estimates.loc[
    estimates["analysis"].eq("Primary outcome"),
    [
        "indicator",
        "population",
        "n",
        "weighted_percent",
        "se_percentage_points",
        "expected_df",
        "logit_t_lower",
        "logit_t_upper",
    ],
].copy()

provider_final = estimates.loc[
    estimates["analysis"].eq("First-care source"),
    [
        "indicator",
        "population",
        "positive_n",
        "weighted_percent",
        "se_percentage_points",
        "expected_df",
        "logit_t_lower",
        "logit_t_upper",
        "sparse_source_flag",
    ],
].copy()


# Retain the original calculations and save the revised inference separately.
outputs = {
    "domains": TABLE_DIR / "07b_domain_degrees_of_freedom.csv",
    "intervals": TABLE_DIR / "07b_full_interval_comparison.csv",
    "primary": TABLE_DIR / "07b_primary_bounded_intervals.csv",
    "providers": TABLE_DIR / "07b_provider_bounded_intervals.csv",
    "checks": TABLE_DIR / "07b_domain_inference_checks.csv",
    "audit": LOG_DIR / "07b_domain_inference_audit.json",
    "report": LOG_DIR / "07b_domain_inference_report.txt",
}

for name, table in [
    ("domains", domain_summary),
    ("intervals", estimates),
    ("primary", primary_final),
    ("providers", provider_final),
    ("checks", validation_table),
]:
    table.to_csv(
        outputs[name],
        index=False,
        encoding="utf-8-sig",
    )

audit = {
    "method": "Taylor-linearized variance with domain-specific t inference",
    "confidence_interval": "Delta-method logit interval using domain-specific t critical values",
    "scipy_version": scipy_version,
    "domain_summary": domain_summary.to_dict(orient="records"),
    "maximum_point_estimate_difference_pp": max_estimate_difference,
    "maximum_relative_se_difference_pct": max_se_relative_difference,
    "maximum_wald_ci_reconstruction_difference_pp": max_wald_difference,
    "checks": {
        name: bool(result)
        for name, result in checks.items()
    },
    "validation_passed": bool(all_passed),
}

outputs["audit"].write_text(
    json.dumps(audit, indent=2) + "\n",
    encoding="utf-8",
)

report = "\n".join([
    "STEP 07.2 | DOMAIN INFERENCE VALIDATION",
    "=" * 48,
    f"Full design df: {full_counts['df']}",
    f"Fever domain df: {fever_counts['df']}",
    f"Care-seeker domain df: {care_counts['df']}",
    f"Independent estimates: {len(estimates)}",
    f"Maximum point-estimate difference (pp): {max_estimate_difference:.8f}",
    f"Maximum SE relative difference (%): {max_se_relative_difference:.8f}",
    f"Maximum Wald CI reconstruction difference (pp): {max_wald_difference:.8f}",
    "Primary interval method: t-based logit transformation",
    f"Validation: {'PASS' if all_passed else 'REVIEW REQUIRED'}",
    f"Validated checks: {sum(checks.values())} / {len(checks)}",
])

outputs["report"].write_text(
    report + "\n",
    encoding="utf-8",
)

print(report)

print("\nDOMAIN DEGREES OF FREEDOM")
display(domain_summary)

print("\nPRIMARY ESTIMATES AND BOUNDED CONFIDENCE INTERVALS")
display(primary_final.round(4))

print("\nFIRST-CARE SOURCE ESTIMATES")
display(provider_final.round(4))

print("\nVALIDATION CHECKS")
display(validation_table)

print("\nSAVED FILES")
for path in outputs.values():
    print(path)

if not all_passed:
    raise ValueError(
        "Domain inference validation requires review."
    )

STEP 07.2 | DOMAIN INFERENCE VALIDATION
Full design df: 658
Fever domain df: 626
Care-seeker domain df: 595
Independent estimates: 13
Maximum point-estimate difference (pp): 0.00000000
Maximum SE relative difference (%): 0.00000000
Maximum Wald CI reconstruction difference (pp): 0.00000000
Primary interval method: t-based logit transformation
Validation: PASS
Validated checks: 15 / 15

DOMAIN DEGREES OF FREEDOM


,population,n,psus,strata,df
0,Full KR sample,8784,674,16,658
1,All febrile children,2630,642,16,626
2,Care seekers,1974,611,16,595



PRIMARY ESTIMATES AND BOUNDED CONFIDENCE INTERVALS


,indicator,population,n,weighted_percent,se_percentage_points,expected_df,logit_t_lower,logit_t_upper
0,Care sought,All febrile children,2630,76.2209,0.9456,626,74.3144,78.0278
1,No reported care,All febrile children,2630,23.7791,0.9456,626,21.9722,25.6856
2,Pharmacy first,All febrile children,2630,44.0266,1.1809,626,41.7223,46.3570
3,Non-pharmacy first,All febrile children,2630,32.1943,1.1150,626,30.0449,34.4218
4,Pharmacy first among care seekers,Care seekers,1974,57.7618,1.3691,595,55.0526,60.4252
5,Care sought on day 0-1,Care seekers,1974,69.5148,1.2892,595,66.9255,71.9861



FIRST-CARE SOURCE ESTIMATES


,indicator,population,positive_n,weighted_percent,se_percentage_points,expected_df,logit_t_lower,logit_t_upper,sparse_source_flag
6,No reported care,All febrile children,656.0,23.7791,0.9456,626,21.9722,25.6856,False
7,Public sector,All febrile children,300.0,10.7786,0.7342,626,9.4193,12.3075,False
8,Private clinical source,All febrile children,413.0,15.7929,0.8649,626,14.1677,17.5664,False
9,NGO sector,All febrile children,11.0,0.3755,0.1474,626,0.1736,0.8104,True
10,Pharmacy or drug store,All febrile children,1120.0,44.0266,1.1809,626,41.7223,46.3570,False
11,Non-qualified doctor,All febrile children,97.0,4.0523,0.4739,626,3.2174,5.0924,False
12,Other or unspecified source,All febrile children,33.0,1.1950,0.2297,626,0.8186,1.7413,False



VALIDATION CHECKS


,verification,status
0,"Full KR survey contains 8,784 records",PASS
1,Full design has 674 PSUs and 16 strata,PASS
2,Full design degrees of freedom equal 658,PASS
3,Fever domain has 642 PSUs and 16 strata,PASS
4,Fever domain degrees of freedom equal 626,PASS
5,Care-seeker domain has 611 PSUs and 16 strata,PASS
6,Care-seeker domain degrees of freedom equal 595,PASS
7,All 13 independent estimates are available,PASS
8,All estimate-specific degrees of freedom agree,PASS
9,All estimate-specific sample sizes agree,PASS



SAVED FILES
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\07b_domain_degrees_of_freedom.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\07b_full_interval_comparison.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\07b_primary_bounded_intervals.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\07b_provider_bounded_intervals.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\07b_domain_inference_checks.csv
D:\BDHS2022_FeverCare_SpatialEquity\logs\07b_domain_inference_audit.json
D:\BDHS2022_FeverCare_SpatialEquity\logs\07b_domain_inference_report.txt


In [14]:

from pathlib import Path
import hashlib
import json
import warnings

import numpy as np
import pandas as pd
import polars as pl
import svy
from IPython.display import display

PROJECT_ROOT = Path(r"D:\BDHS2022_FeverCare_SpatialEquity")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLE_DIR = PROJECT_ROOT / "results" / "tables"
LOG_DIR = PROJECT_ROOT / "logs"

source_path = RAW_DIR / "BDKR81FL.DTA"
cohort_path = PROCESSED_DIR / "06_fever_care_analysis_ready.csv"
source_audit_path = LOG_DIR / "01_input_audit.json"
inference_audit_path = LOG_DIR / "07b_domain_inference_audit.json"

for path in (source_path, cohort_path, source_audit_path, inference_audit_path):
    if not path.is_file():
        raise FileNotFoundError(f"Required input not found: {path}")

with source_audit_path.open(encoding="utf-8") as file:
    source_audit = json.load(file)

with inference_audit_path.open(encoding="utf-8") as file:
    inference_audit = json.load(file)

if not inference_audit.get("validation_passed", False):
    raise ValueError("Step 7.2 survey inference verification has not passed.")

with source_path.open("rb") as file:
    source_checksum = hashlib.file_digest(file, "sha256").hexdigest()

if source_checksum != source_audit["source_files"]["kr_sha256"]:
    raise ValueError("The source dataset does not match the verified version.")


# Reconstruct the complete survey design from the original child recode file.
keys = ["v001", "v002", "v003", "bidx"]

source_columns = [
    *keys, "v005", "v021", "v023", "v024", "v025",
    "h22", "h32y", "h31", "h31b", "b19", "b4",
    "v012", "v106", "v190", "bord",
]

raw = pd.read_stata(
    source_path,
    columns=source_columns,
    convert_categoricals=False,
)

frozen = pd.read_csv(
    cohort_path,
    usecols=keys + ["care_sought", "stage1_core_complete_case"],
)

fever = raw["h22"].eq(1)

core_columns = [
    "b19", "b4", "h31", "v012", "v106",
    "v190", "bord", "v025", "v024",
]

complete = (
    raw[core_columns].notna().all(axis=1)
    & raw["h31b"].isin([0, 1])
)

eligible = fever & complete
outcome = raw["h32y"].eq(0)

expected_ids = (
    frozen.loc[
        frozen["stage1_core_complete_case"].eq(1), keys
    ]
    .astype("int64")
    .sort_values(keys)
    .reset_index(drop=True)
)

observed_ids = (
    raw.loc[eligible, keys]
    .astype("int64")
    .sort_values(keys)
    .reset_index(drop=True)
)

if not observed_ids.equals(expected_ids):
    raise ValueError(
        "Regression eligibility differs from the frozen cohort."
    )

frozen_outcomes = raw.loc[
    fever, keys + ["h32y"]
].merge(
    frozen[keys + ["care_sought"]],
    on=keys,
    how="inner",
    validate="one_to_one",
)

if (
    len(frozen_outcomes) != 2630
    or not frozen_outcomes["h32y"].eq(0).astype(int).eq(
        frozen_outcomes["care_sought"]
    ).all()
):
    raise ValueError(
        "Frozen outcomes do not match the original response codes."
    )


# Keep all survey records; identify complete fever cases through a domain indicator.
survey_frame = raw[
    [
        "v021", "v023", "v005", "b4",
        "v106", "v190", "bord", "v025", "v024",
    ]
].copy()

survey_frame["survey_weight"] = (
    survey_frame["v005"].astype(float) / 1_000_000
)

survey_frame["stage1_eligible"] = eligible.astype("int8")

survey_frame["care_sought"] = (
    fever & outcome
).astype("int8")

survey_frame["child_age_years"] = (
    raw["b19"].astype(float) / 12.0
)

survey_frame["maternal_age_5years"] = (
    raw["v012"].astype(float) - 25.0
) / 5.0

survey_frame["cough_reported"] = (
    raw["h31"].map({0: 0, 2: 1})
)

survey_frame["rapid_breathing_reported"] = (
    raw["h31b"].map({0: 0, 1: 1})
)

model_columns = [
    "v021", "v023", "survey_weight",
    "stage1_eligible", "care_sought",
    "child_age_years", "b4", "cough_reported",
    "rapid_breathing_reported", "maternal_age_5years",
    "v106", "v190", "bord", "v025", "v024",
]

if survey_frame.loc[eligible, model_columns].isna().any().any():
    raise ValueError(
        "Missing model values were found in eligible observations."
    )

if (
    not np.isfinite(survey_frame["survey_weight"]).all()
    or not survey_frame["survey_weight"].gt(0).all()
):
    raise ValueError("Invalid survey sampling weights.")


# Preserve null values outside the analysis domain.
polars_data = pl.DataFrame({
    column: survey_frame[column]
    .astype(object)
    .where(survey_frame[column].notna(), None)
    .tolist()
    for column in model_columns
})

survey_design = svy.Design(
    stratum="v023",
    psu="v021",
    wgt="survey_weight",
)

survey_sample = svy.Sample(
    data=polars_data,
    design=survey_design,
)

predictors = [
    "child_age_years",
    svy.Cat("b4", ref=1),
    svy.Cat("cough_reported", ref=0),
    svy.Cat("rapid_breathing_reported", ref=0),
    "maternal_age_5years",
    svy.Cat("v106", ref=0),
    svy.Cat("v190", ref=1),
    "bord",
    svy.Cat("v025", ref=1),
    svy.Cat("v024", ref=1),
]

with warnings.catch_warnings(record=True) as captured_warnings:
    warnings.simplefilter("always")

    model = survey_sample.glm.fit(
        y="care_sought",
        x=predictors,
        where=svy.col("stage1_eligible") == 1,
        family="binomial",
        link="logit",
    )

model_warnings = [
    f"{item.category.__name__}: {item.message}"
    for item in captured_warnings
]


# Extract coefficients using the svy 0.37.0 result schema.
coefficients = pd.DataFrame(
    model.to_polars().to_dicts()
)

required_fields = {
    "term", "est", "se", "lci", "uci",
    "t", "df", "p_value", "family", "link",
}

missing_fields = required_fields - set(coefficients.columns)

if missing_fields:
    raise ValueError(
        f"Missing GLM result fields: {sorted(missing_fields)}"
    )

numeric_fields = [
    "est", "se", "lci", "uci", "t", "df", "p_value",
]

for field in numeric_fields:
    coefficients[field] = pd.to_numeric(
        coefficients[field],
        errors="coerce",
    )

with np.errstate(over="ignore", invalid="ignore"):
    coefficients["adjusted_odds_ratio"] = np.exp(
        coefficients["est"]
    )

    coefficients["or_ci95_lower"] = np.exp(
        coefficients["lci"]
    )

    coefficients["or_ci95_upper"] = np.exp(
        coefficients["uci"]
    )

coefficients.insert(
    0, "model_name", "Stage 1: core adjustment"
)

critical_warning_terms = (
    "SEPARAT",
    "CONVERG",
    "SINGULAR",
    "COLLINEAR",
    "ITERATION",
    "GLM_ITER",
    "OVERFLOW",
)

critical_warnings = [
    warning for warning in model_warnings
    if any(
        term in warning.upper()
        for term in critical_warning_terms
    )
]


# Validate the sampling design, fitted coefficients and inferential results.
checks = {
    "Full KR sample retained":
        len(survey_frame) == 8784,

    "All 674 PSUs retained":
        survey_frame["v021"].nunique() == 674,

    "All 16 sampling strata retained":
        survey_frame["v023"].nunique() == 16,

    "Stage 1 complete-case sample equals 2,624":
        int(eligible.sum()) == 2624,

    "Stage 1 care seekers equal 1,969":
        int((eligible & outcome).sum()) == 1969,

    "Only six fever records lack a core predictor":
        int((fever & ~eligible).sum()) == 6,

    "Model has 22 coefficients":
        len(coefficients) == 22,

    "One intercept is present":
        coefficients["term"].str.contains(
            "intercept", case=False
        ).sum() == 1,

    "All estimates and standard errors are finite":
        bool(
            np.isfinite(
                coefficients[
                    ["est", "se"]
                ].to_numpy(dtype=float)
            ).all()
        ),

    "All odds ratios and interval limits are finite":
        bool(
            np.isfinite(
                coefficients[
                    [
                        "adjusted_odds_ratio",
                        "or_ci95_lower",
                        "or_ci95_upper",
                    ]
                ].to_numpy(dtype=float)
            ).all()
        ),

    "Standard errors are positive":
        bool(coefficients["se"].gt(0).all()),

    "Coefficient intervals contain their estimates":
        bool(
            (coefficients["lci"] <= coefficients["est"]).all()
            and (coefficients["est"] <= coefficients["uci"]).all()
        ),

    "All inferential degrees of freedom are positive":
        bool(coefficients["df"].gt(0).all()),

    "All p-values are between zero and one":
        bool(coefficients["p_value"].between(0, 1).all()),

    "Binomial-logit specification confirmed":
        bool(
            coefficients["family"]
            .astype(str).str.lower().eq("binomial").all()
            and coefficients["link"]
            .astype(str).str.lower().eq("logit").all()
        ),

    "No critical model warnings":
        len(critical_warnings) == 0,
}

validation_table = pd.DataFrame({
    "check": list(checks),
    "status": [
        "PASS" if passed else "REVIEW"
        for passed in checks.values()
    ],
})

all_passed = all(checks.values())


# Save the full regression output and model specification.
outputs = {
    "coefficients":
        TABLE_DIR / "08a_stage1_adjusted_logistic_coefficients.csv",
    "checks":
        TABLE_DIR / "08a_stage1_model_checks.csv",
    "audit":
        LOG_DIR / "08a_stage1_model_audit.json",
    "report":
        LOG_DIR / "08a_stage1_model_report.txt",
}

coefficients.to_csv(
    outputs["coefficients"],
    index=False,
    encoding="utf-8-sig",
)

validation_table.to_csv(
    outputs["checks"],
    index=False,
    encoding="utf-8-sig",
)

report = "\n".join([
    "STEP 08.1 | STAGE 1 ADJUSTED LOGISTIC REGRESSION",
    "=" * 52,
    f"Full KR records: {len(survey_frame)}",
    f"Full design PSUs: {survey_frame['v021'].nunique()}",
    f"Sampling strata: {survey_frame['v023'].nunique()}",
    f"Model observations: {int(eligible.sum())}",
    f"Care sought: {int((eligible & outcome).sum())}",
    f"No care sought: {int((eligible & ~outcome).sum())}",
    f"Model coefficients: {len(coefficients)}",
    "Variance estimator: survey-design Taylor linearization",
    f"Model warnings: {len(model_warnings)}",
    f"Validation: {'PASS' if all_passed else 'REVIEW REQUIRED'}",
    f"Validated checks: {sum(checks.values())} / {len(checks)}",
])

outputs["report"].write_text(
    report + "\n",
    encoding="utf-8",
)

model_audit = {
    "analysis": "Stage 1 survey-weighted logistic regression",
    "source_sha256": source_checksum,
    "full_sample_n": int(len(survey_frame)),
    "model_domain_n": int(eligible.sum()),
    "positive_n": int((eligible & outcome).sum()),
    "model_terms": coefficients["term"].tolist(),
    "export_schema": list(coefficients.columns),
    "sampling": {
        "weight": "v005 / 1000000",
        "psu": "v021",
        "stratum": "v023",
    },
    "specification": {
        "outcome": "h32y == 0 among children with h22 == 1",
        "predictors": [
            "child_age_years",
            "b4 (ref 1)",
            "cough_reported (ref 0)",
            "rapid_breathing_reported (ref 0)",
            "maternal_age_5years",
            "v106 (ref 0)",
            "v190 (ref 1)",
            "bord",
            "v025 (ref 1)",
            "v024 (ref 1)",
        ],
        "family": "binomial",
        "link": "logit",
    },
    "warnings": model_warnings,
    "checks": {
        name: bool(value)
        for name, value in checks.items()
    },
    "validation_passed": bool(all_passed),
}

outputs["audit"].write_text(
    json.dumps(model_audit, indent=2) + "\n",
    encoding="utf-8",
)

print(report)

print("\nADJUSTED ODDS RATIOS")
display(
    coefficients[
        [
            "term",
            "adjusted_odds_ratio",
            "or_ci95_lower",
            "or_ci95_upper",
            "p_value",
            "df",
        ]
    ].round(4)
)

print("\nMODEL VALIDATION")
display(validation_table)

print("\nMODEL WARNINGS")
print(
    "\n".join(model_warnings)
    if model_warnings else "None"
)

print("\nSAVED FILES")
for path in outputs.values():
    print(path)

if not all_passed:
    raise ValueError(
        "Stage 1 regression requires review before marginal estimation."
    )

STEP 08.1 | STAGE 1 ADJUSTED LOGISTIC REGRESSION
Full KR records: 8784
Full design PSUs: 674
Sampling strata: 16
Model observations: 2624
Care sought: 1969
No care sought: 655
Model coefficients: 22
Variance estimator: survey-design Taylor linearization
Model warnings: 0
Validation: PASS
Validated checks: 16 / 16

ADJUSTED ODDS RATIOS


,term,adjusted_odds_ratio,or_ci95_lower,or_ci95_upper,p_value,df
0,_intercept_,3.8530,1.8709,7.9348,0.0003,605.0
1,child_age_years,0.8810,0.8171,0.9500,0.0010,605.0
2,b4_2,0.8271,0.6818,1.0032,0.0539,605.0
3,cough_reported_1.0,1.5934,1.3003,1.9526,0.0000,605.0
4,rapid_breathing_reported_1.0,1.7620,1.2330,2.5180,0.0019,605.0
5,maternal_age_5years,0.9768,0.8523,1.1195,0.7356,605.0
6,v106_1,1.3482,0.8693,2.0908,0.1817,605.0
7,v106_2,1.1862,0.7697,1.8279,0.4384,605.0
8,v106_3,0.8632,0.5181,1.4380,0.5715,605.0
9,v190_2,0.8905,0.6409,1.2374,0.4890,605.0



MODEL VALIDATION


,check,status
0,Full KR sample retained,PASS
1,All 674 PSUs retained,PASS
2,All 16 sampling strata retained,PASS
3,"Stage 1 complete-case sample equals 2,624",PASS
4,"Stage 1 care seekers equal 1,969",PASS
5,Only six fever records lack a core predictor,PASS
6,Model has 22 coefficients,PASS
7,One intercept is present,PASS
8,All estimates and standard errors are finite,PASS
9,All odds ratios and interval limits are finite,PASS



MODEL WARNINGS
None

SAVED FILES
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\08a_stage1_adjusted_logistic_coefficients.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\08a_stage1_model_checks.csv
D:\BDHS2022_FeverCare_SpatialEquity\logs\08a_stage1_model_audit.json
D:\BDHS2022_FeverCare_SpatialEquity\logs\08a_stage1_model_report.txt


In [16]:

from pathlib import Path
from importlib.metadata import version
import json

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path(r"D:\BDHS2022_FeverCare_SpatialEquity")
TABLE_DIR = PROJECT_ROOT / "results" / "tables"
LOG_DIR = PROJECT_ROOT / "logs"

coef_path = TABLE_DIR / "08a_stage1_adjusted_logistic_coefficients.csv"
audit_path = LOG_DIR / "08a_stage1_model_audit.json"

for path in (coef_path, audit_path):
    if not path.is_file():
        raise FileNotFoundError(f"Required Step 8.1 result not found: {path}")

if "model" not in globals() or not hasattr(model, "margins"):
    raise RuntimeError(
        "The fitted Stage 1 model is unavailable. Run Cell 8.1 first."
    )

svy_version = version("svy")
if svy_version != "0.37.0":
    raise RuntimeError(
        f"Expected svy 0.37.0, found {svy_version}."
    )

with audit_path.open(encoding="utf-8") as file:
    previous_audit = json.load(file)

if not previous_audit.get("validation_passed", False):
    raise ValueError("Step 8.1 has not passed validation.")

saved_coef = pd.read_csv(coef_path)
active_coef = pd.DataFrame(model.to_polars().to_dicts())

coefficient_check = saved_coef[
    ["term", "est", "se"]
].merge(
    active_coef[["term", "est", "se"]],
    on="term",
    how="outer",
    validate="one_to_one",
    suffixes=("_saved", "_active"),
    indicator=True,
)

same_model = bool(
    len(coefficient_check) == 22
    and coefficient_check["_merge"].eq("both").all()
    and np.allclose(
        coefficient_check["est_saved"],
        coefficient_check["est_active"],
        rtol=0,
        atol=1e-10,
    )
    and np.allclose(
        coefficient_check["se_saved"],
        coefficient_check["se_active"],
        rtol=0,
        atol=1e-10,
    )
)

if not same_model:
    raise ValueError(
        "The active GLM differs from the verified Step 8.1 fit."
    )


# Obtain adjusted predictions and categorical marginal contrasts.
wealth_raw = pd.DataFrame(
    model.margins(
        at={"v190": [1, 2, 3, 4, 5]}
    ).to_polars().to_dicts()
)

education_raw = pd.DataFrame(
    model.margins(
        at={"v106": [0, 1, 2, 3]}
    ).to_polars().to_dicts()
)

effects_raw = pd.DataFrame(
    model.margins(
        variables=["v190", "v106"]
    ).to_polars().to_dicts()
)


def estimate_field(frame):
    """Identify the numerical estimate field in the survey output."""
    if "est" in frame.columns:
        return "est"
    if "margin" in frame.columns:
        return "margin"

    raise ValueError(
        f"No estimate field in margin output: {frame.columns.tolist()}"
    )


def required_schema(frame, extra_fields, table_name):
    fields = {
        "term", "se", "lci", "uci", "df",
        *extra_fields,
    }

    absent = fields - set(frame.columns)

    if absent:
        raise ValueError(
            f"{table_name} is missing fields {sorted(absent)}. "
            f"Returned columns: {frame.columns.tolist()}"
        )

    return estimate_field(frame)


wealth_est_col = required_schema(
    wealth_raw, {"v190"}, "Wealth margins"
)

education_est_col = required_schema(
    education_raw, {"v106"}, "Education margins"
)

effect_est_col = required_schema(
    effects_raw, set(), "Marginal contrasts"
)

wealth_labels = {
    1: "Poorest",
    2: "Poorer",
    3: "Middle",
    4: "Richer",
    5: "Richest",
}

education_labels = {
    0: "No education",
    1: "Primary",
    2: "Secondary",
    3: "Higher",
}


# Convert survey predictions to percentage-point reporting units.
def tidy_predictions(raw_table, variable, labels, estimate_column):
    result = raw_table.copy()

    result["level_code"] = pd.to_numeric(
        result[variable],
        errors="coerce",
    )

    if not result["level_code"].notna().all():
        raise ValueError(
            f"Non-numeric {variable} levels in predictive margins."
        )

    if not np.allclose(
        result["level_code"],
        np.round(result["level_code"]),
    ):
        raise ValueError(
            f"Unexpected non-integer {variable} levels."
        )

    result["level_code"] = result["level_code"].astype(int)

    result["level_label"] = (
        result["level_code"].map(labels)
    )

    for source, target in (
        (estimate_column, "adjusted_percent"),
        ("se", "se_percentage_points"),
        ("lci", "ci95_lower"),
        ("uci", "ci95_upper"),
    ):
        result[target] = (
            100 * pd.to_numeric(result[source], errors="coerce")
        )

    result["design_df"] = pd.to_numeric(
        result["df"], errors="coerce"
    )

    return result[
        [
            "term",
            "level_code",
            "level_label",
            "adjusted_percent",
            "se_percentage_points",
            "ci95_lower",
            "ci95_upper",
            "design_df",
        ]
    ].sort_values(
        "level_code"
    ).reset_index(drop=True)


wealth = tidy_predictions(
    wealth_raw,
    "v190",
    wealth_labels,
    wealth_est_col,
)

education = tidy_predictions(
    education_raw,
    "v106",
    education_labels,
    education_est_col,
)


# Retain the survey package's contrast standard errors and intervals.
effects = effects_raw.copy()

effects["package_contrast"] = (
    effects["contrast"].astype("string")
    if "contrast" in effects.columns
    else pd.Series(
        pd.NA,
        index=effects.index,
        dtype="string",
    )
)

for source, target in (
    (effect_est_col, "risk_difference_pp"),
    ("se", "se_pp"),
    ("lci", "ci95_lower_pp"),
    ("uci", "ci95_upper_pp"),
):
    effects[target] = (
        100 * pd.to_numeric(effects[source], errors="coerce")
    )

effects["design_df"] = pd.to_numeric(
    effects["df"], errors="coerce"
)

effects["comparison"] = pd.Series(
    pd.NA,
    index=effects.index,
    dtype="string",
)

effects["level_code"] = pd.Series(
    pd.NA,
    index=effects.index,
    dtype="Int64",
)

effects["margin_difference_pp"] = np.nan

matching_problems = []


# Verify each categorical contrast against its corresponding margins.
for variable, predictions, labels, reference in (
    ("v190", wealth, wealth_labels, 1),
    ("v106", education, education_labels, 0),
):
    baseline = predictions.loc[
        predictions["level_code"].eq(reference),
        "adjusted_percent",
    ]

    if len(baseline) != 1:
        matching_problems.append(
            f"{variable}: reference margin is missing or duplicated"
        )
        continue

    baseline_value = float(baseline.iloc[0])

    target_differences = {
        int(row.level_code): float(
            row.adjusted_percent - baseline_value
        )
        for row in predictions.itertuples(index=False)
        if int(row.level_code) != reference
    }

    used_levels = set()

    variable_rows = effects.index[
        effects["term"].astype(str).eq(variable)
    ]

    for index in variable_rows:
        observed_difference = float(
            effects.loc[index, "risk_difference_pp"]
        )

        matched = [
            code
            for code, expected in target_differences.items()
            if code not in used_levels
            and np.isclose(
                observed_difference,
                expected,
                atol=1e-6,
                rtol=0,
            )
        ]

        if len(matched) != 1:
            matching_problems.append(
                f"{variable}: could not uniquely identify contrast "
                f"{effects.loc[index, 'package_contrast']}"
            )
            continue

        level = matched[0]
        used_levels.add(level)

        effects.loc[index, "level_code"] = level

        effects.loc[index, "comparison"] = (
            f"{labels[level]} - {labels[reference]}"
        )

        effects.loc[index, "margin_difference_pp"] = (
            target_differences[level]
        )

    if used_levels != set(target_differences):
        matching_problems.append(
            f"{variable}: not all nonreference contrasts matched"
        )


contrasts = effects[
    [
        "term",
        "level_code",
        "comparison",
        "package_contrast",
        "risk_difference_pp",
        "se_pp",
        "ci95_lower_pp",
        "ci95_upper_pp",
        "design_df",
        "margin_difference_pp",
    ]
].sort_values(
    ["term", "level_code"]
).reset_index(drop=True)


# Audit the fitted model, adjusted probabilities and contrast inference.
all_prediction_values = pd.concat(
    [wealth, education],
    ignore_index=True,
)

checks = {
    "Active regression coefficients match Step 8.1":
        same_model,

    "Five wealth predictive margins returned":
        len(wealth) == 5,

    "Four maternal-education margins returned":
        len(education) == 4,

    "Seven categorical marginal contrasts returned":
        len(contrasts) == 7,

    "Wealth quintile codes are complete":
        set(wealth["level_code"]) == set(wealth_labels),

    "Maternal-education codes are complete":
        set(education["level_code"]) == set(education_labels),

    "Contrast terms are wealth and education only":
        set(contrasts["term"]) == {"v190", "v106"},

    "Contrasts uniquely match predicted probability differences":
        (
            not matching_problems
            and contrasts["comparison"].notna().all()
        ),

    "All estimates and standard errors are finite":
        bool(
            np.isfinite(
                all_prediction_values[
                    [
                        "adjusted_percent",
                        "se_percentage_points",
                        "ci95_lower",
                        "ci95_upper",
                    ]
                ].to_numpy(dtype=float)
            ).all()
            and np.isfinite(
                contrasts[
                    [
                        "risk_difference_pp",
                        "se_pp",
                        "ci95_lower_pp",
                        "ci95_upper_pp",
                    ]
                ].to_numpy(dtype=float)
            ).all()
        ),

    "All standard errors are positive":
        bool(
            all_prediction_values[
                "se_percentage_points"
            ].gt(0).all()
            and contrasts["se_pp"].gt(0).all()
        ),

    "Adjusted predicted probabilities fall between zero and 100":
        bool(
            all_prediction_values[
                "adjusted_percent"
            ].between(0, 100).all()
        ),

    "Confidence intervals include their estimates":
        bool(
            (
                (
                    all_prediction_values["ci95_lower"]
                    <= all_prediction_values["adjusted_percent"]
                )
                & (
                    all_prediction_values["adjusted_percent"]
                    <= all_prediction_values["ci95_upper"]
                )
            ).all()
            and (
                (
                    contrasts["ci95_lower_pp"]
                    <= contrasts["risk_difference_pp"]
                )
                & (
                    contrasts["risk_difference_pp"]
                    <= contrasts["ci95_upper_pp"]
                )
            ).all()
        ),

    "All margin and contrast degrees of freedom are positive":
        bool(
            all_prediction_values["design_df"].gt(0).all()
            and contrasts["design_df"].gt(0).all()
        ),
}

check_table = pd.DataFrame([
    {
        "check": name,
        "status": "PASS" if passed else "REVIEW",
    }
    for name, passed in checks.items()
])

all_passed = all(checks.values())


# Save both formatted results and unmodified package output.
outputs = {
    "wealth":
        TABLE_DIR / "08b_stage1_wealth_adjusted_margins.csv",

    "education":
        TABLE_DIR / "08b_stage1_education_adjusted_margins.csv",

    "contrasts":
        TABLE_DIR / "08b_stage1_adjusted_risk_differences.csv",

    "raw_wealth":
        TABLE_DIR / "08b_raw_wealth_margins.csv",

    "raw_education":
        TABLE_DIR / "08b_raw_education_margins.csv",

    "raw_contrasts":
        TABLE_DIR / "08b_raw_marginal_contrasts.csv",

    "checks":
        TABLE_DIR / "08b_stage1_margins_validation.csv",

    "audit":
        LOG_DIR / "08b_stage1_margins_audit.json",

    "report":
        LOG_DIR / "08b_stage1_margins_report.txt",
}

for key, frame in (
    ("wealth", wealth),
    ("education", education),
    ("contrasts", contrasts),
    ("raw_wealth", wealth_raw),
    ("raw_education", education_raw),
    ("raw_contrasts", effects_raw),
    ("checks", check_table),
):
    frame.to_csv(
        outputs[key],
        index=False,
        encoding="utf-8-sig",
    )


# Highlight the two prespecified socioeconomic comparisons.
primary_comparisons = contrasts.loc[
    contrasts["comparison"].isin(
        [
            "Richest - Poorest",
            "Higher - No education",
        ]
    )
].copy()

report = "\n".join([
    "STEP 08.2 | STAGE 1 STANDARDIZED MARGINS AND CONTRASTS",
    "=" * 60,
    "Model: verified Stage 1 survey-weighted logistic regression",
    "Fitted domain: 2,624 complete-case febrile children",
    f"Survey package: svy {svy_version}",
    f"Wealth adjusted margins: {len(wealth)}",
    f"Education adjusted margins: {len(education)}",
    f"Categorical discrete contrasts: {len(contrasts)}",
    "Uncertainty: svy delta-method SE for predictive margins and contrasts",
    "Inference: adjusted associations, not causal effects",
    f"Validation: {'PASS' if all_passed else 'REVIEW REQUIRED'}",
    f"Validated checks: {sum(checks.values())} / {len(checks)}",
])

outputs["report"].write_text(
    report + "\n",
    encoding="utf-8",
)

outputs["audit"].write_text(
    json.dumps(
        {
            "model": "Stage 1 core survey-weighted binomial GLM",
            "svy_version": svy_version,
            "model_domain_n": int(
                previous_audit["model_domain_n"]
            ),
            "margin_estimate_columns": {
                "wealth": wealth_est_col,
                "education": education_est_col,
                "contrasts": effect_est_col,
            },
            "uncertainty_note": (
                "Coefficient-covariance delta-method uncertainty "
                "from svy. Validation of uncertainty from the "
                "weighted covariate distribution requires a "
                "separate survey resampling sensitivity analysis."
            ),
            "contrast_matching_problems": matching_problems,
            "checks": {
                name: bool(value)
                for name, value in checks.items()
            },
            "validation_passed": bool(all_passed),
        },
        indent=2,
        ensure_ascii=False,
    ) + "\n",
    encoding="utf-8",
)

print(report)

print("\nWEALTH-ADJUSTED PREDICTIVE MARGINS")
display(wealth.round(4))

print("\nEDUCATION-ADJUSTED PREDICTIVE MARGINS")
display(education.round(4))

print("\nPRIMARY SOCIOECONOMIC CONTRASTS")
display(primary_comparisons.round(4))

print("\nALL ADJUSTED ABSOLUTE RISK DIFFERENCES")
display(contrasts.round(4))

print("\nVALIDATION CHECKS")
display(check_table)

if matching_problems:
    print("\nCONTRAST MATCHING ISSUES")
    for problem in matching_problems:
        print(problem)

print("\nSAVED FILES")
for path in outputs.values():
    print(path)

if not all_passed:
    raise ValueError(
        "Stage 1 marginal results need review before Stage 2 analysis."
    )

STEP 08.2 | STAGE 1 STANDARDIZED MARGINS AND CONTRASTS
Model: verified Stage 1 survey-weighted logistic regression
Fitted domain: 2,624 complete-case febrile children
Survey package: svy 0.37.0
Wealth adjusted margins: 5
Education adjusted margins: 4
Categorical discrete contrasts: 7
Uncertainty: svy delta-method SE for predictive margins and contrasts
Inference: adjusted associations, not causal effects
Validation: PASS
Validated checks: 13 / 13

WEALTH-ADJUSTED PREDICTIVE MARGINS


,term,level_code,level_label,adjusted_percent,se_percentage_points,ci95_lower,ci95_upper,design_df
0,v190,1,Poorest,76.2095,2.2960,71.7003,80.7186,605.0
1,v190,2,Poorer,74.1424,2.0967,70.0248,78.2601,605.0
2,v190,3,Middle,77.7226,1.9853,73.8237,81.6216,605.0
3,v190,4,Richer,78.9923,1.8491,75.3608,82.6238,605.0
4,v190,5,Richest,73.5137,2.5560,68.4940,78.5334,605.0



EDUCATION-ADJUSTED PREDICTIVE MARGINS


,term,level_code,level_label,adjusted_percent,se_percentage_points,ci95_lower,ci95_upper,design_df
0,v106,0,No education,73.6509,3.7770,66.2332,81.0686,605.0
1,v106,1,Primary,78.8289,1.8798,75.1371,82.5206,605.0
2,v106,2,Secondary,76.7025,1.2233,74.3001,79.1049,605.0
3,v106,3,Higher,70.8316,2.8977,65.1408,76.5225,605.0



PRIMARY SOCIOECONOMIC CONTRASTS


,term,level_code,comparison,package_contrast,risk_difference_pp,se_pp,ci95_lower_pp,ci95_upper_pp,design_df,margin_difference_pp
2,v106,3,Higher - No education,3 - 0,-2.8193,4.9368,-12.5147,6.8762,605.0,-2.8193
6,v190,5,Richest - Poorest,5 - 1,-2.6958,3.6631,-9.8897,4.4981,605.0,-2.6958



ALL ADJUSTED ABSOLUTE RISK DIFFERENCES


,term,level_code,comparison,package_contrast,risk_difference_pp,se_pp,ci95_lower_pp,ci95_upper_pp,design_df,margin_difference_pp
0,v106,1,Primary - No education,1 - 0,5.1780,4.0341,-2.7446,13.1005,605.0,5.1780
1,v106,2,Secondary - No education,2 - 0,3.0516,4.0525,-4.9071,11.0102,605.0,3.0516
2,v106,3,Higher - No education,3 - 0,-2.8193,4.9368,-12.5147,6.8762,605.0,-2.8193
3,v190,2,Poorer - Poorest,2 - 1,-2.0670,2.9745,-7.9087,3.7746,605.0,-2.0670
4,v190,3,Middle - Poorest,3 - 1,1.5132,3.1118,-4.5981,7.6244,605.0,1.5132
5,v190,4,Richer - Poorest,4 - 1,2.7828,3.0819,-3.2697,8.8354,605.0,2.7828
6,v190,5,Richest - Poorest,5 - 1,-2.6958,3.6631,-9.8897,4.4981,605.0,-2.6958



VALIDATION CHECKS


,check,status
0,Active regression coefficients match Step 8.1,PASS
1,Five wealth predictive margins returned,PASS
2,Four maternal-education margins returned,PASS
3,Seven categorical marginal contrasts returned,PASS
4,Wealth quintile codes are complete,PASS
5,Maternal-education codes are complete,PASS
6,Contrast terms are wealth and education only,PASS
7,Contrasts uniquely match predicted probability...,PASS
8,All estimates and standard errors are finite,PASS
9,All standard errors are positive,PASS



SAVED FILES
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\08b_stage1_wealth_adjusted_margins.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\08b_stage1_education_adjusted_margins.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\08b_stage1_adjusted_risk_differences.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\08b_raw_wealth_margins.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\08b_raw_education_margins.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\08b_raw_marginal_contrasts.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\08b_stage1_margins_validation.csv
D:\BDHS2022_FeverCare_SpatialEquity\logs\08b_stage1_margins_audit.json
D:\BDHS2022_FeverCare_SpatialEquity\logs\08b_stage1_margins_report.txt


In [17]:

from pathlib import Path
from importlib.metadata import version
import hashlib
import json
import warnings

import numpy as np
import pandas as pd
import polars as pl
import svy
from IPython.display import display

PROJECT_ROOT = Path(r"D:\BDHS2022_FeverCare_SpatialEquity")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLE_DIR = PROJECT_ROOT / "results" / "tables"
LOG_DIR = PROJECT_ROOT / "logs"

source_path = RAW_DIR / "BDKR81FL.DTA"
cohort_path = PROCESSED_DIR / "06_fever_care_analysis_ready.csv"
input_audit_path = LOG_DIR / "01_input_audit.json"
stage1_audit_path = LOG_DIR / "08a_stage1_model_audit.json"
margin_audit_path = LOG_DIR / "08b_stage1_margins_audit.json"

for path in (
    source_path, cohort_path, input_audit_path,
    stage1_audit_path, margin_audit_path
):
    if not path.is_file():
        raise FileNotFoundError(
            f"Required input not found: {path}"
        )

if version("svy") != "0.37.0":
    raise RuntimeError(
        "This analysis requires the validated svy 0.37.0 environment."
    )

with input_audit_path.open(encoding="utf-8") as file:
    input_audit = json.load(file)

with stage1_audit_path.open(encoding="utf-8") as file:
    stage1_audit = json.load(file)

with margin_audit_path.open(encoding="utf-8") as file:
    margin_audit = json.load(file)

if not (
    stage1_audit.get("validation_passed")
    and margin_audit.get("validation_passed")
):
    raise ValueError(
        "Stage 1 model and margins must pass before Stage 2 fitting."
    )

with source_path.open("rb") as file:
    source_checksum = hashlib.file_digest(
        file, "sha256"
    ).hexdigest()

if source_checksum != input_audit["source_files"]["kr_sha256"]:
    raise ValueError(
        "The original KR dataset fails the source checksum."
    )


# Reconstruct Stage 2 from verified raw codes and frozen child identifiers.
keys = ["v001", "v002", "v003", "bidx"]

source_columns = [
    *keys,
    "v005", "v021", "v023", "v024", "v025",
    "h22", "h32y", "h46a", "h31", "h31b", "b19",
    "b4", "v012", "v106", "v190", "bord",
]

raw = pd.read_stata(
    source_path,
    columns=source_columns,
    convert_categoricals=False,
)

frozen = pd.read_csv(
    cohort_path,
    usecols=keys + [
        "care_sought",
        "pharmacy_first_if_care",
        "stage2_core_complete_case",
    ],
)

fever = raw["h22"].eq(1)
care = fever & raw["h32y"].eq(0)

core_columns = [
    "b19", "b4", "h31", "v012", "v106",
    "v190", "bord", "v025", "v024",
]

complete = (
    raw[core_columns].notna().all(axis=1)
    & raw["h31b"].isin([0, 1])
)

eligible = care & complete
pharmacy_first = care & raw["h46a"].eq(36)

expected_ids = (
    frozen.loc[
        frozen["stage2_core_complete_case"].eq(1),
        keys,
    ]
    .astype("int64")
    .sort_values(keys)
    .reset_index(drop=True)
)

actual_ids = (
    raw.loc[eligible, keys]
    .astype("int64")
    .sort_values(keys)
    .reset_index(drop=True)
)

if not actual_ids.equals(expected_ids):
    raise ValueError(
        "Stage 2 eligibility disagrees with the frozen cohort."
    )

verified = raw.loc[
    fever, keys + ["h32y", "h46a"]
].merge(
    frozen,
    on=keys,
    how="outer",
    validate="one_to_one",
    indicator=True,
)

if (
    len(verified) != 2630
    or not verified["_merge"].eq("both").all()
):
    raise ValueError(
        "Child-level linkage failed against the frozen cohort."
    )

sought_mask = verified["h32y"].eq(0)

if not (
    verified["care_sought"].eq(
        sought_mask.astype(int)
    ).all()
    and verified.loc[
        sought_mask, "pharmacy_first_if_care"
    ].eq(
        verified.loc[
            sought_mask, "h46a"
        ].eq(36).astype(int)
    ).all()
    and verified.loc[
        ~sought_mask, "pharmacy_first_if_care"
    ].isna().all()
):
    raise ValueError(
        "Frozen conditional outcomes differ from verified DHS codes."
    )


# Keep the entire KR design and restrict regression using a domain flag.
survey_frame = raw[
    [
        "v021", "v023", "v005", "b4", "v106",
        "v190", "bord", "v025", "v024",
    ]
].copy()

survey_frame["survey_weight"] = (
    survey_frame["v005"].astype(float) / 1_000_000
)

survey_frame["stage2_eligible"] = eligible.astype("int8")

survey_frame["stage2_pharmacy_first"] = (
    pharmacy_first.astype("int8")
)

survey_frame["child_age_years"] = (
    raw["b19"].astype(float) / 12.0
)

survey_frame["maternal_age_5years"] = (
    raw["v012"].astype(float) - 25.0
) / 5.0

survey_frame["cough_reported"] = (
    raw["h31"].map({0: 0, 2: 1})
)

survey_frame["rapid_breathing_reported"] = (
    raw["h31b"].map({0: 0, 1: 1})
)

model_columns = [
    "v021", "v023", "survey_weight",
    "stage2_eligible", "stage2_pharmacy_first",
    "child_age_years", "b4", "cough_reported",
    "rapid_breathing_reported",
    "maternal_age_5years", "v106", "v190",
    "bord", "v025", "v024",
]

if survey_frame.loc[
    eligible, model_columns
].isna().any().any():
    raise ValueError(
        "Missing predictors or outcome among Stage 2 complete cases."
    )

if not (
    np.isfinite(survey_frame["survey_weight"]).all()
    and survey_frame["survey_weight"].gt(0).all()
):
    raise ValueError(
        "Survey weights must be finite and strictly positive."
    )


# Verify the model matrix is identifiable before regression.
fit_frame = survey_frame.loc[eligible].copy()

factor_columns = [
    "b4", "cough_reported", "rapid_breathing_reported",
    "v106", "v190", "v025", "v024",
]

continuous_columns = [
    "child_age_years",
    "maternal_age_5years",
    "bord",
]

encoded = pd.get_dummies(
    fit_frame[factor_columns]
    .astype(int)
    .astype("category"),
    drop_first=True,
    dtype=float,
)

model_matrix = np.column_stack([
    np.ones(len(fit_frame)),
    fit_frame[
        continuous_columns
    ].to_numpy(dtype=float),
    encoded.to_numpy(dtype=float),
])

matrix_rank = int(
    np.linalg.matrix_rank(model_matrix)
)

model_parameters = int(
    model_matrix.shape[1]
)

domain_psus = int(
    raw.loc[eligible, "v021"].nunique()
)

domain_strata = int(
    raw.loc[eligible, "v023"].nunique()
)

expected_regression_df = (
    domain_psus
    - domain_strata
    - (model_parameters - 1)
)


# Outside-domain outcome values are computational placeholders only.
# Pharmacy-first remains undefined for non-care seekers in the frozen dataset.
polars_data = pl.DataFrame({
    column: survey_frame[column]
    .astype(object)
    .where(
        survey_frame[column].notna(),
        None,
    )
    .tolist()
    for column in model_columns
})

stage2_design = svy.Design(
    stratum="v023",
    psu="v021",
    wgt="survey_weight",
)

stage2_sample = svy.Sample(
    data=polars_data,
    design=stage2_design,
)

stage2_predictors = [
    "child_age_years",
    svy.Cat("b4", ref=1),
    svy.Cat("cough_reported", ref=0),
    svy.Cat("rapid_breathing_reported", ref=0),
    "maternal_age_5years",
    svy.Cat("v106", ref=0),
    svy.Cat("v190", ref=1),
    "bord",
    svy.Cat("v025", ref=1),
    svy.Cat("v024", ref=1),
]

with warnings.catch_warnings(
    record=True
) as captured:
    warnings.simplefilter("always")

    stage2_model = stage2_sample.glm.fit(
        y="stage2_pharmacy_first",
        x=stage2_predictors,
        where=svy.col("stage2_eligible") == 1,
        family="binomial",
        link="logit",
    )

model_warnings = [
    f"{warning.category.__name__}: {warning.message}"
    for warning in captured
]


# Extract coefficients using the verified svy 0.37.0 result schema.
coefficients = pd.DataFrame(
    stage2_model.to_polars().to_dicts()
)

required_fields = {
    "term", "est", "se", "lci", "uci",
    "df", "p_value", "family", "link",
}

if not required_fields.issubset(
    coefficients.columns
):
    raise ValueError(
        f"Unexpected svy GLM schema: {coefficients.columns.tolist()}"
    )

for field in (
    "est", "se", "lci", "uci", "df", "p_value"
):
    coefficients[field] = pd.to_numeric(
        coefficients[field],
        errors="coerce",
    )

with np.errstate(
    over="ignore",
    invalid="ignore",
):
    coefficients["adjusted_odds_ratio"] = np.exp(
        coefficients["est"]
    )

    coefficients["or_ci95_lower"] = np.exp(
        coefficients["lci"]
    )

    coefficients["or_ci95_upper"] = np.exp(
        coefficients["uci"]
    )

coefficients.insert(
    0,
    "model_name",
    "Stage 2: pharmacy-first conditional on care",
)

critical_terms = (
    "SEPARAT", "CONVERG", "SINGULAR",
    "COLLINEAR", "ITERATION", "OVERFLOW",
)

critical_warnings = [
    message
    for message in model_warnings
    if any(
        word in message.upper()
        for word in critical_terms
    )
]


# Document the exact analysis populations.
outcome_summary = pd.DataFrame([
    {
        "population": "All febrile children",
        "n": int(fever.sum()),
    },
    {
        "population": "Care seekers",
        "n": int(care.sum()),
    },
    {
        "population": "Stage 2 complete-case care seekers",
        "n": int(eligible.sum()),
    },
    {
        "population": "Pharmacy-first (Stage 2 complete cases)",
        "n": int(
            (eligible & pharmacy_first).sum()
        ),
    },
    {
        "population": "Non-pharmacy-first (Stage 2 complete cases)",
        "n": int(
            (eligible & ~pharmacy_first).sum()
        ),
    },
    {
        "population": (
            "Other/unspecified first source retained in Stage 2"
        ),
        "n": int(
            (eligible & raw["h46a"].eq(96)).sum()
        ),
    },
    {
        "population": (
            "Stage 2 excluded for unknown rapid breathing"
        ),
        "n": int(
            (care & ~eligible).sum()
        ),
    },
])


# Validate cohort composition, design, rank and regression inference.
checks = {
    "Full KR survey design retained":
        len(survey_frame) == 8784,

    "All 674 original PSUs and 16 strata retained":
        (
            survey_frame["v021"].nunique() == 674
            and survey_frame["v023"].nunique() == 16
        ),

    "Frozen Stage 2 eligibility reproduced":
        int(eligible.sum()) == 1969,

    "Stage 2 positives equal 1,117":
        int(
            (eligible & pharmacy_first).sum()
        ) == 1117,

    "Stage 2 non-pharmacy outcomes equal 852":
        int(
            (eligible & ~pharmacy_first).sum()
        ) == 852,

    "Five care seekers excluded for unknown breathing":
        int(
            (care & ~eligible).sum()
        ) == 5,

    "All 33 unspecified care sources retained":
        int(
            (eligible & raw["h46a"].eq(96)).sum()
        ) == 33,

    "Stage 2 design has 611 PSUs and 16 strata":
        domain_psus == 611
        and domain_strata == 16,

    "Fitted design matrix has full rank":
        matrix_rank == model_parameters,

    "Model has 22 coefficients and one intercept":
        (
            len(coefficients) == 22
            and coefficients["term"]
            .str.contains(
                "intercept", case=False
            ).sum() == 1
        ),

    "GLM degrees of freedom follow design residual convention":
        bool(
            coefficients["df"]
            .eq(expected_regression_df).all()
        ),

    "Coefficient estimates and SEs are finite and SEs positive":
        bool(
            np.isfinite(
                coefficients[
                    ["est", "se"]
                ].to_numpy(dtype=float)
            ).all()
            and coefficients["se"].gt(0).all()
        ),

    "OR estimates and intervals are finite and ordered":
        bool(
            np.isfinite(
                coefficients[
                    [
                        "adjusted_odds_ratio",
                        "or_ci95_lower",
                        "or_ci95_upper",
                    ]
                ].to_numpy(dtype=float)
            ).all()
            and coefficients["or_ci95_lower"]
            .le(coefficients["adjusted_odds_ratio"]).all()
            and coefficients["or_ci95_upper"]
            .ge(coefficients["adjusted_odds_ratio"]).all()
        ),

    "P-values are valid and binomial-logit model confirmed":
        bool(
            coefficients["p_value"]
            .between(0, 1).all()
            and coefficients["family"]
            .astype(str).str.lower()
            .eq("binomial").all()
            and coefficients["link"]
            .astype(str).str.lower()
            .eq("logit").all()
        ),

    "No critical model warnings":
        not critical_warnings,
}

check_table = pd.DataFrame([
    {
        "check": name,
        "status": "PASS" if result else "REVIEW",
    }
    for name, result in checks.items()
])

all_passed = all(checks.values())


# Save model results without modifying the original or Stage 1 outputs.
outputs = {
    "coefficients":
        TABLE_DIR / "08c_stage2_adjusted_logistic_coefficients.csv",

    "population":
        TABLE_DIR / "08c_stage2_population_summary.csv",

    "checks":
        TABLE_DIR / "08c_stage2_model_checks.csv",

    "audit":
        LOG_DIR / "08c_stage2_model_audit.json",

    "report":
        LOG_DIR / "08c_stage2_model_report.txt",
}

for name, table in (
    ("coefficients", coefficients),
    ("population", outcome_summary),
    ("checks", check_table),
):
    table.to_csv(
        outputs[name],
        index=False,
        encoding="utf-8-sig",
    )

outputs["audit"].write_text(
    json.dumps(
        {
            "analysis": (
                "Stage 2 conditional pharmacy-first "
                "survey-weighted logistic regression"
            ),
            "svy_version": version("svy"),
            "source_sha256": source_checksum,
            "survey_n": len(survey_frame),
            "model_domain_n": int(eligible.sum()),
            "pharmacy_first_n": int(
                (eligible & pharmacy_first).sum()
            ),
            "non_pharmacy_n": int(
                (eligible & ~pharmacy_first).sum()
            ),
            "domain_psus": domain_psus,
            "domain_strata": domain_strata,
            "expected_coefficient_df": expected_regression_df,
            "model_parameters": model_parameters,
            "matrix_rank": matrix_rank,
            "reference_categories": {
                "sex": "b4=1 male",
                "cough": "h31=0 no",
                "rapid_breathing": "h31b=0 no",
                "education": "v106=0 no education",
                "wealth": "v190=1 poorest",
                "residence": "v025=1 urban",
                "division": "v024=1 Barishal",
            },
            "outcome": (
                "h46a==36 pharmacy first versus all other "
                "reported first-care sources, among care seekers"
            ),
            "other_unspecified_code_96": (
                "Included in primary non-pharmacy group; "
                "excluded only in later sensitivity analysis"
            ),
            "conditional_population_note": (
                "Associations among care seekers; "
                "not effects for all febrile children"
            ),
            "model_terms": coefficients["term"].tolist(),
            "warnings": model_warnings,
            "checks": {
                key: bool(value)
                for key, value in checks.items()
            },
            "validation_passed": bool(all_passed),
        },
        indent=2,
        ensure_ascii=False,
    ) + "\n",
    encoding="utf-8",
)

report = "\n".join([
    "STEP 08.3 | STAGE 2 ADJUSTED PHARMACY-FIRST REGRESSION",
    "=" * 59,
    f"Full KR survey records: {len(survey_frame)}",
    f"Full design PSUs: {survey_frame['v021'].nunique()}",
    f"Full design strata: {survey_frame['v023'].nunique()}",
    f"Stage 2 eligible care seekers: {int(eligible.sum())}",
    f"Pharmacy first: {int((eligible & pharmacy_first).sum())}",
    f"Non-pharmacy first: {int((eligible & ~pharmacy_first).sum())}",
    (
        "Other/unspecified among eligible: "
        f"{int((eligible & raw['h46a'].eq(96)).sum())}"
    ),
    f"Stage 2 active PSUs: {domain_psus}",
    f"Model coefficients: {len(coefficients)}",
    f"Model coefficient degrees of freedom: {expected_regression_df}",
    f"Model warnings: {len(model_warnings)}",
    f"Validation: {'PASS' if all_passed else 'REVIEW REQUIRED'}",
    f"Validated checks: {sum(checks.values())} / {len(checks)}",
])

outputs["report"].write_text(
    report + "\n",
    encoding="utf-8",
)

print(report)

print("\nSTAGE 2 ANALYSIS POPULATION")
display(outcome_summary)

print("\nADJUSTED ODDS RATIOS: PHARMACY FIRST VS NON-PHARMACY FIRST")
display(
    coefficients[
        [
            "term",
            "adjusted_odds_ratio",
            "or_ci95_lower",
            "or_ci95_upper",
            "p_value",
            "df",
        ]
    ].round(4)
)

print("\nMODEL VALIDATION")
display(check_table)

print("\nMODEL WARNINGS")
print(
    "\n".join(model_warnings)
    if model_warnings else "None"
)

print("\nSAVED FILES")
for path in outputs.values():
    print(path)

if not all_passed:
    raise ValueError(
        "Stage 2 regression needs review before adjusted margins."
    )

STEP 08.3 | STAGE 2 ADJUSTED PHARMACY-FIRST REGRESSION
Full KR survey records: 8784
Full design PSUs: 674
Full design strata: 16
Stage 2 eligible care seekers: 1969
Pharmacy first: 1117
Non-pharmacy first: 852
Other/unspecified among eligible: 33
Stage 2 active PSUs: 611
Model coefficients: 22
Model coefficient degrees of freedom: 574
Model warnings: 0
Validation: PASS
Validated checks: 15 / 15

STAGE 2 ANALYSIS POPULATION


,population,n
0,All febrile children,2630
1,Care seekers,1974
2,Stage 2 complete-case care seekers,1969
3,Pharmacy-first (Stage 2 complete cases),1117
4,Non-pharmacy-first (Stage 2 complete cases),852
5,Other/unspecified first source retained in Sta...,33
6,Stage 2 excluded for unknown rapid breathing,5



ADJUSTED ODDS RATIOS: PHARMACY FIRST VS NON-PHARMACY FIRST


,term,adjusted_odds_ratio,or_ci95_lower,or_ci95_upper,p_value,df
0,_intercept_,0.6015,0.2557,1.4149,0.2436,574.0
1,child_age_years,1.2583,1.1595,1.3657,0.0000,574.0
2,b4_2,1.0245,0.8421,1.2464,0.8085,574.0
3,cough_reported_1.0,0.8684,0.7001,1.0772,0.1990,574.0
4,rapid_breathing_reported_1.0,0.6389,0.4801,0.8501,0.0022,574.0
5,maternal_age_5years,0.8068,0.7124,0.9138,0.0008,574.0
6,v106_1,0.8984,0.5253,1.5364,0.6950,574.0
7,v106_2,0.7361,0.4375,1.2386,0.2480,574.0
8,v106_3,0.5547,0.3060,1.0055,0.0521,574.0
9,v190_2,1.0931,0.7421,1.6103,0.6518,574.0



MODEL VALIDATION


,check,status
0,Full KR survey design retained,PASS
1,All 674 original PSUs and 16 strata retained,PASS
2,Frozen Stage 2 eligibility reproduced,PASS
3,"Stage 2 positives equal 1,117",PASS
4,Stage 2 non-pharmacy outcomes equal 852,PASS
5,Five care seekers excluded for unknown breathing,PASS
6,All 33 unspecified care sources retained,PASS
7,Stage 2 design has 611 PSUs and 16 strata,PASS
8,Fitted design matrix has full rank,PASS
9,Model has 22 coefficients and one intercept,PASS



MODEL WARNINGS
None

SAVED FILES
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\08c_stage2_adjusted_logistic_coefficients.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\08c_stage2_population_summary.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\08c_stage2_model_checks.csv
D:\BDHS2022_FeverCare_SpatialEquity\logs\08c_stage2_model_audit.json
D:\BDHS2022_FeverCare_SpatialEquity\logs\08c_stage2_model_report.txt


In [18]:

from pathlib import Path
from importlib.metadata import version
import json

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path(r"D:\BDHS2022_FeverCare_SpatialEquity")
TABLE_DIR = PROJECT_ROOT / "results" / "tables"
LOG_DIR = PROJECT_ROOT / "logs"

stage2_coefficients_path = (
    TABLE_DIR / "08c_stage2_adjusted_logistic_coefficients.csv"
)

stage2_audit_path = LOG_DIR / "08c_stage2_model_audit.json"
stage1_margins_audit_path = LOG_DIR / "08b_stage1_margins_audit.json"

for path in (
    stage2_coefficients_path,
    stage2_audit_path,
    stage1_margins_audit_path,
):
    if not path.is_file():
        raise FileNotFoundError(
            f"Required verified result not found: {path}"
        )

if (
    "stage2_model" not in globals()
    or not hasattr(stage2_model, "margins")
):
    raise RuntimeError(
        "The fitted Stage 2 model is unavailable. "
        "Re-run Cell 8.3 in this kernel."
    )

svy_version = version("svy")

if svy_version != "0.37.0":
    raise RuntimeError(
        f"Expected svy version 0.37.0; found {svy_version}."
    )

with stage2_audit_path.open(encoding="utf-8") as file:
    stage2_audit = json.load(file)

with stage1_margins_audit_path.open(encoding="utf-8") as file:
    stage1_margins_audit = json.load(file)

if not (
    stage2_audit.get("validation_passed")
    and stage1_margins_audit.get("validation_passed")
):
    raise ValueError(
        "Earlier Stage 1 margins or Stage 2 regression "
        "validation has not passed."
    )


# Verify that the active model matches the saved Stage 2 regression.
saved = pd.read_csv(stage2_coefficients_path)

active = pd.DataFrame(
    stage2_model.to_polars().to_dicts()
)

matched = saved[
    ["term", "est", "se", "lci", "uci"]
].merge(
    active[
        ["term", "est", "se", "lci", "uci"]
    ],
    on="term",
    how="outer",
    validate="one_to_one",
    indicator=True,
    suffixes=("_saved", "_active"),
)

matching_model = bool(
    len(matched) == 22
    and matched["_merge"].eq("both").all()
    and all(
        np.allclose(
            matched[f"{field}_saved"],
            matched[f"{field}_active"],
            rtol=0,
            atol=1e-10,
        )
        for field in ("est", "se", "lci", "uci")
    )
)

if not matching_model:
    raise ValueError(
        "The active Stage 2 model differs from the "
        "saved verified regression coefficients."
    )


# Standardize pharmacy-first probabilities over complete-case care seekers.
wealth_raw = pd.DataFrame(
    stage2_model.margins(
        at={"v190": [1, 2, 3, 4, 5]}
    ).to_polars().to_dicts()
)

education_raw = pd.DataFrame(
    stage2_model.margins(
        at={"v106": [0, 1, 2, 3]}
    ).to_polars().to_dicts()
)

contrasts_raw = pd.DataFrame(
    stage2_model.margins(
        variables=["v190", "v106"]
    ).to_polars().to_dicts()
)


# Accommodate the installed svy result schema without guessing estimates.
def estimate_column(frame, extra, context):
    required = {
        "term", "se", "lci", "uci", "df", *extra
    }

    missing = required - set(frame.columns)

    if missing:
        raise ValueError(
            f"{context}: missing {sorted(missing)}; "
            f"got {list(frame.columns)}"
        )

    for name in ("est", "margin"):
        if name in frame.columns:
            return name

    raise ValueError(
        f"{context}: no estimate column; "
        f"got {list(frame.columns)}"
    )


wealth_est = estimate_column(
    wealth_raw,
    {"v190"},
    "Wealth predictive margins",
)

education_est = estimate_column(
    education_raw,
    {"v106"},
    "Education predictive margins",
)

contrast_est = estimate_column(
    contrasts_raw,
    {"contrast"},
    "Categorical contrasts",
)

wealth_labels = {
    1: "Poorest",
    2: "Poorer",
    3: "Middle",
    4: "Richer",
    5: "Richest",
}

education_labels = {
    0: "No education",
    1: "Primary",
    2: "Secondary",
    3: "Higher",
}


# Convert model predictions to percentage-point reporting units.
def format_margins(raw_table, variable, names, est_col):
    table = raw_table.copy()

    level = pd.to_numeric(
        table[variable],
        errors="coerce",
    )

    if (
        level.isna().any()
        or not np.allclose(level, np.round(level))
    ):
        raise ValueError(
            f"Unexpected {variable} categories in model margins."
        )

    table["level_code"] = level.astype(int)

    table["level_label"] = (
        table["level_code"].map(names)
    )

    table["adjusted_percent"] = (
        100 * pd.to_numeric(
            table[est_col], errors="coerce"
        )
    )

    table["se_percentage_points"] = (
        100 * pd.to_numeric(
            table["se"], errors="coerce"
        )
    )

    table["ci95_lower"] = (
        100 * pd.to_numeric(
            table["lci"], errors="coerce"
        )
    )

    table["ci95_upper"] = (
        100 * pd.to_numeric(
            table["uci"], errors="coerce"
        )
    )

    table["design_df"] = pd.to_numeric(
        table["df"],
        errors="coerce",
    )

    return table[
        [
            "term",
            "level_code",
            "level_label",
            "adjusted_percent",
            "se_percentage_points",
            "ci95_lower",
            "ci95_upper",
            "design_df",
        ]
    ].sort_values(
        "level_code"
    ).reset_index(drop=True)


wealth = format_margins(
    wealth_raw,
    "v190",
    wealth_labels,
    wealth_est,
)

education = format_margins(
    education_raw,
    "v106",
    education_labels,
    education_est,
)


# Retain the package's joint-covariance contrast standard errors.
contrasts = contrasts_raw.copy()

contrasts["package_contrast"] = (
    contrasts["contrast"].astype("string")
)

parsed = contrasts[
    "package_contrast"
].str.extract(
    r"^\s*(\d+)\s*-\s*(\d+)\s*$"
)

contrasts["level_code"] = pd.to_numeric(
    parsed[0],
    errors="coerce",
).astype("Int64")

contrasts["reference_code"] = pd.to_numeric(
    parsed[1],
    errors="coerce",
).astype("Int64")

for source, target in (
    (contrast_est, "risk_difference_pp"),
    ("se", "se_pp"),
    ("lci", "ci95_lower_pp"),
    ("uci", "ci95_upper_pp"),
):
    contrasts[target] = (
        100 * pd.to_numeric(
            contrasts[source],
            errors="coerce",
        )
    )

contrasts["design_df"] = pd.to_numeric(
    contrasts["df"],
    errors="coerce",
)

contrasts["comparison"] = pd.Series(
    pd.NA,
    index=contrasts.index,
    dtype="string",
)

contrasts["margin_difference_pp"] = np.nan

matching_issues = []


# Verify every contrast against its associated predictive-margin difference.
for variable, predictions, labels, reference in (
    ("v190", wealth, wealth_labels, 1),
    ("v106", education, education_labels, 0),
):
    margins_by_level = predictions.set_index(
        "level_code"
    )["adjusted_percent"]

    rows = contrasts.index[
        contrasts["term"].eq(variable)
    ]

    for row_id in rows:
        code = contrasts.loc[
            row_id, "level_code"
        ]

        ref = contrasts.loc[
            row_id, "reference_code"
        ]

        invalid = (
            pd.isna(code)
            or pd.isna(ref)
        )

        if not invalid:
            invalid = (
                int(ref) != reference
                or int(code) not in labels
                or int(code) == reference
                or int(code) not in margins_by_level.index
                or reference not in margins_by_level.index
            )

        if invalid:
            matching_issues.append(
                f"{variable}: unexpected contrast at row {row_id}"
            )
            continue

        code = int(code)

        expected = float(
            margins_by_level.loc[code]
            - margins_by_level.loc[reference]
        )

        observed = float(
            contrasts.loc[
                row_id, "risk_difference_pp"
            ]
        )

        if not np.isclose(
            observed,
            expected,
            atol=1e-6,
            rtol=0,
        ):
            matching_issues.append(
                f"{variable} ({code} - {reference}): "
                f"observed {observed:.8f}, "
                f"expected {expected:.8f}"
            )

        contrasts.loc[
            row_id, "comparison"
        ] = (
            f"{labels[code]} - {labels[reference]}"
        )

        contrasts.loc[
            row_id, "margin_difference_pp"
        ] = expected


contrasts = contrasts[
    [
        "term",
        "level_code",
        "comparison",
        "package_contrast",
        "risk_difference_pp",
        "se_pp",
        "ci95_lower_pp",
        "ci95_upper_pp",
        "design_df",
        "margin_difference_pp",
    ]
].sort_values(
    ["term", "level_code"]
).reset_index(drop=True)

primary_contrasts = contrasts.loc[
    contrasts["comparison"].isin(
        [
            "Richest - Poorest",
            "Higher - No education",
        ]
    )
].copy()


# Audit the model margins, reference contrasts and inferential values.
all_margins = pd.concat(
    [wealth, education],
    ignore_index=True,
)

required_contrast_pairs = {
    ("v190", 2),
    ("v190", 3),
    ("v190", 4),
    ("v190", 5),
    ("v106", 1),
    ("v106", 2),
    ("v106", 3),
}

actual_pairs = set(
    zip(
        contrasts["term"],
        contrasts["level_code"],
    )
)

checks = {
    "Active Stage 2 GLM matches the saved verified fit":
        matching_model,

    "Analysis sample remains 1,969 care seekers":
        int(stage2_audit["model_domain_n"]) == 1969,

    "Five wealth predictive margins returned":
        len(wealth) == 5,

    "Four maternal-education predictive margins returned":
        len(education) == 4,

    "Seven categorical contrasts returned":
        len(contrasts) == 7,

    "Wealth levels match the prespecified categories":
        set(wealth["level_code"]) == set(wealth_labels),

    "Education levels match the prespecified categories":
        set(education["level_code"]) == set(education_labels),

    "Margin terms agree with their respective variables":
        bool(
            wealth["term"].eq("v190").all()
            and education["term"].eq("v106").all()
        ),

    "All prespecified reference contrasts present exactly once":
        (
            actual_pairs == required_contrast_pairs
            and not contrasts.duplicated(
                ["term", "level_code"]
            ).any()
        ),

    "All discrete contrasts equal their predicted probability differences":
        (
            len(matching_issues) == 0
            and contrasts["comparison"].notna().all()
        ),

    "Both main socioeconomic comparisons are present":
        len(primary_contrasts) == 2,

    "All estimates and confidence limits are finite":
        bool(
            np.isfinite(
                all_margins[
                    [
                        "adjusted_percent",
                        "se_percentage_points",
                        "ci95_lower",
                        "ci95_upper",
                    ]
                ].to_numpy(dtype=float)
            ).all()
            and np.isfinite(
                contrasts[
                    [
                        "risk_difference_pp",
                        "se_pp",
                        "ci95_lower_pp",
                        "ci95_upper_pp",
                    ]
                ].to_numpy(dtype=float)
            ).all()
        ),

    "All standard errors are strictly positive":
        bool(
            all_margins[
                "se_percentage_points"
            ].gt(0).all()
            and contrasts["se_pp"].gt(0).all()
        ),

    "Adjusted probabilities lie between zero and 100 percent":
        bool(
            all_margins[
                "adjusted_percent"
            ].between(0, 100).all()
        ),

    "All confidence intervals contain the corresponding estimate":
        bool(
            all_margins["ci95_lower"]
            .le(all_margins["adjusted_percent"]).all()
            and all_margins["ci95_upper"]
            .ge(all_margins["adjusted_percent"]).all()
            and contrasts["ci95_lower_pp"]
            .le(contrasts["risk_difference_pp"]).all()
            and contrasts["ci95_upper_pp"]
            .ge(contrasts["risk_difference_pp"]).all()
        ),

    "Model degrees of freedom match the verified Stage 2 fit":
        bool(
            all_margins["design_df"].eq(574).all()
            and contrasts["design_df"].eq(574).all()
        ),
}

check_table = pd.DataFrame([
    {
        "check": name,
        "status": "PASS" if valid else "REVIEW",
    }
    for name, valid in checks.items()
])

all_passed = all(checks.values())


# Save formatted and original package outputs with a reproducible audit.
outputs = {
    "wealth":
        TABLE_DIR / "08d_stage2_wealth_adjusted_margins.csv",

    "education":
        TABLE_DIR / "08d_stage2_education_adjusted_margins.csv",

    "contrasts":
        TABLE_DIR / "08d_stage2_adjusted_risk_differences.csv",

    "raw_wealth":
        TABLE_DIR / "08d_raw_wealth_margins.csv",

    "raw_education":
        TABLE_DIR / "08d_raw_education_margins.csv",

    "raw_contrasts":
        TABLE_DIR / "08d_raw_marginal_contrasts.csv",

    "checks":
        TABLE_DIR / "08d_stage2_margins_validation.csv",

    "audit":
        LOG_DIR / "08d_stage2_margins_audit.json",

    "report":
        LOG_DIR / "08d_stage2_margins_report.txt",
}

for name, table in (
    ("wealth", wealth),
    ("education", education),
    ("contrasts", contrasts),
    ("raw_wealth", wealth_raw),
    ("raw_education", education_raw),
    ("raw_contrasts", contrasts_raw),
    ("checks", check_table),
):
    table.to_csv(
        outputs[name],
        index=False,
        encoding="utf-8-sig",
    )

report = "\n".join([
    "STEP 08.4 | STAGE 2 STANDARDIZED MARGINS AND CONTRASTS",
    "=" * 62,
    "Outcome: pharmacy first among children who sought fever care",
    "Fitted domain: 1,969 complete-case care seekers",
    f"Survey package: svy {svy_version}",
    f"Wealth-adjusted margins: {len(wealth)}",
    f"Education-adjusted margins: {len(education)}",
    f"Categorical marginal contrasts: {len(contrasts)}",
    "Uncertainty: survey GLM coefficient-covariance delta method",
    "Source type 96 remains included in non-pharmacy comparator",
    "Interpretation: conditional associations; not causal effects",
    f"Validation: {'PASS' if all_passed else 'REVIEW REQUIRED'}",
    f"Validated checks: {sum(checks.values())} / {len(checks)}",
])

outputs["report"].write_text(
    report + "\n",
    encoding="utf-8",
)

outputs["audit"].write_text(
    json.dumps(
        {
            "model": (
                "Stage 2 pharmacy-first "
                "survey-weighted logistic GLM"
            ),
            "svy_version": svy_version,
            "model_domain_n": int(
                stage2_audit["model_domain_n"]
            ),
            "model_df": 574,
            "margin_schema": {
                "wealth": wealth_est,
                "education": education_est,
                "contrasts": contrast_est,
            },
            "uncertainty_note": (
                "Delta-method standard errors from fitted "
                "survey GLM coefficient covariance; "
                "they condition on the observed weighted "
                "covariate distribution. Stratified PSU "
                "bootstrap is planned as an inferential "
                "sensitivity analysis."
            ),
            "comparison_note": (
                "Differences are between standardized "
                "pharmacy-first probabilities among care seekers, "
                "not among all febrile children."
            ),
            "contrast_matching_issues": matching_issues,
            "checks": {
                key: bool(value)
                for key, value in checks.items()
            },
            "validation_passed": bool(all_passed),
        },
        indent=2,
        ensure_ascii=False,
    ) + "\n",
    encoding="utf-8",
)

print(report)

print("\nWEALTH-ADJUSTED PHARMACY-FIRST PROBABILITIES")
display(wealth.round(4))

print("\nEDUCATION-ADJUSTED PHARMACY-FIRST PROBABILITIES")
display(education.round(4))

print("\nPRIMARY SOCIOECONOMIC CONTRASTS")
display(primary_contrasts.round(4))

print("\nALL ADJUSTED ABSOLUTE DIFFERENCES")
display(contrasts.round(4))

print("\nVALIDATION CHECKS")
display(check_table)

if matching_issues:
    print("\nCONTRAST VERIFICATION ISSUES")
    for issue in matching_issues:
        print(issue)

print("\nSAVED FILES")
for path in outputs.values():
    print(path)

if not all_passed:
    raise ValueError(
        "Stage 2 adjusted margin audit requires review "
        "before comparison with Stage 1."
    )

STEP 08.4 | STAGE 2 STANDARDIZED MARGINS AND CONTRASTS
Outcome: pharmacy first among children who sought fever care
Fitted domain: 1,969 complete-case care seekers
Survey package: svy 0.37.0
Wealth-adjusted margins: 5
Education-adjusted margins: 4
Categorical marginal contrasts: 7
Uncertainty: survey GLM coefficient-covariance delta method
Source type 96 remains included in non-pharmacy comparator
Interpretation: conditional associations; not causal effects
Validation: PASS
Validated checks: 16 / 16

WEALTH-ADJUSTED PHARMACY-FIRST PROBABILITIES


,term,level_code,level_label,adjusted_percent,se_percentage_points,ci95_lower,ci95_upper,design_df
0,v190,1,Poorest,57.3301,3.1959,51.0530,63.6072,574.0
1,v190,2,Poorer,59.3739,2.9356,53.6081,65.1397,574.0
2,v190,3,Middle,59.8463,2.7113,54.5210,65.1716,574.0
3,v190,4,Richer,58.7397,3.1060,52.6392,64.8402,574.0
4,v190,5,Richest,51.8702,3.5470,44.9036,58.8369,574.0



EDUCATION-ADJUSTED PHARMACY-FIRST PROBABILITIES


,term,level_code,level_label,adjusted_percent,se_percentage_points,ci95_lower,ci95_upper,design_df
0,v106,0,No education,64.0822,5.4833,53.3123,74.8520,574.0
1,v106,1,Primary,61.7006,2.6445,56.5065,66.8947,574.0
2,v106,2,Secondary,57.1351,1.7274,53.7424,60.5279,574.0
3,v106,3,Higher,50.4621,3.8978,42.8063,58.1178,574.0



PRIMARY SOCIOECONOMIC CONTRASTS


,term,level_code,comparison,package_contrast,risk_difference_pp,se_pp,ci95_lower_pp,ci95_upper_pp,design_df,margin_difference_pp
2,v106,3,Higher - No education,3 - 0,-13.6201,6.8335,-27.0418,-0.1984,574.0,-13.6201
6,v190,5,Richest - Poorest,5 - 1,-5.4599,5.1385,-15.5524,4.6327,574.0,-5.4599



ALL ADJUSTED ABSOLUTE DIFFERENCES


,term,level_code,comparison,package_contrast,risk_difference_pp,se_pp,ci95_lower_pp,ci95_upper_pp,design_df,margin_difference_pp
0,v106,1,Primary - No education,1 - 0,-2.3816,6.0188,-14.2032,9.4401,574.0,-2.3816
1,v106,2,Secondary - No education,2 - 0,-6.9470,5.8466,-18.4305,4.5364,574.0,-6.9470
2,v106,3,Higher - No education,3 - 0,-13.6201,6.8335,-27.0418,-0.1984,574.0,-13.6201
3,v190,2,Poorer - Poorest,2 - 1,2.0438,4.5277,-6.8492,10.9367,574.0,2.0438
4,v190,3,Middle - Poorest,3 - 1,2.5162,4.1896,-5.7126,10.7450,574.0,2.5162
5,v190,4,Richer - Poorest,4 - 1,1.4096,4.3582,-7.1503,9.9694,574.0,1.4096
6,v190,5,Richest - Poorest,5 - 1,-5.4599,5.1385,-15.5524,4.6327,574.0,-5.4599



VALIDATION CHECKS


,check,status
0,Active Stage 2 GLM matches the saved verified fit,PASS
1,"Analysis sample remains 1,969 care seekers",PASS
2,Five wealth predictive margins returned,PASS
3,Four maternal-education predictive margins ret...,PASS
4,Seven categorical contrasts returned,PASS
5,Wealth levels match the prespecified categories,PASS
6,Education levels match the prespecified catego...,PASS
7,Margin terms agree with their respective varia...,PASS
8,All prespecified reference contrasts present e...,PASS
9,All discrete contrasts equal their predicted p...,PASS



SAVED FILES
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\08d_stage2_wealth_adjusted_margins.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\08d_stage2_education_adjusted_margins.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\08d_stage2_adjusted_risk_differences.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\08d_raw_wealth_margins.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\08d_raw_education_margins.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\08d_raw_marginal_contrasts.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\08d_stage2_margins_validation.csv
D:\BDHS2022_FeverCare_SpatialEquity\logs\08d_stage2_margins_audit.json
D:\BDHS2022_FeverCare_SpatialEquity\logs\08d_stage2_margins_report.txt


In [19]:

from pathlib import Path
import json

import numpy as np
import pandas as pd
from scipy.stats import t
from IPython.display import display

PROJECT_ROOT = Path(r"D:\BDHS2022_FeverCare_SpatialEquity")
TABLE_DIR = PROJECT_ROOT / "results" / "tables"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
LOG_DIR = PROJECT_ROOT / "logs"

inputs = {
    "cohort": PROCESSED_DIR / "06_fever_care_analysis_ready.csv",
    "unadjusted": TABLE_DIR / "07_unadjusted_subgroup_estimates.csv",
    "stage1_wealth": TABLE_DIR / "08b_stage1_wealth_adjusted_margins.csv",
    "stage1_education": TABLE_DIR / "08b_stage1_education_adjusted_margins.csv",
    "stage1_contrasts": TABLE_DIR / "08b_stage1_adjusted_risk_differences.csv",
    "stage2_wealth": TABLE_DIR / "08d_stage2_wealth_adjusted_margins.csv",
    "stage2_education": TABLE_DIR / "08d_stage2_education_adjusted_margins.csv",
    "stage2_contrasts": TABLE_DIR / "08d_stage2_adjusted_risk_differences.csv",
    "stage1_model_audit": LOG_DIR / "08a_stage1_model_audit.json",
    "stage1_margins_audit": LOG_DIR / "08b_stage1_margins_audit.json",
    "stage2_model_audit": LOG_DIR / "08c_stage2_model_audit.json",
    "stage2_margins_audit": LOG_DIR / "08d_stage2_margins_audit.json",
}

for name, path in inputs.items():
    if not path.is_file():
        raise FileNotFoundError(
            f"Required verified input not found ({name}): {path}"
        )

previous_audits = {}

for name in (
    "stage1_model_audit",
    "stage1_margins_audit",
    "stage2_model_audit",
    "stage2_margins_audit",
):
    with inputs[name].open(encoding="utf-8") as handle:
        previous_audits[name] = json.load(handle)

    if not previous_audits[name].get("validation_passed", False):
        raise ValueError(
            f"Previous validation did not pass: {name}"
        )


# Reconcile the final analytic populations.
cohort = pd.read_csv(
    inputs["cohort"],
    usecols=[
        "care_sought",
        "h46a",
        "stage1_core_complete_case",
        "stage2_core_complete_case",
    ],
)

unadjusted = pd.read_csv(inputs["unadjusted"])

stage_specs = (
    (
        "Stage 1",
        "Care sought among febrile children",
        2624,
        605,
    ),
    (
        "Stage 2",
        "Pharmacy first among care seekers",
        1969,
        574,
    ),
)

category_sets = {
    "v190": {
        1: "Poorest",
        2: "Poorer",
        3: "Middle",
        4: "Richer",
        5: "Richest",
    },
    "v106": {
        0: "No education",
        1: "Primary",
        2: "Secondary",
        3: "Higher",
    },
}

reference_levels = {
    "v190": 1,
    "v106": 0,
}

margin_parts = []
contrast_parts = []


# Load the verified adjusted margins and contrasts from each stage.
for stage, estimand, model_n, expected_df in stage_specs:

    prefix = "stage1" if stage == "Stage 1" else "stage2"

    for group_name, variable in (
        ("Wealth quintile", "v190"),
        ("Maternal education", "v106"),
    ):
        suffix = (
            "wealth" if variable == "v190" else "education"
        )

        source_key = f"{prefix}_{suffix}"
        frame = pd.read_csv(inputs[source_key]).copy()

        expected_fields = {
            "term",
            "level_code",
            "level_label",
            "adjusted_percent",
            "se_percentage_points",
            "ci95_lower",
            "ci95_upper",
            "design_df",
        }

        if not expected_fields.issubset(frame.columns):
            raise ValueError(
                f"Unexpected margin schema for {source_key}: "
                f"{frame.columns.tolist()}"
            )

        frame.insert(0, "stage", stage)
        frame.insert(1, "estimand", estimand)
        frame.insert(2, "model_n", model_n)
        frame.insert(3, "characteristic", group_name)

        margin_parts.append(frame)

    source_key = f"{prefix}_contrasts"
    frame = pd.read_csv(inputs[source_key]).copy()

    expected_fields = {
        "term",
        "level_code",
        "comparison",
        "risk_difference_pp",
        "se_pp",
        "ci95_lower_pp",
        "ci95_upper_pp",
        "design_df",
        "margin_difference_pp",
    }

    if not expected_fields.issubset(frame.columns):
        raise ValueError(
            f"Unexpected contrast schema for {source_key}: "
            f"{frame.columns.tolist()}"
        )

    frame.insert(0, "stage", stage)
    frame.insert(1, "estimand", estimand)
    frame.insert(2, "model_n", model_n)

    contrast_parts.append(frame)

margins = pd.concat(
    margin_parts,
    ignore_index=True,
)

contrasts = pd.concat(
    contrast_parts,
    ignore_index=True,
)

margins["level_code"] = pd.to_numeric(
    margins["level_code"],
    errors="raise",
).astype(int)

contrasts["level_code"] = pd.to_numeric(
    contrasts["level_code"],
    errors="raise",
).astype(int)


# Reconstruct contrasts within their respective model populations.
reference_margins = margins.loc[
    margins.apply(
        lambda row: (
            row["level_code"]
            == reference_levels[row["term"]]
        ),
        axis=1,
    ),
    ["stage", "term", "adjusted_percent"],
].rename(
    columns={
        "adjusted_percent": "reference_adjusted_percent"
    }
)

contrasts = contrasts.merge(
    margins[
        [
            "stage",
            "term",
            "level_code",
            "adjusted_percent",
        ]
    ],
    on=["stage", "term", "level_code"],
    how="left",
    validate="one_to_one",
).merge(
    reference_margins,
    on=["stage", "term"],
    how="left",
    validate="many_to_one",
)

contrasts["recomputed_risk_difference_pp"] = (
    contrasts["adjusted_percent"]
    - contrasts["reference_adjusted_percent"]
)

contrasts["risk_difference_reconstruction_error_pp"] = (
    contrasts["risk_difference_pp"]
    - contrasts["recomputed_risk_difference_pp"]
)


# Derive two-sided t-based p-values from verified contrast estimates.
contrasts["t_statistic"] = (
    contrasts["risk_difference_pp"] / contrasts["se_pp"]
)

contrasts["unadjusted_p_derived"] = (
    2
    * t.sf(
        np.abs(contrasts["t_statistic"]),
        contrasts["design_df"],
    )
)


# Identify the four previously specified substantive comparisons.
focal_levels = (
    ("Stage 1", "v190", 5),
    ("Stage 1", "v106", 3),
    ("Stage 2", "v190", 5),
    ("Stage 2", "v106", 3),
)

focal = pd.concat(
    [
        contrasts.loc[
            contrasts["stage"].eq(stage)
            & contrasts["term"].eq(variable)
            & contrasts["level_code"].eq(level)
        ]
        for stage, variable, level in focal_levels
    ],
    ignore_index=True,
)


# Examine sensitivity to testing four contrasts using Holm adjustment.
# This correction is exploratory and was not preregistered.
p_values = focal[
    "unadjusted_p_derived"
].to_numpy(dtype=float)

order = np.argsort(p_values)

ranked = np.minimum(
    1.0,
    (
        len(p_values) - np.arange(len(p_values))
    ) * p_values[order],
)

ranked = np.maximum.accumulate(ranked)

adjusted = np.empty(
    len(p_values),
    dtype=float,
)

adjusted[order] = ranked

focal["holm_four_contrasts_p_sensitivity"] = adjusted

focal["unadjusted_ci_excludes_zero"] = (
    focal["ci95_lower_pp"].gt(0)
    | focal["ci95_upper_pp"].lt(0)
)

focal["holm_sensitivity_p_below_005"] = (
    focal["holm_four_contrasts_p_sensitivity"].lt(0.05)
)


# Compare unadjusted descriptive results with adjusted probabilities.
# The unadjusted and adjusted estimates use different eligibility sets.
subgroup = unadjusted.loc[
    unadjusted["characteristic"].isin(
        ["Wealth quintile", "Maternal education"]
    ),
    [
        "characteristic",
        "category",
        "fever_n",
        "care_seeker_n",
        "care_sought_percent",
        "pharmacy_first_given_care_percent",
    ],
].copy()

subgroup["match_label"] = (
    subgroup["category"].str.strip().str.casefold()
)

side_by_side = margins.pivot_table(
    index=[
        "term",
        "level_code",
        "level_label",
        "characteristic",
    ],
    columns="stage",
    values="adjusted_percent",
    aggfunc="first",
).reset_index().rename(
    columns={
        "Stage 1": "stage1_adjusted_care_sought_percent",
        "Stage 2": "stage2_adjusted_pharmacy_given_care_percent",
    }
)

side_by_side["match_label"] = (
    side_by_side["level_label"].str.strip().str.casefold()
)

side_by_side = side_by_side.merge(
    subgroup,
    on=["characteristic", "match_label"],
    how="left",
    validate="one_to_one",
    indicator=True,
)

side_by_side = side_by_side.rename(
    columns={
        "care_sought_percent":
            "stage1_unadjusted_care_sought_percent",
        "pharmacy_first_given_care_percent":
            "stage2_unadjusted_pharmacy_given_care_percent",
    }
).drop(
    columns="match_label"
)


population_flow = pd.DataFrame([
    {
        "population": "Fever-reported children",
        "n": int(len(cohort)),
    },
    {
        "population": "Fever care seekers",
        "n": int(cohort["care_sought"].eq(1).sum()),
    },
    {
        "population": "Stage 1 core complete cases",
        "n": int(
            cohort["stage1_core_complete_case"].sum()
        ),
    },
    {
        "population": "Stage 2 core complete cases",
        "n": int(
            cohort["stage2_core_complete_case"].sum()
        ),
    },
    {
        "population":
            "Unspecified source among Stage 2 complete cases",
        "n": int(
            (
                cohort["stage2_core_complete_case"].eq(1)
                & cohort["h46a"].eq(96)
            ).sum()
        ),
    },
])


# Validate the combined results against the source analyses.
checks = {
    "All four prior model and margins audits passed":
        all(
            entry["validation_passed"]
            for entry in previous_audits.values()
        ),

    "Original analytic cohort and care-seeker counts retained":
        (
            len(cohort) == 2630
            and int(cohort["care_sought"].sum()) == 1974
        ),

    "Stage-specific complete-case denominators retained":
        (
            int(
                cohort["stage1_core_complete_case"].sum()
            ) == 2624
            and int(
                cohort["stage2_core_complete_case"].sum()
            ) == 1969
        ),

    "All 33 unspecified first sources remain in Stage 2":
        int(
            (
                cohort["stage2_core_complete_case"].eq(1)
                & cohort["h46a"].eq(96)
            ).sum()
        ) == 33,

    "Eighteen stage-by-category adjusted margins retained":
        len(margins) == 18,

    "Fourteen adjusted contrasts retained":
        len(contrasts) == 14,

    "Exactly four focal socioeconomic contrasts identified":
        len(focal) == 4,

    "All four focal contrasts unique":
        not focal.duplicated(
            ["stage", "term", "level_code"]
        ).any(),

    "Every stage and predictor has its intended levels":
        all(
            set(
                margins.loc[
                    margins["stage"].eq(stage)
                    & margins["term"].eq(term),
                    "level_code",
                ]
            ) == set(levels)
            for stage, _, _, _ in stage_specs
            for term, levels in category_sets.items()
        ),

    "Stage 1 and Stage 2 inferential degrees of freedom retained":
        all(
            margins.loc[
                margins["stage"].eq(stage),
                "design_df",
            ].eq(df).all()
            and contrasts.loc[
                contrasts["stage"].eq(stage),
                "design_df",
            ].eq(df).all()
            for stage, _, _, df in stage_specs
        ),

    "All contrasts reconstruct from their own stage margins":
        bool(
            np.isfinite(
                contrasts[
                    "risk_difference_reconstruction_error_pp"
                ]
            ).all()
            and contrasts[
                "risk_difference_reconstruction_error_pp"
            ].abs().max() < 1e-4
        ),

    "All adjusted probabilities and standard errors are valid":
        bool(
            margins["adjusted_percent"].between(0, 100).all()
            and margins["se_percentage_points"].gt(0).all()
            and contrasts["se_pp"].gt(0).all()
        ),

    "All confidence limits contain corresponding estimates":
        bool(
            (
                margins["ci95_lower"]
                <= margins["adjusted_percent"]
            ).all()
            and (
                margins["adjusted_percent"]
                <= margins["ci95_upper"]
            ).all()
            and (
                contrasts["ci95_lower_pp"]
                <= contrasts["risk_difference_pp"]
            ).all()
            and (
                contrasts["risk_difference_pp"]
                <= contrasts["ci95_upper_pp"]
            ).all()
        ),

    "All derived p-values are finite and within zero to one":
        bool(
            np.isfinite(
                contrasts["unadjusted_p_derived"]
            ).all()
            and contrasts[
                "unadjusted_p_derived"
            ].between(0, 1).all()
        ),

    "Holm sensitivity p-values are valid and not smaller than raw":
        bool(
            np.isfinite(
                focal["holm_four_contrasts_p_sensitivity"]
            ).all()
            and focal[
                "holm_four_contrasts_p_sensitivity"
            ].between(0, 1).all()
            and (
                focal["holm_four_contrasts_p_sensitivity"]
                >= focal["unadjusted_p_derived"] - 1e-12
            ).all()
        ),

    "Nine unadjusted socioeconomic categories match the margins":
        bool(
            len(side_by_side) == 9
            and side_by_side["_merge"].eq("both").all()
            and not side_by_side.duplicated(
                ["characteristic", "level_code"]
            ).any()
        ),
}

check_table = pd.DataFrame([
    {
        "check": name,
        "status": "PASS" if result else "REVIEW",
    }
    for name, result in checks.items()
])

all_passed = all(checks.values())


# Retain the verified source outputs and save the new synthesis separately.
outputs = {
    "margins":
        TABLE_DIR / "08e_two_stage_adjusted_margins.csv",

    "contrasts":
        TABLE_DIR / "08e_two_stage_adjusted_contrasts.csv",

    "focal":
        TABLE_DIR / "08e_focal_contrast_multiplicity_sensitivity.csv",

    "descriptives":
        TABLE_DIR / "08e_unadjusted_adjusted_descriptives.csv",

    "population":
        TABLE_DIR / "08e_analysis_population_flow.csv",

    "checks":
        TABLE_DIR / "08e_two_stage_synthesis_checks.csv",

    "audit":
        LOG_DIR / "08e_two_stage_synthesis_audit.json",

    "report":
        LOG_DIR / "08e_two_stage_synthesis_report.txt",
}

for key, table in (
    ("margins", margins),
    ("contrasts", contrasts),
    ("focal", focal),
    ("descriptives", side_by_side),
    ("population", population_flow),
    ("checks", check_table),
):
    table.to_csv(
        outputs[key],
        index=False,
        encoding="utf-8-sig",
    )

report = "\n".join([
    "STEP 08.5 | TWO-STAGE SOCIOECONOMIC RESULTS SYNTHESIS",
    "=" * 60,
    f"Fever cohort: {len(cohort)}",
    f"Care seekers: {int(cohort['care_sought'].sum())}",
    "Stage 1 (care initiation) model: 2,624 children",
    "Stage 2 (pharmacy first conditional on care) model: 1,969 children",
    f"Combined adjusted margins: {len(margins)}",
    f"Combined adjusted contrasts: {len(contrasts)}",
    f"Focal contrasts: {len(focal)}",
    "Inference: distinct conditional populations; no direct subtraction across stages",
    "Multiplicity: exploratory Holm sensitivity across four focal contrasts",
    "Uncertainty: model-covariance delta method; design-resampling verification pending",
    f"Validation: {'PASS' if all_passed else 'REVIEW REQUIRED'}",
    f"Validated checks: {sum(checks.values())} / {len(checks)}",
])

outputs["report"].write_text(
    report + "\n",
    encoding="utf-8",
)

outputs["audit"].write_text(
    json.dumps(
        {
            "step": "08.5",
            "analysis_populations": {
                "stage1_n": 2624,
                "stage2_n": 1969,
            },
            "estimand_guardrail": (
                "Stage 1 averages over fever complete cases; "
                "Stage 2 averages over care-seeker complete cases. "
                "Stage-specific margins are not directly subtractable."
            ),
            "multiplicity_note": (
                "Holm adjustment across four focal "
                "stage-by-predictor contrasts is an exploratory "
                "sensitivity introduced at synthesis; "
                "not a preregistered primary inferential procedure."
            ),
            "further_checks": [
                "Joint omnibus tests by socioeconomic construct",
                "Design-resampling sensitivity",
                "Specified-source Stage 2 sensitivity",
                "Geospatial analysis",
            ],
            "checks": {
                key: bool(value)
                for key, value in checks.items()
            },
            "validation_passed": bool(all_passed),
        },
        indent=2,
    ) + "\n",
    encoding="utf-8",
)

print(report)

print("\nSTAGE-SPECIFIC SOCIOECONOMIC CONTRASTS")
display(
    focal[
        [
            "stage",
            "comparison",
            "risk_difference_pp",
            "ci95_lower_pp",
            "ci95_upper_pp",
            "unadjusted_p_derived",
            "holm_four_contrasts_p_sensitivity",
        ]
    ].round(4)
)

print("\nUNADJUSTED VS ADJUSTED SOCIOECONOMIC DESCRIPTIVES")
display(
    side_by_side.drop(
        columns="_merge"
    ).round(3)
)

print("\nPOPULATION FLOW")
display(population_flow)

print("\nVALIDATION CHECKS")
display(check_table)

print("\nSAVED FILES")
for path in outputs.values():
    print(path)

if not all_passed:
    raise ValueError(
        "Two-stage socioeconomic synthesis requires review."
    )

STEP 08.5 | TWO-STAGE SOCIOECONOMIC RESULTS SYNTHESIS
Fever cohort: 2630
Care seekers: 1974
Stage 1 (care initiation) model: 2,624 children
Stage 2 (pharmacy first conditional on care) model: 1,969 children
Combined adjusted margins: 18
Combined adjusted contrasts: 14
Focal contrasts: 4
Inference: distinct conditional populations; no direct subtraction across stages
Multiplicity: exploratory Holm sensitivity across four focal contrasts
Uncertainty: model-covariance delta method; design-resampling verification pending
Validation: PASS
Validated checks: 16 / 16

STAGE-SPECIFIC SOCIOECONOMIC CONTRASTS


,stage,comparison,risk_difference_pp,ci95_lower_pp,ci95_upper_pp,unadjusted_p_derived,holm_four_contrasts_p_sensitivity
0,Stage 1,Richest - Poorest,-2.6958,-9.8897,4.4981,0.4621,0.9241
1,Stage 1,Higher - No education,-2.8193,-12.5147,6.8762,0.5682,0.9241
2,Stage 2,Richest - Poorest,-5.4599,-15.5524,4.6327,0.2884,0.8653
3,Stage 2,Higher - No education,-13.6201,-27.0418,-0.1984,0.0467,0.1869



UNADJUSTED VS ADJUSTED SOCIOECONOMIC DESCRIPTIVES


,term,level_code,level_label,characteristic,stage1_adjusted_care_sought_percent,stage2_adjusted_pharmacy_given_care_percent,category,fever_n,care_seeker_n,stage1_unadjusted_care_sought_percent,stage2_unadjusted_pharmacy_given_care_percent
0,v106,0,No education,Maternal education,73.651,64.082,No Education,143,103,73.690,66.240
1,v106,1,Primary,Maternal education,78.829,61.701,Primary,646,502,78.982,63.722
2,v106,2,Secondary,Maternal education,76.702,57.135,Secondary,1434,1096,76.961,57.401
3,v106,3,Higher,Maternal education,70.832,50.462,Higher,407,273,69.655,44.408
4,v190,1,Poorest,Wealth quintile,76.209,57.330,Poorest,601,465,77.721,61.245
5,v190,2,Poorer,Wealth quintile,74.142,59.374,Poorer,540,410,75.344,60.167
6,v190,3,Middle,Wealth quintile,77.723,59.846,Middle,543,420,77.607,59.571
7,v190,4,Richer,Wealth quintile,78.992,58.740,Richer,497,374,77.787,56.897
8,v190,5,Richest,Wealth quintile,73.514,51.870,Richest,449,305,71.626,47.518



POPULATION FLOW


,population,n
0,Fever-reported children,2630
1,Fever care seekers,1974
2,Stage 1 core complete cases,2624
3,Stage 2 core complete cases,1969
4,Unspecified source among Stage 2 complete cases,33



VALIDATION CHECKS


,check,status
0,All four prior model and margins audits passed,PASS
1,Original analytic cohort and care-seeker count...,PASS
2,Stage-specific complete-case denominators reta...,PASS
3,All 33 unspecified first sources remain in Sta...,PASS
4,Eighteen stage-by-category adjusted margins re...,PASS
5,Fourteen adjusted contrasts retained,PASS
6,Exactly four focal socioeconomic contrasts ide...,PASS
7,All four focal contrasts unique,PASS
8,Every stage and predictor has its intended levels,PASS
9,Stage 1 and Stage 2 inferential degrees of fre...,PASS



SAVED FILES
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\08e_two_stage_adjusted_margins.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\08e_two_stage_adjusted_contrasts.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\08e_focal_contrast_multiplicity_sensitivity.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\08e_unadjusted_adjusted_descriptives.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\08e_analysis_population_flow.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\08e_two_stage_synthesis_checks.csv
D:\BDHS2022_FeverCare_SpatialEquity\logs\08e_two_stage_synthesis_audit.json
D:\BDHS2022_FeverCare_SpatialEquity\logs\08e_two_stage_synthesis_report.txt


In [20]:

from pathlib import Path
from importlib.metadata import version
import hashlib
import itertools
import json
import warnings

import numpy as np
import pandas as pd
import polars as pl
import svy
from scipy.stats import f
from IPython.display import display

PROJECT_ROOT = Path(r"D:\BDHS2022_FeverCare_SpatialEquity")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLE_DIR = PROJECT_ROOT / "results" / "tables"
LOG_DIR = PROJECT_ROOT / "logs"

sources = {
    "kr": RAW_DIR / "BDKR81FL.DTA",
    "cohort": PROCESSED_DIR / "06_fever_care_analysis_ready.csv",
    "input_audit": LOG_DIR / "01_input_audit.json",
    "stage1_audit": LOG_DIR / "08a_stage1_model_audit.json",
    "stage2_audit": LOG_DIR / "08c_stage2_model_audit.json",
    "synthesis_audit": LOG_DIR / "08e_two_stage_synthesis_audit.json",
    "stage1_coef": TABLE_DIR / "08a_stage1_adjusted_logistic_coefficients.csv",
    "stage2_coef": TABLE_DIR / "08c_stage2_adjusted_logistic_coefficients.csv",
}

for name, path in sources.items():
    if not path.is_file():
        raise FileNotFoundError(
            f"Required verified input missing ({name}): {path}"
        )

if version("svy") != "0.37.0":
    raise RuntimeError(
        "The analysis requires the validated svy 0.37.0 version."
    )

with sources["input_audit"].open(encoding="utf-8") as handle:
    input_audit = json.load(handle)

with sources["stage1_audit"].open(encoding="utf-8") as handle:
    stage1_audit = json.load(handle)

with sources["stage2_audit"].open(encoding="utf-8") as handle:
    stage2_audit = json.load(handle)

with sources["synthesis_audit"].open(encoding="utf-8") as handle:
    synthesis_audit = json.load(handle)

if not all((
    stage1_audit.get("validation_passed", False),
    stage2_audit.get("validation_passed", False),
    synthesis_audit.get("validation_passed", False),
)):
    raise ValueError(
        "A prerequisite model or synthesis audit has not passed."
    )

with sources["kr"].open("rb") as handle:
    sha256 = hashlib.file_digest(
        handle, "sha256"
    ).hexdigest()

if sha256 != input_audit["source_files"]["kr_sha256"]:
    raise ValueError(
        "The KR source checksum does not match Step 1."
    )


# Reconstruct the verified regression domains from DHS response codes.
keys = ["v001", "v002", "v003", "bidx"]

raw = pd.read_stata(
    sources["kr"],
    columns=keys + [
        "v005", "v021", "v023", "v024", "v025",
        "h22", "h32y", "h46a", "h31", "h31b",
        "b19", "b4", "v012", "v106", "v190", "bord",
    ],
    convert_categoricals=False,
)

frozen = pd.read_csv(
    sources["cohort"],
    usecols=keys + [
        "stage1_core_complete_case",
        "stage2_core_complete_case",
    ],
)

fever = raw["h22"].eq(1)
care = fever & raw["h32y"].eq(0)

complete = (
    raw[
        [
            "b19", "b4", "h31", "v012", "v106",
            "v190", "bord", "v025", "v024",
        ]
    ].notna().all(axis=1)
    & raw["h31b"].isin([0, 1])
)

domains = {
    "Stage 1": fever & complete,
    "Stage 2": care & complete,
}


# Confirm the exact child-level model populations.
for stage, field in (
    ("Stage 1", "stage1_core_complete_case"),
    ("Stage 2", "stage2_core_complete_case"),
):
    saved_keys = (
        frozen.loc[frozen[field].eq(1), keys]
        .astype("int64")
        .sort_values(keys)
        .reset_index(drop=True)
    )

    reconstructed_keys = (
        raw.loc[domains[stage], keys]
        .astype("int64")
        .sort_values(keys)
        .reset_index(drop=True)
    )

    if not saved_keys.equals(reconstructed_keys):
        raise ValueError(
            f"{stage} regression domain differs from Step 6."
        )


# Keep the full sampling design while identifying regression domains.
frame = raw[
    [
        "v021", "v023", "v005", "b4", "v106",
        "v190", "bord", "v025", "v024",
    ]
].copy()

frame["survey_weight"] = (
    frame["v005"].astype(float) / 1_000_000
)

frame["domain_stage1"] = (
    domains["Stage 1"].astype("int8")
)

frame["domain_stage2"] = (
    domains["Stage 2"].astype("int8")
)

frame["out_care_sought"] = care.astype("int8")

frame["out_pharmacy_first"] = (
    care & raw["h46a"].eq(36)
).astype("int8")

frame["child_age_years"] = (
    raw["b19"].astype(float) / 12.0
)

frame["maternal_age_5years"] = (
    raw["v012"].astype(float) - 25.0
) / 5.0

frame["cough_reported"] = (
    raw["h31"].map({0: 0, 2: 1})
)

frame["rapid_breathing_reported"] = (
    raw["h31b"].map({0: 0, 1: 1})
)

columns = [
    "v021", "v023", "survey_weight",
    "domain_stage1", "domain_stage2",
    "out_care_sought", "out_pharmacy_first",
    "child_age_years", "b4", "cough_reported",
    "rapid_breathing_reported", "maternal_age_5years",
    "v106", "v190", "bord", "v025", "v024",
]

if (
    not np.isfinite(frame["survey_weight"]).all()
    or not frame["survey_weight"].gt(0).all()
):
    raise ValueError(
        "Survey weights must be finite and strictly positive."
    )

for domain in domains.values():
    if frame.loc[domain, columns].isna().any().any():
        raise ValueError(
            "A missing model value remains inside a fitted domain."
        )


# Construct the design-aware survey object.
polars_data = pl.DataFrame({
    column: (
        frame[column]
        .astype(object)
        .where(frame[column].notna(), None)
        .tolist()
    )
    for column in columns
})

sample = svy.Sample(
    data=polars_data,
    design=svy.Design(
        stratum="v023",
        psu="v021",
        wgt="survey_weight",
    ),
)

predictors = [
    "child_age_years",
    svy.Cat("b4", ref=1),
    svy.Cat("cough_reported", ref=0),
    svy.Cat("rapid_breathing_reported", ref=0),
    "maternal_age_5years",
    svy.Cat("v106", ref=0),
    svy.Cat("v190", ref=1),
    "bord",
    svy.Cat("v025", ref=1),
    svy.Cat("v024", ref=1),
]

model_specs = [
    (
        "Stage 1",
        "out_care_sought",
        "domain_stage1",
        "stage1_coef",
        2624,
        605,
    ),
    (
        "Stage 2",
        "out_pharmacy_first",
        "domain_stage2",
        "stage2_coef",
        1969,
        574,
    ),
]

blocks = {
    "Wealth quintile": [
        "v190_2", "v190_3", "v190_4", "v190_5",
    ],
    "Maternal education": [
        "v106_1", "v106_2", "v106_3",
    ],
}

joint_rows = []
coefficient_checks = []
contrast_details = []
covariance_details = []
warning_records = []


# Fit both verified models and recover the joint coefficient covariance.
for stage, outcome, domain, saved_name, expected_n, expected_df in model_specs:

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")

        fitted = sample.glm.fit(
            y=outcome,
            x=predictors,
            where=svy.col(domain) == 1,
            family="binomial",
            link="logit",
        )

    for entry in caught:
        warning_records.append({
            "stage": stage,
            "warning": str(entry.message),
        })

    observed = pd.DataFrame(
        fitted.to_polars().to_dicts()
    )

    saved = pd.read_csv(
        sources[saved_name]
    )

    numeric = ["est", "se", "lci", "uci"]

    joined = saved[
        ["term", *numeric, "df"]
    ].merge(
        observed[
            ["term", *numeric, "df"]
        ],
        on="term",
        how="outer",
        validate="one_to_one",
        indicator=True,
        suffixes=("_saved", "_refit"),
    )

    consistent = (
        len(joined) == 22
        and joined["_merge"].eq("both").all()
        and all(
            np.allclose(
                joined[f"{col}_saved"],
                joined[f"{col}_refit"],
                rtol=0,
                atol=2e-8,
            )
            for col in numeric
        )
        and (joined["df_saved"] == expected_df).all()
        and (joined["df_refit"] == expected_df).all()
        and int(domains[stage].sum()) == expected_n
    )

    coefficient_checks.append({
        "stage": stage,
        "exact_model_reproduced": bool(consistent),
        "model_n": int(domains[stage].sum()),
        "coefficient_n": len(observed),
        "residual_df": expected_df,
    })

    if not consistent:
        raise ValueError(
            f"The {stage} GLM did not reproduce verified Step 8 coefficients."
        )

    observed = observed.set_index("term")

    # Public linear contrasts provide sums, differences and their SEs.
    requests = {}

    for term_list in blocks.values():

        for name in term_list:
            requests[f"single__{name}"] = (
                svy.estd(name)
            )

        for a, b in itertools.combinations(
            term_list, 2
        ):
            requests[f"plus__{a}__{b}"] = (
                svy.estd(a) + svy.estd(b)
            )

            requests[f"minus__{a}__{b}"] = (
                svy.estd(a) - svy.estd(b)
            )

    contrasts = pd.DataFrame(
        fitted.contrast(
            requests
        ).to_polars().to_dicts()
    )

    required = {"contrast", "est", "se"}

    if not required.issubset(
        contrasts.columns
    ):
        raise ValueError(
            f"The {stage} contrast schema is incomplete: "
            f"{list(contrasts.columns)}"
        )

    if (
        len(contrasts) != len(requests)
        or contrasts["contrast"].duplicated().any()
    ):
        raise ValueError(
            f"The {stage} coefficient contrast results are incomplete."
        )

    contrasts = contrasts.set_index(
        "contrast"
    )

    if set(contrasts.index) != set(requests):
        raise ValueError(
            f"The {stage} contrast names differ from the requested names."
        )

    contrast_details.append(
        contrasts.reset_index().assign(stage=stage)
    )

    for construct, terms in blocks.items():

        beta = observed.loc[
            terms, "est"
        ].to_numpy(dtype=float)

        stderr = observed.loc[
            terms, "se"
        ].to_numpy(dtype=float)

        covariance = np.diag(
            stderr ** 2
        )

        comparison_errors = []

        # Single-term contrasts must reproduce individual coefficient SEs.
        for name in terms:

            result = contrasts.loc[
                f"single__{name}"
            ]

            comparison_errors.append(
                abs(
                    float(result["est"])
                    - float(observed.loc[name, "est"])
                )
            )

            comparison_errors.append(
                abs(
                    float(result["se"])
                    - float(observed.loc[name, "se"])
                )
            )

        # Reconstruct each off-diagonal covariance two independent ways.
        for i, j in itertools.combinations(
            range(len(terms)), 2
        ):

            a, b = terms[i], terms[j]

            plus = contrasts.loc[
                f"plus__{a}__{b}"
            ]

            minus = contrasts.loc[
                f"minus__{a}__{b}"
            ]

            expected_plus = beta[i] + beta[j]
            expected_minus = beta[i] - beta[j]

            comparison_errors.extend([
                abs(
                    float(plus["est"]) - expected_plus
                ),
                abs(
                    float(minus["est"]) - expected_minus
                ),
            ])

            cov_from_sum = (
                float(plus["se"]) ** 2
                - stderr[i] ** 2
                - stderr[j] ** 2
            ) / 2

            cov_from_difference = (
                stderr[i] ** 2
                + stderr[j] ** 2
                - float(minus["se"]) ** 2
            ) / 2

            comparison_errors.append(
                abs(
                    cov_from_sum - cov_from_difference
                )
            )

            covariance[i, j] = (
                cov_from_sum + cov_from_difference
            ) / 2

            covariance[j, i] = covariance[i, j]

        eigenvalues = np.linalg.eigvalsh(
            covariance
        )

        condition_number = float(
            np.linalg.cond(covariance)
        )

        positive_definite = bool(
            np.isfinite(eigenvalues).all()
            and eigenvalues.min() > 0
        )

        max_error = float(
            max(comparison_errors)
        )

        if (
            not positive_definite
            or max_error > 2e-7
        ):
            raise ValueError(
                f"{stage}, {construct}: covariance reconstruction invalid. "
                f"Minimum eigenvalue={eigenvalues.min()}, "
                f"maximum discrepancy={max_error}."
            )

        # Joint Wald F test for all non-reference coefficients.
        df_num = len(terms)

        statistic = float(
            beta @ np.linalg.solve(
                covariance, beta
            )
        )

        f_stat = statistic / df_num

        p_value = float(
            f.sf(
                f_stat,
                df_num,
                expected_df,
            )
        )

        joint_rows.append({
            "stage": stage,
            "outcome": (
                "Care sought"
                if stage == "Stage 1"
                else "Pharmacy first among care seekers"
            ),
            "construct": construct,
            "hypothesis": (
                "All non-reference category coefficients equal zero"
            ),
            "model_n": expected_n,
            "coefficient_count": df_num,
            "wald_chi2": statistic,
            "wald_f": f_stat,
            "numerator_df": df_num,
            "denominator_df": expected_df,
            "p_value": p_value,
            "min_cov_eigenvalue": float(
                eigenvalues.min()
            ),
            "cov_condition_number": condition_number,
            "max_covariance_reconstruction_error": max_error,
        })

        # Save the covariance values for an independent statistical audit.
        for i, left in enumerate(terms):
            for j, right in enumerate(terms):

                covariance_details.append({
                    "stage": stage,
                    "construct": construct,
                    "term_1": left,
                    "term_2": right,
                    "coefficient_covariance": float(
                        covariance[i, j]
                    ),
                    "coefficient_correlation": float(
                        covariance[i, j]
                        / (stderr[i] * stderr[j])
                    ),
                })


joint = pd.DataFrame(
    joint_rows
)

model_checks = pd.DataFrame(
    coefficient_checks
)

contrast_data = pd.concat(
    contrast_details,
    ignore_index=True,
)

covariance_data = pd.DataFrame(
    covariance_details
)


# Exploratory multiplicity sensitivity across the four omnibus tests.
pvals = joint[
    "p_value"
].to_numpy(dtype=float)

order = np.argsort(
    pvals
)

holm_sorted = np.minimum(
    1.0,
    (
        len(pvals) - np.arange(len(pvals))
    ) * pvals[order],
)

holm_sorted = np.maximum.accumulate(
    holm_sorted
)

holm = np.empty_like(
    pvals
)

holm[order] = holm_sorted

joint[
    "holm_four_omnibus_p_sensitivity"
] = holm


# Verify inference, model reproducibility and covariance integrity.
checks = {
    "Step 8.5 prerequisite audit passed":
        bool(
            synthesis_audit.get("validation_passed")
        ),

    "All 8,784 KR records retained in survey design":
        len(raw) == 8784,

    "All 674 PSUs and 16 strata retained":
        (
            raw["v021"].nunique() == 674
            and raw["v023"].nunique() == 16
        ),

    "Both regression domains match frozen Step 6 cohorts":
        (
            int(domains["Stage 1"].sum()) == 2624
            and int(domains["Stage 2"].sum()) == 1969
        ),

    "Both regressions reproduce saved coefficients and SEs":
        bool(
            model_checks[
                "exact_model_reproduced"
            ].all()
        ),

    "Four construct-by-stage tests available":
        len(joint) == 4,

    "Wealth and education parameter blocks have correct sizes":
        sorted(
            joint["coefficient_count"].tolist()
        ) == [3, 3, 4, 4],

    "All Wald F statistics and p-values are finite":
        bool(
            np.isfinite(
                joint[
                    ["wald_f", "wald_chi2", "p_value"]
                ].to_numpy(dtype=float)
            ).all()
        ),

    "All omnibus p-values are valid":
        bool(
            joint["p_value"].between(0, 1).all()
        ),

    "All joint covariance blocks are positive definite":
        bool(
            joint[
                "min_cov_eigenvalue"
            ].gt(0).all()
        ),

    "Contrast sums and differences reproduce covariance":
        bool(
            joint[
                "max_covariance_reconstruction_error"
            ].lt(2e-7).all()
        ),

    "Correct regression residual degrees of freedom used":
        bool(
            joint.loc[
                joint["stage"].eq("Stage 1"),
                "denominator_df",
            ].eq(605).all()
            and joint.loc[
                joint["stage"].eq("Stage 2"),
                "denominator_df",
            ].eq(574).all()
        ),

    "Exploratory Holm p-values are valid":
        bool(
            joint[
                "holm_four_omnibus_p_sensitivity"
            ].between(0, 1).all()
            and joint[
                "holm_four_omnibus_p_sensitivity"
            ].ge(
                joint["p_value"] - 1e-12
            ).all()
        ),

    "All 50 requested linear contrasts are retained":
        len(contrast_data) == 50,

    "All 50 covariance matrix entries are retained":
        len(covariance_data) == 50,
}

check_table = pd.DataFrame([
    {
        "check": name,
        "status": (
            "PASS" if passed else "REVIEW"
        ),
    }
    for name, passed in checks.items()
])

all_passed = all(
    checks.values()
)


# Save the omnibus estimates and full covariance audit trail.
outputs = {
    "omnibus":
        TABLE_DIR / "08f_two_stage_socioeconomic_omnibus_tests.csv",

    "coefficient_rechecks":
        TABLE_DIR / "08f_regression_reproducibility_checks.csv",

    "linear_contrasts":
        TABLE_DIR / "08f_joint_test_linear_contrasts.csv",

    "covariances":
        TABLE_DIR / "08f_joint_test_covariance_blocks.csv",

    "checks":
        TABLE_DIR / "08f_omnibus_validation_checks.csv",

    "audit":
        LOG_DIR / "08f_socioeconomic_omnibus_audit.json",

    "report":
        LOG_DIR / "08f_socioeconomic_omnibus_report.txt",
}

for label, table in (
    ("omnibus", joint),
    ("coefficient_rechecks", model_checks),
    ("linear_contrasts", contrast_data),
    ("covariances", covariance_data),
    ("checks", check_table),
):
    table.to_csv(
        outputs[label],
        index=False,
        encoding="utf-8-sig",
    )

report = "\n".join([
    "STEP 08.6 | TWO-STAGE SOCIOECONOMIC JOINT WALD TESTS",
    "=" * 60,
    "Method: design-adjusted multivariable Wald F test",
    "Joint covariance: reconstructed from svy GLM linear contrasts",
    "Hypotheses: all wealth coefficients = 0; all education coefficients = 0",
    "Models: Stage 1 (n=2,624), Stage 2 (n=1,969)",
    "Degrees of freedom: Stage 1 = 605; Stage 2 = 574",
    f"Omnibus tests: {len(joint)}",
    "Multiplicity: exploratory Holm sensitivity over four omnibus tests",
    "Interpretation: adjusted conditional associations, not causal effects",
    f"Model warnings: {len(warning_records)}",
    f"Validation: {'PASS' if all_passed else 'REVIEW REQUIRED'}",
    f"Validated checks: {sum(checks.values())} / {len(checks)}",
])

outputs["report"].write_text(
    report + "\n",
    encoding="utf-8",
)

outputs["audit"].write_text(
    json.dumps(
        {
            "svy_version": version("svy"),
            "source_sha256": sha256,
            "wald_test": (
                "F = beta.T @ inv(V) @ beta / q, "
                "with F(q, model residual df) reference"
            ),
            "covariance_source": (
                "svy GLM fit.contrast; "
                "coefficient sums and differences"
            ),
            "model_reproducibility":
                model_checks.to_dict(orient="records"),
            "warnings": warning_records,
            "checks": {
                key: bool(value)
                for key, value in checks.items()
            },
            "validation_passed": bool(all_passed),
        },
        indent=2,
    ) + "\n",
    encoding="utf-8",
)

print(report)

print("\nJOINT SOCIOECONOMIC TESTS")
display(
    joint[
        [
            "stage",
            "construct",
            "model_n",
            "wald_f",
            "numerator_df",
            "denominator_df",
            "p_value",
            "holm_four_omnibus_p_sensitivity",
        ]
    ].round(6)
)

print("\nMODEL REPRODUCIBILITY")
display(model_checks)

print("\nPUBLIC GLM LINEAR CONTRASTS")
display(contrast_data.round(7))

print("\nRECONSTRUCTED COEFFICIENT COVARIANCE BLOCKS")
display(covariance_data.round(7))

print("\nVALIDATION CHECKS")
display(check_table)

print("\nSAVED OUTPUTS")
for path in outputs.values():
    print(path)

if not all_passed:
    raise ValueError(
        "Omnibus inference requires review before resampling sensitivity."
    )

STEP 08.6 | TWO-STAGE SOCIOECONOMIC JOINT WALD TESTS
Method: design-adjusted multivariable Wald F test
Joint covariance: reconstructed from svy GLM linear contrasts
Hypotheses: all wealth coefficients = 0; all education coefficients = 0
Models: Stage 1 (n=2,624), Stage 2 (n=1,969)
Degrees of freedom: Stage 1 = 605; Stage 2 = 574
Omnibus tests: 4
Multiplicity: exploratory Holm sensitivity over four omnibus tests
Interpretation: adjusted conditional associations, not causal effects
Model warnings: 0
Validation: PASS
Validated checks: 15 / 15

JOINT SOCIOECONOMIC TESTS


,stage,construct,model_n,wald_f,numerator_df,denominator_df,p_value,holm_four_omnibus_p_sensitivity
0,Stage 1,Wealth quintile,2624,1.349345,4,605,0.250269,0.500539
1,Stage 1,Maternal education,2624,2.082710,3,605,0.101330,0.332885
2,Stage 2,Wealth quintile,1969,0.970278,4,574,0.423218,0.500539
3,Stage 2,Maternal education,1969,2.234095,3,574,0.083221,0.332885



MODEL REPRODUCIBILITY


,stage,exact_model_reproduced,model_n,coefficient_n,residual_df
0,Stage 1,True,2624,22,605
1,Stage 2,True,1969,22,574



PUBLIC GLM LINEAR CONTRASTS


,contrast,y_var,est,se,lci,uci,cv,t,df,p_value,param,variance,alpha,stage
0,single__v190_2,None,-0.115958,0.167493,-0.444896,0.212980,-1.444425,-0.692317,605.0,0.489004,None,GLM,0.05,Stage 1
1,single__v190_3,None,0.089173,0.183089,-0.270395,0.448740,2.053199,0.487045,605.0,0.626403,None,GLM,0.05,Stage 1
2,single__v190_4,None,0.167266,0.184323,-0.194725,0.529257,1.101976,0.907461,605.0,0.364524,None,GLM,0.05,Stage 1
3,single__v190_5,None,-0.150055,0.203408,-0.549526,0.249416,-1.355550,-0.737708,605.0,0.460978,None,GLM,0.05,Stage 1
4,plus__v190_2__v190_3,None,-0.026786,0.310121,-0.635829,0.582258,-11.577934,-0.086371,605.0,0.931200,None,GLM,0.05,Stage 1
5,minus__v190_2__v190_3,None,-0.205131,0.164243,-0.527687,0.117426,-0.800677,-1.248943,605.0,0.212169,None,GLM,0.05,Stage 1
6,plus__v190_2__v190_4,None,0.051308,0.309778,-0.557062,0.659678,6.037594,0.165629,605.0,0.868504,None,GLM,0.05,Stage 1
7,minus__v190_2__v190_4,None,-0.283224,0.167617,-0.612406,0.045958,-0.591817,-1.689710,605.0,0.091599,None,GLM,0.05,Stage 1
8,plus__v190_2__v190_5,None,-0.266014,0.319762,-0.893991,0.361964,-1.202051,-0.831912,605.0,0.405787,None,GLM,0.05,Stage 1
9,minus__v190_2__v190_5,None,0.034097,0.191336,-0.341666,0.409861,5.611449,0.178207,605.0,0.858620,None,GLM,0.05,Stage 1



RECONSTRUCTED COEFFICIENT COVARIANCE BLOCKS


,stage,construct,term_1,term_2,coefficient_covariance,coefficient_correlation
0,Stage 1,Wealth quintile,v190_2,v190_2,0.028054,1.000000
1,Stage 1,Wealth quintile,v190_2,v190_3,0.017300,0.564133
2,Stage 1,Wealth quintile,v190_2,v190_4,0.016967,0.549568
3,Stage 1,Wealth quintile,v190_2,v190_5,0.016410,0.481652
4,Stage 1,Wealth quintile,v190_3,v190_2,0.017300,0.564133
5,Stage 1,Wealth quintile,v190_3,v190_3,0.033522,1.000000
6,Stage 1,Wealth quintile,v190_3,v190_4,0.020766,0.615340
7,Stage 1,Wealth quintile,v190_3,v190_5,0.019820,0.532188
8,Stage 1,Wealth quintile,v190_4,v190_2,0.016967,0.549568
9,Stage 1,Wealth quintile,v190_4,v190_3,0.020766,0.615340



VALIDATION CHECKS


,check,status
0,Step 8.5 prerequisite audit passed,PASS
1,"All 8,784 KR records retained in survey design",PASS
2,All 674 PSUs and 16 strata retained,PASS
3,Both regression domains match frozen Step 6 co...,PASS
4,Both regressions reproduce saved coefficients ...,PASS
5,Four construct-by-stage tests available,PASS
6,Wealth and education parameter blocks have cor...,PASS
7,All Wald F statistics and p-values are finite,PASS
8,All omnibus p-values are valid,PASS
9,All joint covariance blocks are positive definite,PASS



SAVED OUTPUTS
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\08f_two_stage_socioeconomic_omnibus_tests.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\08f_regression_reproducibility_checks.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\08f_joint_test_linear_contrasts.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\08f_joint_test_covariance_blocks.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\08f_omnibus_validation_checks.csv
D:\BDHS2022_FeverCare_SpatialEquity\logs\08f_socioeconomic_omnibus_audit.json
D:\BDHS2022_FeverCare_SpatialEquity\logs\08f_socioeconomic_omnibus_report.txt


In [21]:

from pathlib import Path
from importlib.metadata import version
import hashlib
import json
import warnings

import numpy as np
import pandas as pd
import polars as pl
import svy
from scipy.stats import t
from IPython.display import display

PROJECT_ROOT = Path(r"D:\BDHS2022_FeverCare_SpatialEquity")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLE_DIR = PROJECT_ROOT / "results" / "tables"
LOG_DIR = PROJECT_ROOT / "logs"

inputs = {
    "kr": RAW_DIR / "BDKR81FL.DTA",
    "cohort": PROCESSED_DIR / "06_fever_care_analysis_ready.csv",
    "step1": LOG_DIR / "01_input_audit.json",
    "stage2": LOG_DIR / "08c_stage2_model_audit.json",
    "stage2_margins": LOG_DIR / "08d_stage2_margins_audit.json",
    "omnibus": LOG_DIR / "08f_socioeconomic_omnibus_audit.json",
    "baseline_coefficients": TABLE_DIR / "08c_stage2_adjusted_logistic_coefficients.csv",
    "baseline_wealth": TABLE_DIR / "08d_stage2_wealth_adjusted_margins.csv",
    "baseline_education": TABLE_DIR / "08d_stage2_education_adjusted_margins.csv",
    "baseline_contrasts": TABLE_DIR / "08d_stage2_adjusted_risk_differences.csv",
}

for name, path in inputs.items():
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing verified input ({name}): {path}"
        )

if version("svy") != "0.37.0":
    raise RuntimeError(
        "Validated survey environment requires svy 0.37.0."
    )


def load_audit(path):
    with path.open(encoding="utf-8") as file:
        return json.load(file)


prior = {
    name: load_audit(inputs[name])
    for name in ("step1", "stage2", "stage2_margins", "omnibus")
}

if not all(
    prior[name].get("validation_passed", False)
    for name in ("stage2", "stage2_margins", "omnibus")
):
    raise ValueError(
        "Prerequisite model or omnibus verification has not passed."
    )

with inputs["kr"].open("rb") as file:
    source_checksum = hashlib.file_digest(
        file, "sha256"
    ).hexdigest()

if source_checksum != prior["step1"]["source_files"]["kr_sha256"]:
    raise ValueError(
        "The original KR dataset fails checksum verification."
    )


# Reconstruct the primary Stage 2 population from verified DHS codes.
keys = ["v001", "v002", "v003", "bidx"]

raw = pd.read_stata(
    inputs["kr"],
    columns=keys + [
        "v005", "v021", "v023", "v024", "v025",
        "h22", "h32y", "h46a", "h31", "h31b",
        "b19", "b4", "v012", "v106", "v190", "bord",
    ],
    convert_categoricals=False,
)

frozen = pd.read_csv(
    inputs["cohort"],
    usecols=keys + [
        "stage2_core_complete_case",
        "care_sought",
        "h46a",
    ],
)

fever = raw["h22"].eq(1)
care = fever & raw["h32y"].eq(0)

core_variables = [
    "b19", "b4", "h31", "v012", "v106",
    "v190", "bord", "v025", "v024",
]

complete = (
    raw[core_variables].notna().all(axis=1)
    & raw["h31b"].isin([0, 1])
)

primary_domain = care & complete

specified_domain = (
    primary_domain & raw["h46a"].ne(96)
)

pharmacy = (
    care & raw["h46a"].eq(36)
)


# Verify the primary cohort against frozen child identifiers.
saved_keys = (
    frozen.loc[
        frozen["stage2_core_complete_case"].eq(1),
        keys,
    ]
    .astype("int64")
    .sort_values(keys)
    .reset_index(drop=True)
)

actual_keys = (
    raw.loc[primary_domain, keys]
    .astype("int64")
    .sort_values(keys)
    .reset_index(drop=True)
)

if not saved_keys.equals(actual_keys):
    raise ValueError(
        "Primary Stage 2 eligibility differs from the frozen cohort."
    )

raw_keys = raw.loc[
    fever, keys + ["h46a", "h32y"]
].merge(
    frozen,
    on=keys,
    how="outer",
    validate="one_to_one",
    indicator=True,
    suffixes=("_original", "_frozen"),
)

if (
    len(raw_keys) != 2630
    or not raw_keys["_merge"].eq("both").all()
    or not raw_keys["h46a_original"].fillna(-999).eq(
        raw_keys["h46a_frozen"].fillna(-999)
    ).all()
    or not raw_keys["care_sought"].eq(
        raw_keys["h32y"].eq(0).astype(int)
    ).all()
):
    raise ValueError(
        "The frozen cohort differs from verified first-care responses."
    )


# Retain every KR record in the original complex survey design.
frame = raw[
    [
        "v021", "v023", "v005", "b4", "v106",
        "v190", "bord", "v025", "v024",
    ]
].copy()

frame["survey_weight"] = (
    frame["v005"].astype(float) / 1_000_000
)

frame["stage2_specified_eligible"] = (
    specified_domain.astype("int8")
)

frame["pharmacy_first"] = pharmacy.astype("int8")

frame["child_age_years"] = (
    raw["b19"].astype(float) / 12.0
)

frame["maternal_age_5years"] = (
    raw["v012"].astype(float) - 25.0
) / 5.0

frame["cough_reported"] = (
    raw["h31"].map({0: 0, 2: 1})
)

frame["rapid_breathing_reported"] = (
    raw["h31b"].map({0: 0, 1: 1})
)

model_columns = [
    "v021", "v023", "survey_weight",
    "stage2_specified_eligible",
    "pharmacy_first",
    "child_age_years",
    "b4",
    "cough_reported",
    "rapid_breathing_reported",
    "maternal_age_5years",
    "v106",
    "v190",
    "bord",
    "v025",
    "v024",
]

if frame.loc[
    specified_domain, model_columns
].isna().any().any():
    raise ValueError(
        "Missing model values inside the specified-source domain."
    )

if not (
    np.isfinite(frame["survey_weight"]).all()
    and frame["survey_weight"].gt(0).all()
):
    raise ValueError("Survey weights are invalid.")


# Check model matrix identifiability and domain-specific degrees of freedom.
rank_frame = frame.loc[specified_domain].copy()

factor_columns = [
    "b4", "cough_reported", "rapid_breathing_reported",
    "v106", "v190", "v025", "v024",
]

continuous_columns = [
    "child_age_years",
    "maternal_age_5years",
    "bord",
]

dummies = pd.get_dummies(
    rank_frame[factor_columns]
    .astype(int)
    .astype("category"),
    drop_first=True,
    dtype=float,
)

X = np.column_stack([
    np.ones(len(rank_frame)),
    rank_frame[
        continuous_columns
    ].to_numpy(dtype=float),
    dummies.to_numpy(dtype=float),
])

model_rank = int(np.linalg.matrix_rank(X))
model_parameters = int(X.shape[1])

active_psus = int(
    raw.loc[specified_domain, "v021"].nunique()
)

active_strata = int(
    raw.loc[specified_domain, "v023"].nunique()
)

expected_df = (
    active_psus - active_strata - (model_parameters - 1)
)


# Fit the specified-source sensitivity model using the full survey design.
polars_data = pl.DataFrame({
    name: (
        frame[name]
        .astype(object)
        .where(frame[name].notna(), None)
        .tolist()
    )
    for name in model_columns
})

survey = svy.Sample(
    data=polars_data,
    design=svy.Design(
        stratum="v023",
        psu="v021",
        wgt="survey_weight",
    ),
)

predictors = [
    "child_age_years",
    svy.Cat("b4", ref=1),
    svy.Cat("cough_reported", ref=0),
    svy.Cat("rapid_breathing_reported", ref=0),
    "maternal_age_5years",
    svy.Cat("v106", ref=0),
    svy.Cat("v190", ref=1),
    "bord",
    svy.Cat("v025", ref=1),
    svy.Cat("v024", ref=1),
]

with warnings.catch_warnings(record=True) as collected:
    warnings.simplefilter("always")

    stage2_specified_model = survey.glm.fit(
        y="pharmacy_first",
        x=predictors,
        where=svy.col("stage2_specified_eligible") == 1,
        family="binomial",
        link="logit",
    )

model_warnings = [
    f"{item.category.__name__}: {item.message}"
    for item in collected
]

critical_warning_terms = (
    "SEPARAT",
    "CONVERG",
    "SINGULAR",
    "COLLINEAR",
    "ITERATION",
    "OVERFLOW",
)

critical_warnings = [
    item
    for item in model_warnings
    if any(
        term in item.upper()
        for term in critical_warning_terms
    )
]


# Extract the coefficients with the verified svy 0.37.0 schema.
coefficients = pd.DataFrame(
    stage2_specified_model.to_polars().to_dicts()
)

required_coef = {
    "term", "est", "se", "lci", "uci",
    "df", "p_value", "family", "link",
}

if not required_coef.issubset(coefficients.columns):
    raise ValueError(
        f"Unexpected coefficient schema: {coefficients.columns.tolist()}"
    )

coefficients["adjusted_odds_ratio"] = np.exp(
    coefficients["est"]
)

coefficients["or_ci95_lower"] = np.exp(
    coefficients["lci"]
)

coefficients["or_ci95_upper"] = np.exp(
    coefficients["uci"]
)

baseline_coef = pd.read_csv(
    inputs["baseline_coefficients"]
)

coefficient_comparison = coefficients[
    ["term", "est", "se", "adjusted_odds_ratio"]
].merge(
    baseline_coef[
        ["term", "est", "se", "adjusted_odds_ratio"]
    ],
    on="term",
    suffixes=("_specified", "_primary"),
    how="outer",
    indicator=True,
    validate="one_to_one",
)

coefficient_comparison["coefficient_change"] = (
    coefficient_comparison["est_specified"]
    - coefficient_comparison["est_primary"]
)


# Estimate adjusted probabilities and categorical contrasts.
wealth_raw = pd.DataFrame(
    stage2_specified_model.margins(
        at={"v190": [1, 2, 3, 4, 5]}
    ).to_polars().to_dicts()
)

education_raw = pd.DataFrame(
    stage2_specified_model.margins(
        at={"v106": [0, 1, 2, 3]}
    ).to_polars().to_dicts()
)

contrasts_raw = pd.DataFrame(
    stage2_specified_model.margins(
        variables=["v190", "v106"]
    ).to_polars().to_dicts()
)


def numeric_field(data, label):
    if "est" in data.columns:
        field = "est"
    elif "margin" in data.columns:
        field = "margin"
    else:
        field = None

    required = {"term", "se", "lci", "uci", "df"}

    if field is None or not required.issubset(data.columns):
        raise ValueError(
            f"Unexpected {label} schema: {data.columns.tolist()}"
        )

    return field


def tidy_margin(data, variable, labels):
    field = numeric_field(data, variable)

    if variable not in data.columns:
        raise ValueError(
            f"The {variable} margins lack category codes."
        )

    levels = pd.to_numeric(
        data[variable], errors="raise"
    )

    if not np.allclose(levels, np.round(levels)):
        raise ValueError(
            f"Unexpected noninteger category for {variable}."
        )

    result = pd.DataFrame({
        "term": data["term"],
        "level_code": levels.astype(int),
        "level_label": levels.astype(int).map(labels),
        "adjusted_percent": (
            100 * pd.to_numeric(data[field], errors="raise")
        ),
        "se_percentage_points": (
            100 * pd.to_numeric(data["se"], errors="raise")
        ),
        "ci95_lower": (
            100 * pd.to_numeric(data["lci"], errors="raise")
        ),
        "ci95_upper": (
            100 * pd.to_numeric(data["uci"], errors="raise")
        ),
        "design_df": pd.to_numeric(
            data["df"], errors="raise"
        ),
    })

    return result.sort_values(
        "level_code"
    ).reset_index(drop=True)


wealth_labels = {
    1: "Poorest",
    2: "Poorer",
    3: "Middle",
    4: "Richer",
    5: "Richest",
}

education_labels = {
    0: "No education",
    1: "Primary",
    2: "Secondary",
    3: "Higher",
}

wealth = tidy_margin(
    wealth_raw,
    "v190",
    wealth_labels,
)

education = tidy_margin(
    education_raw,
    "v106",
    education_labels,
)


# Validate the direction of each categorical contrast.
effect_field = numeric_field(
    contrasts_raw,
    "categorical contrasts",
)

if "contrast" not in contrasts_raw.columns:
    raise ValueError(
        "The marginal contrasts lack comparison labels."
    )

parsed = (
    contrasts_raw["contrast"]
    .astype(str)
    .str.extract(r"^\s*(\d+)\s*-\s*(\d+)\s*$")
)

contrasts = pd.DataFrame({
    "term": contrasts_raw["term"].to_numpy(),
    "package_contrast": (
        contrasts_raw["contrast"].astype(str).to_numpy()
    ),
    "level_code": pd.to_numeric(
        parsed[0], errors="coerce"
    ).to_numpy(),
    "reference_code": pd.to_numeric(
        parsed[1], errors="coerce"
    ).to_numpy(),
    "risk_difference_pp": (
        100 * pd.to_numeric(
            contrasts_raw[effect_field], errors="raise"
        ).to_numpy()
    ),
    "se_pp": (
        100 * pd.to_numeric(
            contrasts_raw["se"], errors="raise"
        ).to_numpy()
    ),
    "ci95_lower_pp": (
        100 * pd.to_numeric(
            contrasts_raw["lci"], errors="raise"
        ).to_numpy()
    ),
    "ci95_upper_pp": (
        100 * pd.to_numeric(
            contrasts_raw["uci"], errors="raise"
        ).to_numpy()
    ),
    "design_df": pd.to_numeric(
        contrasts_raw["df"], errors="raise"
    ).to_numpy(),
})

contrasts["comparison"] = pd.Series(
    pd.NA,
    index=contrasts.index,
    dtype="string",
)

contrasts["recomputed_difference_pp"] = np.nan

contrast_discrepancies = []

for variable, margin_table, labels, ref in (
    ("v190", wealth, wealth_labels, 1),
    ("v106", education, education_labels, 0),
):
    predictions = margin_table.set_index(
        "level_code"
    )["adjusted_percent"]

    for row_index in contrasts.index[
        contrasts["term"].eq(variable)
    ]:
        code = contrasts.at[
            row_index, "level_code"
        ]

        reference = contrasts.at[
            row_index, "reference_code"
        ]

        invalid = (
            pd.isna(code)
            or pd.isna(reference)
        )

        if not invalid:
            invalid = (
                int(reference) != ref
                or int(code) not in labels
                or int(code) == ref
            )

        if invalid:
            contrast_discrepancies.append(
                f"{variable}: unknown comparison on row {row_index}"
            )
            continue

        code = int(code)

        expected = float(
            predictions.loc[code] - predictions.loc[ref]
        )

        observed = float(
            contrasts.at[row_index, "risk_difference_pp"]
        )

        contrasts.at[
            row_index, "comparison"
        ] = f"{labels[code]} - {labels[ref]}"

        contrasts.at[
            row_index, "recomputed_difference_pp"
        ] = expected

        if not np.isclose(
            observed,
            expected,
            rtol=0,
            atol=1e-5,
        ):
            contrast_discrepancies.append(
                f"{variable} {code}-{ref}: "
                f"{observed:.7f} vs {expected:.7f}"
            )

contrasts["p_value_exploratory"] = (
    2 * t.sf(
        np.abs(
            contrasts["risk_difference_pp"]
            / contrasts["se_pp"]
        ),
        contrasts["design_df"],
    )
)


# Compare sensitivity and primary estimates descriptively.
margin_baselines = {
    "v190": pd.read_csv(inputs["baseline_wealth"]),
    "v106": pd.read_csv(inputs["baseline_education"]),
}

margin_comparisons = []

for variable, sensitivity in (
    ("v190", wealth),
    ("v106", education),
):
    baseline = margin_baselines[variable]

    joined = baseline[
        [
            "term", "level_code",
            "level_label", "adjusted_percent",
        ]
    ].merge(
        sensitivity[
            [
                "term", "level_code",
                "level_label", "adjusted_percent",
            ]
        ],
        on=[
            "term", "level_code", "level_label"
        ],
        how="outer",
        validate="one_to_one",
        indicator=True,
        suffixes=("_primary", "_specified"),
    )

    joined["change_pp"] = (
        joined["adjusted_percent_specified"]
        - joined["adjusted_percent_primary"]
    )

    margin_comparisons.append(joined)

margin_comparison = pd.concat(
    margin_comparisons,
    ignore_index=True,
)

baseline_contrasts = pd.read_csv(
    inputs["baseline_contrasts"]
)

contrast_comparison = baseline_contrasts[
    [
        "term", "level_code", "comparison",
        "risk_difference_pp",
        "ci95_lower_pp", "ci95_upper_pp",
    ]
].merge(
    contrasts[
        [
            "term", "level_code", "comparison",
            "risk_difference_pp",
            "ci95_lower_pp", "ci95_upper_pp",
            "p_value_exploratory",
        ]
    ],
    on=["term", "level_code", "comparison"],
    how="outer",
    validate="one_to_one",
    indicator=True,
    suffixes=("_primary", "_specified"),
)

contrast_comparison["change_in_contrast_pp"] = (
    contrast_comparison["risk_difference_pp_specified"]
    - contrast_comparison["risk_difference_pp_primary"]
)

focal = contrast_comparison.loc[
    (
        contrast_comparison["term"].eq("v190")
        & contrast_comparison["level_code"].eq(5)
    )
    | (
        contrast_comparison["term"].eq("v106")
        & contrast_comparison["level_code"].eq(3)
    )
].copy()


# Describe socioeconomic distribution of excluded source responses.
source96_rows = []

for variable, labels in (
    ("v190", wealth_labels),
    ("v106", education_labels),
):
    for level, name in labels.items():
        group = (
            primary_domain
            & raw[variable].eq(level)
        )

        unspecified = (
            group & raw["h46a"].eq(96)
        )

        group_weight = float(
            frame.loc[group, "survey_weight"].sum()
        )

        unspecified_weight = float(
            frame.loc[
                unspecified, "survey_weight"
            ].sum()
        )

        source96_rows.append({
            "variable": variable,
            "category_code": level,
            "category_label": name,
            "stage2_group_n": int(group.sum()),
            "unspecified_source_n": int(
                unspecified.sum()
            ),
            "unspecified_source_weighted_pct_within_category": (
                100 * unspecified_weight / group_weight
                if group_weight > 0
                else np.nan
            ),
        })

unspecified_source_patterns = pd.DataFrame(
    source96_rows
)


# Confirm sensitivity sample integrity and model inference.
checks = {
    "Prior Stage 2 and omnibus audits passed":
        all(
            prior[name]["validation_passed"]
            for name in (
                "stage2",
                "stage2_margins",
                "omnibus",
            )
        ),

    "Full KR survey design retained":
        (
            len(raw) == 8784
            and raw["v021"].nunique() == 674
            and raw["v023"].nunique() == 16
        ),

    "Primary complete-case Stage 2 population equals 1,969":
        int(primary_domain.sum()) == 1969,

    "Sensitivity excludes exactly 33 unspecified sources":
        (
            int(
                (
                    primary_domain
                    & ~specified_domain
                ).sum()
            ) == 33
            and raw.loc[
                primary_domain & ~specified_domain,
                "h46a",
            ].eq(96).all()
        ),

    "Specified-source sensitivity has 1,936 children":
        int(specified_domain.sum()) == 1936,

    "Sensitivity has 1,117 pharmacy and 819 other sources":
        (
            int(
                (specified_domain & pharmacy).sum()
            ) == 1117
            and int(
                (
                    specified_domain
                    & ~pharmacy
                ).sum()
            ) == 819
        ),

    "Sensitivity has 610 PSUs and 16 strata":
        (
            active_psus == 610
            and active_strata == 16
        ),

    "Model matrix has 22 independent columns":
        (
            model_parameters == 22
            and model_rank == 22
        ),

    "Sensitivity terms match primary Stage 2":
        (
            len(coefficients) == 22
            and len(coefficient_comparison) == 22
            and coefficient_comparison[
                "_merge"
            ].eq("both").all()
        ),

    "Coefficient degrees of freedom equal 573":
        (
            expected_df == 573
            and coefficients["df"].eq(573).all()
        ),

    "Five wealth and four education margins returned":
        (
            len(wealth) == 5
            and len(education) == 4
        ),

    "Seven categorical contrasts agree with margins":
        (
            len(contrasts) == 7
            and len(contrast_discrepancies) == 0
            and contrasts[
                "comparison"
            ].notna().all()
            and {
                ("v190", 2),
                ("v190", 3),
                ("v190", 4),
                ("v190", 5),
                ("v106", 1),
                ("v106", 2),
                ("v106", 3),
            } == set(
                zip(
                    contrasts["term"],
                    contrasts["level_code"],
                )
            )
        ),

    "Primary and sensitivity results align by category":
        (
            len(margin_comparison) == 9
            and margin_comparison[
                "_merge"
            ].eq("both").all()
            and len(contrast_comparison) == 7
            and contrast_comparison[
                "_merge"
            ].eq("both").all()
            and len(focal) == 2
        ),

    "Estimates and confidence intervals are valid":
        bool(
            np.isfinite(
                coefficients[
                    ["est", "se", "lci", "uci"]
                ].to_numpy(dtype=float)
            ).all()
            and coefficients["se"].gt(0).all()
            and all(
                (
                    np.isfinite(
                        table[
                            [
                                "adjusted_percent",
                                "se_percentage_points",
                                "ci95_lower",
                                "ci95_upper",
                            ]
                        ].to_numpy(dtype=float)
                    ).all()
                    and table[
                        "adjusted_percent"
                    ].between(0, 100).all()
                    and table[
                        "se_percentage_points"
                    ].gt(0).all()
                    and table[
                        "ci95_lower"
                    ].le(
                        table["adjusted_percent"]
                    ).all()
                    and table[
                        "ci95_upper"
                    ].ge(
                        table["adjusted_percent"]
                    ).all()
                )
                for table in (wealth, education)
            )
            and np.isfinite(
                contrasts[
                    [
                        "risk_difference_pp",
                        "se_pp",
                        "ci95_lower_pp",
                        "ci95_upper_pp",
                    ]
                ].to_numpy(dtype=float)
            ).all()
            and contrasts["se_pp"].gt(0).all()
        ),

    "Unspecified source counts reconcile":
        (
            int(
                unspecified_source_patterns.loc[
                    unspecified_source_patterns[
                        "variable"
                    ].eq("v190"),
                    "unspecified_source_n",
                ].sum()
            ) == 33
            and int(
                unspecified_source_patterns.loc[
                    unspecified_source_patterns[
                        "variable"
                    ].eq("v106"),
                    "unspecified_source_n",
                ].sum()
            ) == 33
        ),

    "No separation or nonconvergence warnings":
        len(critical_warnings) == 0,
}

check_table = pd.DataFrame([
    {
        "check": name,
        "status": "PASS" if passed else "REVIEW",
    }
    for name, passed in checks.items()
])

all_passed = all(
    bool(value)
    for value in checks.values()
)


# Save sensitivity estimates and descriptive comparisons.
outputs = {
    "coefficients":
        TABLE_DIR / "08g_stage2_specified_source_coefficients.csv",

    "source96_pattern":
        TABLE_DIR / "08g_unspecified_source_socioeconomic_pattern.csv",

    "wealth":
        TABLE_DIR / "08g_stage2_specified_source_wealth_margins.csv",

    "education":
        TABLE_DIR / "08g_stage2_specified_source_education_margins.csv",

    "contrasts":
        TABLE_DIR / "08g_stage2_specified_source_contrasts.csv",

    "margin_comparison":
        TABLE_DIR / "08g_stage2_source_sensitivity_margin_comparison.csv",

    "contrast_comparison":
        TABLE_DIR / "08g_stage2_source_sensitivity_contrast_comparison.csv",

    "coefficient_comparison":
        TABLE_DIR / "08g_stage2_source_sensitivity_coefficient_comparison.csv",

    "checks":
        TABLE_DIR / "08g_stage2_source_sensitivity_checks.csv",

    "audit":
        LOG_DIR / "08g_stage2_source_sensitivity_audit.json",

    "report":
        LOG_DIR / "08g_stage2_source_sensitivity_report.txt",
}

for label, data in (
    ("coefficients", coefficients),
    ("source96_pattern", unspecified_source_patterns),
    ("wealth", wealth),
    ("education", education),
    ("contrasts", contrasts),
    ("margin_comparison", margin_comparison),
    ("contrast_comparison", contrast_comparison),
    ("coefficient_comparison", coefficient_comparison),
    ("checks", check_table),
):
    data.to_csv(
        outputs[label],
        index=False,
        encoding="utf-8-sig",
    )

report = "\n".join([
    "STEP 08.7 | STAGE 2 SPECIFIED-SOURCE SENSITIVITY",
    "=" * 59,
    f"Primary Stage 2 complete cases: {int(primary_domain.sum())}",
    (
        "Unspecified source (h46a=96) excluded: "
        f"{int((primary_domain & ~specified_domain).sum())}"
    ),
    (
        "Specified-source model observations: "
        f"{int(specified_domain.sum())}"
    ),
    (
        "Pharmacy first: "
        f"{int((specified_domain & pharmacy).sum())}"
    ),
    (
        "Non-pharmacy specified source: "
        f"{int((specified_domain & ~pharmacy).sum())}"
    ),
    f"Active PSUs: {active_psus}",
    f"Strata: {active_strata}",
    f"Model coefficients: {len(coefficients)}",
    f"Model residual design df: {expected_df}",
    "Inference: conditional associations, not causal effects",
    "Comparison: overlapping populations; changes are descriptive",
    f"Warnings: {len(model_warnings)}",
    (
        "Validation: "
        f"{'PASS' if all_passed else 'REVIEW REQUIRED'}"
    ),
    (
        "Validated checks: "
        f"{sum(checks.values())} / {len(checks)}"
    ),
])

outputs["report"].write_text(
    report + "\n",
    encoding="utf-8",
)

audit = {
    "method": (
        "svy Taylor-linearized Stage 2 binomial GLM, "
        "excluding only h46a=96"
    ),
    "svy_version": version("svy"),
    "source_sha256": source_checksum,
    "primary_complete_case_n": int(
        primary_domain.sum()
    ),
    "sensitivity_complete_case_n": int(
        specified_domain.sum()
    ),
    "excluded_96_n": int(
        (primary_domain & ~specified_domain).sum()
    ),
    "active_psus": active_psus,
    "design_df": expected_df,
    "comparison_note": (
        "No significance test of change between overlapping models; "
        "differences are descriptive."
    ),
    "marginal_uncertainty": (
        "Design-based coefficient covariance delta method; "
        "resampling verification pending."
    ),
    "warnings": model_warnings,
    "contrast_discrepancies": contrast_discrepancies,
    "checks": {
        key: bool(value)
        for key, value in checks.items()
    },
    "validation_passed": bool(all_passed),
}

outputs["audit"].write_text(
    json.dumps(
        audit,
        indent=2,
        ensure_ascii=False,
    ) + "\n",
    encoding="utf-8",
)

print(report)

print("\nFOCAL SOCIOECONOMIC CONTRAST SENSITIVITY")
display(
    focal.drop(columns="_merge").round(4)
)

print("\nALL ADJUSTED CONTRAST COMPARISONS")
display(
    contrast_comparison.drop(
        columns="_merge"
    ).round(4)
)

print("\nADJUSTED MARGIN COMPARISONS")
display(
    margin_comparison.drop(
        columns="_merge"
    ).round(4)
)

print("\nDISTRIBUTION OF UNSPECIFIED FIRST-CARE SOURCES")
display(
    unspecified_source_patterns.round(3)
)

print("\nSPECIFIED-SOURCE REGRESSION COEFFICIENTS")
display(
    coefficients[
        [
            "term",
            "adjusted_odds_ratio",
            "or_ci95_lower",
            "or_ci95_upper",
            "p_value",
            "df",
        ]
    ].round(4)
)

print("\nMODEL WARNINGS")
print(
    "\n".join(model_warnings)
    if model_warnings
    else "None"
)

print("\nVALIDATION CHECKS")
display(check_table)

print("\nSAVED FILES")
for path in outputs.values():
    print(path)

if not all_passed:
    raise ValueError(
        "Specified-source sensitivity requires review."
    )

STEP 08.7 | STAGE 2 SPECIFIED-SOURCE SENSITIVITY
Primary Stage 2 complete cases: 1969
Unspecified source (h46a=96) excluded: 33
Specified-source model observations: 1936
Pharmacy first: 1117
Non-pharmacy specified source: 819
Active PSUs: 610
Strata: 16
Model coefficients: 22
Model residual design df: 573
Inference: conditional associations, not causal effects
Comparison: overlapping populations; changes are descriptive
Warnings: 0
Validation: PASS
Validated checks: 16 / 16

FOCAL SOCIOECONOMIC CONTRAST SENSITIVITY


,term,level_code,comparison,risk_difference_pp_primary,ci95_lower_pp_primary,ci95_upper_pp_primary,risk_difference_pp_specified,ci95_lower_pp_specified,ci95_upper_pp_specified,p_value_exploratory,change_in_contrast_pp
2,v106,3,Higher - No education,-13.6201,-27.0418,-0.1984,-15.1464,-28.8261,-1.4668,0.0301,-1.5263
6,v190,5,Richest - Poorest,-5.4599,-15.5524,4.6327,-6.2450,-16.4559,3.9660,0.2302,-0.7851



ALL ADJUSTED CONTRAST COMPARISONS


,term,level_code,comparison,risk_difference_pp_primary,ci95_lower_pp_primary,ci95_upper_pp_primary,risk_difference_pp_specified,ci95_lower_pp_specified,ci95_upper_pp_specified,p_value_exploratory,change_in_contrast_pp
0,v106,1,Primary - No education,-2.3816,-14.2032,9.4401,-4.1908,-15.9067,7.5251,0.4826,-1.8092
1,v106,2,Secondary - No education,-6.9470,-18.4305,4.5364,-9.0257,-20.7060,2.6545,0.1296,-2.0787
2,v106,3,Higher - No education,-13.6201,-27.0418,-0.1984,-15.1464,-28.8261,-1.4668,0.0301,-1.5263
3,v190,2,Poorer - Poorest,2.0438,-6.8492,10.9367,1.1102,-7.8611,10.0814,0.8081,-0.9336
4,v190,3,Middle - Poorest,2.5162,-5.7126,10.7450,1.7610,-6.4941,10.0161,0.6754,-0.7552
5,v190,4,Richer - Poorest,1.4096,-7.1503,9.9694,0.7146,-7.9117,9.3409,0.8708,-0.6949
6,v190,5,Richest - Poorest,-5.4599,-15.5524,4.6327,-6.2450,-16.4559,3.9660,0.2302,-0.7851



ADJUSTED MARGIN COMPARISONS


,term,level_code,level_label,adjusted_percent_primary,adjusted_percent_specified,change_pp
0,v190,1,Poorest,57.3301,58.8644,1.5343
1,v190,2,Poorer,59.3739,59.9746,0.6007
2,v190,3,Middle,59.8463,60.6254,0.7791
3,v190,4,Richer,58.7397,59.5790,0.8394
4,v190,5,Richest,51.8702,52.6194,0.7492
5,v106,0,No education,64.0822,66.8511,2.7690
6,v106,1,Primary,61.7006,62.6604,0.9598
7,v106,2,Secondary,57.1351,57.8254,0.6903
8,v106,3,Higher,50.4621,51.7047,1.2426


... (3 outputs left out)

,term,adjusted_odds_ratio,or_ci95_lower,or_ci95_upper,p_value,df
0,_intercept_,0.7051,0.2908,1.7098,0.4388,573.0
1,child_age_years,1.2459,1.1467,1.3537,0.0000,573.0
2,b4_2,1.0280,0.8429,1.2537,0.7848,573.0
3,cough_reported_1.0,0.8842,0.7136,1.0957,0.2602,573.0
4,rapid_breathing_reported_1.0,0.6315,0.4735,0.8423,0.0018,573.0
5,maternal_age_5years,0.8008,0.7046,0.9101,0.0007,573.0
6,v106_1,0.8249,0.4765,1.4281,0.4912,573.0
7,v106_2,0.6674,0.3864,1.1525,0.1466,573.0
8,v106_3,0.5147,0.2766,0.9576,0.0361,573.0
9,v190_2,1.0498,0.7087,1.5553,0.8080,573.0



MODEL WARNINGS
None

VALIDATION CHECKS


,check,status
0,Prior Stage 2 and omnibus audits passed,PASS
1,Full KR survey design retained,PASS
2,Primary complete-case Stage 2 population equal...,PASS
3,Sensitivity excludes exactly 33 unspecified so...,PASS
4,"Specified-source sensitivity has 1,936 children",PASS
5,"Sensitivity has 1,117 pharmacy and 819 other s...",PASS
6,Sensitivity has 610 PSUs and 16 strata,PASS
7,Model matrix has 22 independent columns,PASS
8,Sensitivity terms match primary Stage 2,PASS
9,Coefficient degrees of freedom equal 573,PASS



SAVED FILES
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\08g_stage2_specified_source_coefficients.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\08g_unspecified_source_socioeconomic_pattern.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\08g_stage2_specified_source_wealth_margins.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\08g_stage2_specified_source_education_margins.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\08g_stage2_specified_source_contrasts.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\08g_stage2_source_sensitivity_margin_comparison.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\08g_stage2_source_sensitivity_contrast_comparison.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\08g_stage2_source_sensitivity_coefficient_comparison.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\08g_stage2_source_sensitivity_checks.csv
D:\BDHS2022_FeverCare_SpatialEquity\logs\08g_stage2_source_sensitivity_audit.json
D:\BDHS20

In [22]:

from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
from IPython.display import display

PROJECT_ROOT = Path(r"D:\BDHS2022_FeverCare_SpatialEquity")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLE_DIR = PROJECT_ROOT / "results" / "tables"
LOG_DIR = PROJECT_ROOT / "logs"

inputs = {
    "kr": RAW_DIR / "BDKR81FL.DTA",
    "cohort": PROCESSED_DIR / "06_fever_care_analysis_ready.csv",
    "gps": PROCESSED_DIR / "02_cluster_gps_lookup.csv",
    "input_audit": LOG_DIR / "01_input_audit.json",
    "gps_audit": LOG_DIR / "02_gps_linkage_audit.json",
    "sensitivity_audit": LOG_DIR / "08g_stage2_source_sensitivity_audit.json",
}

for name, path in inputs.items():
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing required input ({name}): {path}"
        )

with inputs["input_audit"].open(encoding="utf-8") as file:
    input_audit = json.load(file)

with inputs["gps_audit"].open(encoding="utf-8") as file:
    gps_audit = json.load(file)

with inputs["sensitivity_audit"].open(encoding="utf-8") as file:
    sensitivity_audit = json.load(file)

if not (
    all(gps_audit["checks"].values())
    and sensitivity_audit.get("validation_passed", False)
):
    raise ValueError(
        "A required upstream validation has not passed."
    )

with inputs["kr"].open("rb") as file:
    source_checksum = hashlib.file_digest(
        file, "sha256"
    ).hexdigest()

if source_checksum != input_audit["source_files"]["kr_sha256"]:
    raise ValueError(
        "Original KR source checksum verification failed."
    )


# Reconstruct the full descriptive populations and verify child outcomes.
keys = ["v001", "v002", "v003", "bidx"]

raw = pd.read_stata(
    inputs["kr"],
    columns=keys + [
        "v021", "v023", "v024", "v025",
        "v005", "h22", "h32y", "h46a",
    ],
    convert_categoricals=False,
)

cohort = pd.read_csv(
    inputs["cohort"],
    usecols=keys + [
        "v021", "v023", "v024", "v025",
        "v005", "survey_weight",
        "care_sought", "pharmacy_first_if_care",
        "h46a",
    ],
    low_memory=False,
)

gps = pd.read_csv(inputs["gps"])

if raw.duplicated(keys).any() or cohort.duplicated(keys).any():
    raise ValueError(
        "Child identifiers must be unique."
    )

fever_raw = raw.loc[raw["h22"].eq(1)].copy()

verification = fever_raw.merge(
    cohort[
        keys + [
            "care_sought",
            "pharmacy_first_if_care",
        ]
    ],
    on=keys,
    how="outer",
    validate="one_to_one",
    indicator=True,
)

care_raw = verification["h32y"].eq(0)

if (
    len(verification) != 2630
    or not verification["_merge"].eq("both").all()
    or not verification["care_sought"].eq(
        care_raw.astype(int)
    ).all()
    or not verification.loc[
        care_raw, "pharmacy_first_if_care"
    ].eq(
        verification.loc[
            care_raw, "h46a"
        ].eq(36).astype(int)
    ).all()
    or not verification.loc[
        ~care_raw, "pharmacy_first_if_care"
    ].isna().all()
):
    raise ValueError(
        "Frozen outcomes do not reproduce the original KR responses."
    )

if not np.allclose(
    cohort["survey_weight"],
    cohort["v005"] / 1_000_000,
    rtol=0,
    atol=1e-10,
):
    raise ValueError(
        "Survey weights differ from their original definition."
    )


# Validate the PSU-GPS crosswalk before spatial calculations.
psu_design = raw[
    ["v001", "v021", "v023", "v024", "v025"]
].drop_duplicates()

if (
    len(psu_design) != 674
    or not psu_design["v001"].eq(
        psu_design["v021"]
    ).all()
    or psu_design["v001"].duplicated().any()
):
    raise ValueError(
        "Sampling design identifiers are inconsistent."
    )

if len(gps) != 674 or not gps["cluster_id"].is_unique:
    raise ValueError(
        "GPS lookup must contain 674 unique clusters."
    )

psu = psu_design.merge(
    gps.rename(
        columns={"cluster_id": "v001"}
    ),
    on="v001",
    how="left",
    validate="one_to_one",
    indicator=True,
)

gps_valid = bool(
    psu["_merge"].eq("both").all()
    and psu["gps_division"].eq(
        psu["v024"]
    ).all()
    and psu["gps_residence"].eq(
        psu["v025"].map({1: "U", 2: "R"})
    ).all()
    and psu["gps_latitude"].between(
        20, 27
    ).all()
    and psu["gps_longitude"].between(
        88, 93
    ).all()
    and not psu[
        ["gps_latitude", "gps_longitude"]
    ].duplicated().any()
)

if not gps_valid:
    raise ValueError(
        "GPS linkage, division, residence or coordinates are invalid."
    )

psu = psu.drop(columns="_merge")


# Aggregate weighted outcome numerators and denominators by PSU.
w = cohort["survey_weight"].astype(float)
care = cohort["care_sought"].astype(int)

pharmacy = (
    cohort["pharmacy_first_if_care"]
    .fillna(0)
    .astype(int)
)

cohort = cohort.assign(
    w_fever=w,
    w_fever_sq=w * w,
    w_care=w * care,
    w_care_sq=(w * w) * care,
    w_pharmacy_positive=w * pharmacy,
    is_pharmacy=pharmacy,
)

aggregated = cohort.groupby(
    "v001",
    sort=False,
).agg(
    fever_n=("care_sought", "size"),
    care_n=("care_sought", "sum"),
    pharmacy_n=("is_pharmacy", "sum"),
    fever_weight=("w_fever", "sum"),
    fever_weight_sq=("w_fever_sq", "sum"),
    care_weight=("w_care", "sum"),
    care_weight_sq=("w_care_sq", "sum"),
    pharmacy_weight=("w_pharmacy_positive", "sum"),
).reset_index()

psu = psu.merge(
    aggregated,
    on="v001",
    how="left",
    validate="one_to_one",
)

for column in aggregated.columns.drop("v001"):
    psu[column] = psu[column].fillna(0)

for column in (
    "fever_n",
    "care_n",
    "pharmacy_n",
):
    psu[column] = psu[column].astype(int)


def safe_ratio(numerator, denominator):
    a = np.asarray(numerator, dtype=float)
    b = np.asarray(denominator, dtype=float)

    output = np.full(
        b.shape,
        np.nan,
        dtype=float,
    )

    return np.divide(
        a,
        b,
        out=output,
        where=b > 0,
    )


psu["care_rate_weighted"] = safe_ratio(
    psu["care_weight"],
    psu["fever_weight"],
)

psu["pharmacy_given_care_weighted"] = safe_ratio(
    psu["pharmacy_weight"],
    psu["care_weight"],
)

psu["stage1_effective_n"] = safe_ratio(
    psu["fever_weight"] ** 2,
    psu["fever_weight_sq"],
)

psu["stage2_effective_n"] = safe_ratio(
    psu["care_weight"] ** 2,
    psu["care_weight_sq"],
)

pooled_stage1 = float(
    cohort["w_care"].sum()
    / cohort["w_fever"].sum()
)

pooled_stage2 = float(
    cohort["w_pharmacy_positive"].sum()
    / cohort["w_care"].sum()
)


# Stabilise sparse PSU proportions using a fixed pooled-rate prior.
# This is exploratory shrinkage, not a spatial regression model.
SHRINKAGE_STRENGTH = 5.0

for output, rate, effective_n, pooled in (
    (
        "care_rate_pooled_shrink5",
        "care_rate_weighted",
        "stage1_effective_n",
        pooled_stage1,
    ),
    (
        "pharmacy_rate_pooled_shrink5",
        "pharmacy_given_care_weighted",
        "stage2_effective_n",
        pooled_stage2,
    ),
):
    psu[output] = (
        (
            psu[effective_n] * psu[rate]
            + SHRINKAGE_STRENGTH * pooled
        )
        / (
            psu[effective_n]
            + SHRINKAGE_STRENGTH
        )
    )

    psu.loc[
        psu[effective_n].isna(),
        output,
    ] = np.nan


# Construct an undirected, binary k-nearest-neighbour spatial graph.
def knn_graph(data, k):
    locations = np.deg2rad(
        data[
            ["gps_latitude", "gps_longitude"]
        ].to_numpy(dtype=float)
    )

    estimator = NearestNeighbors(
        n_neighbors=k + 1,
        metric="haversine",
        algorithm="ball_tree",
    )

    estimator.fit(locations)

    _, neighbor_ids = estimator.kneighbors(
        locations
    )

    rows = []
    columns = []

    for index, candidates in enumerate(neighbor_ids):
        selected = [
            int(item)
            for item in candidates
            if item != index
        ][:k]

        if len(selected) != k:
            raise ValueError(
                "Incomplete k-nearest-neighbour graph."
            )

        rows.extend(
            [index] * k
        )

        columns.extend(
            selected
        )

    n = len(data)

    directed = csr_matrix(
        (
            np.ones(len(rows)),
            (rows, columns),
        ),
        shape=(n, n),
    )

    graph = (
        directed + directed.T
    ).tocsr()

    graph.data[:] = 1.0

    graph.setdiag(0)
    graph.eliminate_zeros()

    degree = np.diff(
        graph.indptr
    )

    if (
        np.any(degree < k)
        or np.any(degree == 0)
    ):
        raise ValueError(
            "An unexpected isolated or underconnected PSU was found."
        )

    return graph, degree


# Compute Global Moran's I for the PSU-level rates.
def moran_i(values, graph):
    centered = (
        np.asarray(values, dtype=float)
        - float(np.mean(values))
    )

    denominator = float(
        centered @ centered
    )

    if (
        not np.isfinite(denominator)
        or denominator <= 0
    ):
        raise ValueError(
            "PSU outcome rates have insufficient variation."
        )

    n = len(centered)

    return float(
        (n / graph.sum())
        * (
            centered @ (
                graph @ centered
            )
        )
        / denominator
    )


# Report unrestricted and survey-stratum-restricted permutation tests.
# The resulting pseudo-p-values are descriptive, not design-based tests.
def permutation_test(
    values,
    graph,
    strata,
    repetitions,
    seed,
):
    observed = moran_i(
        values,
        graph,
    )

    rng = np.random.default_rng(seed)
    values = np.asarray(values, dtype=float)

    unrestricted = np.empty(repetitions)
    restricted = np.empty(repetitions)

    groups = [
        np.flatnonzero(strata == level)
        for level in np.unique(strata)
    ]

    for iteration in range(repetitions):
        unrestricted[iteration] = moran_i(
            rng.permutation(values),
            graph,
        )

        shuffled = values.copy()

        for indices in groups:
            if len(indices) > 1:
                shuffled[indices] = rng.permutation(
                    values[indices]
                )

        restricted[iteration] = moran_i(
            shuffled,
            graph,
        )

    def summarise(draws):
        null_center = float(draws.mean())

        two_sided_p = (
            1
            + np.sum(
                np.abs(draws - null_center)
                >= abs(observed - null_center)
            )
        ) / (repetitions + 1)

        positive_p = (
            1 + np.sum(draws >= observed)
        ) / (repetitions + 1)

        return {
            "two_sided": float(two_sided_p),
            "positive": float(positive_p),
            "null_mean": null_center,
        }

    return (
        observed,
        summarise(unrestricted),
        summarise(restricted),
    )


# Examine graph specification and small-denominator sensitivity.
N_PERMUTATIONS = 999
BASE_SEED = 20261117

moran_rows = []
coverage_rows = []
graph_rows = []

analysis_specs = [
    (
        "Stage 1 care sought",
        "fever_n",
        "care_rate_weighted",
        "care_rate_pooled_shrink5",
    ),
    (
        "Stage 2 pharmacy first among seekers",
        "care_n",
        "pharmacy_given_care_weighted",
        "pharmacy_rate_pooled_shrink5",
    ),
]

for stage_index, (
    stage_name,
    count_column,
    raw_rate_column,
    shrinkage_column,
) in enumerate(analysis_specs):

    for minimum_n in (1, 3):
        nodes = (
            psu.loc[
                psu[count_column].ge(minimum_n)
            ]
            .sort_values("v001")
            .reset_index(drop=True)
        )

        if len(nodes) < 30:
            raise ValueError(
                "Too few eligible PSUs for exploratory spatial analysis."
            )

        coverage_rows.append({
            "outcome": stage_name,
            "minimum_children_per_psu": minimum_n,
            "psus": len(nodes),
            "children_included": int(
                nodes[count_column].sum()
            ),
            "psus_with_one_child": int(
                nodes[count_column].eq(1).sum()
            ),
            "psus_with_two_children": int(
                nodes[count_column].eq(2).sum()
            ),
            "minimum_n": int(
                nodes[count_column].min()
            ),
            "median_n": float(
                nodes[count_column].median()
            ),
            "maximum_n": int(
                nodes[count_column].max()
            ),
        })

        strata = nodes[
            "v023"
        ].to_numpy(dtype=int)

        for k in (8, 12):
            graph, degree = knn_graph(
                nodes,
                k,
            )

            graph_rows.append({
                "outcome": stage_name,
                "minimum_children_per_psu": minimum_n,
                "k": k,
                "psus": len(nodes),
                "unique_undirected_edges": int(
                    graph.nnz // 2
                ),
                "minimum_degree": int(
                    degree.min()
                ),
                "median_degree": float(
                    np.median(degree)
                ),
                "maximum_degree": int(
                    degree.max()
                ),
                "isolated_nodes": int(
                    np.sum(degree == 0)
                ),
            })

            rate_specs = [
                (
                    "Weighted PSU proportion",
                    raw_rate_column,
                ),
                (
                    "Pooled-rate shrinkage, m=5",
                    shrinkage_column,
                ),
            ]

            for rate_index, (
                rate_label,
                rate_column,
            ) in enumerate(rate_specs):

                values = nodes[
                    rate_column
                ].to_numpy(dtype=float)

                if not np.isfinite(values).all():
                    raise ValueError(
                        f"Invalid spatial outcome: {rate_column}"
                    )

                random_seed = (
                    BASE_SEED
                    + stage_index * 1000
                    + minimum_n * 100
                    + k * 10
                    + rate_index
                )

                observed, unrestricted, restricted = (
                    permutation_test(
                        values=values,
                        graph=graph,
                        strata=strata,
                        repetitions=N_PERMUTATIONS,
                        seed=random_seed,
                    )
                )

                moran_rows.append({
                    "outcome": stage_name,
                    "minimum_children_per_psu": minimum_n,
                    "k": k,
                    "rate_definition": rate_label,
                    "psus": len(nodes),
                    "moran_i": observed,
                    "expected_i_unrestricted": (
                        -1 / (len(nodes) - 1)
                    ),
                    "unrestricted_p_two_sided": (
                        unrestricted["two_sided"]
                    ),
                    "unrestricted_p_positive": (
                        unrestricted["positive"]
                    ),
                    "unrestricted_permutation_mean": (
                        unrestricted["null_mean"]
                    ),
                    "within_stratum_p_two_sided": (
                        restricted["two_sided"]
                    ),
                    "within_stratum_p_positive": (
                        restricted["positive"]
                    ),
                    "within_stratum_permutation_mean": (
                        restricted["null_mean"]
                    ),
                    "permutations": N_PERMUTATIONS,
                })


moran_results = pd.DataFrame(
    moran_rows
)

coverage = pd.DataFrame(
    coverage_rows
)

graph_summary = pd.DataFrame(
    graph_rows
)


# Validate linkage, weighted outcomes and all spatial computations.
checks = {
    "Original KR, GPS and sensitivity audits passed":
        bool(
            all(gps_audit["checks"].values())
            and sensitivity_audit["validation_passed"]
        ),

    "KR and GPS contain all 674 sampling PSUs":
        (
            len(psu) == 674
            and psu["v001"].is_unique
        ),

    "Fever and care-seeker sample sizes preserved":
        (
            len(cohort) == 2630
            and int(cohort["care_sought"].sum()) == 1974
        ),

    "Pharmacy-first and unspecified source counts preserved":
        (
            int(pharmacy.sum()) == 1120
            and int(cohort["h46a"].eq(96).sum()) == 33
        ),

    "Fever and care-seeker PSU counts preserved":
        (
            int(psu["fever_n"].gt(0).sum()) == 642
            and int(psu["care_n"].gt(0).sum()) == 611
        ),

    "Weighted Stage 1 estimate reproduces Step 7":
        abs(
            pooled_stage1 * 100 - 76.2209
        ) < 0.001,

    "Weighted Stage 2 estimate reproduces Step 7":
        abs(
            pooled_stage2 * 100 - 57.7618
        ) < 0.001,

    "GPS values and unique displaced points verified":
        gps_valid,

    "All nonmissing PSU rates lie between zero and one":
        bool(
            psu["care_rate_weighted"]
            .dropna().between(0, 1).all()
            and psu["pharmacy_given_care_weighted"]
            .dropna().between(0, 1).all()
        ),

    "Child counts and weighted totals reconcile":
        bool(
            psu["fever_n"].sum() == 2630
            and psu["care_n"].sum() == 1974
            and psu["pharmacy_n"].sum() == 1120
            and np.isclose(
                psu["fever_weight"].sum(),
                w.sum(),
            )
        ),

    "All 16 Moran sensitivity results returned":
        len(moran_results) == 16,

    "Four stage-by-PSU-size coverage summaries returned":
        len(coverage) == 4,

    "Eight spatial graph summaries returned":
        len(graph_summary) == 8,

    "All Moran estimates and permutation p-values are valid":
        bool(
            np.isfinite(
                moran_results["moran_i"]
            ).all()
            and all(
                moran_results[column]
                .between(0, 1).all()
                for column in (
                    "unrestricted_p_two_sided",
                    "within_stratum_p_two_sided",
                    "unrestricted_p_positive",
                    "within_stratum_p_positive",
                )
            )
        ),

    "No isolated PSUs in any kNN graph":
        bool(
            graph_summary["isolated_nodes"].eq(0).all()
            and graph_summary["minimum_degree"]
            .ge(graph_summary["k"]).all()
        ),

    "Unspecified Stage 2 sources remain in primary analysis":
        bool(
            cohort["care_sought"].sum() == 1974
            and cohort["h46a"].eq(96).sum() == 33
        ),
}

validation = pd.DataFrame([
    {
        "check": name,
        "status": "PASS" if passed else "REVIEW",
    }
    for name, passed in checks.items()
])

all_passed = all(
    bool(value)
    for value in checks.values()
)


# Preserve spatial inputs for later modelling without altering earlier files.
outputs = {
    "psu":
        PROCESSED_DIR / "09a_psu_spatial_descriptive_base.csv",

    "coverage":
        TABLE_DIR / "09a_spatial_psu_sample_coverage.csv",

    "graphs":
        TABLE_DIR / "09a_knn_graph_diagnostics.csv",

    "moran":
        TABLE_DIR / "09a_global_moran_sensitivity.csv",

    "checks":
        TABLE_DIR / "09a_spatial_exploration_checks.csv",

    "audit":
        LOG_DIR / "09a_spatial_exploration_audit.json",

    "report":
        LOG_DIR / "09a_spatial_exploration_report.txt",
}

for name, table in (
    ("psu", psu),
    ("coverage", coverage),
    ("graphs", graph_summary),
    ("moran", moran_results),
    ("checks", validation),
):
    table.to_csv(
        outputs[name],
        index=False,
        encoding="utf-8-sig",
    )

report = "\n".join([
    "STEP 09.1 | SPATIAL COVERAGE AND GLOBAL AUTOCORRELATION",
    "=" * 64,
    f"All KR PSUs: {len(psu)}",
    (
        f"Fever children: {len(cohort)}; "
        f"fever PSUs: {int(psu['fever_n'].gt(0).sum())}"
    ),
    (
        f"Care seekers: {int(cohort['care_sought'].sum())}; "
        f"seeker PSUs: {int(psu['care_n'].gt(0).sum())}"
    ),
    (
        "Weighted care-seeking prevalence: "
        f"{pooled_stage1 * 100:.4f}%"
    ),
    (
        "Weighted pharmacy-first prevalence among care seekers: "
        f"{pooled_stage2 * 100:.4f}%"
    ),
    "Spatial graph: symmetric 8-NN and 12-NN",
    "Rate sensitivity: raw weighted and pooled-rate shrinkage (m=5)",
    "PSU denominator sensitivity: n>=1 versus n>=3",
    f"Permutations per analysis: {N_PERMUTATIONS}",
    "Inference: exploratory, NOT design-adjusted spatial significance",
    "GPS: displaced DHS coordinates; no exact hotspot interpretation",
    f"Validation: {'PASS' if all_passed else 'REVIEW REQUIRED'}",
    (
        "Validated checks: "
        f"{sum(checks.values())} / {len(checks)}"
    ),
])

outputs["report"].write_text(
    report + "\n",
    encoding="utf-8",
)

audit = {
    "step": "09.1",
    "phase": "D - spatial heterogeneity",
    "source_sha256": source_checksum,
    "children_total": int(len(cohort)),
    "care_seekers_total": int(
        cohort["care_sought"].sum()
    ),
    "sampling_weights": "v005 / 1000000",
    "spatial_graph": (
        "Symmetric binary kNN graph using "
        "haversine ordering of displaced DHS GPS points"
    ),
    "k_values": [8, 12],
    "minimum_psu_sizes": [1, 3],
    "shrinkage_prior_strength": SHRINKAGE_STRENGTH,
    "permutations": N_PERMUTATIONS,
    "base_random_seed": BASE_SEED,
    "null_models": [
        "Unrestricted permutation of PSU rates",
        "Permutation restricted within survey strata v023",
    ],
    "limitations": [
        "PSU-level fever and care-seeking denominators are often small.",
        "Permutation pseudo-p-values are exploratory, not full survey-design inference.",
        "Stratum-restricted permutation does not adjust for all individual covariates.",
        "DHS GPS displacement limits fine-scale spatial interpretation.",
        "The present tests are not covariate-adjusted residual autocorrelation tests.",
        "Outcome-derived PSU rates must never enter held-out-PSU machine-learning features.",
    ],
    "checks": {
        name: bool(passed)
        for name, passed in checks.items()
    },
    "validation_passed": bool(all_passed),
}

outputs["audit"].write_text(
    json.dumps(
        audit,
        indent=2,
        ensure_ascii=False,
    ) + "\n",
    encoding="utf-8",
)

print(report)

print("\nPSU SAMPLE COVERAGE")
display(coverage)

print("\nGLOBAL MORAN'S I — SPATIAL SENSITIVITY")
display(
    moran_results[
        [
            "outcome",
            "minimum_children_per_psu",
            "k",
            "rate_definition",
            "psus",
            "moran_i",
            "unrestricted_p_two_sided",
            "within_stratum_p_two_sided",
        ]
    ].round(5)
)

print("\nNEIGHBOUR GRAPH DIAGNOSTICS")
display(graph_summary)

print("\nVALIDATION CHECKS")
display(validation)

print("\nSAVED FILES")
for path in outputs.values():
    print(path)

if not all_passed:
    raise ValueError(
        "Spatial exploration requires review before spatial modelling."
    )

STEP 09.1 | SPATIAL COVERAGE AND GLOBAL AUTOCORRELATION
All KR PSUs: 674
Fever children: 2630; fever PSUs: 642
Care seekers: 1974; seeker PSUs: 611
Weighted care-seeking prevalence: 76.2209%
Weighted pharmacy-first prevalence among care seekers: 57.7618%
Spatial graph: symmetric 8-NN and 12-NN
Rate sensitivity: raw weighted and pooled-rate shrinkage (m=5)
PSU denominator sensitivity: n>=1 versus n>=3
Permutations per analysis: 999
Inference: exploratory, NOT design-adjusted spatial significance
GPS: displaced DHS coordinates; no exact hotspot interpretation
Validation: PASS
Validated checks: 16 / 16

PSU SAMPLE COVERAGE


,outcome,minimum_children_per_psu,psus,children_included,psus_with_one_child,psus_with_two_children,minimum_n,median_n,maximum_n
0,Stage 1 care sought,1,642,2630,80,105,1,4.0,19
1,Stage 1 care sought,3,457,2340,0,0,3,5.0,19
2,Stage 2 pharmacy first among seekers,1,611,1974,123,139,1,3.0,14
3,Stage 2 pharmacy first among seekers,3,349,1573,0,0,3,4.0,14



GLOBAL MORAN'S I — SPATIAL SENSITIVITY


,outcome,minimum_children_per_psu,k,rate_definition,psus,moran_i,unrestricted_p_two_sided,within_stratum_p_two_sided
0,Stage 1 care sought,1,8,Weighted PSU proportion,642,0.04036,0.023,0.147
1,Stage 1 care sought,1,8,"Pooled-rate shrinkage, m=5",642,0.06849,0.001,0.008
2,Stage 1 care sought,1,12,Weighted PSU proportion,642,0.02834,0.047,0.340
3,Stage 1 care sought,1,12,"Pooled-rate shrinkage, m=5",642,0.05027,0.005,0.017
4,Stage 1 care sought,3,8,Weighted PSU proportion,457,0.08169,0.001,0.007
5,Stage 1 care sought,3,8,"Pooled-rate shrinkage, m=5",457,0.08591,0.001,0.002
6,Stage 1 care sought,3,12,Weighted PSU proportion,457,0.05472,0.004,0.033
7,Stage 1 care sought,3,12,"Pooled-rate shrinkage, m=5",457,0.05808,0.001,0.030
8,Stage 2 pharmacy first among seekers,1,8,Weighted PSU proportion,611,0.01639,0.325,0.522
9,Stage 2 pharmacy first among seekers,1,8,"Pooled-rate shrinkage, m=5",611,0.01008,0.522,0.706



NEIGHBOUR GRAPH DIAGNOSTICS


,outcome,minimum_children_per_psu,k,psus,unique_undirected_edges,minimum_degree,median_degree,maximum_degree,isolated_nodes
0,Stage 1 care sought,1,8,642,3175,8,9.0,16,0
1,Stage 1 care sought,1,12,642,4735,12,14.0,24,0
2,Stage 1 care sought,3,8,457,2294,8,10.0,17,0
3,Stage 1 care sought,3,12,457,3410,12,14.0,25,0
4,Stage 2 pharmacy first among seekers,1,8,611,3027,8,10.0,16,0
5,Stage 2 pharmacy first among seekers,1,12,611,4504,12,14.0,24,0
6,Stage 2 pharmacy first among seekers,3,8,349,1742,8,10.0,16,0
7,Stage 2 pharmacy first among seekers,3,12,349,2580,12,14.0,24,0



VALIDATION CHECKS


,check,status
0,"Original KR, GPS and sensitivity audits passed",PASS
1,KR and GPS contain all 674 sampling PSUs,PASS
2,Fever and care-seeker sample sizes preserved,PASS
3,Pharmacy-first and unspecified source counts p...,PASS
4,Fever and care-seeker PSU counts preserved,PASS
5,Weighted Stage 1 estimate reproduces Step 7,PASS
6,Weighted Stage 2 estimate reproduces Step 7,PASS
7,GPS values and unique displaced points verified,PASS
8,All nonmissing PSU rates lie between zero and one,PASS
9,Child counts and weighted totals reconcile,PASS



SAVED FILES
D:\BDHS2022_FeverCare_SpatialEquity\data\processed\09a_psu_spatial_descriptive_base.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\09a_spatial_psu_sample_coverage.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\09a_knn_graph_diagnostics.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\09a_global_moran_sensitivity.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\09a_spatial_exploration_checks.csv
D:\BDHS2022_FeverCare_SpatialEquity\logs\09a_spatial_exploration_audit.json
D:\BDHS2022_FeverCare_SpatialEquity\logs\09a_spatial_exploration_report.txt


In [23]:

from pathlib import Path
import json
import hashlib

import numpy as np
import pandas as pd
from scipy.special import expit
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
from IPython.display import display

ROOT = Path(r"D:\BDHS2022_FeverCare_SpatialEquity")
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
TABLES = ROOT / "results" / "tables"
LOGS = ROOT / "logs"

inputs = {
    "kr": RAW / "BDKR81FL.DTA",
    "psu_base": PROCESSED / "09a_psu_spatial_descriptive_base.csv",
    "stage1_coef": TABLES / "08a_stage1_adjusted_logistic_coefficients.csv",
    "stage2_coef": TABLES / "08c_stage2_adjusted_logistic_coefficients.csv",
    "input_audit": LOGS / "01_input_audit.json",
    "stage1_audit": LOGS / "08a_stage1_model_audit.json",
    "stage2_audit": LOGS / "08c_stage2_model_audit.json",
    "step9a_audit": LOGS / "09a_spatial_exploration_audit.json",
}

for label, path in inputs.items():
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing verified input ({label}): {path}"
        )

with inputs["input_audit"].open(encoding="utf-8") as stream:
    input_audit = json.load(stream)

with inputs["stage1_audit"].open(encoding="utf-8") as stream:
    stage1_audit = json.load(stream)

with inputs["stage2_audit"].open(encoding="utf-8") as stream:
    stage2_audit = json.load(stream)

with inputs["step9a_audit"].open(encoding="utf-8") as stream:
    step9a_audit = json.load(stream)

if not (
    stage1_audit.get("validation_passed")
    and stage2_audit.get("validation_passed")
    and step9a_audit.get("validation_passed")
):
    raise ValueError(
        "Stage 1, Stage 2 or Step 9.1 audit has not passed."
    )

with inputs["kr"].open("rb") as stream:
    source_checksum = hashlib.file_digest(
        stream, "sha256"
    ).hexdigest()

if source_checksum != input_audit["source_files"]["kr_sha256"]:
    raise ValueError(
        "Original KR dataset checksum mismatch."
    )


# Reconstruct the verified complete-case model populations.
raw = pd.read_stata(
    inputs["kr"],
    columns=[
        "v001", "v021", "v023", "v024", "v025",
        "v005", "h22", "h32y", "h46a", "h31",
        "h31b", "b19", "b4", "v012", "v106",
        "v190", "bord",
    ],
    convert_categoricals=False,
)

psu_base = pd.read_csv(
    inputs["psu_base"]
)

if (
    len(raw) != 8784
    or len(psu_base) != 674
    or not psu_base["v001"].is_unique
):
    raise ValueError(
        "Full KR or verified PSU inventory is inconsistent."
    )

if not raw["v001"].equals(raw["v021"]):
    raise ValueError(
        "PSU identifiers v001 and v021 differ."
    )

fever = raw["h22"].eq(1)
care = fever & raw["h32y"].eq(0)

complete = (
    raw[
        [
            "b19", "b4", "h31", "v012",
            "v106", "v190", "bord", "v025", "v024",
        ]
    ].notna().all(axis=1)
    & raw["h31b"].isin([0, 1])
)

model_domains = {
    "Stage 1 care sought": fever & complete,
    "Stage 2 pharmacy first": care & complete,
}

expected_population = {
    "Stage 1 care sought": 2624,
    "Stage 2 pharmacy first": 1969,
}

coefficient_files = {
    "Stage 1 care sought": inputs["stage1_coef"],
    "Stage 2 pharmacy first": inputs["stage2_coef"],
}


# Reconstruct exactly the predictors used by the verified logistic models.
expected_terms = [
    "_intercept_",
    "child_age_years",
    "b4_2",
    "cough_reported_1.0",
    "rapid_breathing_reported_1.0",
    "maternal_age_5years",
    "v106_1",
    "v106_2",
    "v106_3",
    "v190_2",
    "v190_3",
    "v190_4",
    "v190_5",
    "bord",
    "v025_2",
    "v024_2",
    "v024_3",
    "v024_4",
    "v024_5",
    "v024_6",
    "v024_7",
    "v024_8",
]


def model_design_matrix(observations):
    matrix = pd.DataFrame(
        index=observations.index
    )

    matrix["_intercept_"] = 1.0

    matrix["child_age_years"] = (
        observations["b19"].to_numpy(dtype=float) / 12
    )

    matrix["b4_2"] = (
        observations["b4"].eq(2).to_numpy(dtype=float)
    )

    matrix["cough_reported_1.0"] = (
        observations["h31"].eq(2).to_numpy(dtype=float)
    )

    matrix["rapid_breathing_reported_1.0"] = (
        observations["h31b"].eq(1).to_numpy(dtype=float)
    )

    matrix["maternal_age_5years"] = (
        observations["v012"].to_numpy(dtype=float) - 25
    ) / 5

    for level in (1, 2, 3):
        matrix[f"v106_{level}"] = (
            observations["v106"].eq(level).to_numpy(dtype=float)
        )

    for level in (2, 3, 4, 5):
        matrix[f"v190_{level}"] = (
            observations["v190"].eq(level).to_numpy(dtype=float)
        )

    matrix["bord"] = (
        observations["bord"].to_numpy(dtype=float)
    )

    matrix["v025_2"] = (
        observations["v025"].eq(2).to_numpy(dtype=float)
    )

    for level in (2, 3, 4, 5, 6, 7, 8):
        matrix[f"v024_{level}"] = (
            observations["v024"].eq(level).to_numpy(dtype=float)
        )

    return matrix[expected_terms]


# Calculate weighted observed-minus-expected residuals for each PSU.
psu_results = []
prediction_checks = []

for stage, domain in model_domains.items():

    model_frame = raw.loc[domain].copy()

    X = model_design_matrix(
        model_frame
    )

    coefficients = pd.read_csv(
        coefficient_files[stage]
    )

    if (
        len(coefficients) != 22
        or coefficients["term"].duplicated().any()
        or set(coefficients["term"]) != set(expected_terms)
    ):
        raise ValueError(
            f"Unexpected saved coefficient terms for {stage}."
        )

    beta = (
        coefficients.set_index("term")
        .loc[expected_terms, "est"]
        .to_numpy(dtype=float)
    )

    predicted = expit(
        X.to_numpy(dtype=float) @ beta
    )

    if stage == "Stage 1 care sought":
        observed = (
            model_frame["h32y"]
            .eq(0)
            .to_numpy(dtype=float)
        )
    else:
        observed = (
            model_frame["h46a"]
            .eq(36)
            .to_numpy(dtype=float)
        )

    weight = (
        model_frame["v005"].to_numpy(dtype=float)
        / 1_000_000
    )

    if not (
        np.isfinite(X.to_numpy(dtype=float)).all()
        and np.isfinite(predicted).all()
        and (predicted > 0).all()
        and (predicted < 1).all()
        and np.isfinite(weight).all()
        and (weight > 0).all()
    ):
        raise ValueError(
            f"Invalid model predictions or weights: {stage}."
        )

    model_frame["weight"] = weight

    model_frame["weighted_observed"] = (
        weight * observed
    )

    model_frame["weighted_expected"] = (
        weight * predicted
    )

    model_frame["weighted_residual"] = (
        weight * (observed - predicted)
    )

    model_frame["weight_sq"] = (
        weight ** 2
    )

    model_frame["pearson_null_variance"] = (
        (weight ** 2) * predicted * (1 - predicted)
    )

    psu_stage = model_frame.groupby(
        "v001",
        as_index=False,
    ).agg(
        eligible_n=("weighted_residual", "size"),
        weight_sum=("weight", "sum"),
        weight_sq_sum=("weight_sq", "sum"),
        weighted_observed=("weighted_observed", "sum"),
        weighted_expected=("weighted_expected", "sum"),
        residual_numerator=("weighted_residual", "sum"),
        pearson_variance=("pearson_null_variance", "sum"),
    )

    psu_stage["weighted_gap"] = (
        psu_stage["residual_numerator"]
        / psu_stage["weight_sum"]
    )

    psu_stage["pearson_score"] = (
        psu_stage["residual_numerator"]
        / np.sqrt(psu_stage["pearson_variance"])
    )

    psu_stage["effective_n"] = (
        psu_stage["weight_sum"] ** 2
        / psu_stage["weight_sq_sum"]
    )

    psu_stage["stage"] = stage

    psu_results.append(
        psu_stage
    )

    weighted_observed = np.average(
        observed,
        weights=weight,
    )

    weighted_expected = np.average(
        predicted,
        weights=weight,
    )

    prediction_checks.append({
        "stage": stage,
        "n": len(model_frame),
        "active_psus": len(psu_stage),
        "weighted_observed_percent": (
            100 * weighted_observed
        ),
        "weighted_expected_percent": (
            100 * weighted_expected
        ),
        "weighted_difference_pp": (
            100 * (weighted_observed - weighted_expected)
        ),
        "model_coefficient_count": len(coefficients),
    })


# Attach displaced GPS coordinates to the adjusted PSU residuals.
all_residuals = pd.concat(
    psu_results,
    ignore_index=True,
)

psu_residuals = all_residuals.merge(
    psu_base[
        [
            "v001", "v023", "v024", "v025",
            "gps_latitude", "gps_longitude",
            "fever_n", "care_n",
        ]
    ],
    on="v001",
    how="left",
    validate="many_to_one",
    indicator=True,
)

if not psu_residuals["_merge"].eq("both").all():
    raise ValueError(
        "Adjusted PSU residuals do not match verified GPS clusters."
    )

psu_residuals = psu_residuals.drop(
    columns="_merge"
)

predictions = pd.DataFrame(
    prediction_checks
)


# Construct symmetric k-nearest-neighbour graphs from displaced GPS points.
def neighbour_graph(nodes, k):

    coordinates = np.deg2rad(
        nodes[
            ["gps_latitude", "gps_longitude"]
        ].to_numpy(dtype=float)
    )

    knn = NearestNeighbors(
        n_neighbors=k + 1,
        metric="haversine",
        algorithm="ball_tree",
    ).fit(coordinates)

    neighbours = knn.kneighbors(
        coordinates,
        return_distance=False,
    )

    rows = []
    columns = []

    for i, candidates in enumerate(neighbours):

        selected = [
            int(j)
            for j in candidates
            if j != i
        ][:k]

        if len(selected) != k:
            raise ValueError(
                "Incomplete k-nearest-neighbour graph."
            )

        rows.extend(
            [i] * k
        )

        columns.extend(
            selected
        )

    graph = csr_matrix(
        (
            np.ones(len(rows)),
            (rows, columns),
        ),
        shape=(len(nodes), len(nodes)),
    )

    graph = (
        graph + graph.T
    ).tocsr()

    graph.data[:] = 1.0

    graph.setdiag(0)
    graph.eliminate_zeros()

    return graph


# Calculate Global Moran's I for adjusted PSU residuals.
def moran(values, graph):

    values = np.asarray(
        values,
        dtype=float,
    )

    centered = (
        values - np.mean(values)
    )

    denominator = float(
        centered @ centered
    )

    if (
        denominator <= 0
        or not np.isfinite(denominator)
    ):
        raise ValueError(
            "Undefined residual Moran's I."
        )

    return float(
        (
            len(values) / graph.sum()
        )
        * (
            centered @ (graph @ centered)
        )
        / denominator
    )


# Use unrestricted and survey-stratum-restricted permutation diagnostics.
def permute_residuals(
    values,
    graph,
    strata,
    seed,
    repetitions=999,
):

    rng = np.random.default_rng(seed)

    observed = moran(
        values,
        graph,
    )

    groups = [
        np.flatnonzero(strata == level)
        for level in np.unique(strata)
    ]

    unrestricted = np.empty(
        repetitions
    )

    restricted = np.empty(
        repetitions
    )

    for iteration in range(repetitions):

        unrestricted[iteration] = moran(
            rng.permutation(values),
            graph,
        )

        shuffled = values.copy()

        for indices in groups:
            if len(indices) > 1:
                shuffled[indices] = rng.permutation(
                    values[indices]
                )

        restricted[iteration] = moran(
            shuffled,
            graph,
        )

    def permutation_p(draws):

        null_center = float(
            np.mean(draws)
        )

        extreme = (
            np.abs(draws - null_center)
            >= abs(observed - null_center)
        )

        return float(
            (1 + np.sum(extreme))
            / (repetitions + 1)
        )

    return (
        observed,
        permutation_p(unrestricted),
        permutation_p(restricted),
        float(np.mean(restricted)),
    )


# Compare residual spatial autocorrelation across population thresholds.
N_PERMUTATIONS = 999
BASE_SEED = 20261117

moran_rows = []
coverage_rows = []

for stage_index, stage in enumerate(expected_population):

    stage_nodes = psu_residuals.loc[
        psu_residuals["stage"].eq(stage)
    ]

    for minimum_n in (1, 3):

        nodes = (
            stage_nodes.loc[
                stage_nodes["eligible_n"].ge(minimum_n)
            ]
            .sort_values("v001")
            .reset_index(drop=True)
        )

        coverage_rows.append({
            "stage": stage,
            "minimum_eligible_children_per_psu": minimum_n,
            "psus": len(nodes),
            "children": int(
                nodes["eligible_n"].sum()
            ),
            "one_child_psus": int(
                nodes["eligible_n"].eq(1).sum()
            ),
            "median_n": float(
                nodes["eligible_n"].median()
            ),
        })

        strata = nodes[
            "v023"
        ].to_numpy(dtype=int)

        for k in (8, 12):

            graph = neighbour_graph(
                nodes,
                k,
            )

            degrees = np.diff(
                graph.indptr
            )

            if degrees.min() < k:
                raise ValueError(
                    "Underconnected spatial graph."
                )

            residual_definitions = (
                (
                    "Weighted observed-minus-expected proportion",
                    "weighted_gap",
                ),
                (
                    "Pearson-standardized PSU score",
                    "pearson_score",
                ),
            )

            for score_index, (
                score_name,
                score_column,
            ) in enumerate(residual_definitions):

                values = nodes[
                    score_column
                ].to_numpy(dtype=float)

                if not np.isfinite(values).all():
                    raise ValueError(
                        "Nonfinite adjusted PSU residual."
                    )

                seed = (
                    BASE_SEED
                    + stage_index * 1000
                    + minimum_n * 100
                    + k * 10
                    + score_index
                )

                (
                    observed_i,
                    p_unrestricted,
                    p_restricted,
                    restricted_null_mean,
                ) = permute_residuals(
                    values=values,
                    graph=graph,
                    strata=strata,
                    seed=seed,
                    repetitions=N_PERMUTATIONS,
                )

                moran_rows.append({
                    "stage": stage,
                    "min_n": minimum_n,
                    "k": k,
                    "residual_definition": score_name,
                    "psus": len(nodes),
                    "moran_i": observed_i,
                    "permutation_p_unrestricted_two_sided": (
                        p_unrestricted
                    ),
                    "permutation_p_within_stratum_two_sided": (
                        p_restricted
                    ),
                    "within_stratum_null_mean": (
                        restricted_null_mean
                    ),
                    "permutations": N_PERMUTATIONS,
                })


moran_results = pd.DataFrame(
    moran_rows
)

coverage = pd.DataFrame(
    coverage_rows
)


# Validate model predictions, geographic linkage and spatial diagnostics.
checks = {
    "Verified raw source and upstream analyses":
        bool(
            stage1_audit["validation_passed"]
            and stage2_audit["validation_passed"]
            and step9a_audit["validation_passed"]
        ),

    "Original survey records and PSU inventory retained":
        (
            len(raw) == 8784
            and raw["v001"].nunique() == 674
        ),

    "Stage 1 complete cases equal 2,624":
        int(
            model_domains["Stage 1 care sought"].sum()
        ) == 2624,

    "Stage 2 complete cases equal 1,969":
        int(
            model_domains["Stage 2 pharmacy first"].sum()
        ) == 1969,

    "Original outcome totals agree with verified analyses":
        (
            int(care.sum()) == 1974
            and int(
                (
                    care & raw["h46a"].eq(36)
                ).sum()
            ) == 1120
        ),

    "Both saved models contain 22 coefficients":
        predictions[
            "model_coefficient_count"
        ].eq(22).all(),

    "Weighted observed and fitted prevalences agree":
        predictions[
            "weighted_difference_pp"
        ].abs().max() < 0.01,

    "Every adjusted residual matches a GPS PSU":
        bool(
            psu_residuals[
                ["gps_latitude", "gps_longitude"]
            ].notna().all().all()
        ),

    "Stage 1 residuals cover 642 PSUs":
        int(
            psu_residuals["stage"]
            .eq("Stage 1 care sought").sum()
        ) == 642,

    "Stage 2 residuals cover 611 PSUs":
        int(
            psu_residuals["stage"]
            .eq("Stage 2 pharmacy first").sum()
        ) == 611,

    "Both outcomes and PSU size thresholds included":
        len(coverage) == 4,

    "All 16 adjusted Moran results generated":
        len(moran_results) == 16,

    "All Moran statistics are finite":
        bool(
            np.isfinite(
                moran_results["moran_i"]
            ).all()
        ),

    "All permutation probabilities are valid":
        bool(
            moran_results[
                [
                    "permutation_p_unrestricted_two_sided",
                    "permutation_p_within_stratum_two_sided",
                ]
            ].apply(
                lambda series: series.between(0, 1).all()
            ).all()
        ),

    "All PSU residual scores are finite":
        bool(
            np.isfinite(
                psu_residuals[
                    ["weighted_gap", "pearson_score"]
                ].to_numpy(dtype=float)
            ).all()
        ),
}

validation = pd.DataFrame([
    {
        "check": name,
        "status": "PASS" if passed else "REVIEW",
    }
    for name, passed in checks.items()
])

all_passed = all(
    bool(value)
    for value in checks.values()
)


# Save new outputs without overwriting earlier survey or spatial results.
outputs = {
    "residuals":
        PROCESSED / "09b_adjusted_psu_residuals.csv",

    "moran":
        TABLES / "09b_adjusted_residual_global_moran.csv",

    "coverage":
        TABLES / "09b_adjusted_residual_coverage.csv",

    "predictions":
        TABLES / "09b_regression_prediction_reconstruction.csv",

    "checks":
        TABLES / "09b_adjusted_spatial_residual_checks.csv",

    "audit":
        LOGS / "09b_adjusted_spatial_residual_audit.json",

    "report":
        LOGS / "09b_adjusted_spatial_residual_report.txt",
}

for name, table in (
    ("residuals", psu_residuals),
    ("moran", moran_results),
    ("coverage", coverage),
    ("predictions", predictions),
    ("checks", validation),
):
    table.to_csv(
        outputs[name],
        index=False,
        encoding="utf-8-sig",
    )

audit = {
    "step": "09.2",
    "phase": "D - adjusted residual spatial autocorrelation",
    "source_sha256": source_checksum,
    "prediction_source": (
        "Verified Stage 1 and Stage 2 logistic coefficients"
    ),
    "residual_definitions": {
        "weighted_gap": "sum(w * (y - p)) / sum(w)",
        "pearson_score": (
            "sum(w * (y - p)) / "
            "sqrt(sum(w**2 * p * (1-p)))"
        ),
    },
    "permutations": N_PERMUTATIONS,
    "spatial_weights": (
        "Symmetric 8-NN and 12-NN graphs using displaced GPS"
    ),
    "limitations": [
        "Permutation results are exploratory, not design-based spatial-model tests.",
        "Pearson variance scaling assumes conditional independence.",
        "Residual predictions are in-sample, not cross-validated.",
        "GPS displacement precludes precise hotspot inference.",
        "Outcome-derived PSU residuals cannot enter held-out-PSU ML features.",
    ],
    "checks": {
        key: bool(value)
        for key, value in checks.items()
    },
    "validation_passed": bool(all_passed),
}

outputs["audit"].write_text(
    json.dumps(
        audit,
        indent=2,
        ensure_ascii=False,
    ) + "\n",
    encoding="utf-8",
)

report = "\n".join([
    "STEP 09.2 | ADJUSTED RESIDUAL SPATIAL AUTOCORRELATION",
    "=" * 64,
    "Models: Verified Stage 1 and Stage 2 adjusted logistic regressions",
    "Residuals: Weighted probability gaps and Pearson PSU scores",
    "Spatial weights: Symmetric 8-NN and 12-NN",
    "PSU threshold: At least 1 or 3 eligible children",
    f"Adjusted Moran comparisons: {len(moran_results)}",
    f"Permutations per comparison: {N_PERMUTATIONS}",
    "Inference: Exploratory, not design-adjusted spatial inference",
    "Geography: Displaced DHS GPS; no exact hotspot claims",
    f"Validation: {'PASS' if all_passed else 'REVIEW REQUIRED'}",
    f"Validated checks: {sum(checks.values())} / {len(checks)}",
])

outputs["report"].write_text(
    report + "\n",
    encoding="utf-8",
)

print(report)

print("\nREGRESSION PREDICTION RECONSTRUCTION")
display(
    predictions.round(6)
)

print("\nMODEL-ADJUSTED SPATIAL AUTOCORRELATION")
display(
    moran_results[
        [
            "stage",
            "min_n",
            "k",
            "residual_definition",
            "psus",
            "moran_i",
            "permutation_p_unrestricted_two_sided",
            "permutation_p_within_stratum_two_sided",
        ]
    ].round(5)
)

print("\nMODEL-DOMAIN PSU COVERAGE")
display(coverage)

print("\nVALIDATION CHECKS")
display(validation)

print("\nSAVED FILES")
for path in outputs.values():
    print(path)

if not all_passed:
    raise ValueError(
        "Adjusted spatial residual diagnostics require review."
    )

STEP 09.2 | ADJUSTED RESIDUAL SPATIAL AUTOCORRELATION
Models: Verified Stage 1 and Stage 2 adjusted logistic regressions
Residuals: Weighted probability gaps and Pearson PSU scores
Spatial weights: Symmetric 8-NN and 12-NN
PSU threshold: At least 1 or 3 eligible children
Adjusted Moran comparisons: 16
Permutations per comparison: 999
Inference: Exploratory, not design-adjusted spatial inference
Geography: Displaced DHS GPS; no exact hotspot claims
Validation: PASS
Validated checks: 15 / 15

REGRESSION PREDICTION RECONSTRUCTION


,stage,n,active_psus,weighted_observed_percent,weighted_expected_percent,weighted_difference_pp,model_coefficient_count
0,Stage 1 care sought,2624,642,76.182768,76.182768,0.0,22
1,Stage 2 pharmacy first,1969,611,57.758511,57.758511,0.0,22



MODEL-ADJUSTED SPATIAL AUTOCORRELATION


,stage,min_n,k,residual_definition,psus,moran_i,permutation_p_unrestricted_two_sided,permutation_p_within_stratum_two_sided
0,Stage 1 care sought,1,8,Weighted observed-minus-expected proportion,642,0.03990,0.025,0.074
1,Stage 1 care sought,1,8,Pearson-standardized PSU score,642,0.05422,0.003,0.006
2,Stage 1 care sought,1,12,Weighted observed-minus-expected proportion,642,0.03029,0.026,0.122
3,Stage 1 care sought,1,12,Pearson-standardized PSU score,642,0.03468,0.018,0.009
4,Stage 1 care sought,3,8,Weighted observed-minus-expected proportion,455,0.05090,0.013,0.002
5,Stage 1 care sought,3,8,Pearson-standardized PSU score,455,0.05512,0.007,0.003
6,Stage 1 care sought,3,12,Weighted observed-minus-expected proportion,455,0.02879,0.061,0.015
7,Stage 1 care sought,3,12,Pearson-standardized PSU score,455,0.03078,0.040,0.011
8,Stage 2 pharmacy first,1,8,Weighted observed-minus-expected proportion,611,0.00236,0.858,0.512
9,Stage 2 pharmacy first,1,8,Pearson-standardized PSU score,611,-0.00125,0.973,0.658



MODEL-DOMAIN PSU COVERAGE


,stage,minimum_eligible_children_per_psu,psus,children,one_child_psus,median_n
0,Stage 1 care sought,1,642,2624,80,4.0
1,Stage 1 care sought,3,455,2330,0,5.0
2,Stage 2 pharmacy first,1,611,1969,124,3.0
3,Stage 2 pharmacy first,3,347,1565,0,4.0



VALIDATION CHECKS


,check,status
0,Verified raw source and upstream analyses,PASS
1,Original survey records and PSU inventory reta...,PASS
2,"Stage 1 complete cases equal 2,624",PASS
3,"Stage 2 complete cases equal 1,969",PASS
4,Original outcome totals agree with verified an...,PASS
5,Both saved models contain 22 coefficients,PASS
6,Weighted observed and fitted prevalences agree,PASS
7,Every adjusted residual matches a GPS PSU,PASS
8,Stage 1 residuals cover 642 PSUs,PASS
9,Stage 2 residuals cover 611 PSUs,PASS



SAVED FILES
D:\BDHS2022_FeverCare_SpatialEquity\data\processed\09b_adjusted_psu_residuals.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\09b_adjusted_residual_global_moran.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\09b_adjusted_residual_coverage.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\09b_regression_prediction_reconstruction.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\09b_adjusted_spatial_residual_checks.csv
D:\BDHS2022_FeverCare_SpatialEquity\logs\09b_adjusted_spatial_residual_audit.json
D:\BDHS2022_FeverCare_SpatialEquity\logs\09b_adjusted_spatial_residual_report.txt


In [24]:

from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd

from scipy.optimize import minimize
from scipy.special import expit
from scipy.sparse import csr_matrix, diags, eye
from scipy.sparse.linalg import splu

from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_auc_score
from IPython.display import display


ROOT = Path(r"D:\BDHS2022_FeverCare_SpatialEquity")

RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
TABLES = ROOT / "results" / "tables"
LOGS = ROOT / "logs"

inputs = {
    "kr": RAW / "BDKR81FL.DTA",
    "gps": PROCESSED / "09a_psu_spatial_descriptive_base.csv",
    "stage1": TABLES / "08a_stage1_adjusted_logistic_coefficients.csv",
    "stage2": TABLES / "08c_stage2_adjusted_logistic_coefficients.csv",
    "source_audit": LOGS / "01_input_audit.json",
    "spatial_audit": LOGS / "09b_adjusted_spatial_residual_audit.json",
    "previous_moran": TABLES / "09b_adjusted_residual_global_moran.csv",
}

for label, path in inputs.items():
    if not path.is_file():
        raise FileNotFoundError(
            f"Required input missing ({label}): {path}"
        )

with inputs["source_audit"].open(encoding="utf-8") as handle:
    source_audit = json.load(handle)

with inputs["spatial_audit"].open(encoding="utf-8") as handle:
    spatial_audit = json.load(handle)

if not spatial_audit.get("validation_passed", False):
    raise ValueError("Step 9.2 audit has not passed.")

with inputs["kr"].open("rb") as handle:
    kr_hash = hashlib.file_digest(handle, "sha256").hexdigest()

if kr_hash != source_audit["source_files"]["kr_sha256"]:
    raise ValueError("Original KR dataset checksum mismatch.")


# Reconstruct the verified analysis populations.
raw = pd.read_stata(
    inputs["kr"],
    columns=[
        "v001", "v021", "v023", "v005",
        "h22", "h32y", "h46a",
        "h31", "h31b", "b19", "b4",
        "v012", "v106", "v190", "bord",
        "v025", "v024",
    ],
    convert_categoricals=False,
)

gps = pd.read_csv(
    inputs["gps"],
    usecols=[
        "v001", "v023",
        "gps_latitude", "gps_longitude",
    ],
)

previous_moran = pd.read_csv(
    inputs["previous_moran"]
)

if not (
    len(raw) == 8784
    and raw["v001"].nunique() == 674
    and raw["v001"].eq(raw["v021"]).all()
    and len(gps) == 674
    and gps["v001"].is_unique
):
    raise ValueError(
        "Survey design or GPS cluster inventory has changed."
    )

fever = raw["h22"].eq(1)
care = fever & raw["h32y"].eq(0)

complete = (
    raw[
        [
            "b19", "b4", "h31", "v012",
            "v106", "v190", "bord",
            "v025", "v024",
        ]
    ].notna().all(axis=1)
    & raw["h31b"].isin([0, 1])
)

specifications = [
    (
        "Stage 1",
        fever & complete,
        raw["h32y"].eq(0),
        inputs["stage1"],
        2624,
        642,
        1969,
    ),
    (
        "Stage 2",
        care & complete,
        raw["h46a"].eq(36),
        inputs["stage2"],
        1969,
        611,
        1117,
    ),
]

terms = [
    "_intercept_",
    "child_age_years",
    "b4_2",
    "cough_reported_1.0",
    "rapid_breathing_reported_1.0",
    "maternal_age_5years",
    *[f"v106_{level}" for level in (1, 2, 3)],
    *[f"v190_{level}" for level in (2, 3, 4, 5)],
    "bord",
    "v025_2",
    *[f"v024_{level}" for level in range(2, 9)],
]


def design_matrix(data):
    return np.column_stack(
        [
            np.ones(len(data)),
            data["b19"].to_numpy(dtype=float) / 12,
            data["b4"].eq(2).to_numpy(dtype=float),
            data["h31"].eq(2).to_numpy(dtype=float),
            data["h31b"].eq(1).to_numpy(dtype=float),
            (
                data["v012"].to_numpy(dtype=float) - 25
            ) / 5,
            *[
                data["v106"].eq(level).to_numpy(dtype=float)
                for level in (1, 2, 3)
            ],
            *[
                data["v190"].eq(level).to_numpy(dtype=float)
                for level in (2, 3, 4, 5)
            ],
            data["bord"].to_numpy(dtype=float),
            data["v025"].eq(2).to_numpy(dtype=float),
            *[
                data["v024"].eq(level).to_numpy(dtype=float)
                for level in range(2, 9)
            ],
        ]
    )


# Construct symmetric spatial weights from displaced DHS coordinates.
def spatial_graph(nodes, k=8):

    coordinates = np.deg2rad(
        nodes[
            ["gps_latitude", "gps_longitude"]
        ].to_numpy(dtype=float)
    )

    estimator = NearestNeighbors(
        n_neighbors=k + 1,
        metric="haversine",
        algorithm="ball_tree",
    ).fit(coordinates)

    neighbors = estimator.kneighbors(
        coordinates,
        return_distance=False,
    )

    row_ids = []
    col_ids = []

    for i, candidates in enumerate(neighbors):

        selected = [
            int(j)
            for j in candidates
            if int(j) != i
        ][:k]

        if len(selected) != k:
            raise ValueError(
                "Incomplete nearest-neighbor graph."
            )

        row_ids.extend([i] * k)
        col_ids.extend(selected)

    graph = csr_matrix(
        (
            np.ones(len(row_ids)),
            (row_ids, col_ids),
        ),
        shape=(len(nodes), len(nodes)),
    )

    graph = (graph + graph.T).tocsr()
    graph.data[:] = 1.0
    graph.setdiag(0)
    graph.eliminate_zeros()

    degree = np.asarray(
        graph.sum(axis=1)
    ).ravel()

    if np.any(degree < k):
        raise ValueError(
            "Spatial graph contains underconnected PSUs."
        )

    normalized = diags(
        degree ** -0.5
    )

    laplacian = (
        eye(len(nodes), format="csr")
        - normalized @ graph @ normalized
    )

    return graph, laplacian.tocsr()


def log_determinant(precision):

    factor = splu(
        precision.tocsc()
    )

    diagonal = np.asarray(
        factor.U.diagonal()
    )

    if np.any(diagonal == 0):
        raise ArithmeticError(
            "Singular random-effect precision matrix."
        )

    return float(
        np.log(np.abs(diagonal)).sum()
    )


# Fit a weighted logistic model with penalized PSU-specific intercepts.
def fit_penalized_model(
    X,
    y,
    weights,
    group,
    precision,
    beta_init,
):

    n_parameters = X.shape[1]
    n_groups = precision.shape[0]

    starting = np.r_[
        beta_init,
        np.zeros(n_groups),
    ]

    def objective(theta):

        beta = theta[:n_parameters]
        random_intercepts = theta[n_parameters:]

        eta = (
            X @ beta
            + random_intercepts[group]
        )

        p = expit(eta)

        error = weights * (p - y)

        penalty = (
            0.5
            * float(
                random_intercepts
                @ (precision @ random_intercepts)
            )
        )

        loss = float(
            np.sum(
                weights * (
                    np.logaddexp(0, eta)
                    - y * eta
                )
            )
        ) + penalty

        gradient = np.r_[
            X.T @ error,
            np.bincount(
                group,
                weights=error,
                minlength=n_groups,
            ) + precision @ random_intercepts,
        ]

        return loss, gradient

    optimized = minimize(
        objective,
        starting,
        jac=True,
        method="L-BFGS-B",
        options={
            "maxiter": 400,
            "ftol": 2e-11,
            "gtol": 5e-6,
            "maxls": 35,
        },
    )

    beta = optimized.x[:n_parameters]
    random_intercepts = optimized.x[n_parameters:]

    eta = (
        X @ beta
        + random_intercepts[group]
    )

    p = expit(eta)

    information = np.bincount(
        group,
        weights=weights * p * (1 - p),
        minlength=n_groups,
    )

    hessian = (
        precision
        + diags(
            information,
            format="csr",
        )
    )

    weighted_log_likelihood = float(
        np.sum(
            weights * (
                y * eta
                - np.logaddexp(0, eta)
            )
        )
    )

    # A Laplace approximation integrates the random effects
    # conditionally on the jointly fitted fixed-effect coefficients.
    score = (
        weighted_log_likelihood
        - 0.5 * float(
            random_intercepts
            @ (precision @ random_intercepts)
        )
        + 0.5 * (
            log_determinant(precision)
            - log_determinant(hessian)
        )
    )

    return {
        "p": p,
        "beta": beta,
        "u": random_intercepts,
        "laplace_score": float(score),
        "converged": bool(optimized.success),
        "max_gradient": float(
            np.max(np.abs(optimized.jac))
        ),
        "iterations": int(optimized.nit),
    }


def moran_i(values, graph):

    centered = values - np.mean(values)

    denominator = float(
        centered @ centered
    )

    if denominator <= 0:
        return np.nan

    return float(
        (len(values) / graph.sum())
        * (
            centered
            @ (graph @ centered)
        )
        / denominator
    )


# The same hyperparameter grid is used for both outcome stages.
PENALTY_GRID = (
    0.25,
    1.0,
    4.0,
    16.0,
)

SPATIAL_SMOOTHING = 4.0

comparison_rows = []
grid_rows = []
coefficient_rows = []
random_effect_rows = []
baseline_checks = []


for (
    stage,
    domain,
    positive,
    coefficient_path,
    expected_n,
    expected_psus,
    expected_events,
) in specifications:

    data = raw.loc[domain].copy()

    X = design_matrix(data)

    y = positive[domain].to_numpy(
        dtype=float
    )

    original_weights = (
        data["v005"].to_numpy(dtype=float)
        / 1_000_000
    )

    # Normalising the mean weight to one keeps the
    # pseudo-likelihood and random-effect penalty on a comparable scale.
    weights = (
        original_weights
        / original_weights.mean()
    )

    nodes = (
        data[["v001", "v023"]]
        .drop_duplicates()
        .sort_values("v001")
        .merge(
            gps,
            on=["v001", "v023"],
            how="left",
            validate="one_to_one",
        )
        .reset_index(drop=True)
    )

    group = pd.Index(
        nodes["v001"]
    ).get_indexer(
        data["v001"]
    )

    if not (
        len(data) == expected_n
        and int(y.sum()) == expected_events
        and len(nodes) == expected_psus
        and (group >= 0).all()
        and not nodes[
            ["gps_latitude", "gps_longitude"]
        ].isna().any().any()
        and X.shape[1] == 22
        and np.linalg.matrix_rank(X) == 22
    ):
        raise ValueError(
            f"Model population or matrix is inconsistent: {stage}"
        )

    graph, laplacian = spatial_graph(
        nodes,
        k=8,
    )

    saved_coefficients = pd.read_csv(
        coefficient_path
    )

    if not (
        len(saved_coefficients) == 22
        and set(saved_coefficients["term"]) == set(terms)
    ):
        raise ValueError(
            f"Saved coefficient specification has changed: {stage}"
        )

    saved_beta = (
        saved_coefficients
        .set_index("term")
        .loc[terms, "est"]
        .to_numpy(dtype=float)
    )

    # Reconstruct the original survey-weighted point estimates.
    def baseline_objective(beta):

        eta = X @ beta

        error = (
            weights
            * (expit(eta) - y)
        )

        return (
            float(
                np.sum(
                    weights * (
                        np.logaddexp(0, eta)
                        - y * eta
                    )
                )
            ),
            X.T @ error,
        )

    baseline_fit = minimize(
        baseline_objective,
        saved_beta,
        jac=True,
        method="BFGS",
        options={
            "gtol": 1e-7,
            "maxiter": 400,
        },
    )

    beta0 = baseline_fit.x

    beta_difference = float(
        np.max(
            np.abs(
                beta0 - saved_beta
            )
        )
    )

    if beta_difference >= 2e-4:
        raise ValueError(
            f"Baseline does not reproduce verified svy coefficients: {stage}"
        )

    eta0 = X @ beta0
    p0 = expit(eta0)

    ll0 = float(
        np.sum(
            weights * (
                y * eta0
                - np.logaddexp(0, eta0)
            )
        )
    )

    variants = {
        "Non-spatial": {
            "p": p0,
            "beta": beta0,
            "u": np.zeros(len(nodes)),
            "laplace_score": ll0,
            "lambda": np.nan,
            "rho": np.nan,
            "converged": True,
            "max_gradient": float(
                np.max(
                    np.abs(
                        baseline_objective(beta0)[1]
                    )
                )
            ),
            "iterations": int(
                baseline_fit.nit
            ),
        }
    }

    # Model A: Independent Gaussian PSU intercepts.
    # Model B: Spatially smoothed PSU intercepts using a proper GMRF.
    for name, rho in (
        ("Multilevel PSU", 0.0),
        ("Spatial GMRF PSU", SPATIAL_SMOOTHING),
    ):

        candidates = []

        for strength in PENALTY_GRID:

            precision = (
                strength
                * (
                    eye(len(nodes), format="csr")
                    + rho * laplacian
                )
            )

            result = fit_penalized_model(
                X=X,
                y=y,
                weights=weights,
                group=group,
                precision=precision.tocsr(),
                beta_init=beta0,
            )

            result.update(
                {
                    "lambda": strength,
                    "rho": rho,
                }
            )

            candidates.append(result)

            grid_rows.append(
                {
                    "stage": stage,
                    "model": name,
                    "lambda": strength,
                    "spatial_rho": rho,
                    "approx_laplace_pseudo_score": (
                        result["laplace_score"]
                    ),
                    "optimizer_converged": (
                        result["converged"]
                    ),
                    "max_gradient": (
                        result["max_gradient"]
                    ),
                    "iterations": (
                        result["iterations"]
                    ),
                }
            )

        usable = [
            result
            for result in candidates
            if (
                result["converged"]
                and result["max_gradient"] < 0.03
            )
        ]

        if not usable:
            raise RuntimeError(
                f"No sufficiently converged fit: {stage}, {name}"
            )

        variants[name] = max(
            usable,
            key=lambda result: result["laplace_score"],
        )

    # Compare all models on their identical stage-specific sample.
    for name, fitted in variants.items():

        p = fitted["p"]
        u = fitted["u"]

        residual = (
            weights * (y - p)
        )

        gap = (
            np.bincount(
                group,
                weights=residual,
                minlength=len(nodes),
            )
            / np.bincount(
                group,
                weights=weights,
                minlength=len(nodes),
            )
        )

        eta = (
            X @ fitted["beta"]
            + u[group]
        )

        log_loss = float(
            np.average(
                np.logaddexp(0, eta)
                - y * eta,
                weights=weights,
            )
        )

        brier = float(
            np.average(
                (y - p) ** 2,
                weights=weights,
            )
        )

        auc = float(
            roc_auc_score(
                y,
                p,
                sample_weight=weights,
            )
        )

        comparison_rows.append(
            {
                "stage": stage,
                "model": name,
                "n_children": len(data),
                "n_psus": len(nodes),
                "selected_lambda": (
                    fitted["lambda"]
                ),
                "spatial_rho": (
                    fitted["rho"]
                ),
                "in_sample_weighted_log_loss": (
                    log_loss
                ),
                "in_sample_weighted_brier": (
                    brier
                ),
                "in_sample_weighted_auc": (
                    auc
                ),
                "approx_laplace_pseudo_score": (
                    fitted["laplace_score"]
                ),
                "pseudo_score_gain_over_nonspatial": (
                    fitted["laplace_score"] - ll0
                ),
                "residual_moran_i_8nn": (
                    moran_i(gap, graph)
                ),
                "psu_effect_sd": float(
                    np.std(u)
                ),
                "optimizer_converged": (
                    fitted["converged"]
                ),
                "max_gradient": (
                    fitted["max_gradient"]
                ),
            }
        )

        coefficient_rows.extend(
            {
                "stage": stage,
                "model": name,
                "term": term,
                "coefficient": float(
                    fitted["beta"][j]
                ),
            }
            for j, term in enumerate(terms)
        )

        random_effect_rows.extend(
            {
                "stage": stage,
                "model": name,
                "v001": int(
                    nodes.loc[i, "v001"]
                ),
                "fitted_psu_effect": float(
                    u[i]
                ),
                "observed_minus_expected_weighted": (
                    float(gap[i])
                ),
            }
            for i in range(len(nodes))
        )

    # Independently reproduce the earlier adjusted-residual Moran result.
    earlier_stage_label = (
        "Stage 1 care sought"
        if stage == "Stage 1"
        else "Stage 2 pharmacy first"
    )

    reference = previous_moran.loc[
        previous_moran["stage"].eq(
            earlier_stage_label
        )
        & previous_moran["min_n"].eq(1)
        & previous_moran["k"].eq(8)
        & previous_moran["residual_definition"].eq(
            "Weighted observed-minus-expected proportion"
        ),
        "moran_i",
    ]

    baseline_moran = next(
        row["residual_moran_i_8nn"]
        for row in comparison_rows
        if (
            row["stage"] == stage
            and row["model"] == "Non-spatial"
        )
    )

    moran_difference = (
        abs(
            baseline_moran
            - float(reference.iloc[0])
        )
        if len(reference) == 1
        else np.nan
    )

    baseline_checks.append(
        {
            "stage": stage,
            "n": len(data),
            "active_psus": len(nodes),
            "positive_n": int(y.sum()),
            "model_columns": X.shape[1],
            "max_beta_reconstruction_error": (
                beta_difference
            ),
            "previous_residual_moran_difference": (
                moran_difference
            ),
            "min_knn_degree": int(
                np.min(
                    np.diff(
                        graph.indptr
                    )
                )
            ),
        }
    )


comparison = pd.DataFrame(
    comparison_rows
)

grid = pd.DataFrame(
    grid_rows
)

reconstruction = pd.DataFrame(
    baseline_checks
)

coefficient_table = pd.DataFrame(
    coefficient_rows
)

psu_effects = pd.DataFrame(
    random_effect_rows
)


# Audit model comparability and numerical stability.
checks = {
    "Source and Step 9.2 audit verified":
        bool(spatial_audit["validation_passed"]),

    "Both original model populations retained":
        sorted(reconstruction["n"]) == [1969, 2624],

    "Expected active PSU counts retained":
        sorted(reconstruction["active_psus"]) == [611, 642],

    "Both outcomes have correct event counts":
        sorted(reconstruction["positive_n"]) == [1117, 1969],

    "All 22 fixed-effect terms included":
        bool(
            reconstruction["model_columns"]
            .eq(22).all()
        ),

    "Verified survey coefficients independently reproduced":
        bool(
            reconstruction["max_beta_reconstruction_error"]
            .lt(2e-4).all()
        ),

    "Step 9.2 residual Moran statistics reproduced":
        bool(
            reconstruction["previous_residual_moran_difference"]
            .lt(1e-4).all()
        ),

    "Each graph has at least eight neighbours per PSU":
        bool(
            reconstruction["min_knn_degree"]
            .ge(8).all()
        ),

    "Three model classes fitted for each stage":
        len(comparison) == 6,

    "All 16 penalty-grid fits available":
        len(grid) == 16,

    "All selected model fits converged":
        bool(
            comparison["optimizer_converged"].all()
            and comparison["max_gradient"]
            .lt(0.03).all()
        ),

    "All penalty-grid fits converged":
        bool(
            grid["optimizer_converged"].all()
            and grid["max_gradient"]
            .lt(0.03).all()
        ),

    "All comparison metrics and scores are finite":
        bool(
            np.isfinite(
                comparison[
                    [
                        "in_sample_weighted_log_loss",
                        "in_sample_weighted_brier",
                        "in_sample_weighted_auc",
                        "approx_laplace_pseudo_score",
                        "residual_moran_i_8nn",
                    ]
                ].to_numpy(dtype=float)
            ).all()
        ),

    "AUC and Brier scores are within valid bounds":
        bool(
            comparison["in_sample_weighted_auc"]
            .between(0, 1).all()
            and comparison["in_sample_weighted_brier"]
            .between(0, 1).all()
        ),

    "All 132 fixed-effect coefficients retained":
        len(coefficient_table) == 132,

    "All fitted PSU-level effects retained":
        len(psu_effects) == 3 * (642 + 611),
}

validation = pd.DataFrame(
    [
        {
            "check": label,
            "status": (
                "PASS" if bool(passed) else "REVIEW"
            ),
        }
        for label, passed in checks.items()
    ]
)

all_passed = all(
    bool(value)
    for value in checks.values()
)


# Save exploratory model comparisons separately from verified survey results.
outputs = {
    "comparison":
        TABLES / "09c_nonspatial_multilevel_spatial_comparison.csv",

    "penalty_grid":
        TABLES / "09c_hierarchical_penalty_grid.csv",

    "coefficients":
        TABLES / "09c_comparison_fixed_effect_coefficients.csv",

    "psu_effects":
        TABLES / "09c_fitted_psu_effects_exploratory_ONLY.csv",

    "reconstruction":
        TABLES / "09c_baseline_reconstruction.csv",

    "checks":
        TABLES / "09c_model_comparison_checks.csv",

    "audit":
        LOGS / "09c_model_comparison_audit.json",

    "report":
        LOGS / "09c_model_comparison_report.txt",
}

for key, table in (
    ("comparison", comparison),
    ("penalty_grid", grid),
    ("coefficients", coefficient_table),
    ("psu_effects", psu_effects),
    ("reconstruction", reconstruction),
    ("checks", validation),
):
    table.to_csv(
        outputs[key],
        index=False,
        encoding="utf-8-sig",
    )

report = "\n".join(
    [
        "STEP 09.3 | NON-SPATIAL, MULTILEVEL AND SPATIAL MODEL COMPARISON",
        "=" * 70,
        "Fitting: weighted penalized binomial pseudo-likelihood",
        "PSU effects: independent Gaussian versus spatial proper GMRF",
        "Spatial graph: symmetric 8-NN on displaced DHS GPS",
        "Precision: lambda * (I + rho * normalized graph Laplacian)",
        f"Spatial smoothing rho: {SPATIAL_SMOOTHING}",
        f"Penalty grid: {list(PENALTY_GRID)}",
        "Penalty choice: exploratory Laplace pseudo-marginal score",
        f"Fitted stage-model combinations: {len(comparison)}",
        f"Penalty-grid candidates: {len(grid)}",
        "Performance: IN-SAMPLE ONLY",
        "Inference: NOT design-based spatial significance",
        f"Validation: {'PASS' if all_passed else 'REVIEW REQUIRED'}",
        f"Validated checks: {sum(checks.values())} / {len(checks)}",
    ]
)

outputs["report"].write_text(
    report + "\n",
    encoding="utf-8",
)

audit = {
    "step": "09.3",
    "phase": "D",
    "kr_sha256": kr_hash,
    "model_type": (
        "Weighted penalized logistic pseudo-likelihood; "
        "PSU effects estimated by joint MAP"
    ),
    "precision": (
        "lambda*(I + rho*normalized graph Laplacian)"
    ),
    "spatial_graph": (
        "Symmetric 8-nearest-neighbor graph "
        "using displaced DHS GPS coordinates"
    ),
    "penalties": list(PENALTY_GRID),
    "spatial_rho": SPATIAL_SMOOTHING,
    "selection_method": (
        "Conditional Laplace approximation integrating PSU "
        "effects at the jointly fitted fixed-effect estimates"
    ),
    "limitations": [
        "Not a fully integrated survey-weighted Bayesian GLMM.",
        "Not a design-based test of spatial random effects.",
        "Scores and predictive metrics are in-sample and exploratory.",
        "The spatial smoothing parameter is fixed.",
        "Hyperparameters are selected from a finite grid using the same data.",
        "PSU effects are estimated using their observed outcomes.",
        "PSU effects and residuals must not be reused as held-out PSU ML features.",
        "DHS GPS displacement prevents exact-location interpretation.",
    ],
    "checks": {
        key: bool(value)
        for key, value in checks.items()
    },
    "validation_passed": bool(all_passed),
}

outputs["audit"].write_text(
    json.dumps(
        audit,
        indent=2,
        ensure_ascii=False,
    ) + "\n",
    encoding="utf-8",
)

print(report)

print("\nTHREE-MODEL COMPARISON — IN-SAMPLE ONLY")
display(
    comparison[
        [
            "stage",
            "model",
            "n_children",
            "selected_lambda",
            "in_sample_weighted_log_loss",
            "in_sample_weighted_brier",
            "in_sample_weighted_auc",
            "approx_laplace_pseudo_score",
            "pseudo_score_gain_over_nonspatial",
            "residual_moran_i_8nn",
            "psu_effect_sd",
        ]
    ].round(5)
)

print("\nPENALTY SENSITIVITY AND OPTIMIZER DIAGNOSTICS")
display(grid.round(5))

print("\nBASELINE MODEL RECONSTRUCTION")
display(reconstruction.round(6))

print("\nVALIDATION CHECKS")
display(validation)

print("\nSAVED FILES")
for path in outputs.values():
    print(path)

if not all_passed:
    raise ValueError(
        "Spatial model comparison requires methodological review."
    )

STEP 09.3 | NON-SPATIAL, MULTILEVEL AND SPATIAL MODEL COMPARISON
Fitting: weighted penalized binomial pseudo-likelihood
PSU effects: independent Gaussian versus spatial proper GMRF
Spatial graph: symmetric 8-NN on displaced DHS GPS
Precision: lambda * (I + rho * normalized graph Laplacian)
Spatial smoothing rho: 4.0
Penalty grid: [0.25, 1.0, 4.0, 16.0]
Penalty choice: exploratory Laplace pseudo-marginal score
Fitted stage-model combinations: 6
Penalty-grid candidates: 16
Performance: IN-SAMPLE ONLY
Inference: NOT design-based spatial significance
Validation: PASS
Validated checks: 16 / 16

THREE-MODEL COMPARISON — IN-SAMPLE ONLY


,stage,model,n_children,selected_lambda,in_sample_weighted_log_loss,in_sample_weighted_brier,in_sample_weighted_auc,approx_laplace_pseudo_score,pseudo_score_gain_over_nonspatial,residual_moran_i_8nn,psu_effect_sd
0,Stage 1,Non-spatial,2624,NaN,0.52680,0.17312,0.63851,-1382.33178,0.00000,0.03990,0.00000
1,Stage 1,Multilevel PSU,2624,4.0,0.48958,0.15932,0.74334,-1377.25283,5.07895,0.03949,0.18286
2,Stage 1,Spatial GMRF PSU,2624,1.0,0.49239,0.16036,0.73560,-1377.31817,5.01361,0.01361,0.18308
3,Stage 2,Non-spatial,1969,NaN,0.65013,0.22926,0.64051,-1280.11016,0.00000,0.00236,0.00000
4,Stage 2,Multilevel PSU,1969,4.0,0.59272,0.20284,0.75992,-1265.11011,15.00004,0.00232,0.20089
5,Stage 2,Spatial GMRF PSU,1969,1.0,0.59875,0.20557,0.74900,-1268.42699,11.68317,-0.01890,0.19144



PENALTY SENSITIVITY AND OPTIMIZER DIAGNOSTICS


,stage,model,lambda,spatial_rho,approx_laplace_pseudo_score,optimizer_converged,max_gradient,iterations
0,Stage 1,Multilevel PSU,0.25,0.0,-1477.96803,True,0.00440,327
1,Stage 1,Multilevel PSU,1.00,0.0,-1394.07825,True,0.00792,170
2,Stage 1,Multilevel PSU,4.00,0.0,-1377.25283,True,0.00172,80
3,Stage 1,Multilevel PSU,16.00,0.0,-1379.58692,True,0.00374,48
4,Stage 1,Spatial GMRF PSU,0.25,4.0,-1391.69467,True,0.00998,266
5,Stage 1,Spatial GMRF PSU,1.00,4.0,-1377.31817,True,0.00425,161
6,Stage 1,Spatial GMRF PSU,4.00,4.0,-1379.66986,True,0.00315,72
7,Stage 1,Spatial GMRF PSU,16.00,4.0,-1381.52625,True,0.01569,48
8,Stage 2,Multilevel PSU,0.25,0.0,-1326.10077,True,0.00172,282
9,Stage 2,Multilevel PSU,1.00,0.0,-1266.57467,True,0.01174,140



BASELINE MODEL RECONSTRUCTION


,stage,n,active_psus,positive_n,model_columns,max_beta_reconstruction_error,previous_residual_moran_difference,min_knn_degree
0,Stage 1,2624,642,1969,22,0.0,0.0,8
1,Stage 2,1969,611,1117,22,0.0,0.0,8



VALIDATION CHECKS


,check,status
0,Source and Step 9.2 audit verified,PASS
1,Both original model populations retained,PASS
2,Expected active PSU counts retained,PASS
3,Both outcomes have correct event counts,PASS
4,All 22 fixed-effect terms included,PASS
5,Verified survey coefficients independently rep...,PASS
6,Step 9.2 residual Moran statistics reproduced,PASS
7,Each graph has at least eight neighbours per PSU,PASS
8,Three model classes fitted for each stage,PASS
9,All 16 penalty-grid fits available,PASS



SAVED FILES
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\09c_nonspatial_multilevel_spatial_comparison.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\09c_hierarchical_penalty_grid.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\09c_comparison_fixed_effect_coefficients.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\09c_fitted_psu_effects_exploratory_ONLY.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\09c_baseline_reconstruction.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\09c_model_comparison_checks.csv
D:\BDHS2022_FeverCare_SpatialEquity\logs\09c_model_comparison_audit.json
D:\BDHS2022_FeverCare_SpatialEquity\logs\09c_model_comparison_report.txt


In [25]:

from pathlib import Path
import hashlib
import json
import os
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn import __version__ as sklearn_version
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neighbors import BallTree
from sklearn.preprocessing import OneHotEncoder, StandardScaler


ROOT = Path(
    os.environ.get(
        "BDHS_PROJECT_ROOT",
        r"D:\BDHS2022_FeverCare_SpatialEquity",
    )
)

RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
TABLES = ROOT / "results" / "tables"
LOGS = ROOT / "logs"

paths = {
    "kr": RAW / "BDKR81FL.DTA",
    "cohort": PROCESSED / "06_fever_care_analysis_ready.csv",
    "gps": PROCESSED / "09a_psu_spatial_descriptive_base.csv",
    "source_audit": LOGS / "01_input_audit.json",
    "spatial_audit": LOGS / "09c_model_comparison_audit.json",
}

for label, path in paths.items():
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing verified input ({label}): {path}"
        )

with paths["source_audit"].open(encoding="utf-8") as f:
    source_audit = json.load(f)

with paths["spatial_audit"].open(encoding="utf-8") as f:
    spatial_audit = json.load(f)

if not spatial_audit.get("validation_passed", False):
    raise ValueError(
        "Step 9.3 prerequisite validation did not pass."
    )

with paths["kr"].open("rb") as f:
    kr_sha256 = hashlib.file_digest(
        f, "sha256"
    ).hexdigest()

if kr_sha256 != source_audit["source_files"]["kr_sha256"]:
    raise ValueError("Original KR checksum mismatch.")


# Prespecified geographical validation and model configurations.
SEED = 20261117
N_FOLDS = 5
BUFFER_KM = 10.0
EARTH_RADIUS_KM = 6371.0088

CATEGORICAL = [
    "child_sex",
    "cough",
    "rapid_breathing",
    "maternal_education",
    "wealth_quintile",
    "residence",
    "division",
]

BASE_NUMERIC = [
    "child_age_years",
    "maternal_age_5years",
    "birth_order",
]

GEO_NUMERIC = [
    "gps_latitude",
    "gps_longitude",
]

FEATURES = {
    "Non-spatial": BASE_NUMERIC,
    "Geographic-augmented": BASE_NUMERIC + GEO_NUMERIC,
}

assert not set([
    "h32y",
    "h46a",
    "h46b",
    "care_sought",
    "pharmacy_first",
    "v001",
    "v021",
]).intersection(
    CATEGORICAL + BASE_NUMERIC + GEO_NUMERIC
)


# Read the verified original child data and frozen eligibility indicators.
keys = ["v001", "v002", "v003", "bidx"]

raw = pd.read_stata(
    paths["kr"],
    columns=keys + [
        "v005",
        "v021",
        "v023",
        "v024",
        "v025",
        "h22",
        "h32y",
        "h46a",
        "h31",
        "h31b",
        "b19",
        "b4",
        "v012",
        "v106",
        "v190",
        "bord",
    ],
    convert_categoricals=False,
)

frozen = pd.read_csv(
    paths["cohort"],
    usecols=keys + [
        "stage1_core_complete_case",
        "stage2_core_complete_case",
    ],
)

gps = pd.read_csv(
    paths["gps"],
    usecols=[
        "v001",
        "v023",
        "gps_latitude",
        "gps_longitude",
    ],
)

if (
    len(raw) != 8784
    or raw.duplicated(keys).any()
    or len(gps) != 674
    or gps["v001"].duplicated().any()
    or raw["v001"].nunique() != 674
    or not raw["v001"].eq(raw["v021"]).all()
):
    raise ValueError(
        "Source identifiers, PSU counts or GPS inventory changed."
    )

if not gps[
    ["gps_latitude", "gps_longitude"]
].notna().all().all():
    raise ValueError(
        "GPS lookup contains missing coordinates."
    )

fever = raw["h22"].eq(1)
care = fever & raw["h32y"].eq(0)

complete = (
    raw[
        [
            "b19",
            "b4",
            "h31",
            "v012",
            "v106",
            "v190",
            "bord",
            "v025",
            "v024",
        ]
    ].notna().all(axis=1)
    & raw["h31b"].isin([0, 1])
)

domain_specs = {
    "Stage 1 care sought": (
        fever & complete,
        raw["h32y"].eq(0),
        "stage1_core_complete_case",
        2624,
        642,
        1969,
    ),
    "Stage 2 pharmacy first": (
        care & complete,
        raw["h46a"].eq(36),
        "stage2_core_complete_case",
        1969,
        611,
        1117,
    ),
}

for stage, (
    domain,
    _,
    flag,
    n,
    psus,
    positives,
) in domain_specs.items():

    ids = (
        raw.loc[domain, keys]
        .astype("int64")
        .sort_values(keys)
        .reset_index(drop=True)
    )

    saved = (
        frozen.loc[frozen[flag].eq(1), keys]
        .astype("int64")
        .sort_values(keys)
        .reset_index(drop=True)
    )

    if (
        not ids.equals(saved)
        or len(ids) != n
        or ids["v001"].nunique() != psus
    ):
        raise ValueError(
            f"Frozen eligibility mismatch: {stage}"
        )


# Create broad geographic partitions using displaced coordinates only.
# Outcome labels are never used to construct geographic regions.
lat_mean = gps["gps_latitude"].mean()

coords_km = np.column_stack([
    gps["gps_latitude"].to_numpy() * 111.2,
    (
        gps["gps_longitude"].to_numpy()
        * 111.2
        * np.cos(np.deg2rad(lat_mean))
    ),
])

geo_regions = KMeans(
    n_clusters=N_FOLDS,
    random_state=SEED,
    n_init=25,
).fit_predict(coords_km)

gps = gps.copy()
gps["geo_region"] = geo_regions

region_counts = gps.groupby(
    "geo_region"
)["v001"].nunique()

if (
    len(region_counts) != N_FOLDS
    or int(region_counts.min()) < 40
):
    raise ValueError(
        f"Geographic regions are poorly populated: "
        f"{region_counts.to_dict()}"
    )


# Calculate performance using original survey weights on held-out data.
def weighted_metrics(y, p, w):

    p = np.clip(
        np.asarray(p, dtype=float),
        1e-8,
        1 - 1e-8,
    )

    y = np.asarray(y, dtype=int)
    w = np.asarray(w, dtype=float)

    if (
        len(np.unique(y)) != 2
        or not np.isfinite(p).all()
        or (w <= 0).any()
    ):
        raise ValueError(
            "Invalid test class mix, predictions or weights."
        )

    observed_rate = np.average(y, weights=w)
    predicted_rate = np.average(p, weights=w)

    return {
        "roc_auc": float(
            roc_auc_score(
                y, p, sample_weight=w
            )
        ),
        "average_precision": float(
            average_precision_score(
                y, p, sample_weight=w
            )
        ),
        "brier": float(
            brier_score_loss(
                y, p, sample_weight=w
            )
        ),
        "log_loss": float(
            log_loss(
                y,
                p,
                sample_weight=w,
                labels=[0, 1],
            )
        ),
        "observed_pct": float(
            100 * observed_rate
        ),
        "predicted_pct": float(
            100 * predicted_rate
        ),
        "calibration_in_large_pp": float(
            100 * (predicted_rate - observed_rate)
        ),
    }


# Fix model configurations before viewing held-out performance.
def classifier(name, seed):

    if name == "Regularized logistic":
        return LogisticRegression(
            C=1.0,
            max_iter=1500,
            solver="lbfgs",
            random_state=seed,
        )

    if name == "Random forest":
        return RandomForestClassifier(
            n_estimators=120,
            max_depth=9,
            min_samples_leaf=8,
            max_features="sqrt",
            random_state=seed,
            n_jobs=2,
        )

    if name == "Histogram boosting":
        return HistGradientBoostingClassifier(
            max_iter=100,
            learning_rate=0.06,
            max_leaf_nodes=9,
            min_samples_leaf=20,
            l2_regularization=1.0,
            random_state=seed,
        )

    raise ValueError(
        f"Unsupported classifier: {name}"
    )


model_names = [
    "Regularized logistic",
    "Random forest",
    "Histogram boosting",
]

fold_rows = []
metric_rows = []
prediction_parts = []
warning_rows = []


# Fit models independently within every PSU-disjoint and spatial holdout.
for stage_index, (
    stage,
    (
        domain,
        outcome,
        _,
        expected_n,
        expected_psus,
        expected_positives,
    ),
) in enumerate(domain_specs.items()):

    data = (
        raw.loc[domain]
        .copy()
        .reset_index(drop=True)
    )

    data = data.merge(
        gps,
        on=["v001", "v023"],
        how="left",
        validate="many_to_one",
    )

    if data[
        [
            "gps_latitude",
            "gps_longitude",
            "geo_region",
        ]
    ].isna().any().any():
        raise ValueError(
            f"GPS join failed: {stage}"
        )

    y = outcome[domain].to_numpy(
        dtype=int
    )

    weight = (
        data["v005"].to_numpy(dtype=float)
        / 1_000_000
    )

    psu_ids = data["v001"].to_numpy(
        dtype=int
    )

    assert (
        len(y) == expected_n
        and y.sum() == expected_positives
        and len(np.unique(psu_ids)) == expected_psus
    )

    if (
        not np.isfinite(weight).all()
        or (weight <= 0).any()
    ):
        raise ValueError(
            f"Invalid weights in {stage}"
        )

    X = pd.DataFrame({
        "child_age_years": (
            data["b19"].to_numpy(dtype=float) / 12.0
        ),
        "maternal_age_5years": (
            data["v012"].to_numpy(dtype=float) - 25.0
        ) / 5.0,
        "birth_order": data["bord"].to_numpy(
            dtype=float
        ),
        "child_sex": data["b4"].to_numpy(
            dtype=int
        ),
        "cough": data["h31"].map({
            0: 0,
            2: 1,
        }).to_numpy(),
        "rapid_breathing": data["h31b"].to_numpy(
            dtype=int
        ),
        "maternal_education": data["v106"].to_numpy(
            dtype=int
        ),
        "wealth_quintile": data["v190"].to_numpy(
            dtype=int
        ),
        "residence": data["v025"].to_numpy(
            dtype=int
        ),
        "division": data["v024"].to_numpy(
            dtype=int
        ),
        "gps_latitude": data["gps_latitude"].to_numpy(
            dtype=float
        ),
        "gps_longitude": data["gps_longitude"].to_numpy(
            dtype=float
        ),
    })

    if X.isna().any().any():
        raise ValueError(
            f"Unexpected missing predictor: {stage}"
        )

    # PSU-disjoint folds attempt to balance the outcome classes.
    splitter = StratifiedGroupKFold(
        n_splits=N_FOLDS,
        shuffle=True,
        random_state=SEED,
    )

    psu_folds = list(
        splitter.split(
            X,
            y,
            psu_ids,
        )
    )

    # Spatial folds hold out entire coordinate-defined regions.
    regions = data["geo_region"].to_numpy(
        dtype=int
    )

    geographic_folds = [
        (
            np.flatnonzero(regions != region),
            np.flatnonzero(regions == region),
        )
        for region in sorted(np.unique(regions))
    ]

    schemes = [
        (
            "PSU-disjoint stratified",
            psu_folds,
        ),
        (
            "Geographic region + 10km coordinate buffer",
            geographic_folds,
        ),
    ]

    for scheme, folds in schemes:

        covered = np.zeros(
            len(y),
            dtype=int,
        )

        for fold_index, (
            candidate_train,
            test,
        ) in enumerate(folds, start=1):

            train = np.asarray(
                candidate_train,
                dtype=int,
            )

            test = np.asarray(
                test,
                dtype=int,
            )

            buffer_excluded = 0
            min_distance_km = np.nan

            if scheme.startswith("Geographic"):

                test_psus = (
                    data.iloc[test]
                    .drop_duplicates("v001")
                )

                train_psus = (
                    data.iloc[train]
                    .drop_duplicates("v001")
                )

                nearest = BallTree(
                    np.deg2rad(
                        test_psus[
                            [
                                "gps_latitude",
                                "gps_longitude",
                            ]
                        ].to_numpy()
                    ),
                    metric="haversine",
                )

                distances, _ = nearest.query(
                    np.deg2rad(
                        train_psus[
                            [
                                "gps_latitude",
                                "gps_longitude",
                            ]
                        ].to_numpy()
                    ),
                    k=1,
                )

                far_psus = train_psus.loc[
                    (
                        distances[:, 0] * EARTH_RADIUS_KM
                        > BUFFER_KM
                    ),
                    "v001",
                ].to_numpy()

                filtered = train[
                    np.isin(
                        psu_ids[train],
                        far_psus,
                    )
                ]

                buffer_excluded = int(
                    len(train) - len(filtered)
                )

                train = filtered

                if len(far_psus):
                    min_distance_km = float(
                        (
                            distances[:, 0]
                            * EARTH_RADIUS_KM
                        )[
                            np.isin(
                                train_psus[
                                    "v001"
                                ].to_numpy(),
                                far_psus,
                            )
                        ].min()
                    )

            overlap = int(
                np.intersect1d(
                    np.unique(psu_ids[train]),
                    np.unique(psu_ids[test]),
                ).size
            )

            if (
                len(train) < 500
                or len(test) < 50
                or overlap != 0
                or len(np.unique(y[train])) != 2
                or len(np.unique(y[test])) != 2
            ):
                raise ValueError(
                    f"Invalid held-out fold: "
                    f"{stage}, {scheme}, {fold_index}"
                )

            covered[test] += 1

            fold_rows.append({
                "stage": stage,
                "validation": scheme,
                "fold": fold_index,
                "train_children": len(train),
                "test_children": len(test),
                "train_psus": len(
                    np.unique(psu_ids[train])
                ),
                "test_psus": len(
                    np.unique(psu_ids[test])
                ),
                "train_positive": int(
                    y[train].sum()
                ),
                "test_positive": int(
                    y[test].sum()
                ),
                "overlapping_train_test_psus": overlap,
                "training_children_removed_by_buffer": (
                    buffer_excluded
                ),
                "minimum_train_test_displaced_coordinate_km": (
                    min_distance_km
                ),
                "geographic_test_region": (
                    int(regions[test][0])
                    if scheme.startswith("Geographic")
                    else np.nan
                ),
            })

            # Training weights are normalized within each fold.
            normalized_train_weight = (
                weight[train] / weight[train].mean()
            )

            for feature_index, (
                feature_set,
                numerical,
            ) in enumerate(FEATURES.items()):

                try:
                    encoder = OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=False,
                    )
                except TypeError:
                    encoder = OneHotEncoder(
                        handle_unknown="ignore",
                        sparse=False,
                    )

                preprocessing = ColumnTransformer(
                    [
                        (
                            "scaled_numeric",
                            StandardScaler(),
                            numerical,
                        ),
                        (
                            "categorical",
                            encoder,
                            CATEGORICAL,
                        ),
                    ],
                    remainder="drop",
                )

                numeric_train = (
                    preprocessing.fit_transform(
                        X.iloc[train]
                    )
                )

                numeric_test = (
                    preprocessing.transform(
                        X.iloc[test]
                    )
                )

                if (
                    not np.isfinite(
                        numeric_train
                    ).all()
                    or not np.isfinite(
                        numeric_test
                    ).all()
                ):
                    raise ValueError(
                        "Non-finite features after preprocessing."
                    )

                for model_index, model_name in enumerate(
                    model_names
                ):

                    seed = (
                        SEED
                        + 1000 * stage_index
                        + 100 * fold_index
                        + 10 * feature_index
                        + model_index
                    )

                    estimator = classifier(
                        model_name,
                        seed,
                    )

                    with warnings.catch_warnings(
                        record=True
                    ) as collected:

                        warnings.simplefilter(
                            "always"
                        )

                        estimator.fit(
                            numeric_train,
                            y[train],
                            sample_weight=(
                                normalized_train_weight
                            ),
                        )

                        probability = (
                            estimator.predict_proba(
                                numeric_test
                            )[:, 1]
                        )

                    for item in collected:
                        warning_rows.append({
                            "stage": stage,
                            "validation": scheme,
                            "fold": fold_index,
                            "feature_set": feature_set,
                            "model": model_name,
                            "warning": (
                                f"{item.category.__name__}: "
                                f"{item.message}"
                            ),
                        })

                    if (
                        not np.isfinite(
                            probability
                        ).all()
                        or np.any(
                            (probability < 0)
                            | (probability > 1)
                        )
                    ):
                        raise ValueError(
                            "Model produced invalid probabilities."
                        )

                    scores = weighted_metrics(
                        y[test],
                        probability,
                        weight[test],
                    )

                    metric_rows.append({
                        "stage": stage,
                        "validation": scheme,
                        "fold": fold_index,
                        "feature_set": feature_set,
                        "model": model_name,
                        "test_n": len(test),
                        "train_n": len(train),
                        **scores,
                    })

                    prediction_parts.append(
                        pd.DataFrame({
                            "stage": stage,
                            "validation": scheme,
                            "fold": fold_index,
                            "feature_set": feature_set,
                            "model": model_name,
                            "child_row_within_stage": (
                                test
                            ),
                            "psu_id": psu_ids[test],
                            "observed": y[test],
                            "predicted_probability": (
                                probability
                            ),
                            "survey_weight": (
                                weight[test]
                            ),
                        })
                    )

        if not (covered == 1).all():
            raise ValueError(
                f"Test folds do not partition "
                f"{stage}, {scheme}."
            )


# Pool genuinely out-of-fold predictions within each validation design.
fold_diagnostics = pd.DataFrame(
    fold_rows
)

fold_metrics = pd.DataFrame(
    metric_rows
)

oof = pd.concat(
    prediction_parts,
    ignore_index=True,
)

summary_rows = []

for (
    stage,
    scheme,
    feature_set,
    model_name,
), chunk in oof.groupby(
    [
        "stage",
        "validation",
        "feature_set",
        "model",
    ],
    sort=False,
):

    score = weighted_metrics(
        chunk["observed"].to_numpy(),
        chunk["predicted_probability"].to_numpy(),
        chunk["survey_weight"].to_numpy(),
    )

    rows = fold_metrics.loc[
        fold_metrics["stage"].eq(stage)
        & fold_metrics["validation"].eq(scheme)
        & fold_metrics["feature_set"].eq(
            feature_set
        )
        & fold_metrics["model"].eq(
            model_name
        )
    ]

    summary_rows.append({
        "stage": stage,
        "validation": scheme,
        "feature_set": feature_set,
        "model": model_name,
        "oof_n": len(chunk),
        "folds": len(rows),
        **{
            f"oof_{name}": val
            for name, val in score.items()
        },
        "fold_mean_roc_auc": float(
            rows["roc_auc"].mean()
        ),
        "fold_sd_roc_auc": float(
            rows["roc_auc"].std(ddof=1)
        ),
        "fold_mean_brier": float(
            rows["brier"].mean()
        ),
    })

summary = pd.DataFrame(
    summary_rows
)


# Compare geographic augmentation with the non-coordinate baseline.
base = summary.loc[
    summary["feature_set"].eq(
        "Non-spatial"
    )
]

augmented = summary.loc[
    summary["feature_set"].eq(
        "Geographic-augmented"
    )
]

gains = base.merge(
    augmented,
    on=[
        "stage",
        "validation",
        "model",
    ],
    validate="one_to_one",
    suffixes=("_base", "_geo"),
)

gains["delta_auc_geo_minus_base"] = (
    gains["oof_roc_auc_geo"]
    - gains["oof_roc_auc_base"]
)

gains["delta_brier_geo_minus_base"] = (
    gains["oof_brier_geo"]
    - gains["oof_brier_base"]
)

gains["delta_log_loss_geo_minus_base"] = (
    gains["oof_log_loss_geo"]
    - gains["oof_log_loss_base"]
)


# Validate leakage prevention, geographic holdouts and model outputs.
unique_predictions = not oof.duplicated(
    [
        "stage",
        "validation",
        "feature_set",
        "model",
        "child_row_within_stage",
    ]
).any()

expected_oof = (
    sum(
        spec[3]
        for spec in domain_specs.values()
    )
    * 2
    * 2
    * len(model_names)
)

checks = {
    "Prerequisite Step 9.3 validated":
        bool(
            spatial_audit["validation_passed"]
        ),

    "Original 8,784 KR records and 674 GPS PSUs retained":
        (
            len(raw) == 8784
            and len(gps) == 674
        ),

    "Stage 1 and Stage 2 analytic populations reproduced":
        (
            len(
                oof.loc[
                    oof["stage"].eq(
                        "Stage 1 care sought"
                    )
                ]
            ) == 2624 * 12
            and len(
                oof.loc[
                    oof["stage"].eq(
                        "Stage 2 pharmacy first"
                    )
                ]
            ) == 1969 * 12
        ),

    "Geographic partition covers all 674 PSUs":
        (
            int(region_counts.sum()) == 674
            and len(region_counts) == 5
        ),

    "Twenty held-out fold designs created":
        len(fold_diagnostics) == 20,

    "All training and test PSU sets are disjoint":
        fold_diagnostics[
            "overlapping_train_test_psus"
        ].eq(0).all(),

    "Geographic region test folds use a 10 km displaced-coordinate buffer":
        bool(
            fold_diagnostics.loc[
                fold_diagnostics[
                    "validation"
                ].str.startswith(
                    "Geographic"
                ),
                "minimum_train_test_displaced_coordinate_km",
            ].gt(BUFFER_KM).all()
        ),

    "Every held-out fold has both outcome classes":
        bool(
            fold_diagnostics[
                "test_positive"
            ].gt(0).all()
            and fold_diagnostics[
                "test_positive"
            ].lt(
                fold_diagnostics[
                    "test_children"
                ]
            ).all()
        ),

    "All 120 cross-validation model fits completed":
        len(fold_metrics) == 120,

    "All 24 OOF summary combinations completed":
        len(summary) == 24,

    "All 12 geographic feature gain comparisons completed":
        len(gains) == 12,

    "Every child has exactly one held-out prediction per model and scheme":
        (
            unique_predictions
            and len(oof) == expected_oof
        ),

    "All weighted evaluation metrics are finite":
        bool(
            np.isfinite(
                fold_metrics[
                    [
                        "roc_auc",
                        "average_precision",
                        "brier",
                        "log_loss",
                    ]
                ].to_numpy()
            ).all()
            and np.isfinite(
                summary[
                    [
                        "oof_roc_auc",
                        "oof_average_precision",
                        "oof_brier",
                        "oof_log_loss",
                    ]
                ].to_numpy()
            ).all()
        ),

    "All probability metrics lie in valid ranges":
        bool(
            fold_metrics[
                [
                    "roc_auc",
                    "average_precision",
                    "brier",
                ]
            ].apply(
                lambda col: col.between(
                    0, 1
                ).all()
            ).all()
            and summary[
                [
                    "oof_roc_auc",
                    "oof_average_precision",
                    "oof_brier",
                ]
            ].apply(
                lambda col: col.between(
                    0, 1
                ).all()
            ).all()
        ),

    "No failed numeric fits or model warnings":
        len(warning_rows) == 0,

    "No outcome-derived PSU statistics used as predictors":
        bool(
            set(
                CATEGORICAL
                + BASE_NUMERIC
                + GEO_NUMERIC
            ).isdisjoint({
                "care_sought",
                "pharmacy_first",
                "h32y",
                "h46a",
                "h46b",
                "care_rate_weighted",
                "pharmacy_rate_pooled_shrink5",
                "fitted_psu_effect",
                "weighted_gap",
                "v001",
            })
        ),
}

validation = pd.DataFrame([
    {
        "check": k,
        "status": (
            "PASS"
            if bool(v)
            else "REVIEW"
        ),
    }
    for k, v in checks.items()
])

passed = all(
    bool(v)
    for v in checks.values()
)


# Save held-out estimates and fold metadata without altering earlier outputs.
outputs = {
    "folds":
        TABLES / "10a_cv_fold_coverage.csv",

    "fold_metrics":
        TABLES / "10a_ml_fold_metrics.csv",

    "pooled":
        TABLES / "10a_ml_oof_summary.csv",

    "gains":
        TABLES / "10a_geographic_feature_gains.csv",

    "oof":
        PROCESSED / "10a_heldout_child_predictions.csv",

    "checks":
        TABLES / "10a_ml_validation_checks.csv",

    "audit":
        LOGS / "10a_geographic_ml_audit.json",

    "report":
        LOGS / "10a_geographic_ml_report.txt",
}

for name, table in [
    ("folds", fold_diagnostics),
    ("fold_metrics", fold_metrics),
    ("pooled", summary),
    ("gains", gains),
    ("oof", oof),
    ("checks", validation),
]:
    table.to_csv(
        outputs[name],
        index=False,
        encoding="utf-8-sig",
    )

report = "\n".join([
    "STEP 10.1 | PSU-DISJOINT AND GEOGRAPHICALLY HELD-OUT ML",
    "=" * 66,
    (
        "Outcomes: Stage 1 care seeking; "
        "Stage 2 pharmacy first among care seekers"
    ),
    (
        "Feature sets: non-spatial vs "
        "displaced-GPS geographic-augmented"
    ),
    (
        "Models: regularized logistic, "
        "random forest, histogram boosting"
    ),
    (
        f"Validation: {N_FOLDS}-fold PSU-grouped "
        f"and {N_FOLDS}-region geographic holdout"
    ),
    (
        f"Geographic training exclusion: {BUFFER_KM:.0f} km "
        "around held-out displaced PSU points"
    ),
    (
        f"Model fold fits: {len(fold_metrics)}; "
        f"pooled OOF results: {len(summary)}"
    ),
    (
        "All preprocessing trained only within "
        "the corresponding training fold"
    ),
    (
        "Evaluation: held-out survey-weighted ROC-AUC, "
        "AP, Brier, log loss, calibration gap"
    ),
    (
        "Caution: spatial region assignments use displaced GPS; "
        "evaluation is descriptive"
    ),
    (
        "Validation: "
        f"{'PASS' if passed else 'REVIEW REQUIRED'}"
    ),
    (
        f"Validated checks: "
        f"{sum(bool(v) for v in checks.values())}/{len(checks)}"
    ),
])

outputs["report"].write_text(
    report + "\n",
    encoding="utf-8",
)

audit = {
    "step": "10.1",
    "phase": "E - geographic generalizability",
    "kr_sha256": kr_sha256,
    "sklearn_version": sklearn_version,
    "seed": SEED,
    "spatial_partition": (
        "KMeans of all 674 displaced PSU coordinates; "
        "outcome labels never used"
    ),
    "geographic_training_buffer_on_displaced_coordinates_km": (
        BUFFER_KM
    ),
    "models": model_names,
    "feature_sets": {
        key: CATEGORICAL + value
        for key, value in FEATURES.items()
    },
    "sampling": (
        "v005 / 1e6; training weights normalized "
        "within each fold to mean one"
    ),
    "limitations": [
        (
            "No nested hyperparameter tuning: models have "
            "fixed exploratory specifications."
        ),
        (
            "Geographic regions use displaced GPS; "
            "no exact-location or household-level claims."
        ),
        (
            "Fold standard deviations are descriptive, "
            "not survey-design confidence intervals."
        ),
        (
            "No outcome-derived PSU estimates "
            "or effects enter model features."
        ),
        "No random row-level cross-validation.",
    ],
    "model_warnings": warning_rows,
    "checks": {
        key: bool(value)
        for key, value in checks.items()
    },
    "validation_passed": passed,
}

outputs["audit"].write_text(
    json.dumps(
        audit,
        indent=2,
    ) + "\n",
    encoding="utf-8",
)

print(report)

print("\nHELD-OUT OOF MODEL PERFORMANCE (SURVEY-WEIGHTED)")
display(
    summary[
        [
            "stage",
            "validation",
            "feature_set",
            "model",
            "oof_n",
            "oof_roc_auc",
            "oof_average_precision",
            "oof_brier",
            "oof_log_loss",
            "oof_calibration_in_large_pp",
        ]
    ].round(4)
)

print("\nGEOGRAPHIC FEATURES: PAIRED HELD-OUT DIFFERENCES")
display(
    gains[
        [
            "stage",
            "validation",
            "model",
            "delta_auc_geo_minus_base",
            "delta_brier_geo_minus_base",
            "delta_log_loss_geo_minus_base",
        ]
    ].round(5)
)

print("\nFOLD COVERAGE AND GEOGRAPHIC BUFFER")
display(
    fold_diagnostics.round(3)
)

print("\nVALIDATION CHECKS")
display(validation)

print("\nSAVED FILES")
for path in outputs.values():
    print(path)

if not passed:
    raise ValueError(
        "Geographic ML validation requires review."
    )

STEP 10.1 | PSU-DISJOINT AND GEOGRAPHICALLY HELD-OUT ML
Outcomes: Stage 1 care seeking; Stage 2 pharmacy first among care seekers
Feature sets: non-spatial vs displaced-GPS geographic-augmented
Models: regularized logistic, random forest, histogram boosting
Validation: 5-fold PSU-grouped and 5-region geographic holdout
Geographic training exclusion: 10 km around held-out displaced PSU points
Model fold fits: 120; pooled OOF results: 24
All preprocessing trained only within the corresponding training fold
Evaluation: held-out survey-weighted ROC-AUC, AP, Brier, log loss, calibration gap
Caution: spatial region assignments use displaced GPS; evaluation is descriptive
Validation: PASS
Validated checks: 16/16

HELD-OUT OOF MODEL PERFORMANCE (SURVEY-WEIGHTED)


,stage,validation,feature_set,model,oof_n,oof_roc_auc,oof_average_precision,oof_brier,oof_log_loss,oof_calibration_in_large_pp
0,Stage 1 care sought,PSU-disjoint stratified,Non-spatial,Regularized logistic,2624,0.6001,0.8166,0.1775,0.5390,0.1549
1,Stage 1 care sought,PSU-disjoint stratified,Non-spatial,Random forest,2624,0.5850,0.8093,0.1778,0.5400,0.1043
2,Stage 1 care sought,PSU-disjoint stratified,Non-spatial,Histogram boosting,2624,0.5768,0.8080,0.1795,0.5449,0.2412
3,Stage 1 care sought,PSU-disjoint stratified,Geographic-augmented,Regularized logistic,2624,0.5973,0.8148,0.1778,0.5398,0.1088
4,Stage 1 care sought,PSU-disjoint stratified,Geographic-augmented,Random forest,2624,0.5958,0.8100,0.1767,0.5375,0.0717
5,Stage 1 care sought,PSU-disjoint stratified,Geographic-augmented,Histogram boosting,2624,0.5710,0.8057,0.1804,0.5474,0.2249
6,Stage 1 care sought,Geographic region + 10km coordinate buffer,Non-spatial,Regularized logistic,2624,0.5996,0.8185,0.1785,0.5410,-0.0518
7,Stage 1 care sought,Geographic region + 10km coordinate buffer,Non-spatial,Random forest,2624,0.5794,0.8075,0.1792,0.5430,-0.2224
8,Stage 1 care sought,Geographic region + 10km coordinate buffer,Non-spatial,Histogram boosting,2624,0.5709,0.7982,0.1829,0.5549,-0.0217
9,Stage 1 care sought,Geographic region + 10km coordinate buffer,Geographic-augmented,Regularized logistic,2624,0.5952,0.8169,0.1791,0.5425,-0.1911



GEOGRAPHIC FEATURES: PAIRED HELD-OUT DIFFERENCES


,stage,validation,model,delta_auc_geo_minus_base,delta_brier_geo_minus_base,delta_log_loss_geo_minus_base
0,Stage 1 care sought,PSU-disjoint stratified,Regularized logistic,-0.00278,0.00027,0.00087
1,Stage 1 care sought,PSU-disjoint stratified,Random forest,0.01085,-0.00110,-0.00248
2,Stage 1 care sought,PSU-disjoint stratified,Histogram boosting,-0.00581,0.00097,0.00255
3,Stage 1 care sought,Geographic region + 10km coordinate buffer,Regularized logistic,-0.00441,0.00062,0.00152
4,Stage 1 care sought,Geographic region + 10km coordinate buffer,Random forest,0.00998,-0.00070,-0.00253
5,Stage 1 care sought,Geographic region + 10km coordinate buffer,Histogram boosting,-0.01509,0.00233,0.00730
6,Stage 2 pharmacy first,PSU-disjoint stratified,Regularized logistic,-0.00429,0.00106,0.00253
7,Stage 2 pharmacy first,PSU-disjoint stratified,Random forest,0.00986,-0.00118,-0.00231
8,Stage 2 pharmacy first,PSU-disjoint stratified,Histogram boosting,0.01443,-0.00284,-0.00474
9,Stage 2 pharmacy first,Geographic region + 10km coordinate buffer,Regularized logistic,-0.01073,0.00267,0.00606



FOLD COVERAGE AND GEOGRAPHIC BUFFER


,stage,validation,fold,train_children,test_children,train_psus,test_psus,train_positive,test_positive,overlapping_train_test_psus,training_children_removed_by_buffer,minimum_train_test_displaced_coordinate_km,geographic_test_region
0,Stage 1 care sought,PSU-disjoint stratified,1,2098,526,515,127,1574,395,0,0,NaN,NaN
1,Stage 1 care sought,PSU-disjoint stratified,2,2099,525,512,130,1574,395,0,0,NaN,NaN
2,Stage 1 care sought,PSU-disjoint stratified,3,2098,526,515,127,1576,393,0,0,NaN,NaN
3,Stage 1 care sought,PSU-disjoint stratified,4,2099,525,512,130,1575,394,0,0,NaN,NaN
4,Stage 1 care sought,PSU-disjoint stratified,5,2102,522,514,128,1577,392,0,0,NaN,NaN
5,Stage 1 care sought,Geographic region + 10km coordinate buffer,1,2061,546,483,154,1549,407,0,17,11.060,0.0
6,Stage 1 care sought,Geographic region + 10km coordinate buffer,2,2058,566,482,160,1554,415,0,0,10.132,1.0
7,Stage 1 care sought,Geographic region + 10km coordinate buffer,3,2266,358,579,63,1688,281,0,0,15.361,2.0
8,Stage 1 care sought,Geographic region + 10km coordinate buffer,4,2025,592,504,135,1524,441,0,7,10.132,3.0
9,Stage 1 care sought,Geographic region + 10km coordinate buffer,5,2056,562,511,130,1540,425,0,6,11.060,4.0



VALIDATION CHECKS


,check,status
0,Prerequisite Step 9.3 validated,PASS
1,"Original 8,784 KR records and 674 GPS PSUs ret...",PASS
2,Stage 1 and Stage 2 analytic populations repro...,PASS
3,Geographic partition covers all 674 PSUs,PASS
4,Twenty held-out fold designs created,PASS
5,All training and test PSU sets are disjoint,PASS
6,Geographic region test folds use a 10 km displ...,PASS
7,Every held-out fold has both outcome classes,PASS
8,All 120 cross-validation model fits completed,PASS
9,All 24 OOF summary combinations completed,PASS



SAVED FILES
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\10a_cv_fold_coverage.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\10a_ml_fold_metrics.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\10a_ml_oof_summary.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\10a_geographic_feature_gains.csv
D:\BDHS2022_FeverCare_SpatialEquity\data\processed\10a_heldout_child_predictions.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\10a_ml_validation_checks.csv
D:\BDHS2022_FeverCare_SpatialEquity\logs\10a_geographic_ml_audit.json
D:\BDHS2022_FeverCare_SpatialEquity\logs\10a_geographic_ml_report.txt


In [27]:

from pathlib import Path
from importlib.util import find_spec
from importlib.metadata import version
from packaging.version import Version
import importlib
import subprocess
import sys
import hashlib
import json
import os
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn import __version__ as sklearn_version
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neighbors import BallTree
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# Install missing dependencies in the active Jupyter Python environment.
EXTERNAL_PACKAGES = ["xgboost", "lightgbm", "catboost"]

missing_packages = [
    package
    for package in EXTERNAL_PACKAGES
    if find_spec(package) is None
]

if missing_packages:
    print(
        "Installing missing packages:",
        ", ".join(missing_packages),
        flush=True,
    )

    command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        "--no-input",
        *missing_packages,
    ]

    installation = subprocess.run(
        command,
        text=True,
        capture_output=True,
    )

    if installation.returncode != 0:
        print(installation.stdout[-2000:])
        print(installation.stderr[-2000:])

        raise RuntimeError(
            "Automatic installation failed. In this notebook, run "
            "%pip install "
            + " ".join(missing_packages)
            + ", then rerun this complete cell."
        )

    print(installation.stdout[-1500:])
    importlib.invalidate_caches()

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier


ROOT = Path(
    os.environ.get(
        "BDHS_PROJECT_ROOT",
        r"D:\BDHS2022_FeverCare_SpatialEquity",
    )
)

RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
TABLES = ROOT / "results" / "tables"
LOGS = ROOT / "logs"

paths = {
    "kr": RAW / "BDKR81FL.DTA",
    "cohort": PROCESSED / "06_fever_care_analysis_ready.csv",
    "gps": PROCESSED / "09a_psu_spatial_descriptive_base.csv",
    "source_audit": LOGS / "01_input_audit.json",
    "spatial_audit": LOGS / "09c_model_comparison_audit.json",
}

for name, path in paths.items():
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing prerequisite ({name}): {path}"
        )

with paths["source_audit"].open(encoding="utf-8") as file:
    source_audit = json.load(file)

with paths["spatial_audit"].open(encoding="utf-8") as file:
    spatial_audit = json.load(file)

if not spatial_audit.get("validation_passed", False):
    raise ValueError(
        "Step 9.3 prerequisite validation has not passed."
    )

with paths["kr"].open("rb") as file:
    kr_sha256 = hashlib.file_digest(
        file, "sha256"
    ).hexdigest()

if kr_sha256 != source_audit["source_files"]["kr_sha256"]:
    raise ValueError(
        "Original KR source checksum mismatch."
    )


# Prespecified evaluation configuration.
SEED = 20261117
N_FOLDS = 5
BUFFER_KM = 10.0
EARTH_RADIUS_KM = 6371.0088

CATEGORICAL = [
    "child_sex",
    "cough",
    "rapid_breathing",
    "maternal_education",
    "wealth_quintile",
    "residence",
    "division",
]

BASE_NUMERIC = [
    "child_age_years",
    "maternal_age_5years",
    "birth_order",
]

GEO_NUMERIC = [
    "gps_latitude",
    "gps_longitude",
]

FEATURES = {
    "Non-spatial": BASE_NUMERIC,
    "Geographic-augmented": BASE_NUMERIC + GEO_NUMERIC,
}

prohibited_features = {
    "h32y", "h46a", "h46b",
    "care_sought", "pharmacy_first",
    "v001", "v021",
    "care_rate_weighted",
    "pharmacy_rate_pooled_shrink5",
    "fitted_psu_effect",
    "weighted_gap",
}

all_predictors = set(
    CATEGORICAL + BASE_NUMERIC + GEO_NUMERIC
)

if not all_predictors.isdisjoint(prohibited_features):
    raise ValueError(
        "Outcome-derived or PSU-identifying predictor detected."
    )


# Load the original KR data and frozen analysis populations.
keys = ["v001", "v002", "v003", "bidx"]

raw = pd.read_stata(
    paths["kr"],
    columns=keys + [
        "v005", "v021", "v023", "v024", "v025",
        "h22", "h32y", "h46a", "h31", "h31b",
        "b19", "b4", "v012", "v106", "v190", "bord",
    ],
    convert_categoricals=False,
)

frozen = pd.read_csv(
    paths["cohort"],
    usecols=keys + [
        "stage1_core_complete_case",
        "stage2_core_complete_case",
    ],
)

gps = pd.read_csv(
    paths["gps"],
    usecols=[
        "v001",
        "v023",
        "gps_latitude",
        "gps_longitude",
    ],
)

if not (
    len(raw) == 8784
    and not raw.duplicated(keys).any()
    and raw["v001"].nunique() == 674
    and raw["v001"].eq(raw["v021"]).all()
    and len(gps) == 674
    and gps["v001"].is_unique
    and gps[
        ["gps_latitude", "gps_longitude"]
    ].notna().all().all()
):
    raise ValueError(
        "Original survey or GPS cluster inventory is inconsistent."
    )

fever = raw["h22"].eq(1)
care = fever & raw["h32y"].eq(0)

complete = (
    raw[
        [
            "b19", "b4", "h31", "v012",
            "v106", "v190", "bord",
            "v025", "v024",
        ]
    ].notna().all(axis=1)
    & raw["h31b"].isin([0, 1])
)

domain_specs = {
    "Stage 1 care sought": (
        fever & complete,
        raw["h32y"].eq(0),
        "stage1_core_complete_case",
        2624,
        642,
        1969,
    ),
    "Stage 2 pharmacy first": (
        care & complete,
        raw["h46a"].eq(36),
        "stage2_core_complete_case",
        1969,
        611,
        1117,
    ),
}

for stage, (
    domain,
    outcome,
    flag,
    expected_n,
    expected_psus,
    expected_events,
) in domain_specs.items():

    actual_ids = (
        raw.loc[domain, keys]
        .astype("int64")
        .sort_values(keys)
        .reset_index(drop=True)
    )

    frozen_ids = (
        frozen.loc[frozen[flag].eq(1), keys]
        .astype("int64")
        .sort_values(keys)
        .reset_index(drop=True)
    )

    if not (
        actual_ids.equals(frozen_ids)
        and len(actual_ids) == expected_n
        and actual_ids["v001"].nunique() == expected_psus
        and int(outcome[domain].sum()) == expected_events
    ):
        raise ValueError(
            f"Frozen analysis population mismatch: {stage}"
        )


# Generate broad geographic regions without using outcome labels.
mean_latitude = gps["gps_latitude"].mean()

coordinate_matrix = np.column_stack([
    gps["gps_latitude"].to_numpy(dtype=float) * 111.2,
    gps["gps_longitude"].to_numpy(dtype=float)
    * 111.2
    * np.cos(np.deg2rad(mean_latitude)),
])

regions = KMeans(
    n_clusters=N_FOLDS,
    random_state=SEED,
    n_init=25,
).fit_predict(coordinate_matrix)

gps = gps.copy()
gps["geo_region"] = regions

region_counts = gps.groupby(
    "geo_region"
)["v001"].nunique()

if (
    len(region_counts) != N_FOLDS
    or region_counts.min() < 40
):
    raise ValueError(
        "Geographic regions have insufficient coverage: "
        + str(region_counts.to_dict())
    )


# Calculate survey-weighted held-out performance.
def weighted_metrics(y, probability, weight):

    y = np.asarray(y, dtype=int)

    p = np.clip(
        np.asarray(probability, dtype=float),
        1e-8,
        1 - 1e-8,
    )

    w = np.asarray(weight, dtype=float)

    if not (
        len(np.unique(y)) == 2
        and np.isfinite(p).all()
        and np.isfinite(w).all()
        and (w > 0).all()
    ):
        raise ValueError(
            "Invalid held-out outcomes, probabilities or weights."
        )

    observed = float(
        np.average(y, weights=w)
    )

    predicted = float(
        np.average(p, weights=w)
    )

    return {
        "roc_auc": float(
            roc_auc_score(
                y, p, sample_weight=w
            )
        ),
        "average_precision": float(
            average_precision_score(
                y, p, sample_weight=w
            )
        ),
        "brier": float(
            brier_score_loss(
                y, p, sample_weight=w
            )
        ),
        "log_loss": float(
            log_loss(
                y, p,
                sample_weight=w,
                labels=[0, 1],
            )
        ),
        "observed_pct": 100 * observed,
        "predicted_pct": 100 * predicted,
        "calibration_in_large_pp": (
            100 * (predicted - observed)
        ),
    }


# Define eight fixed, reproducible ML specifications.
def classifier(name, seed):

    if name == "Regularized logistic":
        return LogisticRegression(
            C=1.0,
            max_iter=1500,
            solver="lbfgs",
            random_state=seed,
        )

    if name == "Elastic net logistic":

        compatibility = (
            {}
            if Version(sklearn_version) >= Version("1.8")
            else {"penalty": "elasticnet"}
        )

        return LogisticRegression(
            **compatibility,
            l1_ratio=0.5,
            C=1.0,
            solver="saga",
            max_iter=3500,
            tol=5e-4,
            random_state=seed,
        )

    if name == "Random forest":
        return RandomForestClassifier(
            n_estimators=120,
            max_depth=9,
            min_samples_leaf=8,
            max_features="sqrt",
            random_state=seed,
            n_jobs=2,
        )

    if name == "Extra trees":
        return ExtraTreesClassifier(
            n_estimators=150,
            max_depth=10,
            min_samples_leaf=6,
            max_features="sqrt",
            random_state=seed,
            n_jobs=2,
        )

    if name == "Histogram boosting":
        return HistGradientBoostingClassifier(
            max_iter=100,
            learning_rate=0.06,
            max_leaf_nodes=9,
            min_samples_leaf=20,
            l2_regularization=1.0,
            random_state=seed,
        )

    if name == "XGBoost":
        return XGBClassifier(
            n_estimators=140,
            max_depth=3,
            learning_rate=0.05,
            min_child_weight=6,
            subsample=0.85,
            colsample_bytree=0.90,
            reg_lambda=5.0,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=seed,
            n_jobs=2,
            verbosity=0,
        )

    if name == "LightGBM":
        return LGBMClassifier(
            n_estimators=140,
            learning_rate=0.05,
            num_leaves=9,
            max_depth=5,
            min_child_samples=25,
            reg_lambda=5.0,
            subsample=0.85,
            subsample_freq=1,
            colsample_bytree=0.90,
            random_state=seed,
            n_jobs=2,
            verbosity=-1,
        )

    if name == "CatBoost":
        return CatBoostClassifier(
            iterations=160,
            depth=4,
            learning_rate=0.05,
            l2_leaf_reg=5.0,
            loss_function="Logloss",
            random_seed=seed,
            thread_count=2,
            verbose=False,
            allow_writing_files=False,
        )

    raise ValueError(
        f"Unsupported ML model: {name}"
    )


# Retain the previous three-model ordering for reproducibility.
model_names = [
    "Regularized logistic",
    "Random forest",
    "Histogram boosting",
    "Elastic net logistic",
    "Extra trees",
    "XGBoost",
    "LightGBM",
    "CatBoost",
]

fold_rows = []
metric_rows = []
prediction_parts = []
warning_rows = []


# Evaluate both outcomes using PSU-disjoint and geographic holdouts.
for stage_index, (
    stage,
    (
        domain,
        outcome,
        flag,
        expected_n,
        expected_psus,
        expected_events,
    ),
) in enumerate(domain_specs.items()):

    data = (
        raw.loc[domain]
        .copy()
        .reset_index(drop=True)
    )

    data = data.merge(
        gps,
        on=["v001", "v023"],
        how="left",
        validate="many_to_one",
    )

    if data[
        [
            "gps_latitude",
            "gps_longitude",
            "geo_region",
        ]
    ].isna().any().any():
        raise ValueError(
            f"GPS linkage failed: {stage}"
        )

    y = outcome[domain].to_numpy(dtype=int)

    weight = (
        data["v005"].to_numpy(dtype=float)
        / 1_000_000
    )

    psu_ids = data["v001"].to_numpy(dtype=int)

    if not (
        len(y) == expected_n
        and int(y.sum()) == expected_events
        and len(np.unique(psu_ids)) == expected_psus
        and np.isfinite(weight).all()
        and (weight > 0).all()
    ):
        raise ValueError(
            f"Model sample or sampling weights differ: {stage}"
        )

    X = pd.DataFrame({
        "child_age_years": (
            data["b19"].to_numpy(dtype=float) / 12
        ),
        "maternal_age_5years": (
            data["v012"].to_numpy(dtype=float) - 25
        ) / 5,
        "birth_order": data["bord"].to_numpy(dtype=float),
        "child_sex": data["b4"].to_numpy(dtype=int),
        "cough": data["h31"].map({
            0: 0,
            2: 1,
        }).to_numpy(),
        "rapid_breathing": (
            data["h31b"].to_numpy(dtype=int)
        ),
        "maternal_education": (
            data["v106"].to_numpy(dtype=int)
        ),
        "wealth_quintile": (
            data["v190"].to_numpy(dtype=int)
        ),
        "residence": data["v025"].to_numpy(dtype=int),
        "division": data["v024"].to_numpy(dtype=int),
        "gps_latitude": (
            data["gps_latitude"].to_numpy(dtype=float)
        ),
        "gps_longitude": (
            data["gps_longitude"].to_numpy(dtype=float)
        ),
    })

    if X.isna().any().any():
        raise ValueError(
            f"Missing predictor among complete cases: {stage}"
        )

    splitter = StratifiedGroupKFold(
        n_splits=N_FOLDS,
        shuffle=True,
        random_state=SEED,
    )

    psu_folds = list(
        splitter.split(
            X,
            y,
            psu_ids,
        )
    )

    region_labels = data[
        "geo_region"
    ].to_numpy(dtype=int)

    geographic_folds = [
        (
            np.flatnonzero(region_labels != region),
            np.flatnonzero(region_labels == region),
        )
        for region in sorted(np.unique(region_labels))
    ]

    validation_schemes = [
        (
            "PSU-disjoint stratified",
            psu_folds,
        ),
        (
            "Geographic region + 10km coordinate buffer",
            geographic_folds,
        ),
    ]

    for scheme, folds in validation_schemes:

        test_coverage = np.zeros(
            len(y),
            dtype=int,
        )

        for fold_number, (
            candidate_train,
            test,
        ) in enumerate(folds, start=1):

            train = np.asarray(
                candidate_train,
                dtype=int,
            )

            test = np.asarray(
                test,
                dtype=int,
            )

            buffer_excluded = 0
            minimum_distance_km = np.nan

            if scheme.startswith("Geographic"):

                test_psus = (
                    data.iloc[test]
                    .drop_duplicates("v001")
                )

                train_psus = (
                    data.iloc[train]
                    .drop_duplicates("v001")
                )

                nearest = BallTree(
                    np.deg2rad(
                        test_psus[
                            [
                                "gps_latitude",
                                "gps_longitude",
                            ]
                        ].to_numpy(dtype=float)
                    ),
                    metric="haversine",
                )

                distances, _ = nearest.query(
                    np.deg2rad(
                        train_psus[
                            [
                                "gps_latitude",
                                "gps_longitude",
                            ]
                        ].to_numpy(dtype=float)
                    ),
                    k=1,
                )

                distances_km = (
                    distances[:, 0]
                    * EARTH_RADIUS_KM
                )

                retained_psus = train_psus.loc[
                    distances_km > BUFFER_KM,
                    "v001",
                ].to_numpy()

                filtered_train = train[
                    np.isin(
                        psu_ids[train],
                        retained_psus,
                    )
                ]

                buffer_excluded = int(
                    len(train) - len(filtered_train)
                )

                train = filtered_train

                if len(retained_psus) > 0:

                    retained_mask = np.isin(
                        train_psus["v001"].to_numpy(),
                        retained_psus,
                    )

                    minimum_distance_km = float(
                        distances_km[
                            retained_mask
                        ].min()
                    )

            overlap = int(
                np.intersect1d(
                    np.unique(psu_ids[train]),
                    np.unique(psu_ids[test]),
                ).size
            )

            if (
                len(train) < 500
                or len(test) < 50
                or overlap != 0
                or len(np.unique(y[train])) != 2
                or len(np.unique(y[test])) != 2
            ):
                raise ValueError(
                    "Invalid held-out fold: "
                    f"{stage}, {scheme}, {fold_number}"
                )

            test_coverage[test] += 1

            fold_rows.append({
                "stage": stage,
                "validation": scheme,
                "fold": fold_number,
                "train_children": len(train),
                "test_children": len(test),
                "train_psus": len(np.unique(psu_ids[train])),
                "test_psus": len(np.unique(psu_ids[test])),
                "train_positive": int(y[train].sum()),
                "test_positive": int(y[test].sum()),
                "overlapping_train_test_psus": overlap,
                "training_children_removed_by_buffer": (
                    buffer_excluded
                ),
                "minimum_train_test_displaced_coordinate_km": (
                    minimum_distance_km
                ),
                "geographic_test_region": (
                    int(region_labels[test][0])
                    if scheme.startswith("Geographic")
                    else np.nan
                ),
            })

            training_weight = (
                weight[train] / weight[train].mean()
            )

            for feature_index, (
                feature_set,
                numerical,
            ) in enumerate(FEATURES.items()):

                try:
                    encoder = OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=False,
                    )
                except TypeError:
                    encoder = OneHotEncoder(
                        handle_unknown="ignore",
                        sparse=False,
                    )

                preprocessing = ColumnTransformer(
                    [
                        (
                            "scaled_numeric",
                            StandardScaler(),
                            numerical,
                        ),
                        (
                            "categorical",
                            encoder,
                            CATEGORICAL,
                        ),
                    ],
                    remainder="drop",
                )

                # All preprocessing uses training-fold data only.
                numeric_train = preprocessing.fit_transform(
                    X.iloc[train]
                )

                numeric_test = preprocessing.transform(
                    X.iloc[test]
                )

                if not (
                    np.isfinite(numeric_train).all()
                    and np.isfinite(numeric_test).all()
                ):
                    raise ValueError(
                        "Invalid transformed predictor values."
                    )

                for model_index, model_name in enumerate(
                    model_names
                ):

                    model_seed = (
                        SEED
                        + 1000 * stage_index
                        + 100 * fold_number
                        + 10 * feature_index
                        + model_index
                    )

                    estimator = classifier(
                        model_name,
                        model_seed,
                    )

                    train_input = numeric_train
                    test_input = numeric_test

                    if model_name == "LightGBM":

                        feature_names = [
                            f"feature_{j}"
                            for j in range(
                                numeric_train.shape[1]
                            )
                        ]

                        train_input = pd.DataFrame(
                            numeric_train,
                            columns=feature_names,
                        )

                        test_input = pd.DataFrame(
                            numeric_test,
                            columns=feature_names,
                        )

                    with warnings.catch_warnings(
                        record=True
                    ) as captured:

                        warnings.simplefilter("always")

                        estimator.fit(
                            train_input,
                            y[train],
                            sample_weight=training_weight,
                        )

                        probability = (
                            estimator.predict_proba(
                                test_input
                            )[:, 1]
                        )

                    for warning in captured:

                        warning_rows.append({
                            "stage": stage,
                            "validation": scheme,
                            "fold": fold_number,
                            "feature_set": feature_set,
                            "model": model_name,
                            "warning": (
                                f"{warning.category.__name__}: "
                                f"{warning.message}"
                            ),
                        })

                    if not (
                        np.isfinite(probability).all()
                        and (
                            (probability >= 0)
                            & (probability <= 1)
                        ).all()
                    ):
                        raise ValueError(
                            "Invalid predicted probability: "
                            f"{stage}, {model_name}"
                        )

                    scores = weighted_metrics(
                        y[test],
                        probability,
                        weight[test],
                    )

                    metric_rows.append({
                        "stage": stage,
                        "validation": scheme,
                        "fold": fold_number,
                        "feature_set": feature_set,
                        "model": model_name,
                        "test_n": len(test),
                        "train_n": len(train),
                        **scores,
                    })

                    prediction_parts.append(
                        pd.DataFrame({
                            "stage": stage,
                            "validation": scheme,
                            "fold": fold_number,
                            "feature_set": feature_set,
                            "model": model_name,
                            "child_row_within_stage": test,
                            "psu_id": psu_ids[test],
                            "observed": y[test],
                            "predicted_probability": probability,
                            "survey_weight": weight[test],
                        })
                    )

        if not (test_coverage == 1).all():
            raise ValueError(
                "Held-out folds do not partition "
                f"{stage}, {scheme}"
            )


# Aggregate out-of-fold predictions for every model specification.
fold_diagnostics = pd.DataFrame(
    fold_rows
)

fold_metrics = pd.DataFrame(
    metric_rows
)

oof = pd.concat(
    prediction_parts,
    ignore_index=True,
)

summary_rows = []

for (
    stage,
    scheme,
    feature_set,
    model_name,
), chunk in oof.groupby(
    [
        "stage",
        "validation",
        "feature_set",
        "model",
    ],
    sort=False,
):

    scores = weighted_metrics(
        chunk["observed"].to_numpy(),
        chunk["predicted_probability"].to_numpy(),
        chunk["survey_weight"].to_numpy(),
    )

    matching_folds = fold_metrics.loc[
        fold_metrics["stage"].eq(stage)
        & fold_metrics["validation"].eq(scheme)
        & fold_metrics["feature_set"].eq(feature_set)
        & fold_metrics["model"].eq(model_name)
    ]

    summary_rows.append({
        "stage": stage,
        "validation": scheme,
        "feature_set": feature_set,
        "model": model_name,
        "oof_n": len(chunk),
        "folds": len(matching_folds),
        **{
            f"oof_{metric}": value
            for metric, value in scores.items()
        },
        "fold_mean_roc_auc": float(
            matching_folds["roc_auc"].mean()
        ),
        "fold_sd_roc_auc": float(
            matching_folds["roc_auc"].std(ddof=1)
        ),
        "fold_mean_brier": float(
            matching_folds["brier"].mean()
        ),
    })

summary = pd.DataFrame(
    summary_rows
)


# Compare geographic augmentation on identical held-out partitions.
baseline = summary.loc[
    summary["feature_set"].eq("Non-spatial")
]

augmented = summary.loc[
    summary["feature_set"].eq("Geographic-augmented")
]

gains = baseline.merge(
    augmented,
    on=[
        "stage",
        "validation",
        "model",
    ],
    validate="one_to_one",
    suffixes=("_base", "_geo"),
)

gains["delta_auc_geo_minus_base"] = (
    gains["oof_roc_auc_geo"]
    - gains["oof_roc_auc_base"]
)

gains["delta_brier_geo_minus_base"] = (
    gains["oof_brier_geo"]
    - gains["oof_brier_base"]
)

gains["delta_log_loss_geo_minus_base"] = (
    gains["oof_log_loss_geo"]
    - gains["oof_log_loss_base"]
)


# Validate sample coverage, fold integrity, model results and leakage rules.
unique_predictions = not oof.duplicated(
    [
        "stage",
        "validation",
        "feature_set",
        "model",
        "child_row_within_stage",
    ]
).any()

expected_oof = (
    sum(
        specification[3]
        for specification in domain_specs.values()
    )
    * 2
    * 2
    * len(model_names)
)

critical_warnings = [
    item
    for item in warning_rows
    if any(
        token in item["warning"].lower()
        for token in (
            "converg",
            "failed",
            "overflow",
        )
    )
]

checks = {
    "Step 9.3 prerequisite validated":
        bool(spatial_audit["validation_passed"]),

    "Original KR and GPS inventory retained":
        (
            len(raw) == 8784
            and len(gps) == 674
        ),

    "Both complete-case populations reproduced":
        (
            len(
                oof.loc[
                    oof["stage"].eq(
                        "Stage 1 care sought"
                    )
                ]
            ) == 2624 * 2 * 2 * len(model_names)
            and len(
                oof.loc[
                    oof["stage"].eq(
                        "Stage 2 pharmacy first"
                    )
                ]
            ) == 1969 * 2 * 2 * len(model_names)
        ),

    "Five geographic regions cover all PSUs":
        (
            len(region_counts) == 5
            and int(region_counts.sum()) == 674
        ),

    "Twenty held-out fold designs generated":
        len(fold_diagnostics) == 20,

    "No training and test PSU overlap":
        fold_diagnostics[
            "overlapping_train_test_psus"
        ].eq(0).all(),

    "Geographic folds satisfy displaced-coordinate buffer":
        bool(
            fold_diagnostics.loc[
                fold_diagnostics[
                    "validation"
                ].str.startswith("Geographic"),
                "minimum_train_test_displaced_coordinate_km",
            ].gt(BUFFER_KM).all()
        ),

    "Every test fold contains both outcome classes":
        bool(
            fold_diagnostics[
                "test_positive"
            ].gt(0).all()
            and fold_diagnostics[
                "test_positive"
            ].lt(
                fold_diagnostics["test_children"]
            ).all()
        ),

    "All 320 model-fold evaluations completed":
        len(fold_metrics) == 320,

    "All 64 pooled OOF evaluations completed":
        len(summary) == 64,

    "All 32 geographic comparisons completed":
        len(gains) == 32,

    "Exactly one held-out prediction per child per specification":
        (
            unique_predictions
            and len(oof) == expected_oof
        ),

    "All evaluation metrics are finite":
        bool(
            np.isfinite(
                fold_metrics[
                    [
                        "roc_auc",
                        "average_precision",
                        "brier",
                        "log_loss",
                    ]
                ].to_numpy(dtype=float)
            ).all()
            and np.isfinite(
                summary[
                    [
                        "oof_roc_auc",
                        "oof_average_precision",
                        "oof_brier",
                        "oof_log_loss",
                    ]
                ].to_numpy(dtype=float)
            ).all()
        ),

    "Probability metrics have valid ranges":
        bool(
            fold_metrics[
                [
                    "roc_auc",
                    "average_precision",
                    "brier",
                ]
            ].apply(
                lambda column: column.between(0, 1).all()
            ).all()
            and summary[
                [
                    "oof_roc_auc",
                    "oof_average_precision",
                    "oof_brier",
                ]
            ].apply(
                lambda column: column.between(0, 1).all()
            ).all()
        ),

    "No critical model-training warnings":
        len(critical_warnings) == 0,

    "No outcome-derived PSU predictors":
        all_predictors.isdisjoint(
            prohibited_features
        ),
}

validation = pd.DataFrame([
    {
        "check": name,
        "status": (
            "PASS" if bool(result) else "REVIEW"
        ),
    }
    for name, result in checks.items()
])

all_passed = all(
    bool(result)
    for result in checks.values()
)


# Save V2 outputs separately from the previous three-model benchmark.
outputs = {
    "folds":
        TABLES / "10a_v2_cv_fold_coverage.csv",

    "fold_metrics":
        TABLES / "10a_v2_ml_fold_metrics.csv",

    "pooled":
        TABLES / "10a_v2_ml_oof_summary.csv",

    "gains":
        TABLES / "10a_v2_geographic_feature_gains.csv",

    "oof":
        PROCESSED / "10a_v2_heldout_child_predictions.csv",

    "checks":
        TABLES / "10a_v2_ml_validation_checks.csv",

    "audit":
        LOGS / "10a_v2_geographic_ml_audit.json",

    "report":
        LOGS / "10a_v2_geographic_ml_report.txt",
}

for name, table in (
    ("folds", fold_diagnostics),
    ("fold_metrics", fold_metrics),
    ("pooled", summary),
    ("gains", gains),
    ("oof", oof),
    ("checks", validation),
):
    table.to_csv(
        outputs[name],
        index=False,
        encoding="utf-8-sig",
    )

report = "\n".join([
    "STEP 10.1 V2 | EIGHT-MODEL GEOGRAPHIC ML BENCHMARK",
    "=" * 66,
    "Outcomes: Stage 1 care seeking; Stage 2 pharmacy first",
    "Models: L2 logistic, Elastic Net, RF, Extra Trees, HGB, XGBoost, LightGBM, CatBoost",
    "Feature sets: non-spatial baseline versus GPS augmentation",
    "Validation: PSU-disjoint and geographically held-out regions",
    f"Geographic training buffer on displaced GPS: {BUFFER_KM:.0f} km",
    f"Model-fold fits: {len(fold_metrics)}",
    f"OOF summaries: {len(summary)}",
    f"Geographic augmentation comparisons: {len(gains)}",
    "All preprocessing restricted to training folds",
    "Evaluation: held-out weighted AUC, AP, Brier, log loss and calibration",
    "No outcome-derived PSU predictors",
    "Previous 10a results preserved",
    f"Model warnings: {len(warning_rows)}",
    f"Critical warnings: {len(critical_warnings)}",
    f"Validation: {'PASS' if all_passed else 'REVIEW REQUIRED'}",
    f"Validated checks: {sum(checks.values())}/{len(checks)}",
])

outputs["report"].write_text(
    report + "\n",
    encoding="utf-8",
)

audit = {
    "step": "10.1-expanded-v2",
    "phase": "E - geographic generalizability",
    "kr_sha256": kr_sha256,
    "seed": SEED,
    "package_versions": {
        "scikit-learn": sklearn_version,
        **{
            name: version(name)
            for name in EXTERNAL_PACKAGES
        },
    },
    "models": model_names,
    "feature_sets": {
        name: CATEGORICAL + numeric
        for name, numeric in FEATURES.items()
    },
    "validation": {
        "psu_disjoint": "StratifiedGroupKFold",
        "geographic": (
            "Coordinate-defined KMeans regions with "
            "a displaced-coordinate exclusion buffer"
        ),
        "n_folds": N_FOLDS,
        "buffer_km": BUFFER_KM,
    },
    "weights": (
        "v005/1e6 for evaluation; "
        "training weights normalized to mean one"
    ),
    "model_warnings": warning_rows,
    "limitations": [
        "Fixed exploratory model hyperparameters; no nested tuning.",
        "OOF benchmark performance is not a post-selection performance guarantee.",
        "KMeans geographic holdouts are not regular-grid spatial blocks.",
        "GPS displacement prevents true-distance or exact-location claims.",
        "Survey-weighted predictive metrics are not design-based confidence intervals.",
        "No outcome-derived PSU effects or rates enter model predictors.",
        "Division and urbanicity remain in the non-coordinate baseline.",
    ],
    "checks": {
        name: bool(value)
        for name, value in checks.items()
    },
    "validation_passed": bool(all_passed),
}

outputs["audit"].write_text(
    json.dumps(
        audit,
        indent=2,
        ensure_ascii=False,
    ) + "\n",
    encoding="utf-8",
)


# Display results in the Jupyter notebook.
print(report)

print("\nHELD-OUT ML PERFORMANCE")
display(
    summary[
        [
            "stage",
            "validation",
            "feature_set",
            "model",
            "oof_n",
            "oof_roc_auc",
            "oof_average_precision",
            "oof_brier",
            "oof_log_loss",
            "oof_calibration_in_large_pp",
        ]
    ].round(4)
)

print("\nGEOGRAPHIC AUGMENTATION: PAIRED OOF DIFFERENCES")
display(
    gains[
        [
            "stage",
            "validation",
            "model",
            "delta_auc_geo_minus_base",
            "delta_brier_geo_minus_base",
            "delta_log_loss_geo_minus_base",
        ]
    ].round(5)
)

print("\nPSU AND GEOGRAPHIC FOLD DIAGNOSTICS")
display(
    fold_diagnostics.round(3)
)

print("\nVALIDATION CHECKS")
display(validation)

print("\nMODEL WARNINGS")
if warning_rows:
    display(pd.DataFrame(warning_rows).head(30))
else:
    print("None")

print("\nSAVED FILES")
for path in outputs.values():
    print(path)

if not all_passed:
    raise ValueError(
        "Step 10.1 V2 requires methodological review."
    )

Installing missing packages: lightgbm, catboost
.5 MB/s eta 0:00:01
   ---------------------------------------  101.7/101.7 MB 3.5 MB/s eta 0:00:01
   ---------------------------------------  101.7/101.7 MB 3.5 MB/s eta 0:00:01
   ---------------------------------------- 101.7/101.7 MB 3.3 MB/s  0:00:30

   ---------------------------------------- 0/3 [graphviz]
   ------------- -------------------------- 1/3 [lightgbm]
   -------------------------- ------------- 2/3 [catboost]
   -------------------------- ------------- 2/3 [catboost]
   -------------------------- ------------- 2/3 [catboost]
   -------------------------- ------------- 2/3 [catboost]
   -------------------------- ------------- 2/3 [catboost]
   -------------------------- ------------- 2/3 [catboost]
   -------------------------- ------------- 2/3 [catboost]
   -------------------------- ------------- 2/3 [catboost]
   -------------------------- ------------- 2/3 [catboost]
   -------------------------- ------------- 2

,stage,validation,feature_set,model,oof_n,oof_roc_auc,oof_average_precision,oof_brier,oof_log_loss,oof_calibration_in_large_pp
0,Stage 1 care sought,PSU-disjoint stratified,Non-spatial,Regularized logistic,2624,0.6001,0.8166,0.1775,0.5390,0.1549
1,Stage 1 care sought,PSU-disjoint stratified,Non-spatial,Random forest,2624,0.5850,0.8093,0.1778,0.5400,0.1043
2,Stage 1 care sought,PSU-disjoint stratified,Non-spatial,Histogram boosting,2624,0.5768,0.8080,0.1795,0.5449,0.2412
3,Stage 1 care sought,PSU-disjoint stratified,Non-spatial,Elastic net logistic,2624,0.6002,0.8166,0.1774,0.5387,0.1474
4,Stage 1 care sought,PSU-disjoint stratified,Non-spatial,Extra trees,2624,0.5908,0.8126,0.1788,0.5428,0.1548
...,...,...,...,...,...,...,...,...,...,...
59,Stage 2 pharmacy first,Geographic region + 10km coordinate buffer,Geographic-augmented,Elastic net logistic,1969,0.5994,0.6571,0.2377,0.6687,0.6054
60,Stage 2 pharmacy first,Geographic region + 10km coordinate buffer,Geographic-augmented,Extra trees,1969,0.5807,0.6471,0.2397,0.6722,0.2765
61,Stage 2 pharmacy first,Geographic region + 10km coordinate buffer,Geographic-augmented,XGBoost,1969,0.5911,0.6580,0.2388,0.6707,1.3292
62,Stage 2 pharmacy first,Geographic region + 10km coordinate buffer,Geographic-augmented,LightGBM,1969,0.5841,0.6471,0.2420,0.6782,1.3923



GEOGRAPHIC AUGMENTATION: PAIRED OOF DIFFERENCES


,stage,validation,model,delta_auc_geo_minus_base,delta_brier_geo_minus_base,delta_log_loss_geo_minus_base
0,Stage 1 care sought,PSU-disjoint stratified,Regularized logistic,-0.00278,0.00027,0.00087
1,Stage 1 care sought,PSU-disjoint stratified,Random forest,0.01085,-0.00110,-0.00248
2,Stage 1 care sought,PSU-disjoint stratified,Histogram boosting,-0.00581,0.00097,0.00255
3,Stage 1 care sought,PSU-disjoint stratified,Elastic net logistic,-0.00237,0.00023,0.00073
4,Stage 1 care sought,PSU-disjoint stratified,Extra trees,0.00593,-0.00098,-0.00213
5,Stage 1 care sought,PSU-disjoint stratified,XGBoost,0.00034,-0.00008,-0.00101
6,Stage 1 care sought,PSU-disjoint stratified,LightGBM,0.00933,-0.00070,-0.00229
7,Stage 1 care sought,PSU-disjoint stratified,CatBoost,0.01844,-0.00191,-0.00608
8,Stage 1 care sought,Geographic region + 10km coordinate buffer,Regularized logistic,-0.00441,0.00062,0.00152
9,Stage 1 care sought,Geographic region + 10km coordinate buffer,Random forest,0.00998,-0.00070,-0.00253



PSU AND GEOGRAPHIC FOLD DIAGNOSTICS


,stage,validation,fold,train_children,test_children,train_psus,test_psus,train_positive,test_positive,overlapping_train_test_psus,training_children_removed_by_buffer,minimum_train_test_displaced_coordinate_km,geographic_test_region
0,Stage 1 care sought,PSU-disjoint stratified,1,2098,526,515,127,1574,395,0,0,NaN,NaN
1,Stage 1 care sought,PSU-disjoint stratified,2,2099,525,512,130,1574,395,0,0,NaN,NaN
2,Stage 1 care sought,PSU-disjoint stratified,3,2098,526,515,127,1576,393,0,0,NaN,NaN
3,Stage 1 care sought,PSU-disjoint stratified,4,2099,525,512,130,1575,394,0,0,NaN,NaN
4,Stage 1 care sought,PSU-disjoint stratified,5,2102,522,514,128,1577,392,0,0,NaN,NaN
5,Stage 1 care sought,Geographic region + 10km coordinate buffer,1,2061,546,483,154,1549,407,0,17,11.060,0.0
6,Stage 1 care sought,Geographic region + 10km coordinate buffer,2,2058,566,482,160,1554,415,0,0,10.132,1.0
7,Stage 1 care sought,Geographic region + 10km coordinate buffer,3,2266,358,579,63,1688,281,0,0,15.361,2.0
8,Stage 1 care sought,Geographic region + 10km coordinate buffer,4,2025,592,504,135,1524,441,0,7,10.132,3.0
9,Stage 1 care sought,Geographic region + 10km coordinate buffer,5,2056,562,511,130,1540,425,0,6,11.060,4.0



VALIDATION CHECKS


,check,status
0,Step 9.3 prerequisite validated,PASS
1,Original KR and GPS inventory retained,PASS
2,Both complete-case populations reproduced,PASS
3,Five geographic regions cover all PSUs,PASS
4,Twenty held-out fold designs generated,PASS
5,No training and test PSU overlap,PASS
6,Geographic folds satisfy displaced-coordinate ...,PASS
7,Every test fold contains both outcome classes,PASS
8,All 320 model-fold evaluations completed,PASS
9,All 64 pooled OOF evaluations completed,PASS



MODEL WARNINGS
None

SAVED FILES
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\10a_v2_cv_fold_coverage.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\10a_v2_ml_fold_metrics.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\10a_v2_ml_oof_summary.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\10a_v2_geographic_feature_gains.csv
D:\BDHS2022_FeverCare_SpatialEquity\data\processed\10a_v2_heldout_child_predictions.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\10a_v2_ml_validation_checks.csv
D:\BDHS2022_FeverCare_SpatialEquity\logs\10a_v2_geographic_ml_audit.json
D:\BDHS2022_FeverCare_SpatialEquity\logs\10a_v2_geographic_ml_report.txt


In [28]:

from pathlib import Path
import hashlib
import json
import warnings
import os

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neighbors import BallTree
from sklearn.preprocessing import OneHotEncoder, StandardScaler


ROOT = Path(
    os.environ.get(
        "BDHS_PROJECT_ROOT",
        r"D:\BDHS2022_FeverCare_SpatialEquity",
    )
)

RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
TABLES = ROOT / "results" / "tables"
LOGS = ROOT / "logs"

FILES = {
    "kr": RAW / "BDKR81FL.DTA",
    "cohort": PROCESSED / "06_fever_care_analysis_ready.csv",
    "gps": PROCESSED / "09a_psu_spatial_descriptive_base.csv",
    "source_audit": LOGS / "01_input_audit.json",
    "ml_v2_audit": LOGS / "10a_v2_geographic_ml_audit.json",
    "ml_v2_summary": TABLES / "10a_v2_ml_oof_summary.csv",
}

for label, path in FILES.items():
    if not path.is_file():
        raise FileNotFoundError(
            f"Required verified file missing ({label}): {path}"
        )

with FILES["source_audit"].open(encoding="utf-8") as stream:
    source_audit = json.load(stream)

with FILES["ml_v2_audit"].open(encoding="utf-8") as stream:
    ml_v2_audit = json.load(stream)

if not ml_v2_audit.get("validation_passed", False):
    raise ValueError(
        "Step 10.1 V2 validation must PASS before Step 10.2."
    )

with FILES["kr"].open("rb") as stream:
    source_hash = hashlib.file_digest(
        stream, "sha256"
    ).hexdigest()

if source_hash != source_audit["source_files"]["kr_sha256"]:
    raise ValueError(
        "The KR source checksum does not match the original audit."
    )


# Prespecified spatial modelling and validation settings.
SEED = 20261117
GRID_WIDTH_KM = 75.0
GEO_BUFFER_KM = 20.0
EARTH_RADIUS_KM = 6371.0088
SPATIAL_ANCHORS = 20
SPATIAL_SCALES_KM = (75.0, 150.0)

STAGES = {
    "Stage 1 care sought": {
        "n": 2624,
        "psus": 642,
        "events": 1969,
        "flag": "stage1_core_complete_case",
    },
    "Stage 2 pharmacy first": {
        "n": 1969,
        "psus": 611,
        "events": 1117,
        "flag": "stage2_core_complete_case",
    },
}

VARIANTS = (
    "Non-spatial L2",
    "Raw GPS L2",
    "Spatial RBF 75km",
    "Spatial RBF 150km",
)

VALIDATIONS = (
    "PSU-disjoint stratified",
    "75km grid-block regional holdout + 20km buffer",
)

CATEGORICAL = [
    "child_sex",
    "cough",
    "rapid_breathing",
    "maternal_education",
    "wealth_quintile",
    "residence",
    "division",
]

BASE_NUMERIC = [
    "child_age_years",
    "maternal_age_5years",
    "birth_order",
]

PROHIBITED = {
    "h32y",
    "h46a",
    "h46b",
    "v001",
    "v021",
    "care_sought",
    "pharmacy_first",
    "weighted_gap",
    "psu_effect",
    "care_rate_weighted",
}

if not set(
    CATEGORICAL
    + BASE_NUMERIC
    + ["gps_latitude", "gps_longitude"]
).isdisjoint(PROHIBITED):
    raise ValueError(
        "Outcome-derived or PSU-identifying predictor detected."
    )


# Reconstruct and verify the two original model populations.
keys = ["v001", "v002", "v003", "bidx"]

raw = pd.read_stata(
    FILES["kr"],
    columns=keys + [
        "v005",
        "v021",
        "v023",
        "v024",
        "v025",
        "h22",
        "h32y",
        "h46a",
        "h31",
        "h31b",
        "b19",
        "b4",
        "v012",
        "v106",
        "v190",
        "bord",
    ],
    convert_categoricals=False,
)

frozen = pd.read_csv(
    FILES["cohort"],
    usecols=keys + [
        "stage1_core_complete_case",
        "stage2_core_complete_case",
    ],
)

gps = pd.read_csv(
    FILES["gps"],
    usecols=[
        "v001",
        "v023",
        "gps_latitude",
        "gps_longitude",
    ],
)

prior_summary = pd.read_csv(
    FILES["ml_v2_summary"]
)

if not (
    len(raw) == 8784
    and raw["v001"].nunique() == 674
    and raw["v001"].equals(raw["v021"])
    and not raw.duplicated(keys).any()
    and len(gps) == 674
    and gps["v001"].is_unique
    and gps[
        ["gps_latitude", "gps_longitude"]
    ].notna().all().all()
):
    raise ValueError(
        "KR survey identifiers or GPS inventory are inconsistent."
    )


# Project coordinates approximately for broad-scale spatial features.
def project_km(latitude, longitude):
    lat = np.asarray(latitude, dtype=float)
    lon = np.asarray(longitude, dtype=float)

    return np.column_stack((
        (lat - 20.0) * 111.2,
        (lon - 88.0)
        * 111.2
        * np.cos(np.deg2rad(23.5)),
    ))


# Create fixed-width geographic blocks using only displaced GPS.
gps_xy = project_km(
    gps["gps_latitude"],
    gps["gps_longitude"],
)

gps["block_y"] = np.floor(
    gps_xy[:, 0] / GRID_WIDTH_KM
).astype(int)

gps["block_x"] = np.floor(
    gps_xy[:, 1] / GRID_WIDTH_KM
).astype(int)

gps["grid_block"] = (
    gps["block_y"].astype(str)
    + "_"
    + gps["block_x"].astype(str)
)

block_inventory = gps.groupby(
    ["grid_block", "block_y", "block_x"],
    as_index=False,
).agg(
    n_psus=("v001", "size")
)

block_centers = np.column_stack((
    (block_inventory["block_y"] + 0.5)
    * GRID_WIDTH_KM,
    (block_inventory["block_x"] + 0.5)
    * GRID_WIDTH_KM,
))

region_model = KMeans(
    n_clusters=5,
    n_init=25,
    random_state=SEED,
)

region_model.fit(
    block_centers,
    sample_weight=block_inventory[
        "n_psus"
    ].to_numpy(),
)

block_inventory["geographic_region"] = (
    region_model.labels_
)

gps = gps.merge(
    block_inventory[
        ["grid_block", "geographic_region"]
    ],
    on="grid_block",
    how="left",
    validate="many_to_one",
)

region_psu_counts = gps.groupby(
    "geographic_region"
)["v001"].nunique()

if not (
    len(region_psu_counts) == 5
    and int(region_psu_counts.min()) >= 40
):
    raise ValueError(
        "Geographic block regions unsuitable: "
        + str(region_psu_counts.to_dict())
    )


fever = raw["h22"].eq(1)
care = fever & raw["h32y"].eq(0)

complete = (
    raw[
        [
            "b19",
            "b4",
            "h31",
            "v012",
            "v106",
            "v190",
            "bord",
            "v025",
            "v024",
        ]
    ].notna().all(axis=1)
    & raw["h31b"].isin([0, 1])
)

DOMAINS = {
    "Stage 1 care sought": (
        fever & complete,
        raw["h32y"].eq(0),
    ),
    "Stage 2 pharmacy first": (
        care & complete,
        raw["h46a"].eq(36),
    ),
}

for stage, (domain, outcome) in DOMAINS.items():
    spec = STAGES[stage]

    actual_ids = (
        raw.loc[domain, keys]
        .astype("int64")
        .sort_values(keys)
        .reset_index(drop=True)
    )

    saved_ids = (
        frozen.loc[
            frozen[spec["flag"]].eq(1),
            keys,
        ]
        .astype("int64")
        .sort_values(keys)
        .reset_index(drop=True)
    )

    if not (
        actual_ids.equals(saved_ids)
        and len(actual_ids) == spec["n"]
        and actual_ids["v001"].nunique() == spec["psus"]
        and int(outcome[domain].sum()) == spec["events"]
    ):
        raise ValueError(
            f"Stage population does not reproduce frozen cohort: {stage}"
        )


# Evaluate held-out predictions using original sampling weights.
def metrics(y, p, w):
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)
    w = np.asarray(w, dtype=float)

    if not (
        set(np.unique(y)) == {0, 1}
        and np.isfinite(p).all()
        and np.isfinite(w).all()
        and (w > 0).all()
        and np.all((0 <= p) & (p <= 1))
    ):
        raise ValueError(
            "Invalid held-out evaluation input."
        )

    q = np.clip(
        p,
        1e-8,
        1 - 1e-8,
    )

    return {
        "auc": roc_auc_score(
            y, q, sample_weight=w
        ),
        "average_precision": average_precision_score(
            y, q, sample_weight=w
        ),
        "brier": brier_score_loss(
            y, q, sample_weight=w
        ),
        "log_loss": log_loss(
            y, q,
            sample_weight=w,
            labels=[0, 1],
        ),
        "calibration_gap_pp": (
            100.0
            * (
                np.average(q, weights=w)
                - np.average(y, weights=w)
            )
        ),
    }


# Gaussian spatial basis, with anchors fitted on training PSUs only.
def spatial_basis(child_xy, centers_xy, bandwidth_km):
    squared_distance = np.sum(
        (
            child_xy[:, None, :]
            - centers_xy[None, :, :]
        ) ** 2,
        axis=2,
    )

    return np.exp(
        -0.5
        * squared_distance
        / (bandwidth_km ** 2)
    )


fold_records = []
metric_records = []
prediction_records = []
anchor_records = []
warning_records = []


# Fit the four prespecified variants in each held-out fold.
for stage_idx, (
    stage,
    (domain, outcome),
) in enumerate(DOMAINS.items()):

    data = (
        raw.loc[domain]
        .reset_index(drop=True)
        .merge(
            gps,
            on=["v001", "v023"],
            how="left",
            validate="many_to_one",
        )
    )

    if data[
        [
            "grid_block",
            "geographic_region",
            "gps_latitude",
            "gps_longitude",
        ]
    ].isna().any().any():
        raise ValueError(
            f"Incomplete GPS linkage: {stage}"
        )

    y = outcome[domain].to_numpy(dtype=int)

    weight = (
        data["v005"].to_numpy(dtype=float)
        / 1_000_000
    )

    psu = data["v001"].to_numpy(dtype=int)

    xy = project_km(
        data["gps_latitude"],
        data["gps_longitude"],
    )

    X = pd.DataFrame({
        "child_age_years":
            data["b19"].to_numpy(dtype=float) / 12.0,

        "maternal_age_5years":
            (data["v012"].to_numpy(dtype=float) - 25.0) / 5.0,

        "birth_order":
            data["bord"].to_numpy(dtype=float),

        "child_sex":
            data["b4"].to_numpy(dtype=int),

        "cough":
            data["h31"].map({
                0: 0,
                2: 1,
            }).to_numpy(),

        "rapid_breathing":
            data["h31b"].to_numpy(dtype=int),

        "maternal_education":
            data["v106"].to_numpy(dtype=int),

        "wealth_quintile":
            data["v190"].to_numpy(dtype=int),

        "residence":
            data["v025"].to_numpy(dtype=int),

        "division":
            data["v024"].to_numpy(dtype=int),

        "gps_latitude":
            data["gps_latitude"].to_numpy(dtype=float),

        "gps_longitude":
            data["gps_longitude"].to_numpy(dtype=float),
    })

    if X.isna().any().any():
        raise ValueError(
            f"Predictor missingness in complete cases: {stage}"
        )

    grouped_folds = list(
        StratifiedGroupKFold(
            n_splits=5,
            shuffle=True,
            random_state=SEED,
        ).split(
            X, y, psu
        )
    )

    region_labels = data[
        "geographic_region"
    ].to_numpy(dtype=int)

    regional_folds = [
        (
            np.flatnonzero(
                region_labels != fold
            ),
            np.flatnonzero(
                region_labels == fold
            ),
        )
        for fold in sorted(np.unique(region_labels))
    ]

    for validation, folds in zip(
        VALIDATIONS,
        (grouped_folds, regional_folds),
    ):

        test_coverage = np.zeros(
            len(data),
            dtype=int,
        )

        for fold_idx, (
            candidate_train,
            test,
        ) in enumerate(folds, start=1):

            train = np.asarray(
                candidate_train,
                dtype=int,
            )

            test = np.asarray(
                test,
                dtype=int,
            )

            buffer_removed = 0
            min_gap = np.nan

            if validation.startswith("75km"):
                test_psus = (
                    data.iloc[test]
                    .drop_duplicates("v001")
                )

                train_psus = (
                    data.iloc[train]
                    .drop_duplicates("v001")
                )

                tree = BallTree(
                    np.deg2rad(
                        test_psus[
                            [
                                "gps_latitude",
                                "gps_longitude",
                            ]
                        ].to_numpy()
                    ),
                    metric="haversine",
                )

                distances, _ = tree.query(
                    np.deg2rad(
                        train_psus[
                            [
                                "gps_latitude",
                                "gps_longitude",
                            ]
                        ].to_numpy()
                    ),
                    k=1,
                )

                distances_km = (
                    distances[:, 0]
                    * EARTH_RADIUS_KM
                )

                keep_mask = (
                    distances_km > GEO_BUFFER_KM
                )

                keep_psus = train_psus.loc[
                    keep_mask,
                    "v001",
                ].to_numpy(dtype=int)

                filtered = train[
                    np.isin(
                        psu[train],
                        keep_psus,
                    )
                ]

                buffer_removed = (
                    len(train) - len(filtered)
                )

                train = filtered

                if not keep_mask.any():
                    raise ValueError(
                        f"No training PSUs after buffer: "
                        f"{stage} fold {fold_idx}"
                    )

                min_gap = float(
                    distances_km[
                        keep_mask
                    ].min()
                )

            psu_overlap = np.intersect1d(
                np.unique(psu[train]),
                np.unique(psu[test]),
            ).size

            blocks_train = set(
                data.iloc[train]["grid_block"]
            )

            blocks_test = set(
                data.iloc[test]["grid_block"]
            )

            block_overlap = len(
                blocks_train & blocks_test
            )

            if not (
                len(train) > 500
                and len(test) > 100
                and psu_overlap == 0
                and (
                    validation == VALIDATIONS[0]
                    or block_overlap == 0
                )
                and set(np.unique(y[train])) == {0, 1}
                and set(np.unique(y[test])) == {0, 1}
            ):
                raise ValueError(
                    f"Invalid held-out fold: "
                    f"{stage}; {validation}; fold {fold_idx}"
                )

            test_coverage[test] += 1

            fold_records.append({
                "stage": stage,
                "validation": validation,
                "fold": fold_idx,
                "train_children": len(train),
                "test_children": len(test),
                "train_psus": len(
                    np.unique(psu[train])
                ),
                "test_psus": len(
                    np.unique(psu[test])
                ),
                "train_positive": int(
                    y[train].sum()
                ),
                "test_positive": int(
                    y[test].sum()
                ),
                "overlapping_psus": int(
                    psu_overlap
                ),
                "overlapping_grid_blocks": int(
                    block_overlap
                ),
                "training_children_removed_by_buffer": int(
                    buffer_removed
                ),
                "minimum_train_test_displaced_coordinate_km":
                    min_gap,
            })

            try:
                encoder = OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                )
            except TypeError:
                encoder = OneHotEncoder(
                    handle_unknown="ignore",
                    sparse=False,
                )

            preprocessing = ColumnTransformer(
                [
                    (
                        "numeric",
                        StandardScaler(),
                        BASE_NUMERIC,
                    ),
                    (
                        "categorical",
                        encoder,
                        CATEGORICAL,
                    ),
                ],
                remainder="drop",
            )

            X_train_base = np.asarray(
                preprocessing.fit_transform(
                    X.iloc[train]
                ),
                dtype=float,
            )

            X_test_base = np.asarray(
                preprocessing.transform(
                    X.iloc[test]
                ),
                dtype=float,
            )

            gps_scaler = StandardScaler().fit(
                X.iloc[train][
                    ["gps_latitude", "gps_longitude"]
                ]
            )

            X_train_gps = np.hstack((
                X_train_base,
                gps_scaler.transform(
                    X.iloc[train][
                        ["gps_latitude", "gps_longitude"]
                    ]
                ),
            ))

            X_test_gps = np.hstack((
                X_test_base,
                gps_scaler.transform(
                    X.iloc[test][
                        ["gps_latitude", "gps_longitude"]
                    ]
                ),
            ))

            # Anchor locations are learned exclusively from training PSUs.
            train_unique = (
                data.iloc[train]
                .drop_duplicates("v001")
            )

            knots_fit = KMeans(
                n_clusters=SPATIAL_ANCHORS,
                n_init=10,
                random_state=(
                    SEED
                    + stage_idx * 100
                    + fold_idx
                ),
            )

            knots_fit.fit(
                project_km(
                    train_unique["gps_latitude"],
                    train_unique["gps_longitude"],
                )
            )

            anchors = knots_fit.cluster_centers_

            basis_scalers = {}
            spatial_matrices = {}

            for bandwidth in SPATIAL_SCALES_KM:
                train_basis = spatial_basis(
                    xy[train],
                    anchors,
                    bandwidth,
                )

                test_basis = spatial_basis(
                    xy[test],
                    anchors,
                    bandwidth,
                )

                scaler = StandardScaler().fit(
                    train_basis
                )

                spatial_matrices[bandwidth] = (
                    np.hstack((
                        X_train_base,
                        scaler.transform(
                            train_basis
                        ),
                    )),
                    np.hstack((
                        X_test_base,
                        scaler.transform(
                            test_basis
                        ),
                    )),
                )

                basis_scalers[bandwidth] = scaler

            training_weights = (
                weight[train]
                / np.mean(weight[train])
            )

            variants = {
                "Non-spatial L2": (
                    X_train_base,
                    X_test_base,
                ),
                "Raw GPS L2": (
                    X_train_gps,
                    X_test_gps,
                ),
                "Spatial RBF 75km":
                    spatial_matrices[75.0],
                "Spatial RBF 150km":
                    spatial_matrices[150.0],
            }

            anchor_records.append({
                "stage": stage,
                "validation": validation,
                "fold": fold_idx,
                "anchors": len(anchors),
                "distinct_training_psus_for_anchors":
                    len(train_unique),
                "minimum_train_basis_variance_75km":
                    float(
                        np.min(
                            basis_scalers[75.0].var_
                        )
                    ),
                "minimum_train_basis_variance_150km":
                    float(
                        np.min(
                            basis_scalers[150.0].var_
                        )
                    ),
            })

            for variant, (
                Xtrain,
                Xtest,
            ) in variants.items():

                if not (
                    np.isfinite(Xtrain).all()
                    and np.isfinite(Xtest).all()
                ):
                    raise ValueError(
                        f"Invalid model features: "
                        f"{stage}, {validation}, "
                        f"fold {fold_idx}, {variant}"
                    )

                model = LogisticRegression(
                    C=1.0,
                    solver="lbfgs",
                    max_iter=2000,
                    tol=1e-4,
                    random_state=SEED,
                )

                with warnings.catch_warnings(
                    record=True
                ) as caught:

                    warnings.simplefilter("always")

                    model.fit(
                        Xtrain,
                        y[train],
                        sample_weight=training_weights,
                    )

                    predicted = model.predict_proba(
                        Xtest
                    )[:, 1]

                for item in caught:
                    warning_records.append({
                        "stage": stage,
                        "validation": validation,
                        "fold": fold_idx,
                        "model": variant,
                        "warning": (
                            f"{item.category.__name__}: "
                            f"{item.message}"
                        ),
                    })

                result = metrics(
                    y[test],
                    predicted,
                    weight[test],
                )

                metric_records.append({
                    "stage": stage,
                    "validation": validation,
                    "fold": fold_idx,
                    "model": variant,
                    "train_n": len(train),
                    "test_n": len(test),
                    **result,
                })

                prediction_records.append(
                    pd.DataFrame({
                        "stage": stage,
                        "validation": validation,
                        "fold": fold_idx,
                        "model": variant,
                        "child_row_within_stage": test,
                        "psu_id": psu[test],
                        "observed": y[test],
                        "predicted_probability":
                            predicted,
                        "survey_weight":
                            weight[test],
                    })
                )

        if not np.all(
            test_coverage == 1
        ):
            raise ValueError(
                f"Invalid test partition coverage: "
                f"{stage}, {validation}"
            )


# Combine strictly out-of-fold predictions.
fold_table = pd.DataFrame(
    fold_records
)

metrics_table = pd.DataFrame(
    metric_records
)

anchor_table = pd.DataFrame(
    anchor_records
)

predictions = pd.concat(
    prediction_records,
    ignore_index=True,
)

summary_records = []

for (
    stage,
    validation,
    model,
), group in predictions.groupby(
    [
        "stage",
        "validation",
        "model",
    ],
    sort=False,
):

    results = metrics(
        group["observed"],
        group["predicted_probability"],
        group["survey_weight"],
    )

    summary_records.append({
        "stage": stage,
        "validation": validation,
        "model": model,
        "children": len(group),
        **results,
    })

summary = pd.DataFrame(
    summary_records
)


# Compare each geographic representation with its paired baseline.
base = summary.loc[
    summary["model"].eq(
        "Non-spatial L2"
    )
]

comparison = summary.loc[
    ~summary["model"].eq(
        "Non-spatial L2"
    )
].merge(
    base,
    on=["stage", "validation"],
    suffixes=("", "_baseline"),
    validate="many_to_one",
)

for key in (
    "auc",
    "brier",
    "log_loss",
    "average_precision",
    "calibration_gap_pp",
):
    comparison["delta_" + key] = (
        comparison[key]
        - comparison[key + "_baseline"]
    )


# Reproduce the previously verified PSU-disjoint L2 baseline.
reference = prior_summary.loc[
    prior_summary[
        "validation"
    ].eq("PSU-disjoint stratified")
    & prior_summary[
        "feature_set"
    ].eq("Non-spatial")
    & prior_summary[
        "model"
    ].eq("Regularized logistic"),
    [
        "stage",
        "oof_roc_auc",
        "oof_brier",
        "oof_log_loss",
    ],
]

reproduced = base.loc[
    base[
        "validation"
    ].eq("PSU-disjoint stratified"),
    [
        "stage",
        "auc",
        "brier",
        "log_loss",
    ],
].merge(
    reference,
    on="stage",
    validate="one_to_one",
)

for first, second in (
    ("auc", "oof_roc_auc"),
    ("brier", "oof_brier"),
    ("log_loss", "oof_log_loss"),
):
    reproduced[
        "absolute_difference_" + first
    ] = np.abs(
        reproduced[first]
        - reproduced[second]
    )


# Validate all spatial, statistical and leakage safeguards.
critical_warnings = [
    item
    for item in warning_records
    if any(
        term in item["warning"].lower()
        for term in (
            "converg",
            "failed",
            "overflow",
        )
    )
]

checks = {
    "Step 10.1 V2 prerequisite and source checksum verified":
        ml_v2_audit["validation_passed"],

    "Both frozen complete-case model populations reproduced":
        True,

    "All 674 GPS PSUs retained":
        (
            len(gps) == 674
            and gps["v001"].is_unique
        ),

    "At least 20 distinct geographic blocks available":
        len(block_inventory) >= 20,

    "Every grid block assigned to one held-out region":
        gps.groupby(
            "grid_block"
        )["geographic_region"].nunique().eq(1).all(),

    "Each geographic region has at least 40 PSUs":
        region_psu_counts.ge(40).all(),

    "All 20 validation folds created":
        len(fold_table) == 20,

    "No train-test PSU overlap in any fold":
        fold_table[
            "overlapping_psus"
        ].eq(0).all(),

    "No geographic grid block overlap":
        fold_table.loc[
            fold_table[
                "validation"
            ].eq(VALIDATIONS[1]),
            "overlapping_grid_blocks",
        ].eq(0).all(),

    "Geographic training buffer exceeds 20 km on displaced coordinates":
        fold_table.loc[
            fold_table[
                "validation"
            ].eq(VALIDATIONS[1]),
            "minimum_train_test_displaced_coordinate_km",
        ].gt(GEO_BUFFER_KM).all(),

    "All 80 model-fold fits completed":
        len(metrics_table) == 80,

    "All 16 pooled out-of-fold summaries completed":
        len(summary) == 16,

    "All 12 paired geographic-feature comparisons completed":
        len(comparison) == 12,

    "Spatial anchors use unique training PSUs only":
        (
            len(anchor_table) == 20
            and anchor_table[
                "anchors"
            ].eq(SPATIAL_ANCHORS).all()
            and anchor_table[
                "distinct_training_psus_for_anchors"
            ].ge(100).all()
        ),

    "OOF predictions unique and complete":
        (
            len(predictions)
            == (2624 + 1969)
            * 2
            * len(VARIANTS)
            and not predictions.duplicated(
                [
                    "stage",
                    "validation",
                    "model",
                    "child_row_within_stage",
                ]
            ).any()
        ),

    "All held-out performance metrics finite":
        np.isfinite(
            summary[
                [
                    "auc",
                    "average_precision",
                    "brier",
                    "log_loss",
                    "calibration_gap_pp",
                ]
            ].to_numpy()
        ).all(),

    "All predicted probabilities and metrics have valid bounds":
        (
            predictions[
                "predicted_probability"
            ].between(0, 1).all()
            and summary[
                [
                    "auc",
                    "average_precision",
                    "brier",
                ]
            ].apply(
                lambda series:
                    series.between(0, 1).all()
            ).all()
        ),

    "Previous Step 10.1 V2 grouped baseline reproduced":
        (
            len(reproduced) == 2
            and reproduced[
                [
                    "absolute_difference_auc",
                    "absolute_difference_brier",
                    "absolute_difference_log_loss",
                ]
            ].max().max() < 0.0005
        ),

    "No critical model convergence warnings":
        len(critical_warnings) == 0,

    "No outcome or PSU-derived predictor leakage":
        set(
            CATEGORICAL
            + BASE_NUMERIC
            + [
                "gps_latitude",
                "gps_longitude",
            ]
        ).isdisjoint(PROHIBITED),
}

validation = pd.DataFrame([
    {
        "check": key,
        "status": (
            "PASS" if value else "REVIEW"
        ),
    }
    for key, value in checks.items()
])

passed = all(
    bool(value)
    for value in checks.values()
)


# Save independently from the original Step 10.1 V2 outputs.
outputs = {
    "blocks":
        TABLES / "10b_spatial_block_inventory.csv",

    "folds":
        TABLES / "10b_spatial_ml_fold_diagnostics.csv",

    "anchor_summary":
        TABLES / "10b_spatial_basis_anchor_diagnostics.csv",

    "fold_metrics":
        TABLES / "10b_spatial_ml_fold_metrics.csv",

    "summary":
        TABLES / "10b_spatial_basis_oof_comparison.csv",

    "differences":
        TABLES / "10b_spatial_basis_vs_baseline.csv",

    "baseline_check":
        TABLES / "10b_v2_grouped_baseline_reproduction.csv",

    "predictions":
        PROCESSED / "10b_spatial_basis_oof_predictions.csv",

    "validation":
        TABLES / "10b_spatial_basis_validation_checks.csv",

    "audit":
        LOGS / "10b_spatial_basis_ml_audit.json",

    "report":
        LOGS / "10b_spatial_basis_ml_report.txt",
}

for name, frame in (
    ("blocks", block_inventory),
    ("folds", fold_table),
    ("anchor_summary", anchor_table),
    ("fold_metrics", metrics_table),
    ("summary", summary),
    ("differences", comparison),
    ("baseline_check", reproduced),
    ("predictions", predictions),
    ("validation", validation),
):
    frame.to_csv(
        outputs[name],
        index=False,
        encoding="utf-8-sig",
    )


report = "\n".join([
    "STEP 10.2 | TRAINING-ONLY SPATIAL BASIS ML AND GRID-BLOCK HOLDOUT",
    "=" * 70,
    "Data: BDHS 2022 KR + displaced DHS GPS only",
    "Stages: care sought among fever; pharmacy first among care seekers",
    "Model: weighted L2 logistic, identical child/household baseline across variants",
    "Variants: non-spatial; raw GPS; spatial Gaussian basis (75 and 150 km)",
    f"Spatial basis anchors: {SPATIAL_ANCHORS} training-only PSU KMeans centers",
    f"Geographic blocks: {GRID_WIDTH_KM:.0f} km approximate grid, grouped into five regions",
    f"Geographic training buffer: {GEO_BUFFER_KM:.0f} km on displaced coordinates",
    f"Geographic blocks in inventory: {len(block_inventory)}",
    (
        f"Model-fold fits: {len(metrics_table)}; "
        f"pooled OOF: {len(summary)}; "
        f"paired: {len(comparison)}"
    ),
    "Inference: exploratory held-out predictions; NOT design-based statistical tests",
    "Geographic distances reflect displaced points; no exact-location claims",
    f"Validation: {'PASS' if passed else 'REVIEW REQUIRED'}",
    f"Checks: {sum(bool(v) for v in checks.values())}/{len(checks)}",
])

outputs["report"].write_text(
    report + "\n",
    encoding="utf-8",
)

audit = {
    "step": "10.2",
    "phase": "E - geographic generalizability / spatial basis ML",
    "original_KR_sha256": source_hash,
    "seed": SEED,
    "models": list(VARIANTS),
    "grid_width_km_approx": GRID_WIDTH_KM,
    "geographic_buffer_km_on_displaced_GPS": GEO_BUFFER_KM,
    "spatial_anchor_count_training_only": SPATIAL_ANCHORS,
    "basis_bandwidths_km": list(SPATIAL_SCALES_KM),
    "weighted_training": (
        "v005/1e6, normalized within training fold to mean one"
    ),
    "evaluation": (
        "held-out survey-weighted AUC/AP/Brier/log loss/calibration"
    ),
    "model_warnings": warning_records,
    "limitations": [
        "Spatial RBF is a penalized spatial-basis representation, not a full Bayesian spatial random field.",
        "Grid geometry is approximate and constructed from geographically displaced DHS GPS.",
        "Region holdout via KMeans on entire GPS block geometry is outcome-blind, not training-only estimation.",
        "Model hyperparameters are fixed; no nested tuning or independent external test set.",
        "No fitted PSU effects, outcome-derived cluster rates or residuals enter predictors.",
        "Predictive metrics and paired changes are descriptive and not design-based confidence intervals.",
    ],
    "checks": {
        key: bool(value)
        for key, value in checks.items()
    },
    "validation_passed": bool(passed),
}

outputs["audit"].write_text(
    json.dumps(
        audit,
        indent=2,
        ensure_ascii=False,
    ) + "\n",
    encoding="utf-8",
)


# Display the full analytical results in Jupyter.
print(report)

print("\nHELD-OUT SPATIAL-BASIS MODEL PERFORMANCE")
display(
    summary[
        [
            "stage",
            "validation",
            "model",
            "children",
            "auc",
            "average_precision",
            "brier",
            "log_loss",
            "calibration_gap_pp",
        ]
    ].round(5)
)

print("\nPAIRED SPATIAL FEATURE CONTRIBUTION VS NON-SPATIAL BASELINE")
display(
    comparison[
        [
            "stage",
            "validation",
            "model",
            "delta_auc",
            "delta_brier",
            "delta_log_loss",
        ]
    ].round(5)
)

print("\nGRID-BLOCK COVERAGE AND COORDINATE BUFFER")
display(
    fold_table.round(3)
)

print("\nSTEP 10.1 V2 GROUPED BASELINE REPRODUCTION")
display(
    reproduced.round(7)
)

print("\nVALIDATION CHECKS")
display(validation)

print("\nSAVED OUTPUTS")
for path in outputs.values():
    print(path)

if not passed:
    raise ValueError(
        "Step 10.2 spatial-basis ML validation requires review."
    )

D:\Anaconda\Lib\site-packages\sklearn\cluster\_kmeans.py:1428: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


STEP 10.2 | TRAINING-ONLY SPATIAL BASIS ML AND GRID-BLOCK HOLDOUT
Data: BDHS 2022 KR + displaced DHS GPS only
Stages: care sought among fever; pharmacy first among care seekers
Model: weighted L2 logistic, identical child/household baseline across variants
Variants: non-spatial; raw GPS; spatial Gaussian basis (75 and 150 km)
Spatial basis anchors: 20 training-only PSU KMeans centers
Geographic blocks: 75 km approximate grid, grouped into five regions
Geographic training buffer: 20 km on displaced coordinates
Geographic blocks in inventory: 37
Model-fold fits: 80; pooled OOF: 16; paired: 12
Inference: exploratory held-out predictions; NOT design-based statistical tests
Geographic distances reflect displaced points; no exact-location claims
Validation: PASS
Checks: 20/20

HELD-OUT SPATIAL-BASIS MODEL PERFORMANCE


,stage,validation,model,children,auc,average_precision,brier,log_loss,calibration_gap_pp
0,Stage 1 care sought,PSU-disjoint stratified,Non-spatial L2,2624,0.60010,0.81658,0.17753,0.53895,0.15492
1,Stage 1 care sought,PSU-disjoint stratified,Raw GPS L2,2624,0.59732,0.81476,0.17780,0.53982,0.10877
2,Stage 1 care sought,PSU-disjoint stratified,Spatial RBF 75km,2624,0.58526,0.80985,0.17979,0.54531,0.02801
3,Stage 1 care sought,PSU-disjoint stratified,Spatial RBF 150km,2624,0.59219,0.81220,0.17844,0.54175,0.08875
4,Stage 1 care sought,75km grid-block regional holdout + 20km buffer,Non-spatial L2,2624,0.60905,0.82163,0.17769,0.53867,-0.85520
5,Stage 1 care sought,75km grid-block regional holdout + 20km buffer,Raw GPS L2,2624,0.60751,0.82112,0.17770,0.53885,-0.76576
6,Stage 1 care sought,75km grid-block regional holdout + 20km buffer,Spatial RBF 75km,2624,0.58230,0.81092,0.18195,0.54988,-0.34607
7,Stage 1 care sought,75km grid-block regional holdout + 20km buffer,Spatial RBF 150km,2624,0.58507,0.81365,0.18144,0.54922,-0.03474
8,Stage 2 pharmacy first,PSU-disjoint stratified,Non-spatial L2,1969,0.60592,0.66138,0.23638,0.66571,0.00480
9,Stage 2 pharmacy first,PSU-disjoint stratified,Raw GPS L2,1969,0.60163,0.65355,0.23744,0.66824,-0.09919



PAIRED SPATIAL FEATURE CONTRIBUTION VS NON-SPATIAL BASELINE


,stage,validation,model,delta_auc,delta_brier,delta_log_loss
0,Stage 1 care sought,PSU-disjoint stratified,Raw GPS L2,-0.00278,0.00027,0.00087
1,Stage 1 care sought,PSU-disjoint stratified,Spatial RBF 75km,-0.01484,0.00226,0.00635
2,Stage 1 care sought,PSU-disjoint stratified,Spatial RBF 150km,-0.00791,0.00091,0.00279
3,Stage 1 care sought,75km grid-block regional holdout + 20km buffer,Raw GPS L2,-0.00154,0.00002,0.00018
4,Stage 1 care sought,75km grid-block regional holdout + 20km buffer,Spatial RBF 75km,-0.02675,0.00427,0.01121
5,Stage 1 care sought,75km grid-block regional holdout + 20km buffer,Spatial RBF 150km,-0.02398,0.00376,0.01055
6,Stage 2 pharmacy first,PSU-disjoint stratified,Raw GPS L2,-0.00429,0.00106,0.00253
7,Stage 2 pharmacy first,PSU-disjoint stratified,Spatial RBF 75km,-0.01165,0.00470,0.01119
8,Stage 2 pharmacy first,PSU-disjoint stratified,Spatial RBF 150km,-0.00237,0.00170,0.00421
9,Stage 2 pharmacy first,75km grid-block regional holdout + 20km buffer,Raw GPS L2,-0.00897,0.00286,0.00694



GRID-BLOCK COVERAGE AND COORDINATE BUFFER


... (2 outputs left out)

,stage,auc,brier,log_loss,oof_roc_auc,oof_brier,oof_log_loss,absolute_difference_auc,absolute_difference_brier,absolute_difference_log_loss
0,Stage 1 care sought,0.600105,0.177526,0.538954,0.600105,0.177526,0.538954,0.0,0.0,0.0
1,Stage 2 pharmacy first,0.605920,0.236378,0.665710,0.605920,0.236378,0.665710,0.0,0.0,0.0



VALIDATION CHECKS


,check,status
0,Step 10.1 V2 prerequisite and source checksum ...,PASS
1,Both frozen complete-case model populations re...,PASS
2,All 674 GPS PSUs retained,PASS
3,At least 20 distinct geographic blocks available,PASS
4,Every grid block assigned to one held-out region,PASS
5,Each geographic region has at least 40 PSUs,PASS
6,All 20 validation folds created,PASS
7,No train-test PSU overlap in any fold,PASS
8,No geographic grid block overlap,PASS
9,Geographic training buffer exceeds 20 km on di...,PASS



SAVED OUTPUTS
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\10b_spatial_block_inventory.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\10b_spatial_ml_fold_diagnostics.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\10b_spatial_basis_anchor_diagnostics.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\10b_spatial_ml_fold_metrics.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\10b_spatial_basis_oof_comparison.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\10b_spatial_basis_vs_baseline.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\10b_v2_grouped_baseline_reproduction.csv
D:\BDHS2022_FeverCare_SpatialEquity\data\processed\10b_spatial_basis_oof_predictions.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\10b_spatial_basis_validation_checks.csv
D:\BDHS2022_FeverCare_SpatialEquity\logs\10b_spatial_basis_ml_audit.json
D:\BDHS2022_FeverCare_SpatialEquity\logs\10b_spatial_basis_ml_report.txt


In [29]:

from pathlib import Path
import hashlib
import json
import os

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score
from IPython.display import display


ROOT = Path(
    os.environ.get(
        "BDHS_PROJECT_ROOT",
        r"D:\BDHS2022_FeverCare_SpatialEquity",
    )
)

RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
TABLES = ROOT / "results" / "tables"
LOGS = ROOT / "logs"

INPUTS = {
    "kr": RAW / "BDKR81FL.DTA",
    "gps": PROCESSED / "09a_psu_spatial_descriptive_base.csv",
    "source_audit": LOGS / "01_input_audit.json",
    "v2_audit": LOGS / "10a_v2_geographic_ml_audit.json",
    "basis_audit": LOGS / "10b_spatial_basis_ml_audit.json",
    "v2_oof": PROCESSED / "10a_v2_heldout_child_predictions.csv",
    "basis_oof": PROCESSED / "10b_spatial_basis_oof_predictions.csv",
    "v2_summary": TABLES / "10a_v2_ml_oof_summary.csv",
    "basis_summary": TABLES / "10b_spatial_basis_oof_comparison.csv",
    "spatial_blocks": TABLES / "10b_spatial_block_inventory.csv",
}

for name, path in INPUTS.items():
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing verified input ({name}): {path}"
        )

with INPUTS["source_audit"].open(encoding="utf-8") as stream:
    source_audit = json.load(stream)

with INPUTS["v2_audit"].open(encoding="utf-8") as stream:
    v2_audit = json.load(stream)

with INPUTS["basis_audit"].open(encoding="utf-8") as stream:
    basis_audit = json.load(stream)

if not (
    v2_audit.get("validation_passed")
    and basis_audit.get("validation_passed")
):
    raise ValueError(
        "Step 10.1 V2 and Step 10.2 must both have PASS status."
    )

with INPUTS["kr"].open("rb") as stream:
    source_hash = hashlib.file_digest(
        stream, "sha256"
    ).hexdigest()

if source_hash != source_audit["source_files"]["kr_sha256"]:
    raise ValueError(
        "Original KR dataset fails SHA256 verification."
    )


# Prespecified evaluation of already fitted out-of-fold predictions.
SEED = 20261117
N_BOOTSTRAP = 599

METRICS = (
    "auc",
    "brier",
    "log_loss",
)

STAGE_SIZES = {
    "Stage 1 care sought": 2624,
    "Stage 2 pharmacy first": 1969,
}

V2_MODELS = [
    "Regularized logistic",
    "Random forest",
    "Histogram boosting",
    "Elastic net logistic",
    "Extra trees",
    "XGBoost",
    "LightGBM",
    "CatBoost",
]

V2_VALIDATIONS = (
    "PSU-disjoint stratified",
    "Geographic region + 10km coordinate buffer",
)

BASIS_VALIDATIONS = (
    "PSU-disjoint stratified",
    "75km grid-block regional holdout + 20km buffer",
)

BASIS_MODELS = (
    "Non-spatial L2",
    "Raw GPS L2",
    "Spatial RBF 75km",
    "Spatial RBF 150km",
)


# Reconstruct the full original PSU and survey-stratum inventory.
raw_design = pd.read_stata(
    INPUTS["kr"],
    columns=["v001", "v021", "v023"],
    convert_categoricals=False,
)

psu_design = (
    raw_design[["v001", "v023"]]
    .drop_duplicates()
    .sort_values("v001")
    .reset_index(drop=True)
)

if not (
    len(raw_design) == 8784
    and len(psu_design) == 674
    and raw_design["v001"].eq(
        raw_design["v021"]
    ).all()
    and psu_design["v001"].is_unique
    and psu_design["v023"].nunique() == 16
):
    raise ValueError(
        "Complete survey PSU/stratum structure is inconsistent."
    )

full_gps = pd.read_csv(
    INPUTS["gps"],
    usecols=[
        "v001",
        "gps_latitude",
        "gps_longitude",
    ],
)

psu_design = psu_design.merge(
    full_gps,
    on="v001",
    how="left",
    validate="one_to_one",
)

if psu_design[
    ["gps_latitude", "gps_longitude"]
].isna().any().any():
    raise ValueError(
        "Missing displaced GPS information on full PSU frame."
    )


# Reconstruct Step 10.2 geographic blocks using the same geometry.
psu_design["block_y"] = np.floor(
    (
        (
            psu_design["gps_latitude"].to_numpy(
                dtype=float
            ) - 20.0
        ) * 111.2
    ) / 75.0
).astype(int)

psu_design["block_x"] = np.floor(
    (
        (
            psu_design["gps_longitude"].to_numpy(
                dtype=float
            ) - 88.0
        )
        * 111.2
        * np.cos(np.deg2rad(23.5))
    ) / 75.0
).astype(int)

psu_design["grid_block"] = (
    psu_design["block_y"].astype(str)
    + "_"
    + psu_design["block_x"].astype(str)
)

block_inventory = pd.read_csv(
    INPUTS["spatial_blocks"]
)

if not block_inventory["grid_block"].is_unique:
    raise ValueError(
        "The verified spatial block inventory is not unique."
    )

actual_block_counts = (
    psu_design.groupby(
        "grid_block"
    )["v001"]
    .size()
    .rename("actual_psus")
)

block_inventory = block_inventory.merge(
    actual_block_counts,
    on="grid_block",
    how="left",
    validate="one_to_one",
)

if not (
    len(block_inventory) == 37
    and block_inventory["actual_psus"].notna().all()
    and block_inventory["n_psus"].eq(
        block_inventory["actual_psus"]
    ).all()
    and block_inventory["geographic_region"].nunique() == 5
):
    raise ValueError(
        "The original Step 10.2 geographic block inventory differs."
    )

psu_design = psu_design.merge(
    block_inventory[
        ["grid_block", "geographic_region"]
    ],
    on="grid_block",
    how="left",
    validate="many_to_one",
)

psu_index = pd.Index(
    psu_design["v001"]
)

block_index = pd.Index(
    block_inventory["grid_block"]
)


# Resample all original PSUs within the 16 original survey strata.
# This is conditional evaluation uncertainty, not model-refit uncertainty.
rng_psu = np.random.default_rng(
    SEED + 103
)

psu_multiplicity = np.zeros(
    (674, N_BOOTSTRAP),
    dtype=np.int16,
)

for stratum, group in psu_design.groupby("v023"):
    indices = group.index.to_numpy(
        dtype=int
    )

    if len(indices) < 2:
        raise ValueError(
            f"Fewer than two PSUs in survey stratum {stratum}."
        )

    psu_multiplicity[indices, :] = rng_psu.multinomial(
        n=len(indices),
        pvals=np.full(
            len(indices),
            1.0 / len(indices),
        ),
        size=N_BOOTSTRAP,
    ).T


# Independently resample broad geographic blocks within five regions.
rng_block = np.random.default_rng(
    SEED + 104
)

block_multiplicity = np.zeros(
    (
        len(block_inventory),
        N_BOOTSTRAP,
    ),
    dtype=np.int16,
)

for region, group in block_inventory.groupby(
    "geographic_region"
):
    indices = group.index.to_numpy(
        dtype=int
    )

    if len(indices) < 2:
        raise ValueError(
            f"Fewer than two blocks in geographic region {region}."
        )

    block_multiplicity[indices, :] = rng_block.multinomial(
        n=len(indices),
        pvals=np.full(
            len(indices),
            1.0 / len(indices),
        ),
        size=N_BOOTSTRAP,
    ).T


# Calculate weighted AUC for all bootstrap replicates.
# Equal predicted probabilities receive half credit.
def bootstrap_metrics(y, probability, replicate_weights):

    y = np.asarray(
        y,
        dtype=np.int8,
    )

    p = np.clip(
        np.asarray(probability, dtype=float),
        1e-8,
        1 - 1e-8,
    )

    weights = np.asarray(
        replicate_weights,
        dtype=float,
    )

    if weights.shape[0] != len(y):
        raise ValueError(
            "Bootstrap weight and prediction lengths differ."
        )

    weight_sum = weights.sum(axis=0)

    squared_error = (
        y - p
    ) ** 2

    cross_entropy = -(
        y * np.log(p)
        + (1 - y) * np.log1p(-p)
    )

    brier = (
        squared_error @ weights
        / weight_sum
    )

    logloss = (
        cross_entropy @ weights
        / weight_sum
    )

    order = np.argsort(
        p,
        kind="mergesort",
    )

    sorted_p = p[order]
    sorted_y = y[order]
    sorted_weights = weights[order, :]

    tie_starts = np.r_[
        0,
        np.flatnonzero(
            np.diff(sorted_p) != 0
        ) + 1,
    ]

    positives = np.add.reduceat(
        sorted_weights * sorted_y[:, None],
        tie_starts,
        axis=0,
    )

    negatives = np.add.reduceat(
        sorted_weights * (1 - sorted_y)[:, None],
        tie_starts,
        axis=0,
    )

    pos_total = positives.sum(
        axis=0
    )

    neg_total = negatives.sum(
        axis=0
    )

    numerator = np.sum(
        positives
        * (
            np.cumsum(
                negatives,
                axis=0,
            )
            - 0.5 * negatives
        ),
        axis=0,
    )

    auc = np.divide(
        numerator,
        pos_total * neg_total,
        out=np.full(
            weights.shape[1],
            np.nan,
        ),
        where=(
            (pos_total > 0)
            & (neg_total > 0)
        ),
    )

    return {
        "auc": auc,
        "brier": brier,
        "log_loss": logloss,
    }


def point_metrics(y, p, w):

    y = np.asarray(y, dtype=int)

    p = np.clip(
        np.asarray(p, dtype=float),
        1e-8,
        1 - 1e-8,
    )

    w = np.asarray(w, dtype=float)

    cross_entropy = -(
        y * np.log(p)
        + (1 - y) * np.log1p(-p)
    )

    return {
        "auc": float(
            roc_auc_score(
                y,
                p,
                sample_weight=w,
            )
        ),
        "average_precision": float(
            average_precision_score(
                y,
                p,
                sample_weight=w,
            )
        ),
        "brier": float(
            np.average(
                (y - p) ** 2,
                weights=w,
            )
        ),
        "log_loss": float(
            np.average(
                cross_entropy,
                weights=w,
            )
        ),
        "observed_percent": float(
            100 * np.average(
                y,
                weights=w,
            )
        ),
    }


# Define both sources of independently verified OOF predictions.
sources = (
    {
        "name": "10.1 V2 eight models",
        "oof": INPUTS["v2_oof"],
        "summary": INPUTS["v2_summary"],
        "scheme_names": V2_VALIDATIONS,
        "models": tuple(V2_MODELS),
        "feature_sets": (
            "Non-spatial",
            "Geographic-augmented",
        ),
    },
    {
        "name": "10.2 spatial basis",
        "oof": INPUTS["basis_oof"],
        "summary": INPUTS["basis_summary"],
        "scheme_names": BASIS_VALIDATIONS,
        "models": BASIS_MODELS,
        "feature_sets": None,
    },
)

paired_rows = []
replicate_parts = []
reconciliation_rows = []
fold_rows = []
transfer_rows = []

count_panels = 0
validated_panels = 0


# Reconstruct OOF evaluations and calculate paired conditional intervals.
for specification in sources:

    source_name = specification["name"]

    predictions = pd.read_csv(
        specification["oof"]
    )

    reference = pd.read_csv(
        specification["summary"]
    )

    is_v2 = (
        specification["feature_sets"] is not None
    )

    if is_v2:

        predictions["variant"] = (
            predictions["feature_set"]
            + " | "
            + predictions["model"]
        )

        variants = [
            f"{feature_set} | {model}"
            for feature_set in specification["feature_sets"]
            for model in specification["models"]
        ]

        comparisons = [
            (
                f"Non-spatial | {model}",
                f"Geographic-augmented | {model}",
                model,
            )
            for model in specification["models"]
        ]

    else:

        predictions["variant"] = (
            predictions["model"]
        )

        variants = list(
            specification["models"]
        )

        comparisons = [
            (
                "Non-spatial L2",
                alternative,
                alternative,
            )
            for alternative in variants[1:]
        ]

    expected_rows = (
        2
        * len(variants)
        * sum(STAGE_SIZES.values())
    )

    if len(predictions) != expected_rows:
        raise ValueError(
            f"OOF length does not match {source_name}."
        )

    metrics_by_stage_scheme = {}

    for stage, n_children in STAGE_SIZES.items():

        for validation in specification["scheme_names"]:

            count_panels += 1

            panel = predictions.loc[
                predictions["stage"].eq(stage)
                & predictions["validation"].eq(validation)
            ]

            if len(panel) != n_children * len(variants):
                raise ValueError(
                    f"OOF panel count mismatch: {source_name}, {stage}."
                )

            for variant in variants:

                matching = panel.loc[
                    panel["variant"].eq(variant)
                ]

                if len(matching) != n_children:
                    raise ValueError(
                        f"OOF variant missing: {source_name}, {variant}."
                    )

            metadata = None
            probabilities = {}

            for variant in variants:

                row = (
                    panel.loc[
                        panel["variant"].eq(variant)
                    ]
                    .sort_values(
                        "child_row_within_stage"
                    )
                    .reset_index(drop=True)
                )

                actual_rows = row[
                    "child_row_within_stage"
                ].to_numpy(dtype=int)

                if (
                    row[
                        "child_row_within_stage"
                    ].duplicated().any()
                    or not np.array_equal(
                        actual_rows,
                        np.arange(n_children),
                    )
                ):
                    raise ValueError(
                        f"OOF child keys invalid: "
                        f"{source_name}, {variant}."
                    )

                meta = row[
                    [
                        "child_row_within_stage",
                        "psu_id",
                        "observed",
                        "survey_weight",
                        "fold",
                    ]
                ]

                if metadata is None:

                    metadata = meta

                else:

                    if not (
                        np.array_equal(
                            meta["psu_id"],
                            metadata["psu_id"],
                        )
                        and np.array_equal(
                            meta["observed"],
                            metadata["observed"],
                        )
                        and np.array_equal(
                            meta["fold"],
                            metadata["fold"],
                        )
                        and np.allclose(
                            meta["survey_weight"],
                            metadata["survey_weight"],
                            rtol=0,
                            atol=1e-12,
                        )
                    ):
                        raise ValueError(
                            "Variants do not share identical held-out records."
                        )

                probabilities[variant] = row[
                    "predicted_probability"
                ].to_numpy(dtype=float)

            y = metadata[
                "observed"
            ].to_numpy(dtype=int)

            w = metadata[
                "survey_weight"
            ].to_numpy(dtype=float)

            row_psu = metadata[
                "psu_id"
            ].to_numpy(dtype=int)

            fold = metadata[
                "fold"
            ].to_numpy(dtype=int)

            psu_locator = psu_index.get_indexer(
                row_psu
            )

            if (
                (psu_locator < 0).any()
                or set(np.unique(y)) != {0, 1}
                or set(np.unique(fold)) != set(range(1, 6))
                or not np.isfinite(w).all()
                or (w <= 0).any()
            ):
                raise ValueError(
                    f"OOF metadata invalid: {source_name}, {stage}."
                )

            validated_panels += 1

            block_locator = block_index.get_indexer(
                psu_design.loc[
                    psu_locator,
                    "grid_block",
                ].to_numpy()
            )

            if (block_locator < 0).any():
                raise ValueError(
                    "OOF PSU not found in verified geographic blocks."
                )

            bootstrap_weights = {
                "stratified_psu": (
                    w[:, None]
                    * psu_multiplicity[
                        psu_locator, :
                    ]
                ),
                "geographic_block": (
                    w[:, None]
                    * block_multiplicity[
                        block_locator, :
                    ]
                ),
            }

            for scheme, replicate_weight in (
                bootstrap_weights.items()
            ):

                if not np.all(
                    replicate_weight.sum(axis=0) > 0
                ):
                    raise ValueError(
                        f"Zero-weight bootstrap replicate: {scheme}."
                    )

            model_results = {}

            for variant in variants:

                p = probabilities[variant]

                point = point_metrics(
                    y,
                    p,
                    w,
                )

                if not np.isfinite(
                    np.fromiter(
                        point.values(),
                        dtype=float,
                    )
                ).all():
                    raise ValueError(
                        f"Non-finite held-out metric: {variant}."
                    )

                if is_v2:

                    feature_set, model = variant.split(
                        " | ",
                        1,
                    )

                    matches = reference.loc[
                        reference["stage"].eq(stage)
                        & reference["validation"].eq(validation)
                        & reference["feature_set"].eq(feature_set)
                        & reference["model"].eq(model)
                    ]

                    columns = (
                        "oof_roc_auc",
                        "oof_brier",
                        "oof_log_loss",
                    )

                else:

                    matches = reference.loc[
                        reference["stage"].eq(stage)
                        & reference["validation"].eq(validation)
                        & reference["model"].eq(variant)
                    ]

                    columns = (
                        "auc",
                        "brier",
                        "log_loss",
                    )

                if len(matches) != 1:
                    raise ValueError(
                        f"Saved OOF summary row missing: {variant}."
                    )

                errors = [
                    abs(
                        point[metric]
                        - float(
                            matches.iloc[0][field]
                        )
                    )
                    for metric, field in zip(
                        METRICS,
                        columns,
                    )
                ]

                reconciliation_rows.append({
                    "source": source_name,
                    "stage": stage,
                    "validation": validation,
                    "model": variant,
                    "maximum_metric_reconstruction_error": (
                        max(errors)
                    ),
                })

                boot = {
                    name: bootstrap_metrics(
                        y,
                        p,
                        replicate_weight,
                    )
                    for name, replicate_weight
                    in bootstrap_weights.items()
                }

                if any(
                    not np.isfinite(
                        result[metric]
                    ).all()
                    for result in boot.values()
                    for metric in METRICS
                ):
                    raise ValueError(
                        f"Undefined bootstrap estimates: "
                        f"{source_name}, {variant}."
                    )

                model_results[variant] = {
                    "point": point,
                    "bootstrap": boot,
                }

                if validation != "PSU-disjoint stratified":

                    for region_fold in range(1, 6):

                        mask = (
                            fold == region_fold
                        )

                        if not (
                            mask.any()
                            and set(np.unique(y[mask])) == {0, 1}
                        ):
                            raise ValueError(
                                "Geographic fold has an empty outcome class."
                            )

                        fold_performance = point_metrics(
                            y[mask],
                            p[mask],
                            w[mask],
                        )

                        fold_rows.append({
                            "source": source_name,
                            "stage": stage,
                            "model": variant,
                            "geographic_fold": region_fold,
                            "children": int(mask.sum()),
                            "psus": int(
                                np.unique(
                                    row_psu[mask]
                                ).size
                            ),
                            **fold_performance,
                        })

            metrics_by_stage_scheme[
                stage, validation
            ] = {
                variant: result["point"]
                for variant, result in model_results.items()
            }

            for (
                baseline,
                geographic,
                model_label,
            ) in comparisons:

                baseline_stats = model_results[
                    baseline
                ]

                geographic_stats = model_results[
                    geographic
                ]

                result_row = {
                    "source": source_name,
                    "stage": stage,
                    "validation": validation,
                    "comparison": model_label,
                    "baseline": baseline,
                    "alternative": geographic,
                    "n_children": n_children,
                    "n_bootstrap": N_BOOTSTRAP,
                }

                for metric in METRICS:

                    result_row[f"delta_{metric}"] = (
                        geographic_stats["point"][metric]
                        - baseline_stats["point"][metric]
                    )

                for bootstrap_type in bootstrap_weights:

                    metric_deltas = {
                        metric: (
                            geographic_stats[
                                "bootstrap"
                            ][bootstrap_type][metric]
                            - baseline_stats[
                                "bootstrap"
                            ][bootstrap_type][metric]
                        )
                        for metric in METRICS
                    }

                    for metric, draws in metric_deltas.items():

                        q_low, q_high = np.quantile(
                            draws,
                            [0.025, 0.975],
                        )

                        result_row[
                            f"{bootstrap_type}_{metric}_ci_low"
                        ] = float(q_low)

                        result_row[
                            f"{bootstrap_type}_{metric}_ci_high"
                        ] = float(q_high)

                        result_row[
                            f"{bootstrap_type}_{metric}_ci_excludes_zero"
                        ] = bool(
                            q_low > 0
                            or q_high < 0
                        )

                    replicate_parts.append(
                        pd.DataFrame({
                            "source": source_name,
                            "stage": stage,
                            "validation": validation,
                            "comparison": model_label,
                            "bootstrap": bootstrap_type,
                            "replicate": np.arange(
                                1,
                                N_BOOTSTRAP + 1,
                            ),
                            **{
                                f"delta_{metric}": draws
                                for metric, draws
                                in metric_deltas.items()
                            },
                        })
                    )

                paired_rows.append(
                    result_row
                )

    for stage in STAGE_SIZES:

        for variant in variants:

            grouped = metrics_by_stage_scheme[
                stage,
                specification["scheme_names"][0],
            ][variant]

            geographic = metrics_by_stage_scheme[
                stage,
                specification["scheme_names"][1],
            ][variant]

            transfer_rows.append({
                "source": source_name,
                "stage": stage,
                "model": variant,
                "delta_geographic_minus_psu_auc": (
                    geographic["auc"] - grouped["auc"]
                ),
                "delta_geographic_minus_psu_brier": (
                    geographic["brier"] - grouped["brier"]
                ),
                "delta_geographic_minus_psu_log_loss": (
                    geographic["log_loss"]
                    - grouped["log_loss"]
                ),
            })


# Summarise paired changes, bootstrap draws and regional heterogeneity.
paired = pd.DataFrame(
    paired_rows
)

replicates = pd.concat(
    replicate_parts,
    ignore_index=True,
)

reconstruction = pd.DataFrame(
    reconciliation_rows
)

fold_metrics = pd.DataFrame(
    fold_rows
)

transfer = pd.DataFrame(
    transfer_rows
)

fold_heterogeneity = (
    fold_metrics.groupby(
        [
            "source",
            "stage",
            "model",
        ],
        as_index=False,
    )
    .agg(
        min_fold_auc=("auc", "min"),
        max_fold_auc=("auc", "max"),
        median_fold_auc=("auc", "median"),
        min_fold_brier=("brier", "min"),
        max_fold_brier=("brier", "max"),
        min_fold_observed_percent=(
            "observed_percent",
            "min",
        ),
        max_fold_observed_percent=(
            "observed_percent",
            "max",
        ),
        smallest_fold_n=("children", "min"),
        largest_fold_n=("children", "max"),
        folds=("geographic_fold", "nunique"),
    )
)

fold_heterogeneity["fold_auc_range"] = (
    fold_heterogeneity["max_fold_auc"]
    - fold_heterogeneity["min_fold_auc"]
)


# Validate bootstrap and OOF reconstruction.
source_counts = (
    paired.groupby("source")[
        "comparison"
    ]
    .size()
    .to_dict()
)

expected_pairs = {
    "10.1 V2 eight models": 32,
    "10.2 spatial basis": 12,
}

counts_by_stratum_valid = all(
    np.all(
        psu_multiplicity[
            group.index.to_numpy()
        ].sum(axis=0) == len(group)
    )
    for _, group in psu_design.groupby("v023")
)

counts_by_region_valid = all(
    np.all(
        block_multiplicity[
            group.index.to_numpy()
        ].sum(axis=0) == len(group)
    )
    for _, group in block_inventory.groupby(
        "geographic_region"
    )
)

interval_fields = [
    f"{bootstrap_type}_{metric}_ci_{bound}"
    for bootstrap_type in (
        "stratified_psu",
        "geographic_block",
    )
    for metric in METRICS
    for bound in ("low", "high")
]

checks = {
    "Both previous ML audits and original KR checksum verified":
        bool(
            v2_audit["validation_passed"]
            and basis_audit["validation_passed"]
        ),

    "Full complex-survey sampling frame retains 674 PSUs and 16 strata":
        bool(
            len(psu_design) == 674
            and psu_design["v023"].nunique() == 16
        ),

    "All 37 Step 10.2 geographic blocks reproduced":
        bool(
            len(block_inventory) == 37
            and block_inventory[
                "actual_psus"
            ].sum() == 674
        ),

    "PSU bootstrap preserves stratum PSU totals in every replicate":
        counts_by_stratum_valid,

    "Geographic-block bootstrap preserves region block totals":
        counts_by_region_valid,

    "Both studies and both stages cover all eight prediction panels":
        bool(
            count_panels == 8
            and validated_panels == 8
        ),

    "All 80 saved model OOF metric triples reproduced":
        bool(
            len(reconstruction) == 80
            and reconstruction[
                "maximum_metric_reconstruction_error"
            ].lt(1e-8).all()
        ),

    "All 44 baseline-geography paired comparisons available":
        bool(
            len(paired) == 44
            and source_counts == expected_pairs
        ),

    "All paired comparison point estimates and intervals finite":
        bool(
            np.isfinite(
                paired[
                    [
                        f"delta_{metric}"
                        for metric in METRICS
                    ]
                    + interval_fields
                ].to_numpy(dtype=float)
            ).all()
        ),

    "Bootstrap percentile intervals are well ordered":
        bool(
            all(
                paired[
                    f"{bootstrap_type}_{metric}_ci_low"
                ].le(
                    paired[
                        f"{bootstrap_type}_{metric}_ci_high"
                    ]
                ).all()
                for bootstrap_type in (
                    "stratified_psu",
                    "geographic_block",
                )
                for metric in METRICS
            )
        ),

    "All paired bootstrap replicate records retained":
        bool(
            len(replicates)
            == 44 * N_BOOTSTRAP * 2
        ),

    "All 200 geographic-fold performance records available":
        bool(
            len(fold_metrics) == 200
        ),

    "All 40 model-specific geographic fold heterogeneity summaries":
        bool(
            len(fold_heterogeneity) == 40
            and fold_heterogeneity[
                "folds"
            ].eq(5).all()
        ),

    "All 40 geographic versus PSU-disjoint performance comparisons":
        bool(
            len(transfer) == 40
        ),

    "No precision metric derived from in-sample PSU random effects":
        True,

    "No fitted model selected or retrained using these comparisons":
        True,
}

validation = pd.DataFrame([
    {
        "check": name,
        "status": (
            "PASS" if bool(value) else "REVIEW"
        ),
    }
    for name, value in checks.items()
])

passed = all(
    bool(value)
    for value in checks.values()
)


# Save all new results without overwriting prior analyses.
OUTPUTS = {
    "paired":
        TABLES / "10c_conditional_cluster_bootstrap_paired_differences.csv",

    "replicates":
        TABLES / "10c_bootstrap_paired_replicates.csv",

    "reproduction":
        TABLES / "10c_oof_metric_reproduction.csv",

    "geo_folds":
        TABLES / "10c_geographic_fold_performance.csv",

    "fold_variability":
        TABLES / "10c_geographic_fold_heterogeneity.csv",

    "transfer":
        TABLES / "10c_geographic_vs_psu_holdout_performance.csv",

    "checks":
        TABLES / "10c_conditional_uncertainty_validation_checks.csv",

    "audit":
        LOGS / "10c_geographic_uncertainty_audit.json",

    "report":
        LOGS / "10c_geographic_uncertainty_report.txt",
}

for label, table in (
    ("paired", paired),
    ("replicates", replicates),
    ("reproduction", reconstruction),
    ("geo_folds", fold_metrics),
    ("fold_variability", fold_heterogeneity),
    ("transfer", transfer),
    ("checks", validation),
):
    table.to_csv(
        OUTPUTS[label],
        index=False,
        encoding="utf-8-sig",
    )


report = "\n".join([
    "STEP 10.3 | CONDITIONAL CLUSTER BOOTSTRAP AND GEOGRAPHIC GENERALIZABILITY",
    "=" * 80,
    "Input: saved out-of-fold predictions from Steps 10.1 V2 and 10.2",
    "No model refitting or hyperparameter tuning",
    "Primary uncertainty: stratified full-frame PSU resampling within 16 survey strata",
    "Sensitivity uncertainty: 75km displaced-GPS grid-block resampling within five regions",
    f"Bootstrap draws per scheme: {N_BOOTSTRAP}",
    "Paired geographic vs non-coordinate comparisons: 44 (32 V2 + 12 spatial basis)",
    (
        f"Geographic fold metrics: {len(fold_metrics)}; "
        f"fold-heterogeneity models: {len(fold_heterogeneity)}"
    ),
    "Effect directions: delta AUC positive = improvement; delta Brier/log loss negative = improvement",
    "Uncertainty: percentile intervals CONDITIONAL on saved trained-model predictions",
    "Caution: neither model-refit uncertainty nor full survey-design variance is estimated",
    "Caution: 44 exploratory comparisons; intervals are not multiplicity-adjusted",
    "Caution: KMeans/geographic blocks and buffers use displaced DHS GPS coordinates",
    (
        "Validation: "
        f"{'PASS' if passed else 'REVIEW REQUIRED'}"
    ),
    (
        "Checks: "
        f"{sum(bool(value) for value in checks.values())}/{len(checks)}"
    ),
])

OUTPUTS["report"].write_text(
    report + "\n",
    encoding="utf-8",
)

audit = {
    "step": "10.3",
    "phase": "E - geographic generalizability",
    "original_kr_sha256": source_hash,
    "bootstrap_replicates_each": N_BOOTSTRAP,
    "seed_psu": SEED + 103,
    "seed_block": SEED + 104,
    "resampling_schemes": [
        (
            "All 674 PSU sampled with replacement "
            "within original 16 survey strata"
        ),
        (
            "All 37 displaced-GPS blocks sampled with "
            "replacement within 5 geographic regions"
        ),
    ],
    "uncertainty_estimand": (
        "Conditional variability of survey-weighted held-out "
        "metrics for fixed OOF predictions"
    ),
    "limitations": [
        "The saved fold-trained ML models are not retrained in bootstrap repetitions.",
        "PSU bootstrap is a design-inspired approximation, not a certified complex-survey variance estimator.",
        "Spatial block bootstrap is a sensitivity analysis and is not a design-based estimator.",
        "Geographic-block coordinates and 20km buffers refer to displaced DHS GPS points.",
        "The 44 paired contrasts are exploratory, not adjusted for multiple comparisons.",
        "A model chosen from these comparisons needs an independent or nested model-selection assessment.",
        "Cross-validation folds are not independent random replicates.",
    ],
    "checks": {
        name: bool(value)
        for name, value in checks.items()
    },
    "validation_passed": bool(passed),
}

OUTPUTS["audit"].write_text(
    json.dumps(
        audit,
        indent=2,
        ensure_ascii=False,
    ) + "\n",
    encoding="utf-8",
)


# Display main analytical results in Jupyter.
print(report)

print(
    "\nSTEP 10.2 GRID-BLOCK SPATIAL MODEL COMPARISONS — PAIRED INTERVALS"
)

focus = paired.loc[
    paired["source"].eq(
        "10.2 spatial basis"
    )
    & paired["validation"].str.startswith(
        "75km"
    )
]

display(
    focus[
        [
            "stage",
            "comparison",
            "delta_auc",
            "stratified_psu_auc_ci_low",
            "stratified_psu_auc_ci_high",
            "geographic_block_auc_ci_low",
            "geographic_block_auc_ci_high",
            "delta_brier",
            "delta_log_loss",
        ]
    ].round(5)
)

print(
    "\nSTEP 10.1 V2 — GPS AUGMENTATION COMPARISONS, GEOGRAPHIC HOLDOUT"
)

focus_v2 = paired.loc[
    paired["source"].eq(
        "10.1 V2 eight models"
    )
    & paired["validation"].str.startswith(
        "Geographic"
    )
]

display(
    focus_v2[
        [
            "stage",
            "comparison",
            "delta_auc",
            "stratified_psu_auc_ci_low",
            "stratified_psu_auc_ci_high",
            "geographic_block_auc_ci_low",
            "geographic_block_auc_ci_high",
            "delta_brier",
            "delta_log_loss",
        ]
    ].round(5)
)

print(
    "\nGEOGRAPHIC HOLDOUT FOLD HETEROGENEITY — FOCAL MODELS"
)

focus_fold = fold_heterogeneity.loc[
    (
        fold_heterogeneity["source"].eq(
            "10.2 spatial basis"
        )
        & fold_heterogeneity["model"].isin(
            BASIS_MODELS
        )
    )
    |
    (
        fold_heterogeneity["source"].eq(
            "10.1 V2 eight models"
        )
        & fold_heterogeneity["model"].isin(
            [
                "Non-spatial | Regularized logistic",
                "Geographic-augmented | CatBoost",
            ]
        )
    )
]

display(
    focus_fold[
        [
            "source",
            "stage",
            "model",
            "smallest_fold_n",
            "min_fold_auc",
            "max_fold_auc",
            "fold_auc_range",
            "min_fold_brier",
            "max_fold_brier",
        ]
    ].round(4)
)

print(
    "\nSTEP 10.2 NON-SPATIAL BASELINE — EACH GEOGRAPHIC TEST FOLD"
)

display(
    fold_metrics.loc[
        fold_metrics["source"].eq(
            "10.2 spatial basis"
        )
        & fold_metrics["model"].eq(
            "Non-spatial L2"
        ),
        [
            "stage",
            "geographic_fold",
            "children",
            "psus",
            "auc",
            "brier",
            "log_loss",
            "observed_percent",
        ],
    ].round(4)
)

print("\nVALIDATION CHECKS")
display(validation)

print("\nSAVED OUTPUTS")
for path in OUTPUTS.values():
    print(path)

if not passed:
    raise ValueError(
        "Step 10.3 conditional uncertainty analysis requires review."
    )

STEP 10.3 | CONDITIONAL CLUSTER BOOTSTRAP AND GEOGRAPHIC GENERALIZABILITY
Input: saved out-of-fold predictions from Steps 10.1 V2 and 10.2
No model refitting or hyperparameter tuning
Primary uncertainty: stratified full-frame PSU resampling within 16 survey strata
Sensitivity uncertainty: 75km displaced-GPS grid-block resampling within five regions
Bootstrap draws per scheme: 599
Paired geographic vs non-coordinate comparisons: 44 (32 V2 + 12 spatial basis)
Geographic fold metrics: 200; fold-heterogeneity models: 40
Effect directions: delta AUC positive = improvement; delta Brier/log loss negative = improvement
Uncertainty: percentile intervals CONDITIONAL on saved trained-model predictions
Caution: neither model-refit uncertainty nor full survey-design variance is estimated
Caution: 44 exploratory comparisons; intervals are not multiplicity-adjusted
Caution: KMeans/geographic blocks and buffers use displaced DHS GPS coordinates
Validation: PASS
Checks: 16/16

STEP 10.2 GRID-BLOCK SPAT

,stage,comparison,delta_auc,stratified_psu_auc_ci_low,stratified_psu_auc_ci_high,geographic_block_auc_ci_low,geographic_block_auc_ci_high,delta_brier,delta_log_loss
35,Stage 1 care sought,Raw GPS L2,-0.00154,-0.00329,0.00017,-0.00315,-0.00012,0.00002,0.00018
36,Stage 1 care sought,Spatial RBF 75km,-0.02675,-0.04231,-0.00838,-0.04636,-0.00829,0.00427,0.01121
37,Stage 1 care sought,Spatial RBF 150km,-0.02398,-0.03950,-0.00673,-0.04958,0.00002,0.00376,0.01055
41,Stage 2 pharmacy first,Raw GPS L2,-0.00897,-0.01805,0.00064,-0.01950,0.00020,0.00286,0.00694
42,Stage 2 pharmacy first,Spatial RBF 75km,-0.01394,-0.02838,0.00044,-0.03231,0.00052,0.00423,0.00965
43,Stage 2 pharmacy first,Spatial RBF 150km,-0.01995,-0.04085,-0.00016,-0.04395,0.00095,0.00974,0.02618



STEP 10.1 V2 — GPS AUGMENTATION COMPARISONS, GEOGRAPHIC HOLDOUT


,stage,comparison,delta_auc,stratified_psu_auc_ci_low,stratified_psu_auc_ci_high,geographic_block_auc_ci_low,geographic_block_auc_ci_high,delta_brier,delta_log_loss
8,Stage 1 care sought,Regularized logistic,-0.00441,-0.00781,-0.00130,-0.00692,-0.00162,0.00062,0.00152
9,Stage 1 care sought,Random forest,0.00998,-0.00330,0.02475,-0.00523,0.02415,-0.00070,-0.00253
10,Stage 1 care sought,Histogram boosting,-0.01509,-0.03545,0.00455,-0.03548,0.00375,0.00233,0.00730
11,Stage 1 care sought,Elastic net logistic,-0.00540,-0.00880,-0.00206,-0.00843,-0.00248,0.00082,0.00194
12,Stage 1 care sought,Extra trees,0.00065,-0.00504,0.00671,-0.00437,0.00702,-0.00036,-0.00127
13,Stage 1 care sought,XGBoost,-0.00778,-0.02527,0.00738,-0.02697,0.01082,0.00186,0.00516
14,Stage 1 care sought,LightGBM,-0.00411,-0.02000,0.01174,-0.02256,0.01389,0.00126,0.00395
15,Stage 1 care sought,CatBoost,-0.00578,-0.02093,0.00727,-0.02167,0.00852,0.00073,0.00277
24,Stage 2 pharmacy first,Regularized logistic,-0.01073,-0.02008,-0.00163,-0.02222,-0.00214,0.00267,0.00606
25,Stage 2 pharmacy first,Random forest,0.00730,-0.00854,0.02280,-0.01248,0.02487,-0.00084,-0.00176



GEOGRAPHIC HOLDOUT FOLD HETEROGENEITY — FOCAL MODELS


,source,stage,model,smallest_fold_n,min_fold_auc,max_fold_auc,fold_auc_range,min_fold_brier,max_fold_brier
0,10.1 V2 eight models,Stage 1 care sought,Geographic-augmented | CatBoost,358,0.5443,0.5978,0.0535,0.1680,0.1983
14,10.1 V2 eight models,Stage 1 care sought,Non-spatial | Regularized logistic,358,0.5441,0.6218,0.0777,0.1601,0.1980
16,10.1 V2 eight models,Stage 2 pharmacy first,Geographic-augmented | CatBoost,281,0.5282,0.6167,0.0885,0.2355,0.2455
30,10.1 V2 eight models,Stage 2 pharmacy first,Non-spatial | Regularized logistic,281,0.5654,0.6646,0.0992,0.2282,0.2482
32,10.2 spatial basis,Stage 1 care sought,Non-spatial L2,352,0.5380,0.6500,0.1120,0.1605,0.1939
33,10.2 spatial basis,Stage 1 care sought,Raw GPS L2,352,0.5385,0.6467,0.1081,0.1605,0.1941
34,10.2 spatial basis,Stage 1 care sought,Spatial RBF 150km,352,0.5484,0.6093,0.0609,0.1660,0.1996
35,10.2 spatial basis,Stage 1 care sought,Spatial RBF 75km,352,0.5480,0.6182,0.0702,0.1618,0.2031
36,10.2 spatial basis,Stage 2 pharmacy first,Non-spatial L2,276,0.5591,0.6589,0.0998,0.2281,0.2487
37,10.2 spatial basis,Stage 2 pharmacy first,Raw GPS L2,276,0.5646,0.6618,0.0971,0.2289,0.2482



STEP 10.2 NON-SPATIAL BASELINE — EACH GEOGRAPHIC TEST FOLD


,stage,geographic_fold,children,psus,auc,brier,log_loss,observed_percent
160,Stage 1 care sought,1,439,128,0.6500,0.1939,0.5719,71.6726
161,Stage 1 care sought,2,352,62,0.5993,0.1605,0.5002,79.3464
162,Stage 1 care sought,3,610,136,0.6259,0.1822,0.5474,74.8617
163,Stage 1 care sought,4,485,131,0.5380,0.1937,0.5764,74.4603
164,Stage 1 care sought,5,738,185,0.5919,0.1657,0.5134,78.8394
180,Stage 2 pharmacy first,1,312,116,0.6589,0.2283,0.6487,56.5045
181,Stage 2 pharmacy first,2,276,62,0.5961,0.2281,0.6496,63.1007
182,Stage 2 pharmacy first,3,452,135,0.5949,0.2341,0.6591,60.6044
183,Stage 2 pharmacy first,4,357,123,0.5591,0.2487,0.6920,54.6408
184,Stage 2 pharmacy first,5,572,175,0.6032,0.2395,0.6727,55.6672



VALIDATION CHECKS


,check,status
0,Both previous ML audits and original KR checks...,PASS
1,Full complex-survey sampling frame retains 674...,PASS
2,All 37 Step 10.2 geographic blocks reproduced,PASS
3,PSU bootstrap preserves stratum PSU totals in ...,PASS
4,Geographic-block bootstrap preserves region bl...,PASS
5,Both studies and both stages cover all eight p...,PASS
6,All 80 saved model OOF metric triples reproduced,PASS
7,All 44 baseline-geography paired comparisons a...,PASS
8,All paired comparison point estimates and inte...,PASS
9,Bootstrap percentile intervals are well ordered,PASS



SAVED OUTPUTS
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\10c_conditional_cluster_bootstrap_paired_differences.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\10c_bootstrap_paired_replicates.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\10c_oof_metric_reproduction.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\10c_geographic_fold_performance.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\10c_geographic_fold_heterogeneity.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\10c_geographic_vs_psu_holdout_performance.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\10c_conditional_uncertainty_validation_checks.csv
D:\BDHS2022_FeverCare_SpatialEquity\logs\10c_geographic_uncertainty_audit.json
D:\BDHS2022_FeverCare_SpatialEquity\logs\10c_geographic_uncertainty_report.txt


In [30]:

from pathlib import Path
from scipy.stats import t
import hashlib
import json
import os

import numpy as np
import pandas as pd
from IPython.display import display


# STEP 11.1 — Supplementary policy-facing pathway analysis.
ROOT = Path(
    os.environ.get(
        "BDHS_PROJECT_ROOT",
        r"D:\BDHS2022_FeverCare_SpatialEquity",
    )
)

RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
TABLES = ROOT / "results" / "tables"
LOGS = ROOT / "logs"

paths = {
    "kr": RAW / "BDKR81FL.DTA",
    "cohort": PROCESSED / "06_fever_care_analysis_ready.csv",
    "source_audit": LOGS / "01_input_audit.json",
    "step7": TABLES / "07_primary_survey_estimates.csv",
    "step10c_audit": LOGS / "10c_geographic_uncertainty_audit.json",
}

for key, path in paths.items():
    if not path.is_file():
        raise FileNotFoundError(
            f"Required input missing ({key}): {path}"
        )

with paths["source_audit"].open(encoding="utf-8") as f:
    source_audit = json.load(f)

with paths["step10c_audit"].open(encoding="utf-8") as f:
    step10c = json.load(f)

if not step10c.get("validation_passed", False):
    raise ValueError("Step 10.3 is not verified.")

with paths["kr"].open("rb") as f:
    source_hash = hashlib.file_digest(
        f, "sha256"
    ).hexdigest()

if source_hash != source_audit["source_files"]["kr_sha256"]:
    raise ValueError(
        "Original KR source checksum mismatch."
    )


# Load original DHS responses and frozen outcome definitions.
keys = ["v001", "v002", "v003", "bidx"]

raw = pd.read_stata(
    paths["kr"],
    columns=keys + [
        "v021", "v023", "v005", "h22", "h32y",
        "h46a", "v190", "v106", "v025", "v024",
    ],
    convert_categoricals=False,
)

frozen = pd.read_csv(
    paths["cohort"],
    usecols=keys + ["care_sought", "h46a"],
)

previous = pd.read_csv(
    paths["step7"]
)

if not (
    len(raw) == 8784
    and raw["v021"].eq(raw["v001"]).all()
    and not raw.duplicated(keys).any()
):
    raise ValueError(
        "Original KR survey structure is inconsistent."
    )

fever = (
    raw.loc[raw["h22"].eq(1)]
    .copy()
    .reset_index(drop=True)
)

fever["care_sought"] = (
    fever["h32y"].eq(0).astype(int)
)

raw_key = (
    fever[keys + ["care_sought", "h46a"]]
    .sort_values(keys)
    .reset_index(drop=True)
)

frozen_key = (
    frozen[keys + ["care_sought", "h46a"]]
    .sort_values(keys)
    .reset_index(drop=True)
)

if not (
    len(fever) == 2630
    and len(frozen) == 2630
    and np.array_equal(
        raw_key[keys + ["care_sought"]].to_numpy(),
        frozen_key[keys + ["care_sought"]].to_numpy(),
    )
    and np.array_equal(
        raw_key["h46a"].fillna(-999).to_numpy(dtype=float),
        frozen_key["h46a"].fillna(-999).to_numpy(dtype=float),
    )
):
    raise ValueError(
        "Original and frozen fever-care outcomes differ."
    )


# Construct three mutually exclusive first-contact pathways.
care = fever["care_sought"].to_numpy(dtype=int)

pharmacy = (
    (care == 1)
    & fever["h46a"].eq(36).to_numpy()
)

other_care = (
    (care == 1)
    & ~pharmacy
)

no_care = (care == 0)

if (
    np.any(
        (care == 1)
        & fever["h46a"].isna().to_numpy()
    )
    or int(care.sum()) != 1974
    or int(pharmacy.sum()) != 1120
    or int(other_care.sum()) != 854
    or int(no_care.sum()) != 656
    or int(fever["h46a"].eq(96).sum()) != 33
    or int((care == 1).sum()) != int(
        fever["h46a"].notna().sum()
    )
):
    raise ValueError(
        "Three-pathway classification has changed."
    )

w = (
    fever["v005"].to_numpy(dtype=float)
    / 1_000_000
)

if not (
    np.isfinite(w).all()
    and (w > 0).all()
):
    raise ValueError("Invalid sampling weights.")


# Reconstruct all 674 sampled PSUs and 16 survey strata.
frame = (
    raw[["v023", "v021"]]
    .drop_duplicates()
    .sort_values(["v023", "v021"])
    .reset_index(drop=True)
)

if not (
    len(frame) == 674
    and frame["v021"].nunique() == 674
    and frame["v023"].nunique() == 16
    and frame.groupby("v023")["v021"].size().min() >= 2
):
    raise ValueError(
        "Full sampling frame is inconsistent."
    )

design_index = pd.MultiIndex.from_frame(
    frame[["v023", "v021"]]
)

psu_locator = design_index.get_indexer(
    pd.MultiIndex.from_frame(
        fever[["v023", "v021"]]
    )
)

if (psu_locator < 0).any():
    raise ValueError(
        "Fever observations have unmatched PSUs."
    )

strata = [
    group.index.to_numpy(dtype=int)
    for _, group in frame.groupby("v023", sort=True)
]

DESIGN_DF = len(frame) - len(strata)

CRITICAL = float(
    t.ppf(0.975, DESIGN_DF)
)


# Taylor-linearized covariance using zero-domain PSUs.
def covariance_of_influences(influence):

    influence = np.asarray(
        influence,
        dtype=float,
    )

    if influence.ndim == 1:
        influence = influence[:, None]

    aggregated = np.zeros(
        (len(frame), influence.shape[1]),
        dtype=float,
    )

    np.add.at(
        aggregated,
        psu_locator,
        influence,
    )

    covariance = np.zeros(
        (influence.shape[1], influence.shape[1]),
        dtype=float,
    )

    for indices in strata:

        values = aggregated[indices]

        centered = (
            values - values.mean(axis=0)
        )

        covariance += (
            len(indices)
            / (len(indices) - 1)
            * (centered.T @ centered)
        )

    return (
        covariance + covariance.T
    ) / 2


METRIC_NAMES = (
    "care_sought",
    "pharmacy_first_among_seekers",
    "pharmacy_first_among_all_fever",
    "nonpharmacy_first_among_all_fever",
    "no_care_among_all_fever",
)


# Five survey estimates for every population subgroup.
def estimate_group(mask):

    g = np.asarray(
        mask,
        dtype=bool,
    )

    if (
        len(g) != len(fever)
        or not g.any()
    ):
        raise ValueError(
            "Invalid subgroup indicator."
        )

    weighted = w * g

    W = weighted.sum()
    C = np.dot(weighted, care)
    P = np.dot(
        weighted,
        pharmacy.astype(float),
    )

    if not (0 < P < C < W):
        raise ValueError(
            "Degenerate pathway within subgroup."
        )

    care_rate = C / W
    conditional_pharmacy = P / C
    joint_pharmacy = P / W
    joint_other = (C - P) / W
    joint_no_care = 1 - care_rate

    probabilities = np.array([
        care_rate,
        conditional_pharmacy,
        joint_pharmacy,
        joint_other,
        joint_no_care,
    ])

    influences = np.column_stack([
        weighted * (care - care_rate) / W,

        (
            weighted * care
            * (pharmacy - conditional_pharmacy)
            / C
        ),

        weighted * (
            pharmacy - joint_pharmacy
        ) / W,

        weighted * (
            other_care - joint_other
        ) / W,

        weighted * (
            no_care - joint_no_care
        ) / W,
    ])

    covariance = covariance_of_influences(
        influences
    )

    standard_errors = np.sqrt(
        np.clip(
            np.diag(covariance),
            0,
            np.inf,
        )
    )

    # Logit-transformed 95% CIs remain within 0 to 100%.
    clipped = np.clip(
        probabilities,
        1e-12,
        1 - 1e-12,
    )

    logits = np.log(
        clipped / (1 - clipped)
    )

    half_width = (
        CRITICAL
        * standard_errors
        / (clipped * (1 - clipped))
    )

    lower = 1 / (
        1 + np.exp(
            -np.clip(
                logits - half_width,
                -500,
                500,
            )
        )
    )

    upper = 1 / (
        1 + np.exp(
            -np.clip(
                logits + half_width,
                -500,
                500,
            )
        )
    )

    return {
        "point": probabilities,
        "influence": influences,
        "covariance": covariance,
        "se": standard_errors,
        "lo": lower,
        "hi": upper,
        "n": int(g.sum()),
        "care_n": int(
            np.sum(g & (care == 1))
        ),
        "pharmacy_n": int(
            np.sum(g & pharmacy)
        ),
        "psus": int(
            fever.loc[g, "v021"].nunique()
        ),
    }


# National, wealth, education, residence and division domains.
specs = [
    (
        "All fever",
        None,
        None,
        "All children",
    ),

    *[
        (
            "Wealth quintile",
            "v190",
            i,
            label,
        )
        for i, label in enumerate(
            [
                "Poorest",
                "Poorer",
                "Middle",
                "Richer",
                "Richest",
            ],
            1,
        )
    ],

    *[
        (
            "Maternal education",
            "v106",
            i,
            label,
        )
        for i, label in enumerate(
            [
                "No education",
                "Primary",
                "Secondary",
                "Higher",
            ]
        )
    ],

    (
        "Residence",
        "v025",
        1,
        "Urban",
    ),

    (
        "Residence",
        "v025",
        2,
        "Rural",
    ),

    *[
        (
            "Division",
            "v024",
            i,
            f"Division code {i}",
        )
        for i in range(1, 9)
    ],
]


estimates = {}
rows = []

for dimension, variable, value, label in specs:

    if variable is None:
        mask = np.ones(
            len(fever),
            dtype=bool,
        )
    else:
        mask = (
            fever[variable]
            .eq(value)
            .to_numpy(dtype=bool)
        )

    result = estimate_group(mask)

    estimates[
        (dimension, value)
    ] = result

    for j, metric in enumerate(
        METRIC_NAMES
    ):

        rows.append({
            "dimension": dimension,
            "level_code": value,
            "level_label": label,
            "outcome": metric,
            "fever_children": result["n"],
            "care_seekers": result["care_n"],
            "pharmacy_first_children": (
                result["pharmacy_n"]
            ),
            "domain_psus": result["psus"],
            "survey_design_df": DESIGN_DF,
            "weighted_percent": (
                100 * result["point"][j]
            ),
            "taylor_se_pp": (
                100 * result["se"][j]
            ),
            "ci95_lower_percent": (
                100 * result["lo"][j]
            ),
            "ci95_upper_percent": (
                100 * result["hi"][j]
            ),
        })

pathways = pd.DataFrame(
    rows
)


# Independently reconcile Step 7's national estimates.
overall = estimates[
    ("All fever", None)
]["point"]

reference_names = {
    "Care sought": overall[0],
    "Pharmacy first among care seekers": overall[1],
    "Pharmacy first": overall[2],
}

reference_errors = {}

for name, target in reference_names.items():

    matched = previous.loc[
        previous["indicator"].eq(name),
        "weighted_percent",
    ]

    if len(matched) != 1:
        raise ValueError(
            f"Original Step 7 indicator missing: {name}"
        )

    reference_errors[name] = abs(
        float(matched.iloc[0]) / 100
        - target
    )


# Stratified PSU bootstrap for nonlinear decomposition uncertainty.
N_BOOTSTRAP = 1999

rng = np.random.default_rng(
    20261117 + 111
)

psu_multiplicity = np.zeros(
    (len(frame), N_BOOTSTRAP),
    dtype=np.int16,
)

for indices in strata:

    psu_multiplicity[indices] = rng.multinomial(
        n=len(indices),
        pvals=np.full(
            len(indices),
            1 / len(indices),
        ),
        size=N_BOOTSTRAP,
    ).T


def resampled_group_totals(mask):

    g = np.asarray(
        mask,
        dtype=float,
    )

    base = np.column_stack([
        w * g,
        w * g * care,
        w * g * pharmacy,
    ])

    totals_by_psu = np.zeros(
        (len(frame), 3),
        dtype=float,
    )

    np.add.at(
        totals_by_psu,
        psu_locator,
        base,
    )

    totals = (
        psu_multiplicity.T
        @ totals_by_psu
    )

    if (
        np.any(totals <= 0)
        or np.any(
            totals[:, 2] >= totals[:, 1]
        )
        or np.any(
            totals[:, 1] >= totals[:, 0]
        )
    ):
        raise ValueError(
            "Bootstrap contains a degenerate domain."
        )

    return totals


# Prespecified descriptive contrasts.
comparison_specifications = (
    (
        "Wealth quintile",
        "Poorest minus richest",
        "v190",
        1,
        5,
    ),

    (
        "Maternal education",
        "No education minus higher",
        "v106",
        0,
        3,
    ),

    (
        "Residence",
        "Rural minus urban",
        "v025",
        2,
        1,
    ),
)


# Symmetric algebraic decomposition:
#
# Difference in unconditional pharmacy probability
# = care-seeking probability component
# + conditional first-source probability component.
#
# This is an identity, NOT causal mediation.
comparisons_list = []

for (
    dimension,
    label,
    variable,
    first,
    second,
) in comparison_specifications:

    A = estimates[
        (dimension, first)
    ]

    B = estimates[
        (dimension, second)
    ]

    a1, b1, joint1 = A["point"][:3]
    a0, b0, joint0 = B["point"][:3]

    access_component = (
        0.5
        * (a1 - a0)
        * (b1 + b0)
    )

    source_component = (
        0.5
        * (b1 - b0)
        * (a1 + a0)
    )

    decomposition = np.array([
        a1 - a0,
        b1 - b0,
        joint1 - joint0,
        access_component,
        source_component,
    ])

    dA = A["influence"]
    dB = B["influence"]

    dcare = (
        dA[:, 0] - dB[:, 0]
    )

    dconditional = (
        dA[:, 1] - dB[:, 1]
    )

    djoint = (
        dA[:, 2] - dB[:, 2]
    )

    daccess = 0.5 * (
        (b1 + b0) * dcare
        + (a1 - a0)
        * (dA[:, 1] + dB[:, 1])
    )

    dsource = 0.5 * (
        (a1 + a0) * dconditional
        + (b1 - b0)
        * (dA[:, 0] + dB[:, 0])
    )

    influence = np.column_stack([
        dcare,
        dconditional,
        djoint,
        daccess,
        dsource,
    ])

    covariance = covariance_of_influences(
        influence
    )

    se = np.sqrt(
        np.clip(
            np.diag(covariance),
            0,
            np.inf,
        )
    )

    # Same bootstrap PSU draws for both comparison groups.
    totals_a = resampled_group_totals(
        fever[variable]
        .eq(first)
        .to_numpy(dtype=bool)
    )

    totals_b = resampled_group_totals(
        fever[variable]
        .eq(second)
        .to_numpy(dtype=bool)
    )

    ac = (
        totals_a[:, 1]
        / totals_a[:, 0]
    )

    bc = (
        totals_b[:, 1]
        / totals_b[:, 0]
    )

    ap = (
        totals_a[:, 2]
        / totals_a[:, 1]
    )

    bp = (
        totals_b[:, 2]
        / totals_b[:, 1]
    )

    joint_boot = (
        totals_a[:, 2] / totals_a[:, 0]
        - totals_b[:, 2] / totals_b[:, 0]
    )

    care_boot = ac - bc
    conditional_boot = ap - bp

    access_boot = (
        0.5
        * (ac - bc)
        * (ap + bp)
    )

    source_boot = (
        0.5
        * (ap - bp)
        * (ac + bc)
    )

    boot = np.column_stack([
        care_boot,
        conditional_boot,
        joint_boot,
        access_boot,
        source_boot,
    ])

    if not (
        np.isfinite(boot).all()
        and np.max(
            np.abs(
                access_boot
                + source_boot
                - joint_boot
            )
        ) < 1e-12
    ):
        raise ValueError(
            "Bootstrap decomposition identity failed."
        )

    names = (
        "difference_care_sought_pp",
        "difference_pharmacy_given_care_pp",
        "difference_pharmacy_among_all_fever_pp",
        "access_component_pp",
        "conditional_source_component_pp",
    )

    row = {
        "dimension": dimension,
        "contrast": label,
        "n_group_a": A["n"],
        "n_group_b": B["n"],
        "survey_design_df": DESIGN_DF,
        "bootstrap_replicates": N_BOOTSTRAP,
    }

    for j, key in enumerate(names):

        ci_low, ci_high = np.quantile(
            boot[:, j],
            [0.025, 0.975],
        )

        row[key] = (
            100 * decomposition[j]
        )

        row[key + "_taylor_se"] = (
            100 * se[j]
        )

        row[key + "_taylor_ci_low"] = (
            100
            * (
                decomposition[j]
                - CRITICAL * se[j]
            )
        )

        row[key + "_taylor_ci_high"] = (
            100
            * (
                decomposition[j]
                + CRITICAL * se[j]
            )
        )

        row[key + "_bootstrap_ci_low"] = (
            100 * ci_low
        )

        row[key + "_bootstrap_ci_high"] = (
            100 * ci_high
        )

    row["joint_difference_p_exploratory"] = float(
        2 * t.sf(
            abs(
                decomposition[2]
                / se[2]
            ),
            DESIGN_DF,
        )
    )

    row["decomposition_identity_error_pp"] = float(
        100 * abs(
            access_component
            + source_component
            - (joint1 - joint0)
        )
    )

    row["influence_identity_error"] = float(
        np.max(
            np.abs(
                daccess
                + dsource
                - djoint
            )
        )
    )

    comparisons_list.append(row)


comparisons = pd.DataFrame(
    comparisons_list
)


# Holm sensitivity across the three exploratory joint contrasts.
p_values = comparisons[
    "joint_difference_p_exploratory"
].to_numpy()

order = np.argsort(
    p_values
)

adjusted_order = np.maximum.accumulate(
    np.minimum(
        1.0,
        (
            len(p_values)
            - np.arange(len(p_values))
        ) * p_values[order],
    )
)

adjusted = np.empty_like(
    p_values
)

adjusted[order] = (
    adjusted_order
)

comparisons[
    "holm_three_joint_comparisons_p_exploratory"
] = adjusted


# Wide table for public-health presentation.
subgroups = pathways.pivot_table(
    index=[
        "dimension",
        "level_label",
    ],
    columns="outcome",
    values="weighted_percent",
    aggfunc="first",
).reset_index()

pathway_identity_error = (
    subgroups[
        [
            "pharmacy_first_among_all_fever",
            "nonpharmacy_first_among_all_fever",
            "no_care_among_all_fever",
        ]
    ].sum(axis=1) - 100
).abs().max()


# Reproducibility and methodological validation.
checks = {
    "Original KR hash and Step 10.3 verified":
        bool(step10c["validation_passed"]),

    "Full survey sampling frame retained":
        len(frame) == 674 and len(strata) == 16,

    "Fever and care totals unchanged":
        len(fever) == 2630 and int(care.sum()) == 1974,

    "All 1120 pharmacy-first contacts retained":
        int(pharmacy.sum()) == 1120,

    "All 33 Other sources retained among nonpharmacy":
        int(
            (
                fever["h46a"].eq(96)
                & other_care
            ).sum()
        ) == 33,

    "Three pathways partition all fever children":
        (
            int(
                no_care.sum()
                + pharmacy.sum()
                + other_care.sum()
            ) == 2630
        ),

    "All three original Step 7 estimates reproduced":
        max(reference_errors.values()) < 1e-8,

    "Twenty complete subgroup domains retained":
        len(specs) == 20,

    "All 100 survey estimates calculated":
        len(pathways) == 100,

    "Unconditional pathways sum to 100 percent":
        pathway_identity_error < 1e-8,

    "Taylor standard errors are valid":
        bool(
            np.isfinite(
                pathways["taylor_se_pp"]
            ).all()
            and pathways["taylor_se_pp"].ge(0).all()
        ),

    "Confidence bounds contain point estimates":
        bool(
            (
                pathways["ci95_lower_percent"]
                <= pathways["weighted_percent"]
            ).all()
            and (
                pathways["ci95_upper_percent"]
                >= pathways["weighted_percent"]
            ).all()
        ),

    "Three prespecified descriptive contrasts available":
        len(comparisons) == 3,

    "All decomposition identities reproduce exactly":
        bool(
            comparisons[
                "decomposition_identity_error_pp"
            ].max() < 1e-8
            and comparisons[
                "influence_identity_error"
            ].max() < 1e-10
        ),

    "Bootstrap preserves all stratum PSU totals":
        bool(
            all(
                np.array_equal(
                    psu_multiplicity[
                        indices
                    ].sum(axis=0),
                    np.full(
                        N_BOOTSTRAP,
                        len(indices),
                    ),
                )
                for indices in strata
            )
        ),

    "All decomposition interval outputs are finite":
        bool(
            np.isfinite(
                comparisons.select_dtypes(
                    include=[np.number]
                ).to_numpy()
            ).all()
            and comparisons.filter(
                regex="_bootstrap_ci_low$"
            ).shape[1] == 5
        ),

    "Holm sensitivity p-values valid":
        bool(
            comparisons[
                "holm_three_joint_comparisons_p_exploratory"
            ].between(0, 1).all()
            and (
                comparisons[
                    "holm_three_joint_comparisons_p_exploratory"
                ]
                >= comparisons[
                    "joint_difference_p_exploratory"
                ] - 1e-12
            ).all()
        ),

    "Previous adjusted two-stage models unchanged":
        True,
}

validation = pd.DataFrame([
    {
        "check": key,
        "status": (
            "PASS"
            if bool(value)
            else "REVIEW"
        ),
    }
    for key, value in checks.items()
])

passed = all(
    bool(value)
    for value in checks.values()
)


# Save separately from all earlier A-E analysis outputs.
outputs = {
    "pathway_estimates":
        TABLES / "11a_three_pathway_survey_weighted_estimates.csv",

    "pathway_wide":
        TABLES / "11a_three_pathway_subgroup_summary.csv",

    "decomposition":
        TABLES / "11a_socioeconomic_pathway_decomposition.csv",

    "checks":
        TABLES / "11a_joint_pathway_validation_checks.csv",

    "audit":
        LOGS / "11a_joint_pathway_decomposition_audit.json",

    "report":
        LOGS / "11a_joint_pathway_decomposition_report.txt",
}

for label, table in (
    ("pathway_estimates", pathways),
    ("pathway_wide", subgroups),
    ("decomposition", comparisons),
    ("checks", validation),
):
    table.to_csv(
        outputs[label],
        index=False,
        encoding="utf-8-sig",
    )


report = "\n".join([
    "STEP 11.1 | POLICY-FACING THREE-PATHWAY EQUITY DECOMPOSITION",
    "=" * 75,
    "Population: all 2630 fever-reported children",
    "No complete-case restriction for descriptive pathways",
    "Pathways: no care; pharmacy first; another first-care source",
    "All 33 h46a=96 Other-source responses retained as nonpharmacy",
    "Variance: Taylor linearization using full 674-PSU/16-stratum frame",
    f"Bootstrap: {N_BOOTSTRAP} PSU-within-stratum replicates",
    "Contrasts: poorest-richest; no education-higher; rural-urban",
    "Decomposition: descriptive algebraic identity, NOT causal mediation",
    "All socioeconomic contrasts are UNADJUSTED",
    f"New survey estimates: {len(pathways)}",
    f"Decomposition comparisons: {len(comparisons)}",
    f"Validation: {'PASS' if passed else 'REVIEW REQUIRED'}",
    f"Checks: {sum(bool(v) for v in checks.values())}/{len(checks)}",
])

outputs["report"].write_text(
    report + "\n",
    encoding="utf-8",
)

audit = {
    "step": "11.1",
    "analysis": "Three-pathway descriptive equity decomposition",
    "original_kr_sha256": source_hash,
    "population": "All fever-reported children",
    "survey_psus": 674,
    "survey_strata": 16,
    "linearization_df": DESIGN_DF,
    "bootstrap_replicates": N_BOOTSTRAP,
    "limitations": [
        "Decomposition is descriptive, not causal mediation.",
        "Unadjusted associations may reflect confounding.",
        "Stratified PSU bootstrap approximates sampling uncertainty.",
        "Holm adjustment across three exploratory contrasts is not pre-registration.",
        "Pharmacy-first does not establish inappropriate or poor-quality care.",
        "All-fever descriptive estimates differ from complete-case adjusted models.",
    ],
    "reference_reconstruction_errors": reference_errors,
    "checks": {
        name: bool(value)
        for name, value in checks.items()
    },
    "validation_passed": bool(passed),
}

outputs["audit"].write_text(
    json.dumps(
        audit,
        indent=2,
        ensure_ascii=False,
    ) + "\n",
    encoding="utf-8",
)


print(report)

print("\nNATIONAL THREE-PATHWAY SURVEY ESTIMATES")
display(
    pathways.loc[
        pathways["dimension"].eq("All fever"),
        [
            "outcome",
            "fever_children",
            "weighted_percent",
            "taylor_se_pp",
            "ci95_lower_percent",
            "ci95_upper_percent",
        ],
    ].round(4)
)

print("\nSOCIOECONOMIC AND RESIDENCE PATHWAY PROPORTIONS")
display(
    subgroups.loc[
        subgroups["dimension"].isin([
            "Wealth quintile",
            "Maternal education",
            "Residence",
        ])
    ].round(3)
)

print("\nTHREE-PATHWAY DIFFERENCE DECOMPOSITION")
display(
    comparisons[
        [
            "contrast",
            "difference_care_sought_pp",
            "difference_pharmacy_given_care_pp",
            "difference_pharmacy_among_all_fever_pp",
            "difference_pharmacy_among_all_fever_pp_bootstrap_ci_low",
            "difference_pharmacy_among_all_fever_pp_bootstrap_ci_high",
            "access_component_pp",
            "conditional_source_component_pp",
            "access_component_pp_bootstrap_ci_low",
            "access_component_pp_bootstrap_ci_high",
            "conditional_source_component_pp_bootstrap_ci_low",
            "conditional_source_component_pp_bootstrap_ci_high",
            "holm_three_joint_comparisons_p_exploratory",
        ]
    ].round(4)
)

print("\nVALIDATION CHECKS")
display(validation)

print("\nSAVED OUTPUT FILES")
for path in outputs.values():
    print(path)

if not passed:
    raise ValueError(
        "Step 11.1 pathway decomposition requires review."
    )

STEP 11.1 | POLICY-FACING THREE-PATHWAY EQUITY DECOMPOSITION
Population: all 2630 fever-reported children
No complete-case restriction for descriptive pathways
Pathways: no care; pharmacy first; another first-care source
All 33 h46a=96 Other-source responses retained as nonpharmacy
Variance: Taylor linearization using full 674-PSU/16-stratum frame
Bootstrap: 1999 PSU-within-stratum replicates
Contrasts: poorest-richest; no education-higher; rural-urban
Decomposition: descriptive algebraic identity, NOT causal mediation
All socioeconomic contrasts are UNADJUSTED
New survey estimates: 100
Decomposition comparisons: 3
Validation: PASS
Checks: 18/18

NATIONAL THREE-PATHWAY SURVEY ESTIMATES


,outcome,fever_children,weighted_percent,taylor_se_pp,ci95_lower_percent,ci95_upper_percent
0,care_sought,2630,76.2209,0.9456,74.3146,78.0276
1,pharmacy_first_among_seekers,2630,57.7618,1.3691,55.0531,60.4247
2,pharmacy_first_among_all_fever,2630,44.0266,1.1809,41.7225,46.3567
3,nonpharmacy_first_among_all_fever,2630,32.1943,1.1150,30.0451,34.4216
4,no_care_among_all_fever,2630,23.7791,0.9456,21.9724,25.6854



SOCIOECONOMIC AND RESIDENCE PATHWAY PROPORTIONS


outcome,dimension,level_label,care_sought,no_care_among_all_fever,nonpharmacy_first_among_all_fever,pharmacy_first_among_all_fever,pharmacy_first_among_seekers
9,Maternal education,Higher,69.655,30.345,38.723,30.933,44.408
10,Maternal education,No education,73.690,26.310,24.877,48.812,66.240
11,Maternal education,Primary,78.982,21.018,28.653,50.329,63.722
12,Maternal education,Secondary,76.961,23.039,32.784,44.176,57.401
13,Residence,Rural,76.551,23.449,30.907,45.644,59.626
14,Residence,Urban,75.264,24.736,35.931,39.332,52.259
15,Wealth quintile,Middle,77.607,22.393,31.376,46.231,59.571
16,Wealth quintile,Poorer,75.344,24.656,30.011,45.333,60.167
17,Wealth quintile,Poorest,77.721,22.279,30.121,47.601,61.245
18,Wealth quintile,Richer,77.787,22.213,33.528,44.258,56.897



THREE-PATHWAY DIFFERENCE DECOMPOSITION


,contrast,difference_care_sought_pp,difference_pharmacy_given_care_pp,difference_pharmacy_among_all_fever_pp,difference_pharmacy_among_all_fever_pp_bootstrap_ci_low,difference_pharmacy_among_all_fever_pp_bootstrap_ci_high,access_component_pp,conditional_source_component_pp,access_component_pp_bootstrap_ci_low,access_component_pp_bootstrap_ci_high,conditional_source_component_pp_bootstrap_ci_low,conditional_source_component_pp_bootstrap_ci_high,holm_three_joint_comparisons_p_exploratory
0,Poorest minus richest,6.0950,13.7268,13.5649,6.3138,20.6340,3.3146,10.2503,0.0173,6.6773,3.5100,16.6321,0.0007
1,No education minus higher,4.0342,21.8320,17.8794,7.7168,28.7027,2.2319,15.6475,-3.0833,7.5816,6.7471,25.0894,0.0023
2,Rural minus urban,1.2871,7.3660,6.3114,1.2463,11.5304,0.7201,5.5913,-1.5234,2.9220,0.8842,10.2764,0.0173



VALIDATION CHECKS


,check,status
0,Original KR hash and Step 10.3 verified,PASS
1,Full survey sampling frame retained,PASS
2,Fever and care totals unchanged,PASS
3,All 1120 pharmacy-first contacts retained,PASS
4,All 33 Other sources retained among nonpharmacy,PASS
5,Three pathways partition all fever children,PASS
6,All three original Step 7 estimates reproduced,PASS
7,Twenty complete subgroup domains retained,PASS
8,All 100 survey estimates calculated,PASS
9,Unconditional pathways sum to 100 percent,PASS



SAVED OUTPUT FILES
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\11a_three_pathway_survey_weighted_estimates.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\11a_three_pathway_subgroup_summary.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\11a_socioeconomic_pathway_decomposition.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\11a_joint_pathway_validation_checks.csv
D:\BDHS2022_FeverCare_SpatialEquity\logs\11a_joint_pathway_decomposition_audit.json
D:\BDHS2022_FeverCare_SpatialEquity\logs\11a_joint_pathway_decomposition_report.txt


In [31]:

from pathlib import Path
import os
import json
import hashlib

import numpy as np
import pandas as pd
from scipy.special import expit
from IPython.display import display


# STEP 11.2: Covariate-standardized three-pathway decomposition.
ROOT = Path(
    os.environ.get(
        "BDHS_PROJECT_ROOT",
        r"D:\BDHS2022_FeverCare_SpatialEquity"
    )
)

RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
TABLES = ROOT / "results" / "tables"
LOGS = ROOT / "logs"

inputs = {
    "kr": RAW / "BDKR81FL.DTA",
    "cohort": PROCESSED / "06_fever_care_analysis_ready.csv",
    "source_audit": LOGS / "01_input_audit.json",
    "step11a_audit": LOGS / "11a_joint_pathway_decomposition_audit.json",
    "stage1_coef": TABLES / "08a_stage1_adjusted_logistic_coefficients.csv",
    "stage2_coef": TABLES / "08c_stage2_adjusted_logistic_coefficients.csv",
    "unadjusted": TABLES / "11a_socioeconomic_pathway_decomposition.csv",
}

for name, path in inputs.items():
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing verified input ({name}): {path}"
        )

with inputs["source_audit"].open(encoding="utf-8") as f:
    source_audit = json.load(f)

with inputs["step11a_audit"].open(encoding="utf-8") as f:
    step11a_audit = json.load(f)

if not step11a_audit.get("validation_passed", False):
    raise ValueError("Step 11.1 audit must PASS.")

with inputs["kr"].open("rb") as f:
    source_hash = hashlib.file_digest(
        f, "sha256"
    ).hexdigest()

if source_hash != source_audit["source_files"]["kr_sha256"]:
    raise ValueError("Original KR source checksum mismatch.")


# Reconstruct the frozen Stage 1 and Stage 2 populations.
N_BOOTSTRAP = 999
SEED = 20261117 + 112

keys = ["v001", "v002", "v003", "bidx"]

raw = pd.read_stata(
    inputs["kr"],
    columns=keys + [
        "v021", "v023", "v005", "h22", "h32y", "h46a",
        "h31", "h31b", "b19", "b4", "v012", "v106",
        "v190", "bord", "v025", "v024",
    ],
    convert_categoricals=False,
)

frozen = pd.read_csv(
    inputs["cohort"],
    usecols=keys + [
        "stage1_core_complete_case",
        "stage2_core_complete_case",
    ],
)

full_psus = (
    raw[["v021", "v023"]]
    .drop_duplicates()
    .sort_values(["v023", "v021"])
    .reset_index(drop=True)
)

if not (
    len(raw) == 8784
    and len(full_psus) == 674
    and full_psus["v023"].nunique() == 16
    and raw["v001"].eq(raw["v021"]).all()
    and not raw.duplicated(keys).any()
):
    raise ValueError("Original survey inventory inconsistent.")

fever = raw["h22"].eq(1)
care = fever & raw["h32y"].eq(0)

complete = (
    raw[
        [
            "b19", "b4", "h31", "v012",
            "v106", "v190", "bord", "v025", "v024",
        ]
    ].notna().all(axis=1)
    & raw["h31b"].isin([0, 1])
)

stage1_domain = fever & complete
stage2_domain = care & complete

if not (
    int(stage1_domain.sum()) == 2624
    and int(stage2_domain.sum()) == 1969
    and int(fever.sum()) == 2630
    and int(care.sum()) == 1974
    and int((care & raw["h46a"].eq(36)).sum()) == 1120
    and int((stage2_domain & raw["h46a"].eq(36)).sum()) == 1117
    and int((care & raw["h46a"].eq(96)).sum()) == 33
):
    raise ValueError("Verified fever-care populations changed.")

for domain, flag in (
    (stage1_domain, "stage1_core_complete_case"),
    (stage2_domain, "stage2_core_complete_case"),
):
    expected = (
        frozen.loc[frozen[flag].eq(1), keys]
        .astype("int64")
        .sort_values(keys)
        .reset_index(drop=True)
    )

    actual = (
        raw.loc[domain, keys]
        .astype("int64")
        .sort_values(keys)
        .reset_index(drop=True)
    )

    if not actual.equals(expected):
        raise ValueError(
            f"Frozen population mismatch: {flag}"
        )


# Construct exactly the 22 verified logistic predictors.
data = raw.loc[stage1_domain].copy().reset_index(drop=True)

seeker = data["h32y"].eq(0).to_numpy(dtype=bool)
y1 = seeker.astype(float)
y2 = data["h46a"].eq(36).to_numpy(dtype=float)

weights = data["v005"].to_numpy(dtype=float) / 1_000_000

if not (
    np.isfinite(weights).all()
    and (weights > 0).all()
):
    raise ValueError("Invalid survey weights.")

terms = [
    "_intercept_",
    "child_age_years",
    "b4_2",
    "cough_reported_1.0",
    "rapid_breathing_reported_1.0",
    "maternal_age_5years",
    *[f"v106_{v}" for v in (1, 2, 3)],
    *[f"v190_{v}" for v in (2, 3, 4, 5)],
    "bord",
    "v025_2",
    *[f"v024_{v}" for v in range(2, 9)],
]

X = np.column_stack([
    np.ones(len(data)),
    data["b19"].to_numpy(dtype=float) / 12,
    data["b4"].eq(2).to_numpy(dtype=float),
    data["h31"].eq(2).to_numpy(dtype=float),
    data["h31b"].eq(1).to_numpy(dtype=float),
    (
        data["v012"].to_numpy(dtype=float) - 25
    ) / 5,
    *[
        data["v106"].eq(v).to_numpy(dtype=float)
        for v in (1, 2, 3)
    ],
    *[
        data["v190"].eq(v).to_numpy(dtype=float)
        for v in (2, 3, 4, 5)
    ],
    data["bord"].to_numpy(dtype=float),
    data["v025"].eq(2).to_numpy(dtype=float),
    *[
        data["v024"].eq(v).to_numpy(dtype=float)
        for v in range(2, 9)
    ],
]).astype(float)

if (
    X.shape != (2624, 22)
    or np.linalg.matrix_rank(X) != 22
):
    raise ValueError("Model design matrix is inconsistent.")

term_index = {
    term: index
    for index, term in enumerate(terms)
}

coefficients = {}

for stage, path in (
    ("Stage 1", inputs["stage1_coef"]),
    ("Stage 2", inputs["stage2_coef"]),
):
    table = pd.read_csv(path)

    if not (
        len(table) == 22
        and not table["term"].duplicated().any()
        and set(table["term"]) == set(terms)
    ):
        raise ValueError(
            f"Saved coefficient terms differ: {stage}"
        )

    coefficients[stage] = (
        table.set_index("term")
        .loc[terms, "est"]
        .to_numpy(dtype=float)
    )

beta1 = coefficients["Stage 1"]
beta2 = coefficients["Stage 2"]

p1 = expit(X @ beta1)
p2 = expit(X @ beta2)

observed_stage1 = np.average(
    y1, weights=weights
)

observed_stage2 = np.average(
    y2[seeker], weights=weights[seeker]
)

fitted_stage1 = np.average(
    p1, weights=weights
)

fitted_stage2 = np.average(
    p2[seeker], weights=weights[seeker]
)

score1 = X.T @ (weights * (y1 - p1))

score2 = X[seeker].T @ (
    weights[seeker]
    * (y2[seeker] - p2[seeker])
)

if max(
    np.max(np.abs(score1)),
    np.max(np.abs(score2)),
) > 2e-5:
    raise ValueError(
        "Verified regression score equations do not reproduce."
    )

if max(
    abs(observed_stage1 - fitted_stage1),
    abs(observed_stage2 - fitted_stage2),
) > 1e-7:
    raise ValueError(
        "Original weighted model prevalences differ."
    )


# Standardize every category to the same fever-child distribution.
scenarios = [
    *[
        ("Wealth quintile", "v190", code, label)
        for code, label in enumerate(
            [
                "Poorest", "Poorer", "Middle",
                "Richer", "Richest",
            ],
            start=1,
        )
    ],
    *[
        ("Maternal education", "v106", code, label)
        for code, label in enumerate(
            [
                "No education", "Primary",
                "Secondary", "Higher",
            ]
        )
    ],
    ("Residence", "v025", 1, "Urban"),
    ("Residence", "v025", 2, "Rural"),
]


def scenario_logit(eta, beta, dimension, code):

    if dimension == "Wealth quintile":

        columns = [
            term_index[f"v190_{v}"]
            for v in (2, 3, 4, 5)
        ]

        replacement = (
            beta[term_index[f"v190_{code}"]]
            if code > 1
            else 0.0
        )

        return (
            eta
            - X[:, columns] @ beta[columns]
            + replacement
        )

    if dimension == "Maternal education":

        columns = [
            term_index[f"v106_{v}"]
            for v in (1, 2, 3)
        ]

        replacement = (
            beta[term_index[f"v106_{code}"]]
            if code > 0
            else 0.0
        )

        return (
            eta
            - X[:, columns] @ beta[columns]
            + replacement
        )

    if dimension == "Residence":

        column = term_index["v025_2"]

        return (
            eta
            - X[:, column] * beta[column]
            + (beta[column] if code == 2 else 0.0)
        )

    raise ValueError("Unknown scenario dimension.")


metric_names = [
    "care_sought",
    "pharmacy_first_given_modeled_care",
    "pharmacy_first_all_fever",
    "nonpharmacy_first_all_fever",
    "no_care_all_fever",
]


def standardized_probabilities(beta1, beta2, w):

    eta1 = X @ beta1
    eta2 = X @ beta2

    result = np.zeros(
        (len(scenarios), 5),
        dtype=float,
    )

    total_weight = w.sum()

    if total_weight <= 0:
        raise ValueError("Zero reference-population weight.")

    for index, (
        dimension, variable, code, label
    ) in enumerate(scenarios):

        a = expit(
            scenario_logit(
                eta1, beta1, dimension, code
            )
        )

        b = expit(
            scenario_logit(
                eta2, beta2, dimension, code
            )
        )

        care_rate = np.dot(w, a) / total_weight

        joint_pharmacy = (
            np.dot(w, a * b)
            / total_weight
        )

        result[index] = [
            care_rate,
            joint_pharmacy / care_rate,
            joint_pharmacy,
            care_rate - joint_pharmacy,
            1 - care_rate,
        ]

    return result


point_estimates = standardized_probabilities(
    beta1, beta2, weights
)

scenario_lookup = {
    (dimension, code): index
    for index, (
        dimension, variable, code, label
    ) in enumerate(scenarios)
}

contrast_specs = [
    (
        "Wealth quintile",
        "Poorest minus richest",
        1,
        5,
    ),
    (
        "Maternal education",
        "No education minus higher",
        0,
        3,
    ),
    (
        "Residence",
        "Rural minus urban",
        2,
        1,
    ),
]


def calculate_contrasts(standardized):

    result = np.zeros(
        (len(contrast_specs), 5),
        dtype=float,
    )

    for index, (
        dimension, contrast_label, group_a, group_b
    ) in enumerate(contrast_specs):

        A = standardized[
            scenario_lookup[(dimension, group_a)]
        ]

        B = standardized[
            scenario_lookup[(dimension, group_b)]
        ]

        care_a, conditional_a, joint_a = A[:3]
        care_b, conditional_b, joint_b = B[:3]

        access_component = (
            0.5
            * (care_a - care_b)
            * (conditional_a + conditional_b)
        )

        source_component = (
            0.5
            * (conditional_a - conditional_b)
            * (care_a + care_b)
        )

        result[index] = [
            care_a - care_b,
            conditional_a - conditional_b,
            joint_a - joint_b,
            access_component,
            source_component,
        ]

    return result


contrast_names = [
    "difference_care_sought_pp",
    "difference_pharmacy_given_care_pp",
    "difference_pharmacy_among_all_fever_pp",
    "access_component_pp",
    "conditional_source_component_pp",
]

point_contrasts = calculate_contrasts(
    point_estimates
)

identity_error = np.max(
    np.abs(
        point_contrasts[:, 3]
        + point_contrasts[:, 4]
        - point_contrasts[:, 2]
    )
)

if identity_error > 1e-12:
    raise ValueError(
        "Adjusted pathway decomposition identity failed."
    )


# Joint model-refit bootstrap within the original 16 strata.
psu_index = pd.Index(
    full_psus["v021"]
)

child_psu = psu_index.get_indexer(
    data["v021"]
)

if (child_psu < 0).any():
    raise ValueError("Original PSU linkage failed.")

stratum_indices = [
    group.index.to_numpy(dtype=int)
    for _, group in full_psus.groupby("v023")
]

rng = np.random.default_rng(SEED)

multiplicity = np.zeros(
    (len(full_psus), N_BOOTSTRAP),
    dtype=np.int16,
)

for indices in stratum_indices:

    multiplicity[indices, :] = rng.multinomial(
        n=len(indices),
        pvals=np.full(
            len(indices),
            1 / len(indices),
        ),
        size=N_BOOTSTRAP,
    ).T


# Numerically stable weighted logistic Newton estimation.
def fit_weighted_logistic(
    x, y, w, initial, max_steps=35
):

    beta = initial.copy()
    total_weight = float(w.sum())

    if total_weight <= 0:
        raise ValueError("Empty bootstrap model domain.")

    for iteration in range(max_steps):

        eta = x @ beta
        p = expit(eta)

        score = x.T @ (w * (y - p))

        if np.max(
            np.abs(score)
        ) <= 1e-8 * total_weight:

            return beta, iteration, True

        hessian = x.T @ (
            (w * p * (1 - p))[:, None] * x
        )

        try:
            direction = np.linalg.solve(
                hessian, score
            )

        except np.linalg.LinAlgError:
            return beta, iteration, False

        old_loss = float(
            np.sum(
                w * (
                    np.logaddexp(0, eta)
                    - y * eta
                )
            )
        )

        step = 1.0
        accepted = False

        for _ in range(22):

            candidate = (
                beta + step * direction
            )

            next_eta = x @ candidate

            next_loss = float(
                np.sum(
                    w * (
                        np.logaddexp(0, next_eta)
                        - y * next_eta
                    )
                )
            )

            if (
                np.isfinite(next_loss)
                and next_loss <= old_loss + 1e-9
            ):
                beta = candidate
                accepted = True
                break

            step *= 0.5

        if not accepted:
            return beta, iteration, False

    return beta, max_steps, False


bootstrap_scenarios = np.empty(
    (N_BOOTSTRAP, len(scenarios), 5),
    dtype=float,
)

bootstrap_contrasts = np.empty(
    (N_BOOTSTRAP, len(contrast_specs), 5),
    dtype=float,
)

fit_diagnostics = []

for replicate in range(N_BOOTSTRAP):

    bootstrap_weights = (
        weights
        * multiplicity[child_psu, replicate]
    )

    fitted_beta1, iterations1, success1 = (
        fit_weighted_logistic(
            X,
            y1,
            bootstrap_weights,
            beta1,
        )
    )

    fitted_beta2, iterations2, success2 = (
        fit_weighted_logistic(
            X[seeker],
            y2[seeker],
            bootstrap_weights[seeker],
            beta2,
        )
    )

    if not (success1 and success2):
        raise RuntimeError(
            f"Bootstrap replicate {replicate + 1} "
            "failed joint model convergence."
        )

    probabilities = standardized_probabilities(
        fitted_beta1,
        fitted_beta2,
        bootstrap_weights,
    )

    if not (
        np.isfinite(probabilities).all()
        and np.all(probabilities > 0)
        and np.all(probabilities < 1)
    ):
        raise ValueError(
            f"Invalid bootstrap probabilities "
            f"in replicate {replicate + 1}."
        )

    bootstrap_scenarios[replicate] = (
        probabilities
    )

    bootstrap_contrasts[replicate] = (
        calculate_contrasts(probabilities)
    )

    fit_diagnostics.append({
        "replicate": replicate + 1,
        "stage1_newton_iterations": iterations1,
        "stage2_newton_iterations": iterations2,
    })


# Compute percentile confidence intervals for all adjusted scenarios.
scenario_rows = []

for index, (
    dimension, variable, code, label
) in enumerate(scenarios):

    for metric_index, metric in enumerate(metric_names):

        lower, upper = np.percentile(
            bootstrap_scenarios[
                :, index, metric_index
            ],
            [2.5, 97.5],
        )

        scenario_rows.append({
            "dimension": dimension,
            "level_code": code,
            "level_label": label,
            "pathway": metric,
            "reference_fever_children": len(data),
            "weighted_standardized_percent": (
                100 * point_estimates[index, metric_index]
            ),
            "bootstrap95_lower_percent": (
                100 * lower
            ),
            "bootstrap95_upper_percent": (
                100 * upper
            ),
        })

scenario_table = pd.DataFrame(
    scenario_rows
)


# Compare adjusted and previously verified unadjusted pathways.
unadjusted = pd.read_csv(
    inputs["unadjusted"]
)

comparison_rows = []

for index, (
    dimension, name, group_a, group_b
) in enumerate(contrast_specs):

    matched = unadjusted.loc[
        unadjusted["contrast"].eq(name)
    ]

    if len(matched) != 1:
        raise ValueError(
            f"Previous unadjusted contrast missing: {name}"
        )

    record = {
        "dimension": dimension,
        "contrast": name,
        "reference_n": len(data),
        "bootstrap_draws": N_BOOTSTRAP,
        "unadjusted_joint_pharmacy_gap_pp": float(
            matched.iloc[0][
                "difference_pharmacy_among_all_fever_pp"
            ]
        ),
        "unadjusted_access_component_pp": float(
            matched.iloc[0][
                "access_component_pp"
            ]
        ),
        "unadjusted_source_component_pp": float(
            matched.iloc[0][
                "conditional_source_component_pp"
            ]
        ),
    }

    for metric_index, metric in enumerate(
        contrast_names
    ):

        lower, upper = np.percentile(
            bootstrap_contrasts[
                :, index, metric_index
            ],
            [2.5, 97.5],
        )

        record[metric] = (
            100 * point_contrasts[
                index, metric_index
            ]
        )

        record[metric + "_bootstrap_ci_low"] = (
            100 * lower
        )

        record[metric + "_bootstrap_ci_high"] = (
            100 * upper
        )

    record[
        "joint_gap_change_adjusted_minus_unadjusted_pp"
    ] = (
        record[
            "difference_pharmacy_among_all_fever_pp"
        ]
        - record[
            "unadjusted_joint_pharmacy_gap_pp"
        ]
    )

    record["decomposition_error_pp"] = (
        100
        * abs(
            point_contrasts[index, 3]
            + point_contrasts[index, 4]
            - point_contrasts[index, 2]
        )
    )

    record["bootstrap_decomposition_error_pp"] = (
        100
        * np.max(
            np.abs(
                bootstrap_contrasts[:, index, 3]
                + bootstrap_contrasts[:, index, 4]
                - bootstrap_contrasts[:, index, 2]
            )
        )
    )

    comparison_rows.append(record)

comparison_table = pd.DataFrame(
    comparison_rows
)


# Independently verify previously fitted survey regression estimates.
reconstruction = pd.DataFrame([
    {
        "model": "Stage 1 care sought",
        "n": len(data),
        "events": int(y1.sum()),
        "psus": int(data["v021"].nunique()),
        "weighted_observed_percent": (
            100 * observed_stage1
        ),
        "weighted_fitted_percent": (
            100 * fitted_stage1
        ),
        "max_abs_weighted_score": float(
            np.max(np.abs(score1))
        ),
    },
    {
        "model": "Stage 2 pharmacy among seekers",
        "n": int(seeker.sum()),
        "events": int(y2[seeker].sum()),
        "psus": int(
            data.loc[seeker, "v021"].nunique()
        ),
        "weighted_observed_percent": (
            100 * observed_stage2
        ),
        "weighted_fitted_percent": (
            100 * fitted_stage2
        ),
        "max_abs_weighted_score": float(
            np.max(np.abs(score2))
        ),
    },
])


# Complete reproducibility and methodological validation.
checks = {
    "Original source and Step 11.1 verified":
        bool(step11a_audit["validation_passed"]),

    "Full frame contains 674 PSUs and 16 strata":
        len(full_psus) == 674
        and len(stratum_indices) == 16,

    "Frozen complete-case populations preserved":
        len(data) == 2624
        and int(seeker.sum()) == 1969,

    "Stage outcomes and Other-source handling correct":
        int(y1.sum()) == 1969
        and int(y2[seeker].sum()) == 1117
        and int(
            (
                stage2_domain
                & raw["h46a"].eq(96)
            ).sum()
        ) == 33,

    "Both verified fitted prevalences reproduced":
        max(
            abs(observed_stage1 - fitted_stage1),
            abs(observed_stage2 - fitted_stage2),
        ) < 1e-7,

    "Both models retain 22 full-rank predictors":
        X.shape[1] == 22
        and np.linalg.matrix_rank(X) == 22,

    "All eleven exposure scenarios generated":
        len(scenarios) == 11,

    "All 55 standardized intervals calculated":
        len(scenario_table) == 55,

    "Three original contrasts retained":
        len(comparison_table) == 3,

    "Three pathways sum to one":
        np.max(
            np.abs(
                point_estimates[:, 2]
                + point_estimates[:, 3]
                + point_estimates[:, 4]
                - 1
            )
        ) < 1e-12,

    "Joint and conditional pathway identity valid":
        np.max(
            np.abs(
                point_estimates[:, 0]
                * point_estimates[:, 1]
                - point_estimates[:, 2]
            )
        ) < 1e-12,

    "All 999 replicates refit both logistic models":
        len(fit_diagnostics) == N_BOOTSTRAP,

    "Bootstrap preserves PSU counts within strata":
        all(
            np.all(
                multiplicity[
                    indices, :
                ].sum(axis=0) == len(indices)
            )
            for indices in stratum_indices
        ),

    "All bootstrap probabilities valid":
        np.isfinite(bootstrap_scenarios).all()
        and np.all(
            (bootstrap_scenarios > 0)
            & (bootstrap_scenarios < 1)
        ),

    "All decomposition identities exact":
        comparison_table[
            "decomposition_error_pp"
        ].max() < 1e-8
        and comparison_table[
            "bootstrap_decomposition_error_pp"
        ].max() < 1e-8,

    "Scenario intervals contain point estimates":
        bool(
            (
                scenario_table[
                    "bootstrap95_lower_percent"
                ]
                <= scenario_table[
                    "weighted_standardized_percent"
                ]
            ).all()
            and (
                scenario_table[
                    "bootstrap95_upper_percent"
                ]
                >= scenario_table[
                    "weighted_standardized_percent"
                ]
            ).all()
        ),

    "All adjusted contrast intervals finite":
        bool(
            np.isfinite(
                comparison_table.select_dtypes(
                    include=[np.number]
                ).to_numpy()
            ).all()
        ),

    "Unadjusted results retained for comparison":
        len(unadjusted) == 3
        and all(
            unadjusted["contrast"].isin(
                [
                    item[1]
                    for item in contrast_specs
                ]
            )
        ),
}

validation = pd.DataFrame([
    {
        "check": label,
        "status": (
            "PASS" if bool(value) else "REVIEW"
        ),
    }
    for label, value in checks.items()
])

all_passed = all(
    bool(value)
    for value in checks.values()
)


# Save independently from earlier Stage 11.1 files.
outputs = {
    "scenarios":
        TABLES / "11b_covariate_standardized_three_pathway_probabilities.csv",

    "contrasts":
        TABLES / "11b_adjusted_three_pathway_equity_decomposition.csv",

    "reconstruction":
        TABLES / "11b_two_stage_model_prediction_reconstruction.csv",

    "bootstrap_iterations":
        TABLES / "11b_joint_refit_bootstrap_fit_diagnostics.csv",

    "checks":
        TABLES / "11b_adjusted_pathway_validation_checks.csv",

    "audit":
        LOGS / "11b_adjusted_pathway_decomposition_audit.json",

    "report":
        LOGS / "11b_adjusted_pathway_decomposition_report.txt",
}

for name, table in (
    ("scenarios", scenario_table),
    ("contrasts", comparison_table),
    ("reconstruction", reconstruction),
    ("bootstrap_iterations", pd.DataFrame(fit_diagnostics)),
    ("checks", validation),
):
    table.to_csv(
        outputs[name],
        index=False,
        encoding="utf-8-sig",
    )

report = "\n".join([
    "STEP 11.2 | COVARIATE-STANDARDIZED THREE-PATHWAY DECOMPOSITION",
    "=" * 76,
    "Original population: 2630 fever-reported children",
    "Standardization reference: 2624 complete-case fever children",
    "Stage 2 model: 1969 complete-case care seekers",
    "Models: verified Stage 1 and Stage 2 weighted logistic regressions",
    "Scenarios: wealth quintile, maternal education, residence",
    "Uncertainty: joint refitting of both logistic models in 999 PSU bootstrap replicates",
    "Sampling structure: 674 PSUs within 16 original strata",
    "Decomposition: descriptive algebraic identity, not causal mediation",
    "Contrasts: covariate-standardized associations, not causal effects",
    "Inference: approximate survey-stratified PSU bootstrap",
    "No model selection or additional ML fitting",
    f"Validation: {'PASS' if all_passed else 'REVIEW REQUIRED'}",
    f"Checks: {sum(bool(v) for v in checks.values())}/{len(checks)}",
])

outputs["report"].write_text(
    report + "\n",
    encoding="utf-8",
)

audit = {
    "step": "11.2",
    "original_kr_sha256": source_hash,
    "population_full_fever": 2630,
    "standardized_reference_complete_cases": 2624,
    "stage2_training_n": 1969,
    "bootstrap_replicates": N_BOOTSTRAP,
    "survey_psus": 674,
    "survey_strata": 16,
    "seed": SEED,
    "limitations": [
        "This is not causal mediation or an intervention effect.",
        "Two-stage predicted source probabilities are projected over a common covariate reference distribution.",
        "PSU-within-stratum bootstrap approximates survey uncertainty; finite population correction is unavailable.",
        "Both models are refitted in each bootstrap replicate, but model specification remains fixed.",
        "Positivity and model misspecification remain relevant.",
        "Contrasts are exploratory without multiplicity-adjusted testing.",
        "Step 11.1 used 2630 children whereas standardized Step 11.2 uses 2624 complete cases.",
    ],
    "checks": {
        key: bool(value)
        for key, value in checks.items()
    },
    "validation_passed": bool(all_passed),
}

outputs["audit"].write_text(
    json.dumps(
        audit,
        indent=2,
        ensure_ascii=False,
    ) + "\n",
    encoding="utf-8",
)


# Display outputs in the notebook.
print(report)

print("\nADJUSTED THREE-PATHWAY EQUITY DECOMPOSITION")
display(
    comparison_table[
        [
            "contrast",
            "unadjusted_joint_pharmacy_gap_pp",
            "difference_care_sought_pp",
            "difference_pharmacy_given_care_pp",
            "difference_pharmacy_among_all_fever_pp",
            "difference_pharmacy_among_all_fever_pp_bootstrap_ci_low",
            "difference_pharmacy_among_all_fever_pp_bootstrap_ci_high",
            "access_component_pp",
            "conditional_source_component_pp",
            "access_component_pp_bootstrap_ci_low",
            "access_component_pp_bootstrap_ci_high",
            "conditional_source_component_pp_bootstrap_ci_low",
            "conditional_source_component_pp_bootstrap_ci_high",
        ]
    ].round(4)
)

print("\nCOVARIATE-STANDARDIZED PATHWAY PROBABILITIES")
display(
    scenario_table.loc[
        scenario_table["pathway"].isin([
            "care_sought",
            "pharmacy_first_given_modeled_care",
            "pharmacy_first_all_fever",
        ])
    ].round(3)
)

print("\nVERIFIED TWO-STAGE MODEL RECONSTRUCTION")
display(reconstruction.round(7))

print("\nVALIDATION CHECKS")
display(validation)

print("\nSAVED OUTPUTS")
for path in outputs.values():
    print(path)

if not all_passed:
    raise ValueError(
        "Step 11.2 standardized pathway analysis requires review."
    )

STEP 11.2 | COVARIATE-STANDARDIZED THREE-PATHWAY DECOMPOSITION
Original population: 2630 fever-reported children
Standardization reference: 2624 complete-case fever children
Stage 2 model: 1969 complete-case care seekers
Models: verified Stage 1 and Stage 2 weighted logistic regressions
Scenarios: wealth quintile, maternal education, residence
Uncertainty: joint refitting of both logistic models in 999 PSU bootstrap replicates
Sampling structure: 674 PSUs within 16 original strata
Decomposition: descriptive algebraic identity, not causal mediation
Contrasts: covariate-standardized associations, not causal effects
Inference: approximate survey-stratified PSU bootstrap
No model selection or additional ML fitting
Validation: PASS
Checks: 18/18

ADJUSTED THREE-PATHWAY EQUITY DECOMPOSITION


,contrast,unadjusted_joint_pharmacy_gap_pp,difference_care_sought_pp,difference_pharmacy_given_care_pp,difference_pharmacy_among_all_fever_pp,difference_pharmacy_among_all_fever_pp_bootstrap_ci_low,difference_pharmacy_among_all_fever_pp_bootstrap_ci_high,access_component_pp,conditional_source_component_pp,access_component_pp_bootstrap_ci_low,access_component_pp_bootstrap_ci_high,conditional_source_component_pp_bootstrap_ci_low,conditional_source_component_pp_bootstrap_ci_high
0,Poorest minus richest,13.5649,2.6958,5.4848,5.5772,-2.6404,13.6925,1.4712,4.1060,-2.6068,5.1013,-3.1362,11.4364
1,No education minus higher,17.8794,2.8193,13.6534,11.4738,0.6097,22.2133,1.6103,9.8634,-3.7985,6.9818,0.7457,19.4235
2,Rural minus urban,6.3114,0.1462,7.7049,5.9488,1.1901,11.1497,0.0816,5.8671,-2.3541,2.4097,1.0880,10.6960



COVARIATE-STANDARDIZED PATHWAY PROBABILITIES


,dimension,level_code,level_label,pathway,reference_fever_children,weighted_standardized_percent,bootstrap95_lower_percent,bootstrap95_upper_percent
0,Wealth quintile,1,Poorest,care_sought,2624,76.209,71.202,80.232
1,Wealth quintile,1,Poorest,pharmacy_first_given_modeled_care,2624,57.315,50.857,63.779
2,Wealth quintile,1,Poorest,pharmacy_first_all_fever,2624,43.679,38.209,49.226
5,Wealth quintile,2,Poorer,care_sought,2624,74.142,69.781,78.147
6,Wealth quintile,2,Poorer,pharmacy_first_given_modeled_care,2624,59.333,53.046,64.949
7,Wealth quintile,2,Poorer,pharmacy_first_all_fever,2624,43.991,38.700,49.185
10,Wealth quintile,3,Middle,care_sought,2624,77.723,73.593,81.640
11,Wealth quintile,3,Middle,pharmacy_first_given_modeled_care,2624,59.845,54.994,65.007
12,Wealth quintile,3,Middle,pharmacy_first_all_fever,2624,46.513,42.289,50.679
15,Wealth quintile,4,Richer,care_sought,2624,78.992,75.348,82.480



VERIFIED TWO-STAGE MODEL RECONSTRUCTION


,model,n,events,psus,weighted_observed_percent,weighted_fitted_percent,max_abs_weighted_score
0,Stage 1 care sought,2624,1969,642,76.182768,76.182768,3.000000e-07
1,Stage 2 pharmacy among seekers,1969,1117,611,57.758511,57.758511,0.000000e+00



VALIDATION CHECKS


,check,status
0,Original source and Step 11.1 verified,PASS
1,Full frame contains 674 PSUs and 16 strata,PASS
2,Frozen complete-case populations preserved,PASS
3,Stage outcomes and Other-source handling correct,PASS
4,Both verified fitted prevalences reproduced,PASS
5,Both models retain 22 full-rank predictors,PASS
6,All eleven exposure scenarios generated,PASS
7,All 55 standardized intervals calculated,PASS
8,Three original contrasts retained,PASS
9,Three pathways sum to one,PASS



SAVED OUTPUTS
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\11b_covariate_standardized_three_pathway_probabilities.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\11b_adjusted_three_pathway_equity_decomposition.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\11b_two_stage_model_prediction_reconstruction.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\11b_joint_refit_bootstrap_fit_diagnostics.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\11b_adjusted_pathway_validation_checks.csv
D:\BDHS2022_FeverCare_SpatialEquity\logs\11b_adjusted_pathway_decomposition_audit.json
D:\BDHS2022_FeverCare_SpatialEquity\logs\11b_adjusted_pathway_decomposition_report.txt


In [32]:

# STEP 11.3 | Simultaneous inference sensitivity
# for covariate-standardized three-pathway contrasts.

from pathlib import Path
import os
import json
import hashlib

import numpy as np
import pandas as pd
from scipy.special import expit
from IPython.display import display


ROOT = Path(
    os.environ.get(
        "BDHS_PROJECT_ROOT",
        r"D:\BDHS2022_FeverCare_SpatialEquity"
    )
)

RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
TABLES = ROOT / "results" / "tables"
LOGS = ROOT / "logs"

INPUTS = {
    "kr": RAW / "BDKR81FL.DTA",
    "cohort": PROC / "06_fever_care_analysis_ready.csv",
    "source_audit": LOGS / "01_input_audit.json",
    "previous_audit": LOGS / "11b_adjusted_pathway_decomposition_audit.json",
    "previous_contrasts": TABLES / "11b_adjusted_three_pathway_equity_decomposition.csv",
    "stage1_coefficients": TABLES / "08a_stage1_adjusted_logistic_coefficients.csv",
    "stage2_coefficients": TABLES / "08c_stage2_adjusted_logistic_coefficients.csv",
}

for name, path in INPUTS.items():
    if not path.is_file():
        raise FileNotFoundError(
            f"Required input missing ({name}): {path}"
        )

with INPUTS["source_audit"].open(encoding="utf-8") as file:
    original_audit = json.load(file)

with INPUTS["previous_audit"].open(encoding="utf-8") as file:
    previous_audit = json.load(file)

if not previous_audit.get("validation_passed", False):
    raise ValueError(
        "Step 11.2 validation must PASS before this step."
    )

with INPUTS["kr"].open("rb") as file:
    source_checksum = hashlib.file_digest(
        file, "sha256"
    ).hexdigest()

if source_checksum != original_audit["source_files"]["kr_sha256"]:
    raise ValueError(
        "Original KR checksum mismatch."
    )


# Reuse exactly the Step 11.2 model and bootstrap design.
B = 999
SEED = 20261117 + 112
ALPHA = 0.05

keys = ["v001", "v002", "v003", "bidx"]

raw = pd.read_stata(
    INPUTS["kr"],
    convert_categoricals=False,
    columns=keys + [
        "v021", "v023", "v005",
        "h22", "h32y", "h46a",
        "h31", "h31b", "b19", "b4",
        "v012", "v106", "v190",
        "bord", "v025", "v024",
    ],
)

frozen = pd.read_csv(
    INPUTS["cohort"],
    usecols=keys + [
        "stage1_core_complete_case",
        "stage2_core_complete_case",
    ],
)

full_psus = (
    raw[["v021", "v023"]]
    .drop_duplicates()
    .sort_values(["v023", "v021"])
    .reset_index(drop=True)
)

if not (
    len(raw) == 8784
    and not raw.duplicated(keys).any()
    and raw["v001"].eq(raw["v021"]).all()
    and len(full_psus) == 674
    and full_psus["v023"].nunique() == 16
):
    raise ValueError(
        "Verified survey design is inconsistent."
    )

fever = raw["h22"].eq(1)
care = fever & raw["h32y"].eq(0)

complete = (
    raw[
        [
            "b19", "b4", "h31", "v012",
            "v106", "v190", "bord",
            "v025", "v024",
        ]
    ].notna().all(axis=1)
    & raw["h31b"].isin([0, 1])
)

stage1 = fever & complete
stage2 = care & complete

if not (
    fever.sum() == 2630
    and care.sum() == 1974
    and stage1.sum() == 2624
    and stage2.sum() == 1969
    and (stage2 & raw["h46a"].eq(36)).sum() == 1117
    and (care & raw["h46a"].eq(96)).sum() == 33
):
    raise ValueError(
        "Stage domain or source code inventory changed."
    )

for domain, flag in (
    (stage1, "stage1_core_complete_case"),
    (stage2, "stage2_core_complete_case"),
):
    ids = (
        raw.loc[domain, keys]
        .astype("int64")
        .sort_values(keys)
        .reset_index(drop=True)
    )

    verified = (
        frozen.loc[frozen[flag].eq(1), keys]
        .astype("int64")
        .sort_values(keys)
        .reset_index(drop=True)
    )

    if not ids.equals(verified):
        raise ValueError(
            f"Frozen population mismatch: {flag}"
        )


# Reconstruct both verified weighted logistic models.
data = (
    raw.loc[stage1]
    .copy()
    .reset_index(drop=True)
)

seekers = (
    data["h32y"]
    .eq(0)
    .to_numpy(dtype=bool)
)

y1 = seekers.astype(float)
y2 = data["h46a"].eq(36).to_numpy(dtype=float)

w = (
    data["v005"].to_numpy(dtype=float)
    / 1e6
)

if not (
    len(w) == 2624
    and seekers.sum() == 1969
    and y2[seekers].sum() == 1117
    and np.isfinite(w).all()
    and (w > 0).all()
):
    raise ValueError(
        "Invalid analysis weights or outcomes."
    )

terms = [
    "_intercept_",
    "child_age_years",
    "b4_2",
    "cough_reported_1.0",
    "rapid_breathing_reported_1.0",
    "maternal_age_5years",
    *[f"v106_{v}" for v in (1, 2, 3)],
    *[f"v190_{v}" for v in (2, 3, 4, 5)],
    "bord",
    "v025_2",
    *[f"v024_{v}" for v in range(2, 9)],
]

X = np.column_stack([
    np.ones(len(data)),
    data["b19"].to_numpy(float) / 12,
    data["b4"].eq(2).to_numpy(float),
    data["h31"].eq(2).to_numpy(float),
    data["h31b"].eq(1).to_numpy(float),
    (data["v012"].to_numpy(float) - 25) / 5,

    *[
        data["v106"].eq(v).to_numpy(float)
        for v in (1, 2, 3)
    ],

    *[
        data["v190"].eq(v).to_numpy(float)
        for v in (2, 3, 4, 5)
    ],

    data["bord"].to_numpy(float),
    data["v025"].eq(2).to_numpy(float),

    *[
        data["v024"].eq(v).to_numpy(float)
        for v in range(2, 9)
    ],
]).astype(float)

if (
    X.shape != (2624, 22)
    or np.linalg.matrix_rank(X) != 22
):
    raise ValueError(
        "Rebuilt 22-column design matrix mismatch."
    )

term_index = {
    name: j
    for j, name in enumerate(terms)
}

coefs = {}

for name, path in (
    ("stage1", INPUTS["stage1_coefficients"]),
    ("stage2", INPUTS["stage2_coefficients"]),
):
    table = pd.read_csv(path)

    if not (
        len(table) == 22
        and not table["term"].duplicated().any()
        and set(table["term"]) == set(terms)
    ):
        raise ValueError(
            f"Verified coefficient terms differ: {name}"
        )

    coefs[name] = (
        table.set_index("term")
        .loc[terms, "est"]
        .to_numpy(float)
    )

b1 = coefs["stage1"]
b2 = coefs["stage2"]

p1 = expit(X @ b1)
p2 = expit(X @ b2)

score1 = X.T @ (
    w * (y1 - p1)
)

score2 = X[seekers].T @ (
    w[seekers]
    * (y2[seekers] - p2[seekers])
)

if max(
    np.max(np.abs(score1)),
    np.max(np.abs(score2)),
) > 2e-5:
    raise ValueError(
        "Saved models fail weighted score reproduction."
    )

if max(
    abs(
        np.average(y1, weights=w)
        - np.average(p1, weights=w)
    ),

    abs(
        np.average(
            y2[seekers],
            weights=w[seekers],
        )
        - np.average(
            p2[seekers],
            weights=w[seekers],
        )
    ),
) > 1e-7:
    raise ValueError(
        "Saved model weighted prevalences mismatch."
    )


# Same complete-case standardization reference as Step 11.2.
scenarios = [
    *[
        ("Wealth quintile", code)
        for code in range(1, 6)
    ],

    *[
        ("Maternal education", code)
        for code in range(4)
    ],

    ("Residence", 1),
    ("Residence", 2),
]

scenario_idx = {
    item: j
    for j, item in enumerate(scenarios)
}

comparisons = [
    (
        "Wealth quintile",
        "Poorest minus richest",
        1, 5,
    ),
    (
        "Maternal education",
        "No education minus higher",
        0, 3,
    ),
    (
        "Residence",
        "Rural minus urban",
        2, 1,
    ),
]

metric_columns = [
    "difference_care_sought_pp",
    "difference_pharmacy_given_care_pp",
    "difference_pharmacy_among_all_fever_pp",
    "access_component_pp",
    "conditional_source_component_pp",
]


def replace_exposure(
    eta, beta, dimension, code
):

    if dimension == "Wealth quintile":

        cols = [
            term_index[f"v190_{v}"]
            for v in (2, 3, 4, 5)
        ]

        add = (
            beta[term_index[f"v190_{code}"]]
            if code > 1
            else 0.0
        )

        return (
            eta
            - X[:, cols] @ beta[cols]
            + add
        )

    if dimension == "Maternal education":

        cols = [
            term_index[f"v106_{v}"]
            for v in (1, 2, 3)
        ]

        add = (
            beta[term_index[f"v106_{code}"]]
            if code > 0
            else 0.0
        )

        return (
            eta
            - X[:, cols] @ beta[cols]
            + add
        )

    if dimension == "Residence":

        col = term_index["v025_2"]

        return (
            eta
            - X[:, col] * beta[col]
            + (
                beta[col]
                if code == 2
                else 0.0
            )
        )

    raise ValueError(
        "Unknown exposure scenario."
    )


def standardized(
    beta1, beta2, reference_weights
):

    eta1 = X @ beta1
    eta2 = X @ beta2

    total = reference_weights.sum()

    if total <= 0:
        raise ValueError(
            "Bootstrap reference-weight total is zero."
        )

    results = np.empty(
        (len(scenarios), 3)
    )

    for j, (
        dimension, code
    ) in enumerate(scenarios):

        a = expit(
            replace_exposure(
                eta1, beta1, dimension, code
            )
        )

        b = expit(
            replace_exposure(
                eta2, beta2, dimension, code
            )
        )

        care_rate = (
            np.dot(reference_weights, a)
            / total
        )

        joint = (
            np.dot(reference_weights, a * b)
            / total
        )

        results[j] = [
            care_rate,
            joint / care_rate,
            joint,
        ]

    return results


def decompositions(rates):

    out = np.empty(
        (
            len(comparisons),
            len(metric_columns),
        ),
        dtype=float,
    )

    for j, (
        dimension,
        label,
        code_a,
        code_b,
    ) in enumerate(comparisons):

        a = rates[
            scenario_idx[(dimension, code_a)]
        ]

        b = rates[
            scenario_idx[(dimension, code_b)]
        ]

        access = (
            0.5
            * (a[0] - b[0])
            * (a[1] + b[1])
        )

        source = (
            0.5
            * (a[1] - b[1])
            * (a[0] + b[0])
        )

        out[j] = [
            a[0] - b[0],
            a[1] - b[1],
            a[2] - b[2],
            access,
            source,
        ]

    return out


point = 100 * decompositions(
    standardized(b1, b2, w)
)

if np.max(
    np.abs(
        point[:, 3]
        + point[:, 4]
        - point[:, 2]
    )
) > 1e-9:
    raise ValueError(
        "Observed standardized decomposition identity failed."
    )


# Survey-stratified PSU bootstrap, refitting both stage models.
psu_index = pd.Index(
    full_psus["v021"]
)

child_psu = psu_index.get_indexer(
    data["v021"]
)

if (child_psu < 0).any():
    raise ValueError(
        "Survey PSU linkage invalid."
    )

stratum_indices = [
    group.index.to_numpy(dtype=int)
    for _, group in full_psus.groupby("v023")
]

rng = np.random.default_rng(SEED)

multiplicity = np.zeros(
    (674, B),
    dtype=np.int16,
)

for idx in stratum_indices:

    multiplicity[idx, :] = rng.multinomial(
        n=len(idx),
        pvals=np.full(
            len(idx),
            1 / len(idx),
        ),
        size=B,
    ).T


def fit_weighted_logistic(
    x, y, weight, start, max_steps=35
):

    beta = start.copy()
    total_w = weight.sum()

    if total_w <= 0:
        return beta, 0, False

    for it in range(max_steps):

        eta = x @ beta
        p = expit(eta)

        score = x.T @ (
            weight * (y - p)
        )

        if np.max(
            np.abs(score)
        ) <= 1e-8 * total_w:
            return beta, it, True

        h = x.T @ (
            (
                weight
                * p
                * (1 - p)
            )[:, None] * x
        )

        try:
            step_direction = np.linalg.solve(
                h, score
            )
        except np.linalg.LinAlgError:
            return beta, it, False

        previous_loss = float(
            np.sum(
                weight
                * (
                    np.logaddexp(0, eta)
                    - y * eta
                )
            )
        )

        scale = 1.0
        accepted = False

        for _ in range(22):

            trial = (
                beta
                + scale * step_direction
            )

            trial_eta = x @ trial

            trial_loss = float(
                np.sum(
                    weight
                    * (
                        np.logaddexp(0, trial_eta)
                        - y * trial_eta
                    )
                )
            )

            if (
                np.isfinite(trial_loss)
                and trial_loss <= previous_loss + 1e-9
            ):
                beta = trial
                accepted = True
                break

            scale *= 0.5

        if not accepted:
            return beta, it, False

    return beta, max_steps, False


boot = np.empty(
    (B, 3, 5),
    dtype=float,
)

iterations = []

for rep in range(B):

    wr = (
        w
        * multiplicity[
            child_psu, rep
        ]
    )

    z1, it1, ok1 = fit_weighted_logistic(
        X, y1, wr, b1
    )

    z2, it2, ok2 = fit_weighted_logistic(
        X[seekers],
        y2[seekers],
        wr[seekers],
        b2,
    )

    if not (ok1 and ok2):
        raise RuntimeError(
            f"PSU bootstrap {rep + 1}: "
            "joint logistic refit failed."
        )

    boot[rep] = (
        100
        * decompositions(
            standardized(
                z1, z2, wr
            )
        )
    )

    iterations.append(
        (rep + 1, it1, it2)
    )

if (
    not np.isfinite(boot).all()
    or np.max(
        np.abs(
            boot[:, :, 3]
            + boot[:, :, 4]
            - boot[:, :, 2]
        )
    ) > 1e-8
):
    raise ValueError(
        "Bootstrap decomposition numerical checks failed."
    )


# Reproduce ALL Step 11.2 point estimates and percentile intervals.
reference = pd.read_csv(
    INPUTS["previous_contrasts"]
)

if (
    len(reference) != 3
    or reference["contrast"].duplicated().any()
):
    raise ValueError(
        "Saved Step 11.2 contrast table is malformed."
    )

reconcile = []

for j, (
    dimension,
    label,
    _,
    _,
) in enumerate(comparisons):

    row = reference.loc[
        reference["contrast"].eq(label)
    ]

    if len(row) != 1:
        raise ValueError(
            f"Step 11.2 contrast missing: {label}"
        )

    row = row.iloc[0]

    for k, metric in enumerate(metric_columns):

        bounds = np.percentile(
            boot[:, j, k],
            [2.5, 97.5],
        )

        reconcile.append({
            "contrast": label,
            "metric": metric,

            "point_error_pp": abs(
                point[j, k]
                - float(row[metric])
            ),

            "percentile_lower_error_pp": abs(
                bounds[0]
                - float(
                    row[
                        metric
                        + "_bootstrap_ci_low"
                    ]
                )
            ),

            "percentile_upper_error_pp": abs(
                bounds[1]
                - float(
                    row[
                        metric
                        + "_bootstrap_ci_high"
                    ]
                )
            ),
        })

reconciliation = pd.DataFrame(
    reconcile
)


# Centered bootstrap simultaneous max-|Z| inference.
# The familywise procedure compares fixed standardized contrasts.
# It is exploratory, NOT a prespecified confirmatory hypothesis test.
def holm_adjust(p_values):

    values = np.asarray(
        p_values,
        dtype=float,
    )

    order = np.argsort(values)

    adjusted = np.empty(
        len(values)
    )

    adjusted[order] = np.minimum(
        1,
        np.maximum.accumulate(
            (
                len(values)
                - np.arange(len(values))
            )
            * values[order]
        ),
    )

    return adjusted


def family_inference(
    point_vector,
    bootstrap_matrix,
):

    observed = np.asarray(
        point_vector,
        dtype=float,
    )

    draws = np.asarray(
        bootstrap_matrix,
        dtype=float,
    )

    if (
        draws.ndim != 2
        or draws.shape != (
            B, len(observed)
        )
    ):
        raise ValueError(
            "Simultaneous inference dimensions inconsistent."
        )

    se = np.std(
        draws,
        axis=0,
        ddof=1,
    )

    if not (
        np.isfinite(se).all()
        and (se > 1e-8).all()
    ):
        raise ValueError(
            "Zero or non-finite bootstrap SE."
        )

    # Bootstrap deviations centered at original point estimates.
    centered_z = (
        draws - observed[None, :]
    ) / se[None, :]

    max_z = np.max(
        np.abs(centered_z),
        axis=1,
    )

    critical = float(
        np.quantile(
            max_z,
            1 - ALPHA,
        )
    )

    observed_z = np.abs(
        observed / se
    )

    unadjusted_p = (
        1
        + np.sum(
            np.abs(centered_z)
            >= observed_z[None, :],
            axis=0,
        )
    ) / (B + 1)

    max_z_p = (
        1
        + np.sum(
            max_z[:, None]
            >= observed_z[None, :],
            axis=0,
        )
    ) / (B + 1)

    holm_p = holm_adjust(
        unadjusted_p
    )

    return {
        "bootstrap_se_pp": se,

        "pointwise_centered_bootstrap_p":
            unadjusted_p,

        "holm_adjusted_p":
            holm_p,

        "maxz_familywise_adjusted_p":
            max_z_p,

        "pointwise_normal_approx_ci_low_pp":
            observed - 1.959963984540054 * se,

        "pointwise_normal_approx_ci_high_pp":
            observed + 1.959963984540054 * se,

        "maxz_simultaneous95_ci_low_pp":
            observed - critical * se,

        "maxz_simultaneous95_ci_high_pp":
            observed + critical * se,

        "maxz_critical": critical,
    }


# Family A: the three unconditional pharmacy-first gaps.
primary = family_inference(
    point[:, 2],
    boot[:, :, 2],
)

primary_rows = []

for j, (
    dimension,
    label,
    _,
    _,
) in enumerate(comparisons):

    matched = reference.loc[
        reference["contrast"].eq(label)
    ].iloc[0]

    primary_rows.append({
        "contrast": label,
        "dimension": dimension,

        "estimate_joint_gap_pp":
            point[j, 2],

        "step11b_percentile_ci_low_pp": float(
            matched[
                "difference_pharmacy_among_all_fever_pp_bootstrap_ci_low"
            ]
        ),

        "step11b_percentile_ci_high_pp": float(
            matched[
                "difference_pharmacy_among_all_fever_pp_bootstrap_ci_high"
            ]
        ),

        **{
            k: (
                float(v[j])
                if isinstance(v, np.ndarray)
                else float(v)
            )
            for k, v in primary.items()
        },
    })

primary_table = pd.DataFrame(
    primary_rows
)


# Family B: all 15 adjusted pathway estimands.
# This is the stricter, post-hoc multiplicity sensitivity.
all15 = family_inference(
    point.reshape(-1),
    boot.reshape(B, -1),
)

full_rows = []

for j, (
    _, label, _, _
) in enumerate(comparisons):

    for k, metric in enumerate(metric_columns):

        idx = (
            j * len(metric_columns)
            + k
        )

        full_rows.append({
            "contrast": label,
            "estimand": metric,
            "estimate_pp": point[j, k],

            **{
                name: (
                    float(value[idx])
                    if isinstance(value, np.ndarray)
                    else float(value)
                )
                for name, value in all15.items()
            },
        })

full_table = pd.DataFrame(
    full_rows
)


# Focus on access and conditional source-choice sensitivity.
secondary_table = full_table.loc[
    full_table["estimand"].isin([
        "difference_pharmacy_given_care_pp",
        "access_component_pp",
        "conditional_source_component_pp",
    ])
].copy()


# Reproducibility and inferential safeguards.
checks = {
    "Original KR source and Step 11.2 audit verified":
        bool(previous_audit["validation_passed"]),

    "Original full-frame 674 PSU and 16 strata preserved":
        len(full_psus) == 674
        and len(stratum_indices) == 16,

    "Both complete-case cohorts matched frozen eligibility":
        len(data) == 2624
        and int(seekers.sum()) == 1969,

    "Stage 2 pharmacy and Other source codes retained":
        int(y2[seekers].sum()) == 1117
        and int(
            (
                stage2
                & raw["h46a"].eq(96)
            ).sum()
        ) == 33,

    "Both saved weighted model scores reproduced":
        max(
            np.max(np.abs(score1)),
            np.max(np.abs(score2)),
        ) < 2e-5,

    "All bootstrap replicates jointly refit both models":
        len(iterations) == B,

    "PSU bootstrap counts preserved within strata":
        all(
            np.all(
                multiplicity[idx, :].sum(axis=0)
                == len(idx)
            )
            for idx in stratum_indices
        ),

    "All Step 11.2 contrasts and intervals reproduced":
        len(reconciliation) == 15
        and reconciliation.select_dtypes(
            include=[np.number]
        ).max().max() < 1e-6,

    "Point and bootstrap pathway identities satisfied":
        np.max(
            np.abs(
                point[:, 3]
                + point[:, 4]
                - point[:, 2]
            )
        ) < 1e-8
        and np.max(
            np.abs(
                boot[:, :, 3]
                + boot[:, :, 4]
                - boot[:, :, 2]
            )
        ) < 1e-8,

    "Three primary joint pharmacy-gap contrasts evaluated":
        len(primary_table) == 3,

    "Fifteen comprehensive estimands evaluated":
        len(full_table) == 15,

    "Nine source-choice/access sensitivity estimands":
        len(secondary_table) == 9,

    "Simultaneous intervals contain point estimates":
        bool(
            (
                primary_table[
                    "maxz_simultaneous95_ci_low_pp"
                ]
                <= primary_table[
                    "estimate_joint_gap_pp"
                ]
            ).all()
            and (
                primary_table[
                    "maxz_simultaneous95_ci_high_pp"
                ]
                >= primary_table[
                    "estimate_joint_gap_pp"
                ]
            ).all()
        ),

    "Simultaneous intervals not narrower than pointwise":
        bool(
            (
                primary_table[
                    "maxz_simultaneous95_ci_low_pp"
                ]
                <= primary_table[
                    "pointwise_normal_approx_ci_low_pp"
                ]
            ).all()
            and (
                primary_table[
                    "maxz_simultaneous95_ci_high_pp"
                ]
                >= primary_table[
                    "pointwise_normal_approx_ci_high_pp"
                ]
            ).all()
        ),

    "Adjusted p-values not smaller than pointwise":
        bool(
            (
                primary_table[
                    "holm_adjusted_p"
                ]
                >= primary_table[
                    "pointwise_centered_bootstrap_p"
                ] - 1e-12
            ).all()
            and (
                primary_table[
                    "maxz_familywise_adjusted_p"
                ]
                >= primary_table[
                    "pointwise_centered_bootstrap_p"
                ] - 1e-12
            ).all()
        ),

    "Fifteen-contrast threshold no smaller than three-contrast":
        float(all15["maxz_critical"])
        >= float(primary["maxz_critical"]) - 1e-12,

    "All inference estimates and intervals finite":
        bool(
            np.isfinite(
                primary_table.select_dtypes(
                    include=[np.number]
                ).to_numpy()
            ).all()
            and np.isfinite(
                full_table.select_dtypes(
                    include=[np.number]
                ).to_numpy()
            ).all()
        ),

    "No additional outer-fold ML model selection":
        True,
}

validation = pd.DataFrame([
    {
        "check": name,
        "status": (
            "PASS"
            if bool(value)
            else "REVIEW"
        ),
    }
    for name, value in checks.items()
])

passed = all(
    bool(value)
    for value in checks.values()
)


# Save independent Step 11.3 outputs.
OUTPUTS = {
    "primary":
        TABLES / "11c_three_joint_gaps_simultaneous_inference.csv",

    "all15":
        TABLES / "11c_fifteen_estimands_familywise_sensitivity.csv",

    "secondary":
        TABLES / "11c_source_choice_component_sensitivity.csv",

    "replicates":
        PROC / "11c_joint_refit_bootstrap_contrast_replicates.csv",

    "reproduction":
        TABLES / "11c_step11b_reproduction_checks.csv",

    "convergence":
        TABLES / "11c_joint_refit_fit_diagnostics.csv",

    "validation":
        TABLES / "11c_simultaneous_inference_validation_checks.csv",

    "audit":
        LOGS / "11c_simultaneous_inference_audit.json",

    "report":
        LOGS / "11c_simultaneous_inference_report.txt",
}

replicate_frames = []

for j, (
    _, label, _, _
) in enumerate(comparisons):

    replicate_frames.append(
        pd.DataFrame({
            "replicate": np.arange(1, B + 1),
            "contrast": label,

            **{
                metric: boot[:, j, k]
                for k, metric in enumerate(
                    metric_columns
                )
            },
        })
    )

replicate_table = pd.concat(
    replicate_frames,
    ignore_index=True,
)

for name, frame in (
    ("primary", primary_table),
    ("all15", full_table),
    ("secondary", secondary_table),
    ("replicates", replicate_table),
    ("reproduction", reconciliation),

    (
        "convergence",
        pd.DataFrame(
            iterations,
            columns=[
                "replicate",
                "stage1_iterations",
                "stage2_iterations",
            ],
        ),
    ),

    ("validation", validation),
):

    frame.to_csv(
        OUTPUTS[name],
        index=False,
        encoding="utf-8-sig",
    )


report = "\n".join([
    "STEP 11.3 | MULTIPLICITY-AWARE TWO-STAGE PATHWAY INFERENCE",
    "=" * 73,

    "Data: BDHS 2022 KR; verified Stage 11.2 model specification",
    "Reference: 2624 complete-case children; Stage 2 care seekers: 1969",

    "Main family: 3 adjusted unconditional pharmacy-first contrasts",
    "Conservative sensitivity family: all 15 related adjusted contrasts/components",

    f"Bootstrap: same {B} draws as Step 11.2; both logit models refitted in each draw",

    "Methods: centered bootstrap max-|Z| simultaneous intervals/p values; Holm adjusted p",

    "Multiplicity family was defined AFTER inspection of Step 11.2 results: exploratory only",

    "Uncertainty is approximate stratified PSU bootstrap; not exact DHS design-based CI",

    "Contrasts and decomposition are associational, not causal mediation",

    "NO geographic model tuning or new outcome-derived predictors introduced",

    (
        "Step 11.2 reconstruction maximum discrepancy (pp): "
        f"{reconciliation.select_dtypes(include=[np.number]).max().max():.3g}"
    ),

    f"Critical max-Z for primary family: {primary['maxz_critical']:.4f}",
    f"Critical max-Z for all 15 comparisons: {all15['maxz_critical']:.4f}",

    f"Validation: {'PASS' if passed else 'REVIEW REQUIRED'}",
    f"Checks: {sum(checks.values())}/{len(checks)}",
])

OUTPUTS["report"].write_text(
    report + "\n",
    encoding="utf-8",
)

audit = {
    "step": "11.3",
    "purpose": "Post-hoc simultaneous-inference sensitivity",
    "kr_sha256": source_checksum,
    "bootstraps": B,
    "seed": SEED,

    "primary_family":
        "3 unconditional pharmacy-first adjusted differences",

    "secondary_family":
        "15 differences and algebraic pathway components",

    "maxz_primary_critical":
        float(primary["maxz_critical"]),

    "maxz_all15_critical":
        float(all15["maxz_critical"]),

    "limitations": [
        "Post hoc choice of reported contrast family; not prespecified or confirmatory.",
        "All 15 endpoints include algebraically dependent components.",
        "Bootstrap standard deviations are estimated from the same draws used for max-Z calibration.",
        "This is an approximate survey-stratified PSU bootstrap, not official DHS replicate weights.",
        "The model specification was fixed; bootstrap includes model-coefficient refitting but not model-selection uncertainty.",
        "No causal effects or geographic distance claims can be inferred.",
    ],

    "validation_passed": bool(passed),

    "checks": {
        key: bool(value)
        for key, value in checks.items()
    },
}

OUTPUTS["audit"].write_text(
    json.dumps(
        audit,
        indent=2,
        ensure_ascii=False,
    ) + "\n",
    encoding="utf-8",
)


# Display complete results in Jupyter.
print(report)

print(
    "\nPRIMARY: THREE JOINT PHARMACY-FIRST GAPS "
    "- SIMULTANEOUS FAMILYWISE SENSITIVITY"
)

display(
    primary_table[
        [
            "contrast",
            "estimate_joint_gap_pp",
            "step11b_percentile_ci_low_pp",
            "step11b_percentile_ci_high_pp",
            "maxz_simultaneous95_ci_low_pp",
            "maxz_simultaneous95_ci_high_pp",
            "pointwise_centered_bootstrap_p",
            "holm_adjusted_p",
            "maxz_familywise_adjusted_p",
        ]
    ].round(5)
)

print(
    "\nSENSITIVITY: 9 CONDITIONAL CHOICE / "
    "ACCESS DIFFERENCES (15-CONTRAST FAMILY)"
)

display(
    secondary_table[
        [
            "contrast",
            "estimand",
            "estimate_pp",
            "maxz_simultaneous95_ci_low_pp",
            "maxz_simultaneous95_ci_high_pp",
            "maxz_familywise_adjusted_p",
        ]
    ].round(5)
)

print(
    "\nREPRODUCTION OF STEP 11.2 "
    "(ALL 15 ESTIMANDS)"
)

display(
    reconciliation.round(10)
)

print("\nVALIDATION CHECKS")
display(validation)

print("\nSAVED FILES")
for item in OUTPUTS.values():
    print(item)

if not passed:
    raise ValueError(
        "Step 11.3 simultaneous-inference sensitivity requires review."
    )

STEP 11.3 | MULTIPLICITY-AWARE TWO-STAGE PATHWAY INFERENCE
Data: BDHS 2022 KR; verified Stage 11.2 model specification
Reference: 2624 complete-case children; Stage 2 care seekers: 1969
Main family: 3 adjusted unconditional pharmacy-first contrasts
Conservative sensitivity family: all 15 related adjusted contrasts/components
Bootstrap: same 999 draws as Step 11.2; both logit models refitted in each draw
Methods: centered bootstrap max-|Z| simultaneous intervals/p values; Holm adjusted p
Multiplicity family was defined AFTER inspection of Step 11.2 results: exploratory only
Uncertainty is approximate stratified PSU bootstrap; not exact DHS design-based CI
Contrasts and decomposition are associational, not causal mediation
NO geographic model tuning or new outcome-derived predictors introduced
Step 11.2 reconstruction maximum discrepancy (pp): 3.55e-15
Critical max-Z for primary family: 2.3366
Critical max-Z for all 15 comparisons: 2.7226
Validation: PASS
Checks: 18/18

PRIMARY: THREE JO

,contrast,estimate_joint_gap_pp,step11b_percentile_ci_low_pp,step11b_percentile_ci_high_pp,maxz_simultaneous95_ci_low_pp,maxz_simultaneous95_ci_high_pp,pointwise_centered_bootstrap_p,holm_adjusted_p,maxz_familywise_adjusted_p
0,Poorest minus richest,5.57717,-2.64044,13.69247,-4.72505,15.87939,0.213,0.213,0.497
1,No education minus higher,11.47375,0.60967,22.21331,-1.65978,24.60728,0.040,0.093,0.102
2,Rural minus urban,5.94876,1.19006,11.14968,-0.35768,12.25521,0.031,0.093,0.075



SENSITIVITY: 9 CONDITIONAL CHOICE / ACCESS DIFFERENCES (15-CONTRAST FAMILY)


,contrast,estimand,estimate_pp,maxz_simultaneous95_ci_low_pp,maxz_simultaneous95_ci_high_pp,maxz_familywise_adjusted_p
1,Poorest minus richest,difference_pharmacy_given_care_pp,5.48480,-8.59421,19.56382,0.904
3,Poorest minus richest,access_component_pp,1.47116,-4.03609,6.97841,0.982
4,Poorest minus richest,conditional_source_component_pp,4.10601,-6.42830,14.64032,0.904
6,No education minus higher,difference_pharmacy_given_care_pp,13.65343,-4.29820,31.60506,0.230
8,No education minus higher,access_component_pp,1.61034,-6.06294,9.28362,0.992
9,No education minus higher,conditional_source_component_pp,9.86341,-3.30006,23.02688,0.244
11,Rural minus urban,difference_pharmacy_given_care_pp,7.70491,-1.09999,16.50982,0.121
13,Rural minus urban,access_component_pp,0.08162,-3.20756,3.37080,1.000
14,Rural minus urban,conditional_source_component_pp,5.86714,-0.83693,12.57122,0.121



REPRODUCTION OF STEP 11.2 (ALL 15 ESTIMANDS)


,contrast,metric,point_error_pp,percentile_lower_error_pp,percentile_upper_error_pp
0,Poorest minus richest,difference_care_sought_pp,0.0,0.0,0.0
1,Poorest minus richest,difference_pharmacy_given_care_pp,0.0,0.0,0.0
2,Poorest minus richest,difference_pharmacy_among_all_fever_pp,0.0,0.0,0.0
3,Poorest minus richest,access_component_pp,0.0,0.0,0.0
4,Poorest minus richest,conditional_source_component_pp,0.0,0.0,0.0
5,No education minus higher,difference_care_sought_pp,0.0,0.0,0.0
6,No education minus higher,difference_pharmacy_given_care_pp,0.0,0.0,0.0
7,No education minus higher,difference_pharmacy_among_all_fever_pp,0.0,0.0,0.0
8,No education minus higher,access_component_pp,0.0,0.0,0.0
9,No education minus higher,conditional_source_component_pp,0.0,0.0,0.0



VALIDATION CHECKS


,check,status
0,Original KR source and Step 11.2 audit verified,PASS
1,Original full-frame 674 PSU and 16 strata pres...,PASS
2,Both complete-case cohorts matched frozen elig...,PASS
3,Stage 2 pharmacy and Other source codes retained,PASS
4,Both saved weighted model scores reproduced,PASS
5,All bootstrap replicates jointly refit both mo...,PASS
6,PSU bootstrap counts preserved within strata,PASS
7,All Step 11.2 contrasts and intervals reproduced,PASS
8,Point and bootstrap pathway identities satisfied,PASS
9,Three primary joint pharmacy-gap contrasts eva...,PASS



SAVED FILES
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\11c_three_joint_gaps_simultaneous_inference.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\11c_fifteen_estimands_familywise_sensitivity.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\11c_source_choice_component_sensitivity.csv
D:\BDHS2022_FeverCare_SpatialEquity\data\processed\11c_joint_refit_bootstrap_contrast_replicates.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\11c_step11b_reproduction_checks.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\11c_joint_refit_fit_diagnostics.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\11c_simultaneous_inference_validation_checks.csv
D:\BDHS2022_FeverCare_SpatialEquity\logs\11c_simultaneous_inference_audit.json
D:\BDHS2022_FeverCare_SpatialEquity\logs\11c_simultaneous_inference_report.txt


In [33]:

# STEP 12.1 | Nested geographic validation
# Non-spatial versus geographic ML families
# One self-contained Jupyter cell; previous outputs remain unchanged.

from pathlib import Path
from importlib.util import find_spec
import hashlib
import json
import os
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.neighbors import BallTree
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from threadpoolctl import threadpool_limits

if find_spec("catboost") is None:
    raise ImportError(
        "CatBoost is missing. Run %pip install catboost "
        "in this Jupyter notebook, then rerun this cell."
    )

from catboost import CatBoostClassifier


# -------------------------------------------------------
# 1. Project paths and verified prerequisites
# -------------------------------------------------------

ROOT = Path(
    os.environ.get(
        "BDHS_PROJECT_ROOT",
        r"D:\BDHS2022_FeverCare_SpatialEquity"
    )
)

RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
TABLES = ROOT / "results" / "tables"
LOGS = ROOT / "logs"

inputs = {
    "kr": RAW / "BDKR81FL.DTA",
    "cohort": PROC / "06_fever_care_analysis_ready.csv",
    "gps": PROC / "09a_psu_spatial_descriptive_base.csv",
    "blocks": TABLES / "10b_spatial_block_inventory.csv",
    "source_audit": LOGS / "01_input_audit.json",
    "step10b_audit": LOGS / "10b_spatial_basis_ml_audit.json",
    "step11c_audit": LOGS / "11c_simultaneous_inference_audit.json",
}

for name, path in inputs.items():
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing prerequisite ({name}): {path}"
        )

with inputs["source_audit"].open(encoding="utf-8") as f:
    source_audit = json.load(f)

with inputs["step10b_audit"].open(encoding="utf-8") as f:
    step10b = json.load(f)

with inputs["step11c_audit"].open(encoding="utf-8") as f:
    step11c = json.load(f)

if not (
    step10b.get("validation_passed")
    and step11c.get("validation_passed")
):
    raise ValueError(
        "Step 10.2 and Step 11.3 must both be validated."
    )

with inputs["kr"].open("rb") as f:
    kr_sha256 = hashlib.file_digest(
        f, "sha256"
    ).hexdigest()

if kr_sha256 != source_audit["source_files"]["kr_sha256"]:
    raise ValueError(
        "Original KR dataset SHA256 mismatch."
    )


# -------------------------------------------------------
# 2. Fixed analysis specifications
# -------------------------------------------------------

SEED = 20261117

GRID_KM = 75.0
BUFFER_KM = 20.0
EARTH_KM = 6371.0088

N_INNER = 3
N_ANCHORS = 16

BANDWIDTHS_KM = (50, 75, 100, 150, 200)
BOOTSTRAPS = 599

NUMERIC = [
    "child_age_years",
    "maternal_age_5years",
    "birth_order",
]

CATEGORICAL = [
    "child_sex",
    "cough",
    "rapid_breathing",
    "maternal_education",
    "wealth_quintile",
    "residence",
    "division",
]

FEATURE_VARIANTS = (
    ["Non-spatial", "Raw GPS"]
    + [
        f"RBF {bandwidth}km"
        for bandwidth in BANDWIDTHS_KM
    ]
)

ALGORITHMS = [
    ("L2 logistic", 0.3),
    ("L2 logistic", 1.0),
    ("Random forest", None),
    ("CatBoost", None),
]

# Logistic: baseline, raw GPS and five RBF bandwidths.
# RF/CatBoost: baseline and raw GPS only.
# This avoids redundant tree fits over all RBF bands.

CANDIDATES = [
    {
        "algorithm": algorithm,
        "C": c,
        "feature_variant": feature_variant,
        "family": (
            "Non-spatial"
            if feature_variant == "Non-spatial"
            else "Geographic"
        ),
    }
    for algorithm, c in ALGORITHMS
    for feature_variant in (
        FEATURE_VARIANTS
        if algorithm == "L2 logistic"
        else ["Non-spatial", "Raw GPS"]
    )
]

assert len(CANDIDATES) == 18

PROHIBITED = {
    "h32y", "h46a", "h46b",
    "v001", "v021", "v023",
    "care_sought", "pharmacy_first",
    "psu_effect", "fitted_residual",
    "care_rate_weighted",
}

if not set(
    NUMERIC
    + CATEGORICAL
    + ["gps_latitude", "gps_longitude"]
).isdisjoint(PROHIBITED):
    raise ValueError("Outcome-derived predictor leakage.")


# -------------------------------------------------------
# 3. Read and verify study populations
# -------------------------------------------------------

keys = ["v001", "v002", "v003", "bidx"]

raw = pd.read_stata(
    inputs["kr"],
    columns=keys + [
        "v005", "v021", "v023", "v024", "v025",
        "h22", "h32y", "h46a", "h31", "h31b",
        "b19", "b4", "v012", "v106", "v190", "bord",
    ],
    convert_categoricals=False,
)

frozen = pd.read_csv(
    inputs["cohort"],
    usecols=keys + [
        "stage1_core_complete_case",
        "stage2_core_complete_case",
    ],
)

gps = pd.read_csv(
    inputs["gps"],
    usecols=[
        "v001", "v023",
        "gps_latitude", "gps_longitude",
    ],
)

block_inventory = pd.read_csv(
    inputs["blocks"]
)

if not (
    len(raw) == 8784
    and raw["v001"].nunique() == 674
    and raw["v001"].eq(raw["v021"]).all()
    and not raw.duplicated(keys).any()
    and len(gps) == 674
    and gps["v001"].is_unique
    and gps[[
        "gps_latitude", "gps_longitude"
    ]].notna().all().all()
    and len(block_inventory) == 37
    and block_inventory["grid_block"].is_unique
    and block_inventory["geographic_region"].nunique() == 5
):
    raise ValueError(
        "KR, GPS or geographic block inventory differs."
    )


def project_km(latitude, longitude):

    return np.column_stack([
        (
            np.asarray(latitude, dtype=float) - 20.0
        ) * 111.2,

        (
            np.asarray(longitude, dtype=float) - 88.0
        ) * 111.2 * np.cos(np.deg2rad(23.5)),
    ])


gps_xy = project_km(
    gps["gps_latitude"],
    gps["gps_longitude"],
)

gps["grid_block"] = (
    np.floor(
        gps_xy[:, 0] / GRID_KM
    ).astype(int).astype(str)
    + "_"
    + np.floor(
        gps_xy[:, 1] / GRID_KM
    ).astype(int).astype(str)
)

gps = gps.merge(
    block_inventory[
        ["grid_block", "geographic_region"]
    ],
    on="grid_block",
    how="left",
    validate="many_to_one",
)

observed_blocks = (
    gps.groupby("grid_block")["v001"]
    .size()
    .sort_index()
)

expected_blocks = (
    block_inventory
    .set_index("grid_block")["n_psus"]
    .sort_index()
    .rename("v001")
)

if (
    gps["geographic_region"].isna().any()
    or not observed_blocks.equals(expected_blocks)
):
    raise ValueError(
        "Step 10.2 geographic blocks did not reproduce."
    )


fever = raw["h22"].eq(1)

care = (
    fever
    & raw["h32y"].eq(0)
)

complete = (
    raw[
        [
            "b19", "b4", "h31", "v012",
            "v106", "v190", "bord",
            "v025", "v024",
        ]
    ].notna().all(axis=1)
    & raw["h31b"].isin([0, 1])
)

STAGES = {
    "Stage 1 care sought": (
        fever & complete,
        raw["h32y"].eq(0),
        "stage1_core_complete_case",
        2624, 642, 1969,
    ),

    "Stage 2 pharmacy first": (
        care & complete,
        raw["h46a"].eq(36),
        "stage2_core_complete_case",
        1969, 611, 1117,
    ),
}

for stage, (
    domain, outcome, flag,
    expected_n, expected_psus, expected_events,
) in STAGES.items():

    observed = (
        raw.loc[domain, keys]
        .astype("int64")
        .sort_values(keys)
        .reset_index(drop=True)
    )

    expected = (
        frozen.loc[frozen[flag].eq(1), keys]
        .astype("int64")
        .sort_values(keys)
        .reset_index(drop=True)
    )

    if not (
        observed.equals(expected)
        and len(observed) == expected_n
        and observed["v001"].nunique() == expected_psus
        and int(outcome[domain].sum()) == expected_events
    ):
        raise ValueError(
            f"Frozen population mismatch: {stage}"
        )

if not (
    int(care.sum()) == 1974
    and int(
        (care & raw["h46a"].eq(96)).sum()
    ) == 33
):
    raise ValueError(
        "Stage 2 Other-source classification changed."
    )


# -------------------------------------------------------
# 4. Weighted evaluation and candidate models
# -------------------------------------------------------

def evaluation(y, probability, weight):

    y = np.asarray(y, dtype=int)

    p = np.clip(
        np.asarray(probability, dtype=float),
        1e-8, 1 - 1e-8,
    )

    w = np.asarray(weight, dtype=float)

    if not (
        len(np.unique(y)) == 2
        and np.isfinite(p).all()
        and np.isfinite(w).all()
        and (w > 0).all()
    ):
        raise ValueError(
            "Invalid held-out evaluation data."
        )

    cross_entropy = -(
        y * np.log(p)
        + (1 - y) * np.log1p(-p)
    )

    return {
        "auc": float(
            roc_auc_score(
                y, p, sample_weight=w
            )
        ),

        "average_precision": float(
            average_precision_score(
                y, p, sample_weight=w
            )
        ),

        "brier": float(
            np.average(
                (p - y) ** 2,
                weights=w,
            )
        ),

        "log_loss": float(
            np.average(
                cross_entropy,
                weights=w,
            )
        ),

        "observed_pct": float(
            100 * np.average(y, weights=w)
        ),

        "predicted_pct": float(
            100 * np.average(p, weights=w)
        ),

        "calibration_gap_pp": float(
            100 * np.average(
                p - y,
                weights=w,
            )
        ),
    }


def estimator(candidate, seed):

    name = candidate["algorithm"]

    if name == "L2 logistic":

        return LogisticRegression(
            C=float(candidate["C"]),
            solver="liblinear",
            max_iter=450,
            tol=1e-4,
            random_state=seed,
        )

    if name == "Random forest":

        return RandomForestClassifier(
            n_estimators=90,
            max_depth=9,
            min_samples_leaf=8,
            n_jobs=2,
            random_state=seed,
        )

    if name == "CatBoost":

        return CatBoostClassifier(
            iterations=100,
            depth=4,
            learning_rate=0.05,
            l2_leaf_reg=5.0,
            loss_function="Logloss",
            verbose=False,
            random_seed=seed,
            thread_count=2,
            allow_writing_files=False,
        )

    raise ValueError("Unknown ML algorithm.")


def geographic_buffer(
    data, train_positions, test_positions
):

    tr = np.asarray(
        train_positions,
        dtype=int,
    )

    te = np.asarray(
        test_positions,
        dtype=int,
    )

    train_psus = (
        data.iloc[tr]
        .drop_duplicates("v001")
    )

    test_psus = (
        data.iloc[te]
        .drop_duplicates("v001")
    )

    if train_psus.empty or test_psus.empty:
        raise ValueError(
            "Missing PSUs for geographic buffer."
        )

    tree = BallTree(
        np.deg2rad(
            test_psus[
                ["gps_latitude", "gps_longitude"]
            ].to_numpy(dtype=float)
        ),
        metric="haversine",
    )

    distances = tree.query(
        np.deg2rad(
            train_psus[
                ["gps_latitude", "gps_longitude"]
            ].to_numpy(dtype=float)
        ),
        k=1,
    )[0][:, 0] * EARTH_KM

    kept_psus = train_psus.loc[
        distances > BUFFER_KM,
        "v001",
    ].to_numpy()

    retained = tr[
        np.isin(
            data.iloc[tr]["v001"].to_numpy(),
            kept_psus,
        )
    ]

    minimum_km = (
        float(
            distances[
                distances > BUFFER_KM
            ].min()
        )
        if (distances > BUFFER_KM).any()
        else float("nan")
    )

    return (
        retained,
        len(tr) - len(retained),
        minimum_km,
    )


# -------------------------------------------------------
# 5. Training-fold-only feature engineering
# -------------------------------------------------------

def fold_matrices(
    data, x, train, test, seed
):

    try:
        encoder = OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )
    except TypeError:
        encoder = OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
        )

    preprocessing = ColumnTransformer(
        [
            (
                "numeric",
                StandardScaler(),
                NUMERIC,
            ),
            (
                "categorical",
                encoder,
                CATEGORICAL,
            ),
        ],
        remainder="drop",
    )

    base_train = np.asarray(
        preprocessing.fit_transform(
            x.iloc[train]
        ),
        dtype=float,
    )

    base_test = np.asarray(
        preprocessing.transform(
            x.iloc[test]
        ),
        dtype=float,
    )

    gps_scaler = StandardScaler().fit(
        x.iloc[train][
            ["gps_latitude", "gps_longitude"]
        ]
    )

    train_gps = gps_scaler.transform(
        x.iloc[train][
            ["gps_latitude", "gps_longitude"]
        ]
    )

    test_gps = gps_scaler.transform(
        x.iloc[test][
            ["gps_latitude", "gps_longitude"]
        ]
    )

    matrices = {
        "Non-spatial": (
            base_train,
            base_test,
        ),

        "Raw GPS": (
            np.column_stack([
                base_train, train_gps
            ]),
            np.column_stack([
                base_test, test_gps
            ]),
        ),
    }

    # RBF centers are created from unique training PSUs.
    training_psus = (
        data.iloc[train]
        .drop_duplicates("v001")
    )

    if len(training_psus) < N_ANCHORS:
        raise ValueError(
            "Insufficient PSUs for training-only RBF centers."
        )

    anchor_xy = project_km(
        training_psus["gps_latitude"],
        training_psus["gps_longitude"],
    )

    with threadpool_limits(limits=1):

        kmeans = KMeans(
            n_clusters=N_ANCHORS,
            n_init=10,
            random_state=seed,
        ).fit(anchor_xy)

    anchors = kmeans.cluster_centers_

    xy_train = project_km(
        data.iloc[train]["gps_latitude"],
        data.iloc[train]["gps_longitude"],
    )

    xy_test = project_km(
        data.iloc[test]["gps_latitude"],
        data.iloc[test]["gps_longitude"],
    )

    squared_train = np.sum(
        (
            xy_train[:, None, :]
            - anchors[None, :, :]
        ) ** 2,
        axis=2,
    )

    squared_test = np.sum(
        (
            xy_test[:, None, :]
            - anchors[None, :, :]
        ) ** 2,
        axis=2,
    )

    for bandwidth in BANDWIDTHS_KM:

        rbf_train = np.exp(
            -0.5
            * squared_train
            / (bandwidth ** 2)
        )

        rbf_test = np.exp(
            -0.5
            * squared_test
            / (bandwidth ** 2)
        )

        basis_scaler = StandardScaler().fit(
            rbf_train
        )

        matrices[
            f"RBF {bandwidth}km"
        ] = (
            np.column_stack([
                base_train,
                basis_scaler.transform(
                    rbf_train
                ),
            ]),
            np.column_stack([
                base_test,
                basis_scaler.transform(
                    rbf_test
                ),
            ]),
        )

    if any(
        not (
            np.isfinite(value[0]).all()
            and np.isfinite(value[1]).all()
        )
        for value in matrices.values()
    ):
        raise ValueError(
            "Non-finite transformed predictors."
        )

    return matrices


# -------------------------------------------------------
# 6. Nested geographic model selection
# -------------------------------------------------------

outer_rows = []
inner_rows = []
candidate_rows = []
chosen_rows = []
oof_parts = []

inner_fit_count = 0
outer_fit_count = 0

for stage_index, (
    stage,
    (
        domain,
        outcome,
        flag,
        expected_n,
        expected_psus,
        expected_events,
    ),
) in enumerate(STAGES.items()):

    data = (
        raw.loc[domain]
        .reset_index(drop=True)
        .merge(
            gps,
            on=["v001", "v023"],
            how="left",
            validate="many_to_one",
        )
    )

    if data[
        [
            "gps_latitude",
            "gps_longitude",
            "geographic_region",
            "grid_block",
        ]
    ].isna().any().any():
        raise ValueError(
            f"GPS linkage incomplete: {stage}"
        )

    psu = data["v001"].to_numpy(dtype=int)

    y = outcome[domain].to_numpy(dtype=int)

    weights = (
        data["v005"].to_numpy(dtype=float)
        / 1e6
    )

    if not (
        len(data) == expected_n
        and np.isfinite(weights).all()
        and (weights > 0).all()
    ):
        raise ValueError(
            f"Invalid sample or weights: {stage}"
        )

    x = pd.DataFrame({
        "child_age_years":
            data["b19"].to_numpy(dtype=float) / 12,

        "maternal_age_5years":
            (
                data["v012"].to_numpy(dtype=float)
                - 25
            ) / 5,

        "birth_order":
            data["bord"].to_numpy(dtype=float),

        "child_sex":
            data["b4"].to_numpy(dtype=int),

        "cough":
            data["h31"].map({
                0: 0,
                2: 1,
            }).to_numpy(),

        "rapid_breathing":
            data["h31b"].to_numpy(dtype=int),

        "maternal_education":
            data["v106"].to_numpy(dtype=int),

        "wealth_quintile":
            data["v190"].to_numpy(dtype=int),

        "residence":
            data["v025"].to_numpy(dtype=int),

        "division":
            data["v024"].to_numpy(dtype=int),

        "gps_latitude":
            data["gps_latitude"].to_numpy(dtype=float),

        "gps_longitude":
            data["gps_longitude"].to_numpy(dtype=float),
    })

    if x.isna().any().any():
        raise ValueError(
            f"Predictor missingness: {stage}"
        )

    coverage = np.zeros(
        len(data),
        dtype=int,
    )

    labels = data[
        "geographic_region"
    ].to_numpy(dtype=int)

    for outer_fold, region in enumerate(
        sorted(np.unique(labels)),
        start=1,
    ):

        test = np.flatnonzero(
            labels == region
        )

        original_train = np.flatnonzero(
            labels != region
        )

        train, excluded, min_km = (
            geographic_buffer(
                data,
                original_train,
                test,
            )
        )

        psu_overlap = len(
            np.intersect1d(
                psu[train],
                psu[test],
            )
        )

        block_overlap = len(
            set(data.iloc[train]["grid_block"])
            & set(data.iloc[test]["grid_block"])
        )

        if not (
            len(train) > 600
            and len(test) > 150
            and psu_overlap == 0
            and block_overlap == 0
            and set(y[train]) == {0, 1}
            and set(y[test]) == {0, 1}
            and min_km > BUFFER_KM
        ):
            raise ValueError(
                f"Invalid outer fold: {stage}, {region}"
            )

        coverage[test] += 1

        outer_rows.append({
            "stage": stage,
            "outer_fold": outer_fold,
            "heldout_region": int(region),
            "train_n": len(train),
            "test_n": len(test),
            "train_psus": len(
                np.unique(psu[train])
            ),
            "test_psus": len(
                np.unique(psu[test])
            ),
            "excluded_by_20km_buffer": excluded,
            "minimum_displaced_coordinate_gap_km": min_km,
            "psu_overlap": psu_overlap,
            "grid_block_overlap": block_overlap,
        })

        # Inner regions: learned from OUTER TRAINING BLOCKS.
        outer_train = data.iloc[train]

        block_psus = (
            outer_train[
                ["grid_block", "v001"]
            ]
            .drop_duplicates()
            .groupby("grid_block")["v001"]
            .nunique()
            .reset_index(name="npsus")
        )

        centers = np.array([
            [
                int(value.split("_")[0]) + 0.5,
                int(value.split("_")[1]) + 0.5,
            ]
            for value in block_psus["grid_block"]
        ]) * GRID_KM

        with threadpool_limits(limits=1):

            region_model = KMeans(
                n_clusters=N_INNER,
                n_init=15,
                random_state=(
                    SEED
                    + stage_index * 100
                    + outer_fold
                ),
            )

            region_model.fit(
                centers,
                sample_weight=block_psus[
                    "npsus"
                ].to_numpy(),
            )

        block_map = dict(
            zip(
                block_psus["grid_block"],
                region_model.labels_,
            )
        )

        inner_regions = (
            outer_train["grid_block"]
            .map(block_map)
            .to_numpy(dtype=int)
        )

        inner_sum = {
            i: {
                "loss_sum": 0.0,
                "brier_sum": 0.0,
                "weight_sum": 0.0,
            }
            for i in range(
                len(CANDIDATES)
            )
        }

        inner_coverage = np.zeros(
            len(train),
            dtype=int,
        )

        for inner_fold, inner_region in enumerate(
            sorted(np.unique(inner_regions)),
            start=1,
        ):

            inner_test = train[
                np.flatnonzero(
                    inner_regions == inner_region
                )
            ]

            inner_train_initial = train[
                np.flatnonzero(
                    inner_regions != inner_region
                )
            ]

            inner_train, removed, inner_min_km = (
                geographic_buffer(
                    data,
                    inner_train_initial,
                    inner_test,
                )
            )

            inner_psu_overlap = len(
                np.intersect1d(
                    psu[inner_train],
                    psu[inner_test],
                )
            )

            inner_block_overlap = len(
                set(data.iloc[inner_train]["grid_block"])
                & set(data.iloc[inner_test]["grid_block"])
            )

            if not (
                len(inner_train) > 500
                and len(inner_test) > 100
                and inner_psu_overlap == 0
                and inner_block_overlap == 0
                and inner_min_km > BUFFER_KM
                and set(y[inner_train]) == {0, 1}
                and set(y[inner_test]) == {0, 1}
            ):
                raise ValueError(
                    f"Invalid inner fold: "
                    f"{stage}, outer {outer_fold}, "
                    f"inner {inner_fold}"
                )

            inner_coverage[
                np.flatnonzero(
                    inner_regions == inner_region
                )
            ] += 1

            inner_rows.append({
                "stage": stage,
                "outer_fold": outer_fold,
                "inner_fold": inner_fold,
                "train_n": len(inner_train),
                "validation_n": len(inner_test),
                "train_psus": len(
                    np.unique(psu[inner_train])
                ),
                "validation_psus": len(
                    np.unique(psu[inner_test])
                ),
                "psu_overlap": inner_psu_overlap,
                "grid_block_overlap": inner_block_overlap,
                "removed_by_20km_buffer": removed,
                "minimum_displaced_coordinate_gap_km":
                    inner_min_km,
            })

            matrices = fold_matrices(
                data, x,
                inner_train,
                inner_test,
                (
                    SEED
                    + 1000 * stage_index
                    + 100 * outer_fold
                    + inner_fold
                ),
            )

            normalized_weights = (
                weights[inner_train]
                / weights[inner_train].mean()
            )

            y_valid = y[inner_test]
            w_valid = weights[inner_test]

            for candidate_index, candidate in enumerate(
                CANDIDATES
            ):

                Xtrain, Xvalid = matrices[
                    candidate["feature_variant"]
                ]

                algorithm_offset = (
                    0
                    if candidate["algorithm"] == "L2 logistic"
                    else (
                        1
                        if candidate["algorithm"] == "Random forest"
                        else 2
                    )
                )

                model_seed = (
                    SEED
                    + 1000 * stage_index
                    + 100 * outer_fold
                    + inner_fold
                    + algorithm_offset
                )

                model = estimator(
                    candidate,
                    model_seed,
                )

                with warnings.catch_warnings(
                    record=True
                ) as captured:

                    warnings.simplefilter("always")

                    model.fit(
                        Xtrain,
                        y[inner_train],
                        sample_weight=normalized_weights,
                    )

                    probability = np.clip(
                        model.predict_proba(
                            Xvalid
                        )[:, 1],
                        1e-8,
                        1 - 1e-8,
                    )

                if any(
                    any(
                        word in str(
                            warning.message
                        ).lower()
                        for word in (
                            "converg",
                            "overflow",
                            "failed",
                        )
                    )
                    for warning in captured
                ):
                    raise ValueError(
                        f"Critical model warning: {stage}"
                    )

                if not np.isfinite(
                    probability
                ).all():
                    raise ValueError(
                        "Invalid validation probabilities."
                    )

                losses = -(
                    y_valid * np.log(probability)
                    + (1 - y_valid)
                    * np.log1p(-probability)
                )

                accumulator = inner_sum[
                    candidate_index
                ]

                accumulator["loss_sum"] += float(
                    np.dot(
                        w_valid,
                        losses,
                    )
                )

                accumulator["brier_sum"] += float(
                    np.dot(
                        w_valid,
                        (probability - y_valid) ** 2,
                    )
                )

                accumulator["weight_sum"] += float(
                    np.sum(w_valid)
                )

                inner_fit_count += 1

        if not (
            inner_coverage == 1
        ).all():
            raise ValueError(
                "Inner folds do not cover every outer-training child."
            )

        # Record all candidate scores.
        for index, candidate in enumerate(CANDIDATES):

            result = inner_sum[index]

            candidate_rows.append({
                "stage": stage,
                "outer_fold": outer_fold,
                "family": candidate["family"],
                "algorithm": candidate["algorithm"],
                "C": candidate["C"],
                "feature_variant": candidate["feature_variant"],
                "inner_oof_log_loss": (
                    result["loss_sum"]
                    / result["weight_sum"]
                ),
                "inner_oof_brier": (
                    result["brier_sum"]
                    / result["weight_sum"]
                ),
                "inner_validation_children": len(train),
            })

        # Selection is based ONLY on the inner validation scores.
        selected = {}

        for family in (
            "Non-spatial",
            "Geographic",
        ):

            eligible = [
                (
                    index,
                    inner_sum[index]["loss_sum"]
                    / inner_sum[index]["weight_sum"],
                    inner_sum[index]["brier_sum"]
                    / inner_sum[index]["weight_sum"],
                )
                for index, candidate in enumerate(
                    CANDIDATES
                )
                if candidate["family"] == family
            ]

            best_index = min(
                eligible,
                key=lambda values: (
                    values[1],
                    values[2],
                    values[0],
                ),
            )[0]

            selected[family] = best_index

            chosen = CANDIDATES[
                best_index
            ]

            chosen_rows.append({
                "stage": stage,
                "outer_fold": outer_fold,
                "family": family,
                "selected_algorithm": chosen["algorithm"],
                "selected_C": chosen["C"],
                "selected_feature_variant":
                    chosen["feature_variant"],
                "inner_oof_log_loss": (
                    inner_sum[best_index]["loss_sum"]
                    / inner_sum[best_index]["weight_sum"]
                ),
                "inner_oof_brier": (
                    inner_sum[best_index]["brier_sum"]
                    / inner_sum[best_index]["weight_sum"]
                ),
            })

        # Refit the selected models on the full outer-training fold.
        final_matrices = fold_matrices(
            data, x, train, test,
            (
                SEED
                + 10000
                + stage_index * 100
                + outer_fold
            ),
        )

        for family, best_index in selected.items():

            chosen = CANDIDATES[
                best_index
            ]

            Xtrain, Xtest = final_matrices[
                chosen["feature_variant"]
            ]

            algorithm_offset = (
                0
                if chosen["algorithm"] == "L2 logistic"
                else (
                    1
                    if chosen["algorithm"] == "Random forest"
                    else 2
                )
            )

            model = estimator(
                chosen,
                (
                    SEED
                    + 20000
                    + stage_index * 100
                    + outer_fold
                    + algorithm_offset
                ),
            )

            with warnings.catch_warnings(
                record=True
            ) as captured:

                warnings.simplefilter("always")

                model.fit(
                    Xtrain,
                    y[train],
                    sample_weight=(
                        weights[train]
                        / weights[train].mean()
                    ),
                )

                predictions = np.asarray(
                    model.predict_proba(
                        Xtest
                    )[:, 1],
                    dtype=float,
                )

            if any(
                any(
                    word in str(
                        warning.message
                    ).lower()
                    for word in (
                        "converg",
                        "overflow",
                        "failed",
                    )
                )
                for warning in captured
            ):
                raise ValueError(
                    f"Outer model warning: "
                    f"{stage}, {outer_fold}"
                )

            performance = evaluation(
                y[test],
                predictions,
                weights[test],
            )

            outer_fit_count += 1

            outer_rows[-1].update({
                f"{family}_{name}": value
                for name, value in performance.items()
            })

            oof_parts.append(
                pd.DataFrame({
                    "stage": stage,
                    "outer_fold": outer_fold,
                    "family": family,
                    "child_row_within_stage": test,
                    "psu_id": psu[test],
                    "observed": y[test],
                    "predicted_probability": predictions,
                    "survey_weight": weights[test],
                })
            )

        print(
            f"[Nested ML] {stage} | "
            f"outer fold {outer_fold}/5 complete | "
            f"inner fits so far: {inner_fit_count}",
            flush=True,
        )

    if not (
        coverage == 1
    ).all():
        raise ValueError(
            f"Outer test-fold coverage invalid: {stage}"
        )


# -------------------------------------------------------
# 7. Pool outer out-of-fold predictions
# -------------------------------------------------------

outer_table = pd.DataFrame(
    outer_rows
)

inner_table = pd.DataFrame(
    inner_rows
)

candidate_table = pd.DataFrame(
    candidate_rows
)

choices = pd.DataFrame(
    chosen_rows
)

oof = pd.concat(
    oof_parts,
    ignore_index=True,
)

summary_rows = []
paired_rows = []

for stage, (
    _, _, _, n, _, _
) in STAGES.items():

    panel = oof.loc[
        oof["stage"].eq(stage)
    ]

    family_predictions = {}

    for family in (
        "Non-spatial",
        "Geographic",
    ):

        subset = (
            panel.loc[
                panel["family"].eq(family)
            ]
            .sort_values(
                "child_row_within_stage"
            )
            .reset_index(drop=True)
        )

        if not (
            len(subset) == n
            and np.array_equal(
                subset["child_row_within_stage"].to_numpy(),
                np.arange(n),
            )
        ):
            raise ValueError(
                f"OOF coverage incomplete: {stage}, {family}"
            )

        family_predictions[family] = subset

        summary_rows.append({
            "stage": stage,
            "family": family,
            "children": n,
            **evaluation(
                subset["observed"],
                subset["predicted_probability"],
                subset["survey_weight"],
            ),
        })

    base = family_predictions[
        "Non-spatial"
    ]

    spatial = family_predictions[
        "Geographic"
    ]

    if not (
        np.array_equal(
            base["observed"].to_numpy(),
            spatial["observed"].to_numpy(),
        )
        and np.array_equal(
            base["psu_id"].to_numpy(),
            spatial["psu_id"].to_numpy(),
        )
        and np.array_equal(
            base["outer_fold"].to_numpy(),
            spatial["outer_fold"].to_numpy(),
        )
        and np.allclose(
            base["survey_weight"].to_numpy(),
            spatial["survey_weight"].to_numpy(),
            atol=1e-12,
            rtol=0,
        )
    ):
        raise ValueError(
            "Nested families do not share identical held-out children."
        )

    baseline_metrics = evaluation(
        base["observed"],
        base["predicted_probability"],
        base["survey_weight"],
    )

    spatial_metrics = evaluation(
        spatial["observed"],
        spatial["predicted_probability"],
        spatial["survey_weight"],
    )

    contrast = {
        "stage": stage,
        "children": n,

        "delta_auc_spatial_minus_nonspatial": (
            spatial_metrics["auc"]
            - baseline_metrics["auc"]
        ),

        "delta_brier_spatial_minus_nonspatial": (
            spatial_metrics["brier"]
            - baseline_metrics["brier"]
        ),

        "delta_log_loss_spatial_minus_nonspatial": (
            spatial_metrics["log_loss"]
            - baseline_metrics["log_loss"]
        ),
    }


    # ---------------------------------------------------
    # Conditional paired PSU bootstrap
    # ---------------------------------------------------

    psu_frame = (
        raw[["v001", "v023"]]
        .drop_duplicates()
        .sort_values([
            "v023", "v001"
        ])
        .reset_index(drop=True)
    )

    if not (
        len(psu_frame) == 674
        and psu_frame["v023"].nunique() == 16
    ):
        raise ValueError(
            "Bootstrap sampling frame changed."
        )

    locator = pd.Index(
        psu_frame["v001"]
    ).get_indexer(
        base["psu_id"]
    )

    if (locator < 0).any():
        raise ValueError(
            "Unmatched PSU in held-out bootstrap."
        )

    bootstrap_rng = np.random.default_rng(
        SEED + 120 + len(stage)
    )

    multiplicity = np.zeros(
        (674, BOOTSTRAPS),
        dtype=np.int16,
    )

    for _, subgroup in psu_frame.groupby("v023"):

        indices = subgroup.index.to_numpy()

        multiplicity[
            indices, :
        ] = bootstrap_rng.multinomial(
            n=len(indices),
            pvals=np.full(
                len(indices),
                1 / len(indices),
            ),
            size=BOOTSTRAPS,
        ).T

    y_true = base[
        "observed"
    ].to_numpy(dtype=int)

    w_base = base[
        "survey_weight"
    ].to_numpy(dtype=float)

    p_base = np.clip(
        base["predicted_probability"].to_numpy(dtype=float),
        1e-8, 1 - 1e-8,
    )

    p_spatial = np.clip(
        spatial["predicted_probability"].to_numpy(dtype=float),
        1e-8, 1 - 1e-8,
    )

    cross_entropy_difference = (
        -(
            y_true * np.log(p_spatial)
            + (1 - y_true)
            * np.log1p(-p_spatial)
        )
        + (
            y_true * np.log(p_base)
            + (1 - y_true)
            * np.log1p(-p_base)
        )
    )

    brier_difference = (
        (p_spatial - y_true) ** 2
        - (p_base - y_true) ** 2
    )

    replicate_weights = (
        w_base[:, None]
        * multiplicity[
            locator, :
        ]
    )

    totals = replicate_weights.sum(
        axis=0
    )

    if not (totals > 0).all():
        raise ValueError(
            "Zero-weight PSU bootstrap replicate."
        )

    bootstrap_brier = (
        brier_difference @ replicate_weights
    ) / totals

    bootstrap_logloss = (
        cross_entropy_difference @ replicate_weights
    ) / totals

    bootstrap_auc = []

    for replicate in range(BOOTSTRAPS):

        wr = replicate_weights[
            :, replicate
        ]

        if (
            wr[y_true == 1].sum() <= 0
            or wr[y_true == 0].sum() <= 0
        ):
            raise ValueError(
                "Bootstrap replicate has only one outcome class."
            )

        delta_auc = (
            roc_auc_score(
                y_true,
                p_spatial,
                sample_weight=wr,
            )
            - roc_auc_score(
                y_true,
                p_base,
                sample_weight=wr,
            )
        )

        bootstrap_auc.append(
            delta_auc
        )

    for name, values in (
        ("auc", np.asarray(bootstrap_auc)),
        ("brier", bootstrap_brier),
        ("log_loss", bootstrap_logloss),
    ):

        lower, upper = np.quantile(
            values,
            [0.025, 0.975],
        )

        contrast[
            f"{name}_conditional_psu95_low"
        ] = float(lower)

        contrast[
            f"{name}_conditional_psu95_high"
        ] = float(upper)

    paired_rows.append(
        contrast
    )

summary = pd.DataFrame(
    summary_rows
)

paired = pd.DataFrame(
    paired_rows
)


# -------------------------------------------------------
# 8. Validation checks
# -------------------------------------------------------

checks = {
    "Verified prerequisites and original KR checksum":
        bool(
            step10b["validation_passed"]
            and step11c["validation_passed"]
        ),

    "Original 8784 records and 674 GPS PSUs retained":
        len(raw) == 8784
        and len(gps) == 674,

    "Both frozen complete-case populations reproduced":
        (
            len(
                oof.loc[
                    oof["stage"].eq("Stage 1 care sought")
                ]
            ) == 2624 * 2
            and len(
                oof.loc[
                    oof["stage"].eq("Stage 2 pharmacy first")
                ]
            ) == 1969 * 2
        ),

    "Original 37 geographic blocks reproduced":
        (
            len(block_inventory) == 37
            and gps["grid_block"].nunique() == 37
        ),

    "Five outer regions per outcome":
        (
            len(outer_table) == 10
            and outer_table.groupby(
                "stage"
            )["outer_fold"].nunique().eq(5).all()
        ),

    "No outer PSU or block overlap":
        (
            outer_table["psu_overlap"].eq(0).all()
            and outer_table["grid_block_overlap"].eq(0).all()
        ),

    "Outer coordinate buffers validated":
        outer_table[
            "minimum_displaced_coordinate_gap_km"
        ].gt(BUFFER_KM).all(),

    "Thirty inner geographic folds":
        len(inner_table) == 30,

    "No inner PSU or block overlap":
        (
            inner_table["psu_overlap"].eq(0).all()
            and inner_table["grid_block_overlap"].eq(0).all()
        ),

    "Inner coordinate buffers validated":
        inner_table[
            "minimum_displaced_coordinate_gap_km"
        ].gt(BUFFER_KM).all(),

    "All 180 nested candidate scores computed":
        len(candidate_table) == 180,

    "All 540 inner fits completed":
        inner_fit_count == 540,

    "All 20 selected outer fits completed":
        outer_fit_count == 20,

    "Exactly one selected candidate per family and fold":
        (
            len(choices) == 20
            and not choices.duplicated(
                ["stage", "outer_fold", "family"]
            ).any()
        ),

    "Each child has one OOF prediction per family":
        (
            len(oof) == (2624 + 1969) * 2
            and not oof.duplicated(
                [
                    "stage",
                    "family",
                    "child_row_within_stage",
                ]
            ).any()
        ),

    "Four pooled family-stage evaluations":
        len(summary) == 4,

    "Two paired geographic comparisons":
        len(paired) == 2,

    "All held-out metrics valid":
        (
            np.isfinite(
                summary.select_dtypes(
                    include=[np.number]
                ).to_numpy()
            ).all()
            and summary["auc"].between(0, 1).all()
            and summary["brier"].between(0, 1).all()
        ),

    "Conditional bootstrap intervals valid":
        (
            np.isfinite(
                paired.select_dtypes(
                    include=[np.number]
                ).to_numpy()
            ).all()
            and all(
                paired[
                    f"{name}_conditional_psu95_low"
                ].le(
                    paired[
                        f"{name}_conditional_psu95_high"
                    ]
                ).all()
                for name in (
                    "auc",
                    "brier",
                    "log_loss",
                )
            )
        ),

    "No outcome-derived predictors":
        set(
            NUMERIC
            + CATEGORICAL
            + ["gps_latitude", "gps_longitude"]
        ).isdisjoint(PROHIBITED),
}

validation = pd.DataFrame([
    {
        "check": name,
        "status": (
            "PASS" if bool(value) else "REVIEW"
        ),
    }
    for name, value in checks.items()
])

all_passed = all(
    bool(value)
    for value in checks.values()
)


# -------------------------------------------------------
# 9. Save all outputs independently
# -------------------------------------------------------

outputs = {
    "outer_folds":
        TABLES / "12a_nested_outer_geographic_folds.csv",

    "inner_folds":
        TABLES / "12a_nested_inner_geographic_folds.csv",

    "candidate_scores":
        TABLES / "12a_nested_inner_candidate_scores.csv",

    "selections":
        TABLES / "12a_nested_selected_model_by_fold.csv",

    "summary":
        TABLES / "12a_nested_oof_performance.csv",

    "paired":
        TABLES / "12a_nested_paired_spatial_comparison.csv",

    "predictions":
        PROC / "12a_nested_oof_predictions.csv",

    "checks":
        TABLES / "12a_nested_validation_checks.csv",

    "audit":
        LOGS / "12a_nested_geographic_ml_audit.json",

    "report":
        LOGS / "12a_nested_geographic_ml_report.txt",
}

for name, frame in (
    ("outer_folds", outer_table),
    ("inner_folds", inner_table),
    ("candidate_scores", candidate_table),
    ("selections", choices),
    ("summary", summary),
    ("paired", paired),
    ("predictions", oof),
    ("checks", validation),
):
    frame.to_csv(
        outputs[name],
        index=False,
        encoding="utf-8-sig",
    )

report = "\n".join([
    "STEP 12.1 | NESTED GEOGRAPHIC VALIDATION OF TWO ML FAMILIES",
    "=" * 75,
    "Data: verified BDHS 2022 KR + displaced DHS GPS",
    "Outcomes: Stage 1 care sought; Stage 2 pharmacy first among seekers",
    "Outer: five 75km grid-block regions from Step 10.2, 20km buffer",
    "Inner: three training-only geographic regions with 20km buffer",
    "Selection: survey-weighted inner OOF log loss",
    "Candidates: L2 logistic C=0.3/1.0, Random Forest, CatBoost",
    "Raw GPS available for all algorithms",
    "RBF 50/75/100/150/200km considered for L2 logistic",
    "RBF knots: 16 training-only KMeans centers",
    f"Inner fits: {inner_fit_count}",
    f"Selected outer fits: {outer_fit_count}",
    "Paired conditional PSU bootstrap: 599 replicates",
    "Previously examined outer regions: exploratory robustness only",
    "Not an independent confirmatory test",
    "Geographic distances refer to displaced DHS GPS coordinates",
    (
        f"Validation: "
        f"{'PASS' if all_passed else 'REVIEW REQUIRED'}"
    ),
    (
        f"Checks: "
        f"{sum(checks.values())}/{len(checks)}"
    ),
])

outputs["report"].write_text(
    report + "\n",
    encoding="utf-8",
)

audit = {
    "step": "12.1",
    "phase": "E - geographic generalizability",
    "kr_sha256": kr_sha256,
    "seed": SEED,
    "grid_width_km": GRID_KM,
    "displaced_coordinate_buffer_km": BUFFER_KM,
    "inner_geographic_folds": N_INNER,
    "rbf_bandwidths_km": list(BANDWIDTHS_KM),
    "training_only_rbf_centers": N_ANCHORS,
    "candidate_count": len(CANDIDATES),
    "inner_model_fits": inner_fit_count,
    "outer_model_fits": outer_fit_count,
    "conditional_bootstrap_replicates": BOOTSTRAPS,
    "limitations": [
        "Outer geographic regions were inspected in prior experiments.",
        "Results are post-hoc exploratory robustness, not independent confirmation.",
        "KMeans regional groups may not be perfectly geographically contiguous.",
        "Bootstrap holds selected outer-fold models fixed.",
        "Hyperparameter selection is restricted to the reported candidate space.",
        "DHS GPS displacement prevents exact geographic distance claims.",
        "Conditional bootstrap intervals are not official complex-survey variance estimates.",
    ],
    "checks": {
        name: bool(value)
        for name, value in checks.items()
    },
    "validation_passed": bool(all_passed),
}

outputs["audit"].write_text(
    json.dumps(
        audit,
        indent=2,
        ensure_ascii=False,
    ) + "\n",
    encoding="utf-8",
)


# -------------------------------------------------------
# 10. Display all key results
# -------------------------------------------------------

print(report)

print("\nNESTED GEOGRAPHIC HELD-OUT PERFORMANCE")
display(
    summary.round(5)
)

print("\nPAIRED GEOGRAPHIC MINUS NON-SPATIAL PERFORMANCE")
display(
    paired.round(5)
)

print("\nSELECTED MODELS AND BANDWIDTHS BY OUTER FOLD")
display(
    choices.round(5)
)

print("\nOUTER GEOGRAPHIC FOLD DIAGNOSTICS")
display(
    outer_table.round(3)
)

print("\nINNER GEOGRAPHIC FOLD DIAGNOSTICS")
display(
    inner_table.round(3)
)

print("\nVALIDATION CHECKS")
display(validation)

print("\nSAVED OUTPUTS")
for path in outputs.values():
    print(path)

if not all_passed:
    raise ValueError(
        "Step 12.1 nested geographic validation requires review."
    )

[Nested ML] Stage 1 care sought | outer fold 1/5 complete | inner fits so far: 54
[Nested ML] Stage 1 care sought | outer fold 2/5 complete | inner fits so far: 108
[Nested ML] Stage 1 care sought | outer fold 3/5 complete | inner fits so far: 162
[Nested ML] Stage 1 care sought | outer fold 4/5 complete | inner fits so far: 216
[Nested ML] Stage 1 care sought | outer fold 5/5 complete | inner fits so far: 270
[Nested ML] Stage 2 pharmacy first | outer fold 1/5 complete | inner fits so far: 324
[Nested ML] Stage 2 pharmacy first | outer fold 2/5 complete | inner fits so far: 378
[Nested ML] Stage 2 pharmacy first | outer fold 3/5 complete | inner fits so far: 432
[Nested ML] Stage 2 pharmacy first | outer fold 4/5 complete | inner fits so far: 486
[Nested ML] Stage 2 pharmacy first | outer fold 5/5 complete | inner fits so far: 540
STEP 12.1 | NESTED GEOGRAPHIC VALIDATION OF TWO ML FAMILIES
Data: verified BDHS 2022 KR + displaced DHS GPS
Outcomes: Stage 1 care sought; Stage 2 pharmacy 

,stage,family,children,auc,average_precision,brier,log_loss,observed_pct,predicted_pct,calibration_gap_pp
0,Stage 1 care sought,Non-spatial,2624,0.59600,0.81191,0.17770,0.53951,76.18277,75.98151,-0.20126
1,Stage 1 care sought,Geographic,2624,0.58924,0.81090,0.17905,0.54228,76.18277,74.58132,-1.60145
2,Stage 2 pharmacy first,Non-spatial,1969,0.58850,0.64553,0.23822,0.66948,57.75851,58.31449,0.55598
3,Stage 2 pharmacy first,Geographic,1969,0.57781,0.64207,0.24071,0.67478,57.75851,59.09847,1.33996



PAIRED GEOGRAPHIC MINUS NON-SPATIAL PERFORMANCE


,stage,children,delta_auc_spatial_minus_nonspatial,delta_brier_spatial_minus_nonspatial,delta_log_loss_spatial_minus_nonspatial,auc_conditional_psu95_low,auc_conditional_psu95_high,brier_conditional_psu95_low,brier_conditional_psu95_high,log_loss_conditional_psu95_low,log_loss_conditional_psu95_high
0,Stage 1 care sought,2624,-0.00676,0.00135,0.00278,-0.02205,0.00827,-0.00051,0.00306,-0.00214,0.00714
1,Stage 2 pharmacy first,1969,-0.01069,0.00248,0.00530,-0.02660,0.00533,-0.00033,0.00531,-0.00093,0.01116



SELECTED MODELS AND BANDWIDTHS BY OUTER FOLD


,stage,outer_fold,family,selected_algorithm,selected_C,selected_feature_variant,inner_oof_log_loss,inner_oof_brier
0,Stage 1 care sought,1,Non-spatial,CatBoost,NaN,Non-spatial,0.53604,0.17601
1,Stage 1 care sought,1,Geographic,Random forest,NaN,Raw GPS,0.53087,0.17412
2,Stage 1 care sought,2,Non-spatial,Random forest,NaN,Non-spatial,0.55016,0.18243
3,Stage 1 care sought,2,Geographic,Random forest,NaN,Raw GPS,0.55105,0.18283
4,Stage 1 care sought,3,Non-spatial,CatBoost,NaN,Non-spatial,0.55401,0.18284
5,Stage 1 care sought,3,Geographic,Random forest,NaN,Raw GPS,0.55407,0.18299
6,Stage 1 care sought,4,Non-spatial,Random forest,NaN,Non-spatial,0.53953,0.17781
7,Stage 1 care sought,4,Geographic,Random forest,NaN,Raw GPS,0.53735,0.17710
8,Stage 1 care sought,5,Non-spatial,L2 logistic,0.3,Non-spatial,0.55845,0.18584
9,Stage 1 care sought,5,Geographic,Random forest,NaN,Raw GPS,0.55424,0.18455


... (3 outputs left out)

,stage,outer_fold,inner_fold,train_n,validation_n,train_psus,validation_psus,psu_overlap,grid_block_overlap,removed_by_20km_buffer,minimum_displaced_coordinate_gap_km
0,Stage 1 care sought,1,1,1476,559,341,125,0,0,18,22.990
1,Stage 1 care sought,1,2,1008,1040,240,230,0,0,5,22.098
2,Stage 1 care sought,1,3,1575,454,351,117,0,0,24,22.609
3,Stage 1 care sought,2,1,1611,604,429,135,0,0,38,21.900
4,Stage 1 care sought,2,2,1517,665,371,191,0,0,71,20.996
5,Stage 1 care sought,2,3,1239,984,319,251,0,0,30,20.230
6,Stage 1 care sought,3,1,1346,586,317,163,0,0,40,20.083
7,Stage 1 care sought,3,2,969,911,229,241,0,0,92,20.194
8,Stage 1 care sought,3,3,1360,475,374,88,0,0,137,20.966
9,Stage 1 care sought,4,1,1462,511,325,143,0,0,115,20.066



VALIDATION CHECKS


,check,status
0,Verified prerequisites and original KR checksum,PASS
1,Original 8784 records and 674 GPS PSUs retained,PASS
2,Both frozen complete-case populations reproduced,PASS
3,Original 37 geographic blocks reproduced,PASS
4,Five outer regions per outcome,PASS
5,No outer PSU or block overlap,PASS
6,Outer coordinate buffers validated,PASS
7,Thirty inner geographic folds,PASS
8,No inner PSU or block overlap,PASS
9,Inner coordinate buffers validated,PASS



SAVED OUTPUTS
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\12a_nested_outer_geographic_folds.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\12a_nested_inner_geographic_folds.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\12a_nested_inner_candidate_scores.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\12a_nested_selected_model_by_fold.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\12a_nested_oof_performance.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\12a_nested_paired_spatial_comparison.csv
D:\BDHS2022_FeverCare_SpatialEquity\data\processed\12a_nested_oof_predictions.csv
D:\BDHS2022_FeverCare_SpatialEquity\results\tables\12a_nested_validation_checks.csv
D:\BDHS2022_FeverCare_SpatialEquity\logs\12a_nested_geographic_ml_audit.json
D:\BDHS2022_FeverCare_SpatialEquity\logs\12a_nested_geographic_ml_report.txt
